# RQ2

## RQ2 development setup, reproducibility, leakage guards, and configuration status

In [78]:
# RQ2: Can adaptive frontier pruning reduce unnecessary graph exploration more
# effectively than unpruned RoG and simple fixed-pruning strategies?
#
# IMPORTANT:
# - This cell freezes only methodological/protocol invariants.
# - Model architecture and hyperparameters are DEVELOPMENT choices, not final values.
# - TRAIN is used for branch supervision.
# - VALIDATION is used for model/configuration selection.
# - TEST must not influence AFP development.

import os
import sys
import json
import random
import hashlib
import platform
import numpy as np
import pandas as pd
import torch
import networkx as nx
import datasets

from datasets import load_dataset

# =========================================================
# RQ2 stage and output directories
# =========================================================
RQ2_STAGE = "RQ2_DEVELOPMENT"
RQ2_VERSION = "v1"

RQ2_DIR = f"/kaggle/working/step3_rq2_dev_{RQ2_VERSION}"
RQ2_PLAN_DIR = os.path.join(RQ2_DIR, "01_train_plans")
RQ2_LABEL_DIR = os.path.join(RQ2_DIR, "02_branch_labels")
RQ2_FEATURE_DIR = os.path.join(RQ2_DIR, "03_features")
RQ2_MODEL_DIR = os.path.join(RQ2_DIR, "04_models")
RQ2_TUNING_DIR = os.path.join(RQ2_DIR, "05_validation_tuning")
RQ2_COMPARE_DIR = os.path.join(RQ2_DIR, "06_rq2_comparison")
RQ2_ABLATION_DIR = os.path.join(RQ2_DIR, "07_ablation")
RQ2_MANIFEST_DIR = os.path.join(RQ2_DIR, "manifests")

for d in [
    RQ2_DIR,
    RQ2_PLAN_DIR,
    RQ2_LABEL_DIR,
    RQ2_FEATURE_DIR,
    RQ2_MODEL_DIR,
    RQ2_TUNING_DIR,
    RQ2_COMPARE_DIR,
    RQ2_ABLATION_DIR,
    RQ2_MANIFEST_DIR,
]:
    os.makedirs(d, exist_ok=True)

# =========================================================
# Leakage-safe split policy
# =========================================================
RQ2_TRAIN_SPLIT = "train"
RQ2_VALIDATION_SPLIT = "validation"
RQ2_TEST_SPLIT = "test"

RQ2_ALLOWED_DEV_SPLITS = {
    RQ2_TRAIN_SPLIT,
    RQ2_VALIDATION_SPLIT,
}

def assert_rq2_dev_split(split_name):
    assert split_name in RQ2_ALLOWED_DEV_SPLITS, (
        f"RQ2 LEAKAGE GUARD: split='{split_name}' is forbidden during development. "
        f"Only TRAIN and VALIDATION may influence AFP."
    )

assert_rq2_dev_split("train")
assert_rq2_dev_split("validation")

# =========================================================
# Predeclared reproducibility seeds
# These can be frozen now because they should NOT be selected
# according to downstream performance.
# =========================================================
RQ2_SEEDS = [42, 43, 44]
RQ2_CANONICAL_SEED = 42

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(RQ2_CANONICAL_SEED)

# =========================================================
# Verify required frozen RoG components already exist
# =========================================================
required_globals = [
    "DATASET_NAME",
    "CWQ_DATASET_NAME",
    "MODEL_PATH",
    "N_BEAM",
    "INSTRUCTION",
    "build_graph",
    "suffix_reachable_dp",
    "bfs_with_rule_profiled_v2",
]

missing_globals = [
    name for name in required_globals
    if name not in globals()
]

assert len(missing_globals) == 0, (
    "Missing required RoG objects/functions: "
    + ", ".join(missing_globals)
)

RQ2_WEBQSP_DATASET = DATASET_NAME
RQ2_CWQ_DATASET = CWQ_DATASET_NAME
RQ2_PLANNER_MODEL = MODEL_PATH
RQ2_N_BEAM = N_BEAM

RQ2_INSTRUCTION_HASH = hashlib.sha256(
    INSTRUCTION.encode("utf-8")
).hexdigest()

# =========================================================
# Load development splits only
# =========================================================
if "webqsp_train" not in globals():
    webqsp_train = load_dataset(
        RQ2_WEBQSP_DATASET,
        split=RQ2_TRAIN_SPLIT
    )

if "cwq_train" not in globals():
    cwq_train = load_dataset(
        RQ2_CWQ_DATASET,
        split=RQ2_TRAIN_SPLIT
    )

if "webqsp_val" not in globals():
    webqsp_val = load_dataset(
        RQ2_WEBQSP_DATASET,
        split=RQ2_VALIDATION_SPLIT
    )

if "cwq_val" not in globals():
    cwq_val = load_dataset(
        RQ2_CWQ_DATASET,
        split=RQ2_VALIDATION_SPLIT
    )

# =========================================================
# METHOD INVARIANTS -- fixed now
# =========================================================
AFP_FINAL_HOP_PROTECTION = True
AFP_INTERMEDIATE_ONLY = True
AFP_LABEL_RULE = "suffix_gold_reachability"
AFP_EXCLUDE_ALL_NEGATIVE_GROUPS = True

AFP_FEATURE_GROUPS = [
    "semantic",
    "path_context",
    "structural",
    "search_progress",
]

AFP_USE_FUTURE_NEIGHBORHOOD_FEATURES = False
AFP_USE_KGE_CORE = False

RQ2_PRIMARY_COST_METRIC = "edges_examined"
RQ2_PRIMARY_OUTCOME = "SSR"

RQ2_METHODS = [
    "RoG",
    "Fixed-Top-B",
    "Fixed-Threshold",
    "Random-B",
    "Adaptive-Budget-Random",
    "AFP",
]

RQ2_INVARIANTS = {
    "same_questions": True,
    "same_topic_entities": True,
    "same_relation_plans": True,
    "same_question_graph": True,
    "same_graph_construction": True,
    "same_relation_matching": True,
    "same_final_hop_policy": True,
    "same_candidate_generation": True,
    "same_retrieval_semantics_except_selection": True,
    "test_used_for_development": False,
}

# =========================================================
# DEVELOPMENT CONFIGURATION -- NOT FROZEN YET
# =========================================================

# Lightweight semantic encoder from current methodology.
# Keep as primary candidate for now, but NOT frozen.
AFP_SEMANTIC_ENCODER_CANDIDATE = (
    "sentence-transformers/all-MiniLM-L6-v2"
)
AFP_SEMANTIC_DIM = 384

# Scorer architecture candidates.
# 64 remains the manuscript's initial configuration,
# but we allow evidence-based adjustment.
AFP_HIDDEN_DIM_CANDIDATES = [32, 64, 128]

AFP_ACTIVATION_CANDIDATE = "ReLU"
AFP_LOSS_CANDIDATE = "BCEWithLogitsLoss"
AFP_OPTIMIZER_CANDIDATE = "AdamW"

# No final optimizer values yet
AFP_LR_SELECTED = None
AFP_WEIGHT_DECAY_SELECTED = None
AFP_BATCH_SIZE_SELECTED = None
AFP_EPOCHS_SELECTED = None
AFP_HIDDEN_DIM_SELECTED = None
AFP_SEMANTIC_ENCODER_SELECTED = None

# =========================================================
# Fixed-pruning validation grids
# Revised from the manuscript placeholder because RQ1 showed
# intermediate candidate sets well above B=8.
# =========================================================
TOP_B_GRID = [1, 2, 4, 8, 16, 32]

# Wider threshold search than the draft placeholder {0.3,0.5,0.7}.
# Final threshold will be selected on validation only.
THRESHOLD_GRID = [
    0.1, 0.2, 0.3, 0.4, 0.5,
    0.6, 0.7, 0.8, 0.9
]

# Adaptive-policy grids should NOT be chosen yet.
# Their useful ranges depend on the trained scorer's validation
# logit/probability distribution.
AFP_T_GRID = None
AFP_GAMMA_MIN_GRID = None

AFP_T_SELECTED = None
AFP_GAMMA_MIN_SELECTED = None

# =========================================================
# RQ2 structural metrics
# =========================================================
RQ2_STRUCTURAL_METRICS = [
    "active_prefixes",
    "unique_expanded_nodes",
    "edges_examined",
    "candidate_branches",
    "unique_frontier_nodes",
    "peak_frontier",
    "retrieved_paths",
]

def compute_ssr(method_edges, rog_edges):
    assert rog_edges > 0, "RoG examined-edge denominator must be > 0."
    return 1.0 - (method_edges / rog_edges)

# =========================================================
# Hard methodological assertions
# =========================================================
assert AFP_FINAL_HOP_PROTECTION is True
assert AFP_INTERMEDIATE_ONLY is True
assert AFP_EXCLUDE_ALL_NEGATIVE_GROUPS is True
assert AFP_USE_FUTURE_NEIGHBORHOOD_FEATURES is False
assert AFP_USE_KGE_CORE is False
assert RQ2_INVARIANTS["test_used_for_development"] is False
assert callable(suffix_reachable_dp)

# =========================================================
# Development manifest
# =========================================================
RQ2_CONFIG = {
    "stage": RQ2_STAGE,
    "version": RQ2_VERSION,

    "dataset": {
        "webqsp_name": RQ2_WEBQSP_DATASET,
        "cwq_name": RQ2_CWQ_DATASET,
        "webqsp_train_questions": len(webqsp_train),
        "webqsp_validation_questions": len(webqsp_val),
        "cwq_train_questions": len(cwq_train),
        "cwq_validation_questions": len(cwq_val),
        "test_allowed_for_development": False,
    },

    "frozen_rog": {
        "planner_model": RQ2_PLANNER_MODEL,
        "top_k_relation_plans": RQ2_N_BEAM,
        "instruction_sha256": RQ2_INSTRUCTION_HASH,
    },

    "method_invariants": {
        "final_hop_protection": AFP_FINAL_HOP_PROTECTION,
        "intermediate_only": AFP_INTERMEDIATE_ONLY,
        "label_rule": AFP_LABEL_RULE,
        "exclude_all_negative_groups":
            AFP_EXCLUDE_ALL_NEGATIVE_GROUPS,
        "feature_groups": AFP_FEATURE_GROUPS,
        "future_neighborhood_features":
            AFP_USE_FUTURE_NEIGHBORHOOD_FEATURES,
        "kge_core": AFP_USE_KGE_CORE,
    },

    "development_candidates": {
        "semantic_encoder":
            AFP_SEMANTIC_ENCODER_CANDIDATE,
        "hidden_dims":
            AFP_HIDDEN_DIM_CANDIDATES,
        "activation":
            AFP_ACTIVATION_CANDIDATE,
        "loss":
            AFP_LOSS_CANDIDATE,
        "optimizer":
            AFP_OPTIMIZER_CANDIDATE,
        "top_b_grid":
            TOP_B_GRID,
        "threshold_grid":
            THRESHOLD_GRID,
        "temperature_grid":
            AFP_T_GRID,
        "gamma_min_grid":
            AFP_GAMMA_MIN_GRID,
    },

    "rq2": {
        "methods": RQ2_METHODS,
        "primary_cost_metric":
            RQ2_PRIMARY_COST_METRIC,
        "primary_outcome":
            RQ2_PRIMARY_OUTCOME,
        "structural_metrics":
            RQ2_STRUCTURAL_METRICS,
    },

    "reproducibility": {
        "seeds": RQ2_SEEDS,
        "canonical_seed":
            RQ2_CANONICAL_SEED,
        "python":
            sys.version,
        "platform":
            platform.platform(),
        "torch":
            torch.__version__,
        "numpy":
            np.__version__,
        "pandas":
            pd.__version__,
        "networkx":
            nx.__version__,
        "datasets":
            datasets.__version__,
        "cuda_available":
            torch.cuda.is_available(),
        "gpu": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
    },

    "controlled_comparison_invariants":
        RQ2_INVARIANTS,
}

RQ2_CONFIG_PATH = os.path.join(
    RQ2_MANIFEST_DIR,
    "rq2_initial_development_config.json"
)

with open(RQ2_CONFIG_PATH, "w") as f:
    json.dump(
        RQ2_CONFIG,
        f,
        indent=2
    )

# =========================================================
# Setup report
# =========================================================
print("\n" + "=" * 80)
print("RQ2 DEVELOPMENT SETUP")
print("=" * 80)

print(f"Output directory:    {RQ2_DIR}")

print("\nDevelopment datasets:")
print(f"WebQSP train:        {len(webqsp_train)}")
print(f"WebQSP validation:   {len(webqsp_val)}")
print(f"CWQ train:           {len(cwq_train)}")
print(f"CWQ validation:      {len(cwq_val)}")

print("\nFrozen RoG:")
print(f"Planner model:       {RQ2_PLANNER_MODEL}")
print(f"Top-K plans:         {RQ2_N_BEAM}")
print(f"Instruction SHA256:  {RQ2_INSTRUCTION_HASH[:16]}...")

print("\nAFP methodological invariants:")
print(f"Final-hop protection:             {AFP_FINAL_HOP_PROTECTION}")
print(f"Intermediate pruning only:        {AFP_INTERMEDIATE_ONLY}")
print(f"All-negative groups excluded:     {AFP_EXCLUDE_ALL_NEGATIVE_GROUPS}")
print(f"Future-neighborhood features:     {AFP_USE_FUTURE_NEIGHBORHOOD_FEATURES}")
print(f"KGE required in core model:       {AFP_USE_KGE_CORE}")

print("\nDevelopment candidates -- NOT frozen:")
print(f"Semantic encoder:    {AFP_SEMANTIC_ENCODER_CANDIDATE}")
print(f"Hidden dims:         {AFP_HIDDEN_DIM_CANDIDATES}")
print(f"Top-B grid:          {TOP_B_GRID}")
print(f"Threshold grid:      {THRESHOLD_GRID}")
print("T grid:              deferred until scorer validation")
print("gamma_min grid:      deferred until scorer validation")

print("\nControlled RQ2 methods:")
for method in RQ2_METHODS:
    print(f"  - {method}")

print(f"\nPrimary structural cost: {RQ2_PRIMARY_COST_METRIC}")
print(f"Primary RQ2 outcome:     {RQ2_PRIMARY_OUTCOME}")
print(f"Seeds:                   {RQ2_SEEDS}")

print("\n=== RQ2 CELL 1: PASSED ===")
print("TRAIN + VALIDATION only.")
print("No test-set information may influence AFP development.")
print("Next: generate/resume and freeze full TRAIN relation plans.")


RQ2 DEVELOPMENT SETUP
Output directory:    /kaggle/working/step3_rq2_dev_v1

Development datasets:
WebQSP train:        2826
WebQSP validation:   246
CWQ train:           27639
CWQ validation:      3519

Frozen RoG:
Planner model:       rmanluo/RoG
Top-K plans:         3
Instruction SHA256:  e3687b4a5081c22c...

AFP methodological invariants:
Final-hop protection:             True
Intermediate pruning only:        True
All-negative groups excluded:     True
Future-neighborhood features:     False
KGE required in core model:       False

Development candidates -- NOT frozen:
Semantic encoder:    sentence-transformers/all-MiniLM-L6-v2
Hidden dims:         [32, 64, 128]
Top-B grid:          [1, 2, 4, 8, 16, 32]
Threshold grid:      [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
T grid:              deferred until scorer validation
gamma_min grid:      deferred until scorer validation

Controlled RQ2 methods:
  - RoG
  - Fixed-Top-B
  - Fixed-Threshold
  - Random-B
  - Adaptive-Budget-Rand

In [79]:
# Check partial CWQ shard 42 after interruption

import os
import json

path = (
    "/kaggle/working/step3_rq2_dev_v1/"
    "01_train_plans/cwq/shards/"
    "shard_0041_010250_010499.jsonl"
)

print("Exists:", os.path.exists(path))

valid = 0

if os.path.exists(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                json.loads(line)
                valid += 1
            except json.JSONDecodeError:
                break

print("Valid saved records:", valid)
print("Remaining:", 250 - valid)

Exists: True
Valid saved records: 250
Remaining: 0


In [80]:
import os

meta_path = (
    "/kaggle/working/step3_rq2_dev_v1/"
    "01_train_plans/cwq/shards/"
    "shard_0041_010250_010499.meta.json"
)

print("Frozen metadata exists:", os.path.exists(meta_path))

Frozen metadata exists: True


In [81]:
# ## Utility — backup current RQ2 progress
# Run BEFORE resuming the long RQ2 Cell 2.

import os
import shutil
from datetime import datetime

SRC = "/kaggle/working/step3_rq2_dev_v1"

assert os.path.exists(SRC), f"RQ2 directory not found: {SRC}"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

BACKUP_BASE = (
    f"/kaggle/working/"
    f"rq2_backup_after_shard42_{timestamp}"
)

backup_file = shutil.make_archive(
    BACKUP_BASE,
    "zip",
    root_dir="/kaggle/working",
    base_dir="step3_rq2_dev_v1"
)

print("=" * 70)
print("RQ2 BACKUP CREATED")
print("=" * 70)
print("Source :", SRC)
print("Backup :", backup_file)
print(
    "Size   :",
    round(os.path.getsize(backup_file) / (1024 ** 2), 2),
    "MB"
)
print("\nIMPORTANT: save a Kaggle notebook version/output")
print("to make this backup survive a full session reset.")

RQ2 BACKUP CREATED
Source : /kaggle/working/step3_rq2_dev_v1
Backup : /kaggle/working/rq2_backup_after_shard42_20260831_073227.zip
Size   : 3.86 MB

IMPORTANT: save a Kaggle notebook version/output
to make this backup survive a full session reset.


## Generate/resume and freeze full TRAIN relation plans

In [82]:
# Produces deterministic, resumable RoG relation-plan shards for WebQSP/CWQ TRAIN.
#
# IMPORTANT:
# - Uses TRAIN only.
# - Reuses the SAME RoG planner semantics used for frozen validation planning.
# - Performs a cheap validation-plan fidelity gate BEFORE the long training run.
# - Saves minimal planner outputs only; does NOT duplicate full per-question graphs.
# - source_index is the primary key.
# - Completed shards are immutable.
# - Full training plans are frozen before branch-label construction.

import os
import json
import hashlib
from datetime import datetime, timezone
from tqdm.auto import tqdm

RQ2_CELL2_CODE_TAG = "rq2_train_plan_freeze_v1"
TRAIN_PLAN_SHARD_SIZE = 250
RQ2_PLANNER_DO_SAMPLE = False
RQ2_PLANNER_MAX_NEW_TOKENS = int(
    globals().get("MAX_NEW_TOKENS", 100)
)
PLANNER_PREFLIGHT_N = 3

# =========================================================
# 1. Resolve EXACT planner components already used earlier
# =========================================================
assert "generate_seq" in globals() and callable(generate_seq), (
    "generate_seq() is missing. Re-run the earlier RoG planning-definition cell."
)
assert "parse_prediction" in globals() and callable(parse_prediction), (
    "parse_prediction() is missing. Re-run the earlier RoG planning-definition cell."
)

def first_existing_global(names):
    for name in names:
        if name in globals() and globals()[name] is not None:
            return globals()[name], name
    return None, None

planner_model, planner_model_var = first_existing_global([
    "planner_model", "rog_model", "model"
])

planner_tokenizer, planner_tokenizer_var = first_existing_global([
    "planner_tokenizer", "rog_tokenizer", "tokenizer"
])

assert planner_model is not None, (
    "Could not locate the already-loaded RoG planner model."
)
assert planner_tokenizer is not None, (
    "Could not locate the already-loaded RoG tokenizer."
)
assert hasattr(planner_model, "generate"), (
    f"Resolved '{planner_model_var}', but it has no .generate() method."
)

planner_model.eval()

print(f"Planner model object:     {planner_model_var}")
print(f"Planner tokenizer object: {planner_tokenizer_var}")

# =========================================================
# 2. Resolve the SAME prompt formatter used for validation
# =========================================================
def infer_prompt_template_from_frozen_validation():
    for records in [
        globals().get("webqsp_val_planning"),
        globals().get("cwq_val_planning"),
    ]:
        if records is None or len(records) == 0:
            continue

        rec = records[0]

        if "input" not in rec or "question" not in rec:
            continue

        question = rec["question"]
        full_input = rec["input"]

        pos = full_input.rfind(question)

        if pos >= 0:
            prefix = full_input[:pos]
            suffix = full_input[pos + len(question):]

            def builder(q):
                return prefix + q + suffix

            assert builder(question) == full_input
            return builder, "inferred_from_frozen_validation_input"

    return None, None


def resolve_prompt_builder():
    # Preferred: an explicit function from the existing notebook
    for fn_name in [
        "format_planning_prompt",
        "build_planning_prompt",
        "format_planner_input",
    ]:
        fn = globals().get(fn_name)

        if callable(fn):
            return fn, fn_name

    # Next: RoG-style InstructFormater object
    for obj_name in [
        "prompter",
        "planner_prompter",
        "planning_prompter",
    ]:
        obj = globals().get(obj_name)

        if obj is None or not hasattr(obj, "format"):
            continue

        def builder(q, formatter=obj):
            return formatter.format(
                instruction=INSTRUCTION,
                message=q
            )

        try:
            _ = builder("RQ2_PROMPT_TEST")
            return builder, obj_name
        except Exception:
            pass

    # Last safe option: infer the exact frozen template
    return infer_prompt_template_from_frozen_validation()


planner_prompt_builder, planner_prompt_source = resolve_prompt_builder()

assert planner_prompt_builder is not None, (
    "Could not safely recover the SAME planning prompt formatter used for "
    "validation. Do NOT invent a new prompt here. Re-run the earlier "
    "validation-planning setup cell so `prompter` or the prompt builder exists."
)

print(f"Planner prompt source:    {planner_prompt_source}")

# Prompt-template fingerprint
prompt_probe = planner_prompt_builder(
    "__RQ2_QUESTION_PLACEHOLDER__"
)

RQ2_PROMPT_TEMPLATE_HASH = hashlib.sha256(
    prompt_probe.encode("utf-8")
).hexdigest()

# =========================================================
# 3. Planner configuration fingerprint
# =========================================================
RQ2_TRAIN_PLANNER_CONFIG = {
    "code_tag": RQ2_CELL2_CODE_TAG,
    "model_path": RQ2_PLANNER_MODEL,
    "n_beam": int(RQ2_N_BEAM),
    "do_sample": bool(RQ2_PLANNER_DO_SAMPLE),
    "max_new_tokens": int(RQ2_PLANNER_MAX_NEW_TOKENS),
    "instruction_sha256": RQ2_INSTRUCTION_HASH,
    "prompt_template_sha256": RQ2_PROMPT_TEMPLATE_HASH,
}

RQ2_TRAIN_PLANNER_SIGNATURE = hashlib.sha256(
    json.dumps(
        RQ2_TRAIN_PLANNER_CONFIG,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

print(
    "Planner configuration:   "
    f"{RQ2_TRAIN_PLANNER_SIGNATURE[:16]}..."
)

# =========================================================
# 4. Exact deterministic RoG planning adapter
# =========================================================
def normalize_paths(paths):
    if paths is None:
        return []

    return [
        [str(rel) for rel in path]
        for path in paths
    ]


def generate_frozen_rog_plan(question):
    input_text = planner_prompt_builder(question)

    with torch.inference_mode():
        raw_output = generate_seq(
            planner_model,
            input_text,
            planner_tokenizer,
            num_beam=RQ2_N_BEAM,
            do_sample=RQ2_PLANNER_DO_SAMPLE,
            max_new_tokens=RQ2_PLANNER_MAX_NEW_TOKENS,
        )

    assert isinstance(raw_output, dict)
    assert "paths" in raw_output

    predicted_paths = normalize_paths(
        parse_prediction(raw_output["paths"])
    )

    return {
        "predicted_paths": predicted_paths,
        "raw_paths": [
            str(x) for x in raw_output.get("paths", [])
        ],
        "scores": [
            float(x) for x in raw_output.get("scores", [])
        ],
        "norm_scores": [
            float(x) for x in raw_output.get("norm_scores", [])
        ],
        "planner_input_sha256": hashlib.sha256(
            input_text.encode("utf-8")
        ).hexdigest(),
    }

# =========================================================
# 5. CHEAP FIDELITY GATE against frozen validation plans
# =========================================================
def planner_fidelity_gate(frozen_records, dataset_label, n=3):
    assert frozen_records is not None
    assert len(frozen_records) > 0

    checked = 0
    mismatches = 0

    # Deterministically take first usable records
    for rec in frozen_records:
        if checked >= n:
            break

        if "predicted_paths" not in rec:
            continue

        generated = generate_frozen_rog_plan(
            rec["question"]
        )["predicted_paths"]

        expected = normalize_paths(
            rec["predicted_paths"]
        )

        checked += 1

        if generated != expected:
            mismatches += 1
            print(
                f"\n[{dataset_label}] Planner mismatch "
                f"for question id={rec['id']}"
            )
            print("Expected:", expected)
            print("Generated:", generated)

    assert checked > 0, (
        f"{dataset_label}: no frozen validation records "
        f"were available for planner fidelity testing."
    )

    print(
        f"[{dataset_label}] Planner fidelity: "
        f"{checked} checked | mismatches={mismatches}"
    )

    assert mismatches == 0, (
        f"{dataset_label}: TRAIN planning configuration does not "
        f"reproduce the frozen validation planner. STOP before "
        f"generating training plans."
    )


planner_fidelity_gate(
    webqsp_val_planning,
    "WebQSP validation",
    PLANNER_PREFLIGHT_N
)

planner_fidelity_gate(
    cwq_val_planning,
    "CWQ validation",
    PLANNER_PREFLIGHT_N
)

print("\n=== TRAIN PLANNER FIDELITY GATE: PASSED ===")

# =========================================================
# 6. Dataset fingerprint
# =========================================================
def dataset_planning_fingerprint(dataset):
    ids = dataset["id"]
    questions = dataset["question"]

    h = hashlib.sha256()

    for i, (qid, q) in enumerate(zip(ids, questions)):
        h.update(
            f"{i}\t{qid}\t{q}\n".encode(
                "utf-8",
                errors="replace"
            )
        )

    return h.hexdigest()


webqsp_train_fingerprint = dataset_planning_fingerprint(
    webqsp_train
)

cwq_train_fingerprint = dataset_planning_fingerprint(
    cwq_train
)

print(
    "WebQSP train fingerprint:",
    webqsp_train_fingerprint[:16] + "..."
)

print(
    "CWQ train fingerprint:   ",
    cwq_train_fingerprint[:16] + "..."
)

# =========================================================
# 7. Safe JSON utilities
# =========================================================
def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def atomic_write_json(obj, path):
    tmp = path + ".tmp"

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2
        )
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)


def load_jsonl_recover(path):
    if not os.path.exists(path):
        return []

    rows = []
    corrupted_tail = False

    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                corrupted_tail = True

                print(
                    f"Recovering {path}: invalid JSON "
                    f"at line {line_no}; dropping tail."
                )
                break

    # Rewrite only valid records after interrupted write
    if corrupted_tail:
        tmp = path + ".recover.tmp"

        with open(tmp, "w", encoding="utf-8") as f:
            for row in rows:
                f.write(
                    json.dumps(
                        row,
                        ensure_ascii=False
                    ) + "\n"
                )

            f.flush()
            os.fsync(f.fileno())

        os.replace(tmp, path)

    return rows


def canonicalize_jsonl(rows, path):
    rows = sorted(
        rows,
        key=lambda x: int(x["source_index"])
    )

    tmp = path + ".tmp"

    with open(tmp, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                ) + "\n"
            )

        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)

# =========================================================
# 8. Per-record planning output
# =========================================================
def plan_training_record(
    dataset,
    source_index,
    dataset_label,
    dataset_fingerprint
):
    qid = dataset["id"][source_index]
    question = dataset["question"][source_index]

    result = generate_frozen_rog_plan(question)

    return {
        "dataset": dataset_label,
        "split": "train",
        "source_index": int(source_index),
        "id": qid,
        "question": question,
        "predicted_paths": result["predicted_paths"],
        "raw_paths": result["raw_paths"],
        "scores": result["scores"],
        "norm_scores": result["norm_scores"],
        "planner_input_sha256":
            result["planner_input_sha256"],
        "planner_signature":
            RQ2_TRAIN_PLANNER_SIGNATURE,
        "dataset_fingerprint":
            dataset_fingerprint,
    }

# =========================================================
# 9. Resumable deterministic shard generation
# =========================================================
def generate_train_plan_shards(
    dataset,
    dataset_label,
    dataset_fingerprint,
    shard_size=250
):
    dataset_dir = os.path.join(
        RQ2_PLAN_DIR,
        dataset_label
    )

    shard_dir = os.path.join(
        dataset_dir,
        "shards"
    )

    os.makedirs(shard_dir, exist_ok=True)

    n = len(dataset)
    n_shards = (n + shard_size - 1) // shard_size
    shard_metadata = []

    print(
        f"\n{dataset_label}: {n} TRAIN questions | "
        f"{n_shards} shards | shard_size={shard_size}"
    )

    for shard_id in range(n_shards):
        start = shard_id * shard_size
        end = min(start + shard_size, n)

        shard_name = (
            f"shard_{shard_id:04d}_"
            f"{start:06d}_{end - 1:06d}"
        )

        shard_path = os.path.join(
            shard_dir,
            shard_name + ".jsonl"
        )

        meta_path = os.path.join(
            shard_dir,
            shard_name + ".meta.json"
        )

        # -------------------------------------------------
        # Completed shard = immutable
        # -------------------------------------------------
        if os.path.exists(meta_path):
            with open(
                meta_path,
                "r",
                encoding="utf-8"
            ) as f:
                meta = json.load(f)

            assert (
                meta["planner_signature"]
                == RQ2_TRAIN_PLANNER_SIGNATURE
            ), (
                f"{dataset_label} shard {shard_id}: "
                f"planner configuration drift."
            )

            assert (
                meta["dataset_fingerprint"]
                == dataset_fingerprint
            ), (
                f"{dataset_label} shard {shard_id}: "
                f"dataset fingerprint drift."
            )

            assert meta["start"] == start
            assert meta["end"] == end
            assert os.path.exists(shard_path)

            actual_hash = sha256_file(shard_path)

            assert (
                actual_hash == meta["sha256"]
            ), (
                f"{dataset_label} shard {shard_id}: "
                f"completed shard hash mismatch."
            )

            shard_metadata.append(meta)

            print(
                f"[{dataset_label}] shard "
                f"{shard_id + 1}/{n_shards}: "
                f"verified frozen ({end-start} records)"
            )

            continue

        # -------------------------------------------------
        # Partial shard: recover/resume by source_index
        # -------------------------------------------------
        existing_rows = load_jsonl_recover(
            shard_path
        )

        existing_by_index = {}

        for row in existing_rows:
            idx = int(row["source_index"])

            assert start <= idx < end, (
                f"{dataset_label} shard {shard_id}: "
                f"record index {idx} outside shard range."
            )

            assert (
                row["planner_signature"]
                == RQ2_TRAIN_PLANNER_SIGNATURE
            )

            assert (
                row["dataset_fingerprint"]
                == dataset_fingerprint
            )

            expected_id = dataset["id"][idx]

            assert str(row["id"]) == str(expected_id), (
                f"{dataset_label} index {idx}: "
                f"dataset ID mismatch."
            )

            assert idx not in existing_by_index, (
                f"{dataset_label} shard {shard_id}: "
                f"duplicate source_index={idx}."
            )

            existing_by_index[idx] = row

        missing_indices = [
            i for i in range(start, end)
            if i not in existing_by_index
        ]

        print(
            f"\n[{dataset_label}] shard "
            f"{shard_id + 1}/{n_shards} "
            f"[{start}:{end}] | "
            f"resume={len(existing_by_index)} | "
            f"remaining={len(missing_indices)}"
        )

        if missing_indices:
            with open(
                shard_path,
                "a",
                encoding="utf-8"
            ) as f:

                for j, idx in enumerate(
                    tqdm(
                        missing_indices,
                        desc=(
                            f"{dataset_label} "
                            f"train shard {shard_id}"
                        ),
                        leave=False
                    ),
                    start=1
                ):
                    row = plan_training_record(
                        dataset,
                        idx,
                        dataset_label,
                        dataset_fingerprint
                    )

                    f.write(
                        json.dumps(
                            row,
                            ensure_ascii=False
                        ) + "\n"
                    )

                    # Make interruption recovery practical
                    f.flush()

                    if j % 25 == 0:
                        os.fsync(f.fileno())

                os.fsync(f.fileno())

        # -------------------------------------------------
        # Verify complete shard before freezing
        # -------------------------------------------------
        complete_rows = load_jsonl_recover(
            shard_path
        )

        assert len(complete_rows) == end - start, (
            f"{dataset_label} shard {shard_id}: "
            f"expected {end-start} records, "
            f"found {len(complete_rows)}."
        )

        complete_indices = sorted(
            int(r["source_index"])
            for r in complete_rows
        )

        assert complete_indices == list(
            range(start, end)
        ), (
            f"{dataset_label} shard {shard_id}: "
            f"missing or duplicate source indices."
        )

        # Canonical deterministic ordering
        canonicalize_jsonl(
            complete_rows,
            shard_path
        )

        shard_hash = sha256_file(
            shard_path
        )

        meta = {
            "dataset": dataset_label,
            "split": "train",
            "shard_id": shard_id,
            "start": start,
            "end": end,
            "n_records": end - start,
            "dataset_fingerprint":
                dataset_fingerprint,
            "planner_signature":
                RQ2_TRAIN_PLANNER_SIGNATURE,
            "sha256":
                shard_hash,
            "code_tag":
                RQ2_CELL2_CODE_TAG,
            "created_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }

        atomic_write_json(
            meta,
            meta_path
        )

        shard_metadata.append(meta)

        print(
            f"[{dataset_label}] shard "
            f"{shard_id + 1}/{n_shards}: "
            f"FROZEN | sha256={shard_hash[:12]}..."
        )

    return shard_metadata

# =========================================================
# 10. Run resumable TRAIN planning
# =========================================================
webqsp_train_shards = generate_train_plan_shards(
    webqsp_train,
    "webqsp",
    webqsp_train_fingerprint,
    shard_size=TRAIN_PLAN_SHARD_SIZE
)

cwq_train_shards = generate_train_plan_shards(
    cwq_train,
    "cwq",
    cwq_train_fingerprint,
    shard_size=TRAIN_PLAN_SHARD_SIZE
)

# =========================================================
# 11. Combine verified shards into immutable frozen files
# =========================================================
def freeze_combined_train_plans(
    dataset,
    dataset_label,
    dataset_fingerprint,
    shard_metadata
):
    dataset_dir = os.path.join(
        RQ2_PLAN_DIR,
        dataset_label
    )

    shard_dir = os.path.join(
        dataset_dir,
        "shards"
    )

    frozen_path = os.path.join(
        dataset_dir,
        f"{dataset_label}_train_plans_frozen.jsonl"
    )

    manifest_path = os.path.join(
        dataset_dir,
        f"{dataset_label}_train_plans_frozen_manifest.json"
    )

    # ---------------------------------------------
    # If already frozen, verify rather than rewrite
    # ---------------------------------------------
    if os.path.exists(manifest_path):
        with open(
            manifest_path,
            "r",
            encoding="utf-8"
        ) as f:
            manifest = json.load(f)

        assert (
            manifest["planner_signature"]
            == RQ2_TRAIN_PLANNER_SIGNATURE
        )

        assert (
            manifest["dataset_fingerprint"]
            == dataset_fingerprint
        )

        assert manifest["n_records"] == len(dataset)
        assert os.path.exists(frozen_path)

        actual_hash = sha256_file(
            frozen_path
        )

        assert (
            actual_hash
            == manifest["combined_sha256"]
        ), (
            f"{dataset_label}: frozen combined "
            f"training-plan hash mismatch."
        )

        print(
            f"\n[{dataset_label}] Existing combined "
            f"TRAIN plan file verified."
        )

        return frozen_path, manifest

    # ---------------------------------------------
    # Create deterministic combined file
    # ---------------------------------------------
    tmp_path = frozen_path + ".tmp"
    total_written = 0

    with open(
        tmp_path,
        "w",
        encoding="utf-8"
    ) as fout:

        for meta in sorted(
            shard_metadata,
            key=lambda x: x["shard_id"]
        ):
            shard_name = (
                f"shard_{meta['shard_id']:04d}_"
                f"{meta['start']:06d}_"
                f"{meta['end'] - 1:06d}.jsonl"
            )

            shard_path = os.path.join(
                shard_dir,
                shard_name
            )

            assert (
                sha256_file(shard_path)
                == meta["sha256"]
            )

            rows = load_jsonl_recover(
                shard_path
            )

            rows = sorted(
                rows,
                key=lambda x: int(
                    x["source_index"]
                )
            )

            for row in rows:
                fout.write(
                    json.dumps(
                        row,
                        ensure_ascii=False
                    ) + "\n"
                )

                total_written += 1

        fout.flush()
        os.fsync(fout.fileno())

    assert total_written == len(dataset), (
        f"{dataset_label}: combined file has "
        f"{total_written} records; expected {len(dataset)}."
    )

    os.replace(
        tmp_path,
        frozen_path
    )

    combined_rows = load_jsonl_recover(
        frozen_path
    )

    assert len(combined_rows) == len(dataset)

    indices = [
        int(r["source_index"])
        for r in combined_rows
    ]

    assert indices == list(
        range(len(dataset))
    ), (
        f"{dataset_label}: combined training plans "
        f"are not in exact dataset order."
    )

    # Verify IDs against source dataset
    for i, row in enumerate(combined_rows):
        assert str(row["id"]) == str(
            dataset["id"][i]
        )

    combined_hash = sha256_file(
        frozen_path
    )

    manifest = {
        "dataset": dataset_label,
        "split": "train",
        "n_records": len(dataset),
        "n_shards": len(shard_metadata),
        "shard_size": TRAIN_PLAN_SHARD_SIZE,
        "dataset_fingerprint":
            dataset_fingerprint,
        "planner_signature":
            RQ2_TRAIN_PLANNER_SIGNATURE,
        "planner_config":
            RQ2_TRAIN_PLANNER_CONFIG,
        "combined_sha256":
            combined_hash,
        "code_tag":
            RQ2_CELL2_CODE_TAG,
        "frozen_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    atomic_write_json(
        manifest,
        manifest_path
    )

    print(
        f"\n[{dataset_label}] TRAIN plans FROZEN: "
        f"{len(dataset)} records"
    )

    print(
        f"[{dataset_label}] combined SHA256: "
        f"{combined_hash}"
    )

    return frozen_path, manifest


WEBQSP_TRAIN_PLAN_FILE, webqsp_train_plan_manifest = (
    freeze_combined_train_plans(
        webqsp_train,
        "webqsp",
        webqsp_train_fingerprint,
        webqsp_train_shards
    )
)

CWQ_TRAIN_PLAN_FILE, cwq_train_plan_manifest = (
    freeze_combined_train_plans(
        cwq_train,
        "cwq",
        cwq_train_fingerprint,
        cwq_train_shards
    )
)

# =========================================================
# 12. Load minimal frozen planner rows
# =========================================================
webqsp_train_plan_rows = load_jsonl_recover(
    WEBQSP_TRAIN_PLAN_FILE
)

cwq_train_plan_rows = load_jsonl_recover(
    CWQ_TRAIN_PLAN_FILE
)

assert len(webqsp_train_plan_rows) == len(
    webqsp_train
)

assert len(cwq_train_plan_rows) == len(
    cwq_train
)

# =========================================================
# 13. Memory-efficient view for future branch-label cells
# =========================================================
class FrozenPlanningView:
    """
    Lazily attaches frozen predicted_paths to the original
    HuggingFace dataset record without duplicating all graphs
    into a second in-memory list.
    """

    def __init__(self, dataset, plan_rows):
        assert len(dataset) == len(plan_rows)

        self.dataset = dataset
        self.plan_rows = plan_rows

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            return [
                self[i]
                for i in range(
                    *idx.indices(len(self))
                )
            ]

        rec = dict(self.dataset[int(idx)])
        plan_row = self.plan_rows[int(idx)]

        assert int(
            plan_row["source_index"]
        ) == int(idx)

        assert str(
            rec["id"]
        ) == str(
            plan_row["id"]
        )

        rec["predicted_paths"] = normalize_paths(
            plan_row["predicted_paths"]
        )

        rec["_plan_source_index"] = int(idx)
        rec["_planner_signature"] = (
            plan_row["planner_signature"]
        )

        return rec

    def __iter__(self):
        for i in range(len(self)):
            yield self[i]


webqsp_train_planning = FrozenPlanningView(
    webqsp_train,
    webqsp_train_plan_rows
)

cwq_train_planning = FrozenPlanningView(
    cwq_train,
    cwq_train_plan_rows
)

# =========================================================
# 14. Final completeness and plan-distribution sanity report
# =========================================================
def summarize_frozen_train_plans(
    plan_rows,
    dataset_label
):
    n_questions = len(plan_rows)

    plan_counts = [
        len(r["predicted_paths"])
        for r in plan_rows
    ]

    plan_lengths = [
        len(path)
        for r in plan_rows
        for path in r["predicted_paths"]
    ]

    empty_plan_questions = sum(
        1 for x in plan_counts
        if x == 0
    )

    print(
        f"\n[{dataset_label}] FROZEN TRAIN PLAN SUMMARY"
    )

    print(
        f"Questions:              {n_questions}"
    )

    print(
        f"Questions with 0 plans: {empty_plan_questions}"
    )

    print(
        f"Total predicted plans:  {sum(plan_counts)}"
    )

    print(
        f"Mean plans/question:    "
        f"{np.mean(plan_counts):.3f}"
    )

    if len(plan_lengths) > 0:
        print(
            f"Plan length min/max:    "
            f"{min(plan_lengths)} / "
            f"{max(plan_lengths)}"
        )

        print(
            f"Mean plan length:       "
            f"{np.mean(plan_lengths):.3f}"
        )

    print(
        f"Planner signature:      "
        f"{RQ2_TRAIN_PLANNER_SIGNATURE[:16]}..."
    )


summarize_frozen_train_plans(
    webqsp_train_plan_rows,
    "WebQSP"
)

summarize_frozen_train_plans(
    cwq_train_plan_rows,
    "CWQ"
)

print("\n" + "=" * 80)
print("=== RQ2 CELL 2: TRAIN RELATION PLANS FROZEN ===")
print("=" * 80)
print(f"WebQSP: {len(webqsp_train_plan_rows)} / {len(webqsp_train)}")
print(f"CWQ:    {len(cwq_train_plan_rows)} / {len(cwq_train)}")
print("Planner fidelity gate: PASSED")
print("Completed shards: verified and immutable")
print("Combined TRAIN plan files: frozen with SHA256 manifests")
print("No test data used.")
print("Next: construct intermediate branch supervision from TRAIN only.")

Planner model object:     model
Planner tokenizer object: tokenizer
Planner prompt source:    prompter
Planner configuration:   2c36bd8621e44901...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


[WebQSP validation] Planner fidelity: 3 checked | mismatches=0
[CWQ validation] Planner fidelity: 3 checked | mismatches=0

=== TRAIN PLANNER FIDELITY GATE: PASSED ===
WebQSP train fingerprint: 461cebcb68f95041...
CWQ train fingerprint:    bdc1e12cb379a627...

webqsp: 2826 TRAIN questions | 12 shards | shard_size=250
[webqsp] shard 1/12: verified frozen (250 records)
[webqsp] shard 2/12: verified frozen (250 records)
[webqsp] shard 3/12: verified frozen (250 records)
[webqsp] shard 4/12: verified frozen (250 records)
[webqsp] shard 5/12: verified frozen (250 records)
[webqsp] shard 6/12: verified frozen (250 records)
[webqsp] shard 7/12: verified frozen (250 records)
[webqsp] shard 8/12: verified frozen (250 records)
[webqsp] shard 9/12: verified frozen (250 records)
[webqsp] shard 10/12: verified frozen (250 records)
[webqsp] shard 11/12: verified frozen (250 records)
[webqsp] shard 12/12: verified frozen (76 records)

cwq: 27639 TRAIN questions | 111 shards | shard_size=250
[cwq] sha

cwq train shard 57:   0%|          | 0/33 [00:00<?, ?it/s]

[cwq] shard 58/111: FROZEN | sha256=fc366ca0a26d...

[cwq] shard 59/111 [14500:14750] | resume=0 | remaining=250


...

[cwq] shard 109/111: FROZEN | sha256=284950243d7e...

[cwq] shard 110/111 [27250:27500] | resume=0 | remaining=250


cwq train shard 109:   0%|          | 0/250 [00:00<?, ?it/s]

[cwq] shard 110/111: FROZEN | sha256=b2b0d5239227...

[cwq] shard 111/111 [27500:27639] | resume=0 | remaining=139


cwq train shard 110:   0%|          | 0/139 [00:00<?, ?it/s]

[cwq] shard 111/111: FROZEN | sha256=f7db18193eb3...

[webqsp] TRAIN plans FROZEN: 2826 records
[webqsp] combined SHA256: ac4d388a9a24314b103b3a376812f07a7e4271358a318a02328b2668335cd5cd

[cwq] TRAIN plans FROZEN: 27639 records
[cwq] combined SHA256: 517502c8509aa44773a9a9c2915d15303623dd10410880965a2c49449b579aa1

[WebQSP] FROZEN TRAIN PLAN SUMMARY
Questions:              2826
Questions with 0 plans: 0
Total predicted plans:  8398
Mean plans/question:    2.972
Plan length min/max:    0 / 4
Mean plan length:       1.421
Planner signature:      2c36bd8621e44901...

[CWQ] FROZEN TRAIN PLAN SUMMARY
Questions:              27639
Questions with 0 plans: 0
Total predicted plans:  82738
Mean plans/question:    2.994
Plan length min/max:    0 / 5
Mean plan length:       1.809
Planner signature:      2c36bd8621e44901...

=== RQ2 CELL 2: TRAIN RELATION PLANS FROZEN ===
WebQSP: 2826 / 2826
CWQ:    27639 / 27639
Planner fidelity gate: PASSED
Completed shards: verified and immutable
Combined TRAIN 

In [84]:
#  Frozen TRAIN-plan sanity diagnostic

def inspect_frozen_plans(plan_rows, dataset_name):
    total_plans = 0
    empty_plans = 0
    duplicate_plan_questions = 0
    length_counts = {}

    examples = []

    for row in plan_rows:
        plans = row["predicted_paths"]
        total_plans += len(plans)

        normalized = [tuple(p) for p in plans]

        if len(normalized) != len(set(normalized)):
            duplicate_plan_questions += 1

        for plan_idx, plan in enumerate(plans):
            L = len(plan)
            length_counts[L] = length_counts.get(L, 0) + 1

            if L == 0:
                empty_plans += 1

                if len(examples) < 5:
                    examples.append({
                        "source_index": row["source_index"],
                        "question_id": row["id"],
                        "question": row["question"],
                        "plan_index": plan_idx,
                        "predicted_paths": plans,
                    })

    print("\n" + "=" * 70)
    print(f"{dataset_name} FROZEN TRAIN PLAN DIAGNOSTIC")
    print("=" * 70)
    print("Questions:                ", len(plan_rows))
    print("Total predicted plans:    ", total_plans)
    print("Empty relation plans:     ", empty_plans)
    print("Duplicate-plan questions: ", duplicate_plan_questions)
    print("Plan-length distribution: ", dict(sorted(length_counts.items())))

    if examples:
        print("\nExample questions containing an empty plan:")
        for x in examples:
            print(x)

inspect_frozen_plans(
    webqsp_train_plan_rows,
    "WebQSP"
)

inspect_frozen_plans(
    cwq_train_plan_rows,
    "CWQ"
)


WebQSP FROZEN TRAIN PLAN DIAGNOSTIC
Questions:                 2826
Total predicted plans:     8398
Empty relation plans:      5
Duplicate-plan questions:  15
Plan-length distribution:  {0: 5, 1: 4916, 2: 3415, 3: 60, 4: 2}

Example questions containing an empty plan:
{'source_index': 120, 'question_id': 'WebQTrn-168', 'question': 'what was the name of the original seattle baseball team', 'plan_index': 1, 'predicted_paths': [['sports.defunct_sports_team.later_known_as'], [], ['common.topic.notable_types', 'common.topic.notable_types']]}
{'source_index': 806, 'question_id': 'WebQTrn-1097', 'question': 'what is the new orleans hornets new name', 'plan_index': 0, 'predicted_paths': [[], ['common.topic.notable_types', 'common.topic.notable_types'], ['sports.defunct_sports_team.later_known_as']]}
{'source_index': 2441, 'question_id': 'WebQTrn-3312', 'question': 'which legend of zelda game is the first', 'plan_index': 0, 'predicted_paths': [[], ['common.topic.notable_types'], ['film.film.st

## Construct leakage-free TRAIN branch supervision using validated suffix DP

In [85]:
# ## 3.3 Construct leakage-free TRAIN branch supervision using validated suffix DP
# Positive branch:
#   candidate endpoint can reach >=1 TRAIN gold answer through the exact
#   remaining suffix of the SAME frozen RoG relation plan.
#
# Main label file:
#   contains branches ONLY from feasible intermediate plan-hop groups.
#
# Infeasible groups:
#   candidate set exists but no candidate supports a gold continuation.
#   Logged separately and EXCLUDED from scorer loss.
#
# Empty relation plans:
#   preserved in frozen planner outputs but skipped here because they
#   contain no relation transition and therefore no AFP supervision.
#
# Duplicate predicted plans:
#   preserved exactly as generated by the frozen RoG planner.
#
# Final hop:
#   NEVER labeled because the main AFP configuration does not prune it.

import os
import json
import hashlib
import inspect
from datetime import datetime, timezone
from tqdm import tqdm

RQ2_CELL3_CODE_TAG = "rq2_branch_labels_suffix_dp_v2"
BRANCH_LABEL_SHARD_SIZE = 250

# =========================================================
# 1. Required frozen artifacts / methodological guards
# =========================================================
required = [
    "webqsp_train",
    "cwq_train",
    "webqsp_train_plan_rows",
    "cwq_train_plan_rows",
    "webqsp_train_plan_manifest",
    "cwq_train_plan_manifest",
    "build_graph",
    "suffix_reachable_dp",
    "RQ2_LABEL_DIR",
]

missing = [
    x for x in required
    if x not in globals()
]

assert not missing, (
    "Run RQ2 Cell 2 to COMPLETE first. Missing: "
    + ", ".join(missing)
)

assert AFP_FINAL_HOP_PROTECTION is True
assert AFP_INTERMEDIATE_ONLY is True
assert AFP_EXCLUDE_ALL_NEGATIVE_GROUPS is True

assert len(webqsp_train_plan_rows) == len(webqsp_train) == 2826
assert len(cwq_train_plan_rows) == len(cwq_train) == 27639

WEBQSP_PLAN_SHA256 = (
    webqsp_train_plan_manifest["combined_sha256"]
)

CWQ_PLAN_SHA256 = (
    cwq_train_plan_manifest["combined_sha256"]
)

# Hash the already-validated suffix-DP implementation
try:
    SUFFIX_DP_SOURCE = inspect.getsource(
        suffix_reachable_dp
    )

    SUFFIX_DP_SHA256 = hashlib.sha256(
        SUFFIX_DP_SOURCE.encode("utf-8")
    ).hexdigest()

except Exception:
    SUFFIX_DP_SHA256 = "source_unavailable"

print(
    "Suffix-DP signature:",
    inspect.signature(suffix_reachable_dp)
)

print(
    "Suffix-DP SHA256:   ",
    SUFFIX_DP_SHA256[:16] + "..."
)

# =========================================================
# 2. Dataset / frozen-plan alignment gate
# =========================================================
def validate_train_schema(
    dataset,
    plan_rows,
    dataset_name
):
    required_fields = {
        "id",
        "question",
        "q_entity",
        "a_entity",
        "graph",
    }

    actual_fields = set(
        dataset.column_names
    )

    missing_fields = (
        required_fields - actual_fields
    )

    assert not missing_fields, (
        f"{dataset_name}: missing fields "
        f"{sorted(missing_fields)}\n"
        f"Available: {sorted(actual_fields)}"
    )

    assert len(dataset) == len(plan_rows)

    check_indices = [
        0,
        len(dataset) // 2,
        len(dataset) - 1,
    ]

    for i in check_indices:
        assert (
            str(dataset[i]["id"])
            == str(plan_rows[i]["id"])
        )

        assert (
            int(plan_rows[i]["source_index"])
            == i
        )

        assert (
            plan_rows[i]["split"]
            == "train"
        )

    print(
        f"[{dataset_name}] "
        f"schema/alignment gate: PASSED | "
        f"{len(dataset)} questions"
    )


validate_train_schema(
    webqsp_train,
    webqsp_train_plan_rows,
    "WebQSP"
)

validate_train_schema(
    cwq_train,
    cwq_train_plan_rows,
    "CWQ"
)

# =========================================================
# 3. Relation-valid neighbor matching
# Uses reproduced RoG graph/relation semantics.
# =========================================================
def relation_valid_neighbors(
    G,
    node,
    required_relation
):
    if node not in G:
        return []

    matched = []

    for nbr in G.neighbors(node):
        edge_data = G.get_edge_data(
            node,
            nbr
        )

        assert edge_data is not None

        assert "relation" in edge_data, (
            "Expected edge attribute "
            "'relation' not found."
        )

        edge_relation = (
            edge_data["relation"]
        )

        if isinstance(
            edge_relation,
            (list, tuple, set)
        ):
            is_match = (
                required_relation
                in edge_relation
            )
        else:
            is_match = (
                edge_relation
                == required_relation
            )

        if is_match:
            matched.append(nbr)

    return matched

# =========================================================
# 4. Adapter to validated suffix-reachability DP
#
# Concept:
# reachable_dp[t][e] =
#   can entity e reach any gold answer by executing plan[t:]?
#
# Candidate generated at hop h already executed plan[h].
# Therefore its remaining suffix begins at h+1.
# =========================================================
def call_suffix_dp(
    G,
    plan,
    gold_answers
):
    sig = inspect.signature(
        suffix_reachable_dp
    )

    params = list(
        sig.parameters.keys()
    )

    if len(params) != 3:
        raise RuntimeError(
            "suffix_reachable_dp does not have "
            "the expected 3-argument interface. "
            f"Current signature: {sig}"
        )

    values = {}

    for p in params:
        low = p.lower()

        if (
            low in {"g", "kg", "graph"}
            or "graph" in low
        ):
            values[p] = G

        elif (
            "plan" in low
            or "rule" in low
            or "relation_path" in low
            or low in {
                "relations",
                "path",
            }
        ):
            values[p] = plan

        elif (
            "gold" in low
            or "answer" in low
            or "target" in low
        ):
            values[p] = gold_answers

        else:
            raise RuntimeError(
                f"Cannot safely map DP "
                f"parameter '{p}'. "
                f"Signature: {sig}"
            )

    reachable_dp = (
        suffix_reachable_dp(**values)
    )

    assert (
        len(reachable_dp)
        == len(plan) + 1
    ), (
        f"Suffix DP returned "
        f"{len(reachable_dp)} layers "
        f"for plan length {len(plan)}; "
        f"expected L+1."
    )

    return reachable_dp


def dp_is_reachable(
    reachable_dp,
    state_idx,
    entity
):
    layer = reachable_dp[state_idx]

    if isinstance(layer, dict):
        return bool(
            layer.get(entity, False)
        )

    if isinstance(
        layer,
        (set, frozenset)
    ):
        return entity in layer

    try:
        return bool(layer[entity])

    except (
        KeyError,
        IndexError,
        TypeError,
    ):
        return False

# =========================================================
# 5. Terminal-state DP sanity gate
# At DP state L, reachability must equal gold membership.
# =========================================================
def suffix_dp_terminal_gate(
    dataset,
    plan_rows,
    dataset_name
):
    checked = 0

    for source_index in range(
        min(len(dataset), 100)
    ):
        rec = dataset[source_index]

        plans = (
            plan_rows[source_index]
            ["predicted_paths"]
        )

        if not plans:
            continue

        G = build_graph(
            rec["graph"]
        )

        gold_answers = set(
            rec["a_entity"]
        )

        for plan in plans:
            plan = list(plan)

            # Empty relation plans contain no
            # AFP-supervisable transition.
            if len(plan) == 0:
                continue

            reachable_dp = call_suffix_dp(
                G,
                plan,
                gold_answers
            )

            L = len(plan)

            nodes_to_check = list(
                gold_answers
            )[:10]

            if len(nodes_to_check) < 20:
                nodes_to_check += list(
                    G.nodes()
                )[:20]

            for node in nodes_to_check:
                observed = dp_is_reachable(
                    reachable_dp,
                    L,
                    node
                )

                expected = (
                    node in gold_answers
                )

                assert observed == expected, (
                    f"{dataset_name}: "
                    f"suffix-DP terminal mismatch\n"
                    f"node={node}\n"
                    f"observed={observed}\n"
                    f"expected={expected}"
                )

                checked += 1

            if checked >= 50:
                print(
                    f"[{dataset_name}] "
                    f"suffix-DP terminal gate: "
                    f"PASSED | checked={checked}"
                )

                return

    assert checked > 0

    print(
        f"[{dataset_name}] "
        f"suffix-DP terminal gate: "
        f"PASSED | checked={checked}"
    )


suffix_dp_terminal_gate(
    webqsp_train,
    webqsp_train_plan_rows,
    "WebQSP"
)

suffix_dp_terminal_gate(
    cwq_train,
    cwq_train_plan_rows,
    "CWQ"
)

print(
    "\n=== SUFFIX-DP "
    "TRAINING-LABEL PREFLIGHT: PASSED ==="
)

# =========================================================
# 6. JSON / hashing helpers
# =========================================================
def json_safe(x):
    if isinstance(x, dict):
        return {
            str(k): json_safe(v)
            for k, v in x.items()
        }

    if isinstance(
        x,
        (list, tuple, set)
    ):
        return [
            json_safe(v)
            for v in x
        ]

    if isinstance(x, np.integer):
        return int(x)

    if isinstance(x, np.floating):
        return float(x)

    if isinstance(x, np.bool_):
        return bool(x)

    return x


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def atomic_json(
    obj,
    path
):
    tmp = path + ".tmp"

    with open(
        tmp,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            json_safe(obj),
            f,
            ensure_ascii=False,
            indent=2
        )

        f.flush()
        os.fsync(f.fileno())

    os.replace(
        tmp,
        path
    )


def atomic_jsonl(
    rows,
    path
):
    tmp = path + ".tmp"

    with open(
        tmp,
        "w",
        encoding="utf-8"
    ) as f:
        for row in rows:
            f.write(
                json.dumps(
                    json_safe(row),
                    ensure_ascii=False
                )
                + "\n"
            )

        f.flush()
        os.fsync(f.fileno())

    os.replace(
        tmp,
        path
    )

# =========================================================
# 7. Label ONE training question
# =========================================================
def label_one_train_question(
    rec,
    plan_row,
    dataset_name,
    source_index
):
    assert (
        str(rec["id"])
        == str(plan_row["id"])
    )

    assert (
        int(plan_row["source_index"])
        == source_index
    )

    G = build_graph(
        rec["graph"]
    )

    topic_entities = list(
        rec["q_entity"]
    )

    gold_answers = set(
        rec["a_entity"]
    )

    plans = [
        list(p)
        for p in plan_row[
            "predicted_paths"
        ]
    ]

    branch_rows = []
    group_rows = []

    question_stats = {
        "plans": len(plans),

        # NEW: explicit audit of empty plans
        "empty_plans_skipped": 0,

        "topic_entities":
            len(topic_entities),

        "intermediate_groups": 0,
        "feasible_groups": 0,
        "infeasible_groups": 0,

        "empty_terminated": 0,
        "final_hop_groups": 0,
        "decision_groups": 0,

        "labeled_branches": 0,
        "positive_branches": 0,
        "negative_branches": 0,
    }

    # -----------------------------------------------------
    # Preserve EVERY frozen predicted plan, including
    # duplicate plan instances. We do not deduplicate.
    # -----------------------------------------------------
    for plan_idx, plan in enumerate(
        plans
    ):
        L = len(plan)

        # ---------------------------------------------
        # Frozen planner occasionally produced [].
        # Keep it in planner artifact, but it has
        # no relation transition and no AFP label.
        # ---------------------------------------------
        if L == 0:
            question_stats[
                "empty_plans_skipped"
            ] += 1
            continue

        # Plan-specific DP computed once,
        # reused across topic entities.
        reachable_dp = call_suffix_dp(
            G,
            plan,
            gold_answers
        )

        for (
            topic_idx,
            topic_entity
        ) in enumerate(
            topic_entities
        ):
            # Path multiplicity is preserved.
            active_prefixes = [
                (topic_entity,)
            ]

            for h in range(L):
                required_relation = (
                    plan[h]
                )

                candidates = []

                # -----------------------------------------
                # Construct exact relation-valid C_h
                # -----------------------------------------
                for (
                    parent_idx,
                    prefix
                ) in enumerate(
                    active_prefixes
                ):
                    current_entity = (
                        prefix[-1]
                    )

                    neighbors = (
                        relation_valid_neighbors(
                            G,
                            current_entity,
                            required_relation
                        )
                    )

                    for nbr in neighbors:
                        candidates.append({
                            "parent_prefix_index":
                                parent_idx,

                            "prefix_entities":
                                prefix,

                            "candidate_entity":
                                nbr,

                            "branch_entities":
                                prefix + (nbr,),
                        })

                group_id = (
                    f"{dataset_name}|"
                    f"{source_index}|"
                    f"p{plan_idx}|"
                    f"t{topic_idx}|"
                    f"h{h}"
                )

                # -----------------------------------------
                # C_h = empty:
                # terminate THIS topic-plan traversal.
                # -----------------------------------------
                if len(candidates) == 0:
                    group_rows.append({
                        "dataset":
                            dataset_name,

                        "split":
                            "train",

                        "source_index":
                            source_index,

                        "question_id":
                            rec["id"],

                        "group_id":
                            group_id,

                        "plan_index":
                            plan_idx,

                        "topic_index":
                            topic_idx,

                        "topic_entity":
                            topic_entity,

                        "hop":
                            h,

                        "plan_length":
                            L,

                        "required_relation":
                            required_relation,

                        "candidate_count":
                            0,

                        "positive_count":
                            0,

                        "negative_count":
                            0,

                        "decision_opportunity":
                            False,

                        "status":
                            "empty_terminated",
                    })

                    question_stats[
                        "empty_terminated"
                    ] += 1

                    break

                # -----------------------------------------
                # FINAL HOP:
                # deliberately excluded from supervision.
                # -----------------------------------------
                if h == L - 1:
                    group_rows.append({
                        "dataset":
                            dataset_name,

                        "split":
                            "train",

                        "source_index":
                            source_index,

                        "question_id":
                            rec["id"],

                        "group_id":
                            group_id,

                        "plan_index":
                            plan_idx,

                        "topic_index":
                            topic_idx,

                        "topic_entity":
                            topic_entity,

                        "hop":
                            h,

                        "plan_length":
                            L,

                        "required_relation":
                            required_relation,

                        "candidate_count":
                            len(candidates),

                        "positive_count":
                            None,

                        "negative_count":
                            None,

                        "decision_opportunity":
                            False,

                        "status":
                            "final_hop_excluded",
                    })

                    question_stats[
                        "final_hop_groups"
                    ] += 1

                    break

                # -----------------------------------------
                # INTERMEDIATE SUPERVISION
                #
                # Candidate already executed plan[h].
                # Remaining suffix = plan[h+1:].
                #
                # y_i = reachable_dp[h+1][candidate]
                # -----------------------------------------
                labels = [
                    int(
                        dp_is_reachable(
                            reachable_dp,
                            h + 1,
                            c[
                                "candidate_entity"
                            ]
                        )
                    )
                    for c in candidates
                ]

                n_pos = int(
                    sum(labels)
                )

                n_neg = (
                    len(labels) - n_pos
                )

                feasible = (
                    n_pos > 0
                )

                decision = (
                    len(candidates) > 1
                )

                question_stats[
                    "intermediate_groups"
                ] += 1

                question_stats[
                    "decision_groups"
                ] += int(decision)

                group_status = (
                    "feasible"
                    if feasible
                    else
                    "infeasible_all_negative"
                )

                group_rows.append({
                    "dataset":
                        dataset_name,

                    "split":
                        "train",

                    "source_index":
                        source_index,

                    "question_id":
                        rec["id"],

                    "group_id":
                        group_id,

                    "plan_index":
                        plan_idx,

                    "topic_index":
                        topic_idx,

                    "topic_entity":
                        topic_entity,

                    "hop":
                        h,

                    "plan_length":
                        L,

                    "required_relation":
                        required_relation,

                    "candidate_count":
                        len(candidates),

                    "positive_count":
                        n_pos,

                    "negative_count":
                        n_neg,

                    "decision_opportunity":
                        decision,

                    "status":
                        group_status,
                })

                if feasible:
                    question_stats[
                        "feasible_groups"
                    ] += 1

                    # -------------------------------------
                    # ONLY feasible groups enter
                    # scorer-supervision dataset.
                    # -------------------------------------
                    for (
                        candidate_idx,
                        (cand, y)
                    ) in enumerate(
                        zip(
                            candidates,
                            labels
                        )
                    ):
                        branch_rows.append({
                            "dataset":
                                dataset_name,

                            "split":
                                "train",

                            "source_index":
                                source_index,

                            "question_id":
                                rec["id"],

                            "group_id":
                                group_id,

                            "plan_index":
                                plan_idx,

                            "plan":
                                plan,

                            "plan_length":
                                L,

                            "topic_index":
                                topic_idx,

                            "topic_entity":
                                topic_entity,

                            "hop":
                                h,

                            "required_relation":
                                required_relation,

                            "remaining_suffix":
                                plan[h + 1:],

                            "candidate_index":
                                candidate_idx,

                            "parent_prefix_index":
                                cand[
                                    "parent_prefix_index"
                                ],

                            "prefix_entities":
                                list(
                                    cand[
                                        "prefix_entities"
                                    ]
                                ),

                            "candidate_entity":
                                cand[
                                    "candidate_entity"
                                ],

                            "branch_entities":
                                list(
                                    cand[
                                        "branch_entities"
                                    ]
                                ),

                            "candidate_count":
                                len(candidates),

                            "decision_opportunity":
                                decision,

                            "label":
                                int(y),
                        })

                    question_stats[
                        "labeled_branches"
                    ] += len(candidates)

                    question_stats[
                        "positive_branches"
                    ] += n_pos

                    question_stats[
                        "negative_branches"
                    ] += n_neg

                else:
                    # -------------------------------------
                    # Planner/graph coverage failure:
                    # logged but excluded from BCE labels.
                    # -------------------------------------
                    question_stats[
                        "infeasible_groups"
                    ] += 1

                # -----------------------------------------
                # CRITICAL:
                # Label construction follows UNPRUNED RoG.
                #
                # Gold labels NEVER influence traversal.
                # All relation-valid candidates propagate.
                # -----------------------------------------
                active_prefixes = [
                    tuple(
                        c["branch_entities"]
                    )
                    for c in candidates
                ]

    return (
        branch_rows,
        group_rows,
        question_stats
    )

# =========================================================
# 8. Resumable label-shard generation
# =========================================================
def generate_branch_label_shards(
    dataset,
    plan_rows,
    dataset_name,
    plan_sha256,
    shard_size=250
):
    dataset_dir = os.path.join(
        RQ2_LABEL_DIR,
        dataset_name
    )

    shard_dir = os.path.join(
        dataset_dir,
        "shards"
    )

    os.makedirs(
        shard_dir,
        exist_ok=True
    )

    n = len(dataset)

    n_shards = (
        n + shard_size - 1
    ) // shard_size

    metadata = []

    print(
        f"\n{dataset_name.upper()}: "
        f"{n} TRAIN questions | "
        f"{n_shards} label shards"
    )

    for shard_id in range(
        n_shards
    ):
        start = (
            shard_id * shard_size
        )

        end = min(
            start + shard_size,
            n
        )

        stem = (
            f"shard_{shard_id:04d}_"
            f"{start:06d}_"
            f"{end - 1:06d}"
        )

        labels_path = os.path.join(
            shard_dir,
            stem + ".labels.jsonl"
        )

        groups_path = os.path.join(
            shard_dir,
            stem + ".groups.jsonl"
        )

        meta_path = os.path.join(
            shard_dir,
            stem + ".meta.json"
        )

        # ---------------------------------------------
        # Frozen shard: verify, never regenerate.
        # ---------------------------------------------
        if os.path.exists(meta_path):
            with open(
                meta_path,
                "r",
                encoding="utf-8"
            ) as f:
                meta = json.load(f)

            assert (
                meta["plan_sha256"]
                == plan_sha256
            )

            assert (
                meta["suffix_dp_sha256"]
                == SUFFIX_DP_SHA256
            )

            assert (
                meta["start"]
                == start
            )

            assert (
                meta["end"]
                == end
            )

            assert (
                sha256_file(
                    labels_path
                )
                == meta[
                    "labels_sha256"
                ]
            )

            assert (
                sha256_file(
                    groups_path
                )
                == meta[
                    "groups_sha256"
                ]
            )

            metadata.append(meta)

            print(
                f"[{dataset_name}] "
                f"label shard "
                f"{shard_id + 1}/"
                f"{n_shards}: "
                f"verified frozen"
            )

            continue

        # Remove stale temp files
        for p in [
            labels_path + ".tmp",
            groups_path + ".tmp",
        ]:
            if os.path.exists(p):
                os.remove(p)

        all_labels = []
        all_groups = []

        stats = {
            "questions": 0,

            # NEW
            "empty_plans_skipped": 0,

            "intermediate_groups": 0,
            "feasible_groups": 0,
            "infeasible_groups": 0,

            "empty_terminated": 0,
            "final_hop_groups": 0,
            "decision_groups": 0,

            "labeled_branches": 0,
            "positive_branches": 0,
            "negative_branches": 0,
        }

        for idx in tqdm(
            range(start, end),
            desc=(
                f"{dataset_name} "
                f"label shard {shard_id}"
            ),
            leave=False
        ):
            rec = dataset[idx]

            plan_row = (
                plan_rows[idx]
            )

            (
                labels,
                groups,
                qstats
            ) = (
                label_one_train_question(
                    rec,
                    plan_row,
                    dataset_name,
                    idx
                )
            )

            all_labels.extend(
                labels
            )

            all_groups.extend(
                groups
            )

            stats[
                "questions"
            ] += 1

            for key in stats:
                if key == "questions":
                    continue

                stats[key] += (
                    qstats[key]
                )

        # ---------------------------------------------
        # Atomic shard freeze
        # ---------------------------------------------
        atomic_jsonl(
            all_labels,
            labels_path
        )

        atomic_jsonl(
            all_groups,
            groups_path
        )

        labels_hash = (
            sha256_file(
                labels_path
            )
        )

        groups_hash = (
            sha256_file(
                groups_path
            )
        )

        meta = {
            "dataset":
                dataset_name,

            "split":
                "train",

            "shard_id":
                shard_id,

            "start":
                start,

            "end":
                end,

            "n_questions":
                end - start,

            "plan_sha256":
                plan_sha256,

            "suffix_dp_sha256":
                SUFFIX_DP_SHA256,

            "code_tag":
                RQ2_CELL3_CODE_TAG,

            **stats,

            "labels_sha256":
                labels_hash,

            "groups_sha256":
                groups_hash,

            "created_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }

        atomic_json(
            meta,
            meta_path
        )

        metadata.append(meta)

        print(
            f"[{dataset_name}] "
            f"label shard "
            f"{shard_id + 1}/{n_shards}: "
            f"FROZEN | "
            f"branches="
            f"{stats['labeled_branches']} | "
            f"pos="
            f"{stats['positive_branches']} | "
            f"neg="
            f"{stats['negative_branches']} | "
            f"infeasible="
            f"{stats['infeasible_groups']} | "
            f"empty_plans="
            f"{stats['empty_plans_skipped']}"
        )

    return metadata

# =========================================================
# 9. Generate TRAIN branch supervision
# =========================================================
webqsp_branch_label_shards = (
    generate_branch_label_shards(
        webqsp_train,
        webqsp_train_plan_rows,
        "webqsp",
        WEBQSP_PLAN_SHA256,
        shard_size=
            BRANCH_LABEL_SHARD_SIZE
    )
)

cwq_branch_label_shards = (
    generate_branch_label_shards(
        cwq_train,
        cwq_train_plan_rows,
        "cwq",
        CWQ_PLAN_SHA256,
        shard_size=
            BRANCH_LABEL_SHARD_SIZE
    )
)

# =========================================================
# 10. Combine and freeze label artifacts
# =========================================================
def combine_label_artifacts(
    dataset_name,
    shard_metadata,
    plan_sha256
):
    dataset_dir = os.path.join(
        RQ2_LABEL_DIR,
        dataset_name
    )

    shard_dir = os.path.join(
        dataset_dir,
        "shards"
    )

    labels_out = os.path.join(
        dataset_dir,
        f"{dataset_name}_"
        f"train_branch_labels_frozen.jsonl"
    )

    groups_out = os.path.join(
        dataset_dir,
        f"{dataset_name}_"
        f"train_branch_groups_frozen.jsonl"
    )

    infeasible_out = os.path.join(
        dataset_dir,
        f"{dataset_name}_"
        f"train_infeasible_groups_frozen.jsonl"
    )

    label_count = 0
    group_count = 0
    infeasible_count = 0

    with \
        open(
            labels_out + ".tmp",
            "w",
            encoding="utf-8"
        ) as lf, \
        open(
            groups_out + ".tmp",
            "w",
            encoding="utf-8"
        ) as gf, \
        open(
            infeasible_out + ".tmp",
            "w",
            encoding="utf-8"
        ) as inf:

        for meta in sorted(
            shard_metadata,
            key=lambda x:
                x["shard_id"]
        ):
            stem = (
                f"shard_"
                f"{meta['shard_id']:04d}_"
                f"{meta['start']:06d}_"
                f"{meta['end'] - 1:06d}"
            )

            labels_path = os.path.join(
                shard_dir,
                stem + ".labels.jsonl"
            )

            groups_path = os.path.join(
                shard_dir,
                stem + ".groups.jsonl"
            )

            assert (
                sha256_file(
                    labels_path
                )
                == meta[
                    "labels_sha256"
                ]
            )

            assert (
                sha256_file(
                    groups_path
                )
                == meta[
                    "groups_sha256"
                ]
            )

            with open(
                labels_path,
                "r",
                encoding="utf-8"
            ) as f:

                for line in f:
                    if line.strip():
                        lf.write(line)
                        label_count += 1

            with open(
                groups_path,
                "r",
                encoding="utf-8"
            ) as f:

                for line in f:
                    if not line.strip():
                        continue

                    gf.write(line)
                    group_count += 1

                    row = json.loads(
                        line
                    )

                    if (
                        row["status"]
                        ==
                        "infeasible_all_negative"
                    ):
                        inf.write(line)
                        infeasible_count += 1

        for f in [
            lf,
            gf,
            inf
        ]:
            f.flush()
            os.fsync(f.fileno())

    os.replace(
        labels_out + ".tmp",
        labels_out
    )

    os.replace(
        groups_out + ".tmp",
        groups_out
    )

    os.replace(
        infeasible_out + ".tmp",
        infeasible_out
    )

    totals = {
        key: sum(
            int(
                m.get(
                    key,
                    0
                )
            )
            for m in shard_metadata
        )
        for key in [
            "questions",

            # NEW
            "empty_plans_skipped",

            "intermediate_groups",
            "feasible_groups",
            "infeasible_groups",
            "empty_terminated",
            "final_hop_groups",
            "decision_groups",
            "labeled_branches",
            "positive_branches",
            "negative_branches",
        ]
    }

    assert (
        label_count
        == totals[
            "labeled_branches"
        ]
    )

    assert (
        infeasible_count
        == totals[
            "infeasible_groups"
        ]
    )

    assert (
        totals[
            "labeled_branches"
        ]
        ==
        totals[
            "positive_branches"
        ]
        +
        totals[
            "negative_branches"
        ]
    )

    manifest = {
        "dataset":
            dataset_name,

        "split":
            "train",

        "code_tag":
            RQ2_CELL3_CODE_TAG,

        "plan_sha256":
            plan_sha256,

        "suffix_dp_sha256":
            SUFFIX_DP_SHA256,

        **totals,

        "labels_file":
            labels_out,

        "groups_file":
            groups_out,

        "infeasible_file":
            infeasible_out,

        "labels_sha256":
            sha256_file(
                labels_out
            ),

        "groups_sha256":
            sha256_file(
                groups_out
            ),

        "infeasible_sha256":
            sha256_file(
                infeasible_out
            ),

        "frozen_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    manifest_path = os.path.join(
        dataset_dir,
        f"{dataset_name}_"
        f"train_branch_labels_manifest.json"
    )

    atomic_json(
        manifest,
        manifest_path
    )

    return manifest

# =========================================================
# 11. Freeze combined artifacts
# =========================================================
webqsp_branch_manifest = (
    combine_label_artifacts(
        "webqsp",
        webqsp_branch_label_shards,
        WEBQSP_PLAN_SHA256
    )
)

cwq_branch_manifest = (
    combine_label_artifacts(
        "cwq",
        cwq_branch_label_shards,
        CWQ_PLAN_SHA256
    )
)

# =========================================================
# 12. Scientific sanity checks / report
# =========================================================
def report_branch_manifest(m):
    total = (
        m["labeled_branches"]
    )

    pos = (
        m["positive_branches"]
    )

    neg = (
        m["negative_branches"]
    )

    assert total == pos + neg
    assert m["feasible_groups"] > 0
    assert pos > 0
    assert neg >= 0

    pos_rate = (
        100.0 * pos / total
        if total > 0
        else 0.0
    )

    decision_rate = (
        100.0
        * m["decision_groups"]
        / m["intermediate_groups"]
        if m[
            "intermediate_groups"
        ] > 0
        else 0.0
    )

    print("\n" + "=" * 78)

    print(
        f"{m['dataset'].upper()} "
        f"TRAIN BRANCH SUPERVISION"
    )

    print("=" * 78)

    print(
        f"Questions:                   "
        f"{m['questions']}"
    )

    print(
        f"Empty relation plans skipped:"
        f" {m['empty_plans_skipped']}"
    )

    print(
        f"Intermediate groups:         "
        f"{m['intermediate_groups']}"
    )

    print(
        f"Feasible groups:             "
        f"{m['feasible_groups']}"
    )

    print(
        f"Infeasible groups:           "
        f"{m['infeasible_groups']}"
    )

    print(
        f"Decision groups |C_h|>1:     "
        f"{m['decision_groups']}"
    )

    print(
        f"Decision-group rate:         "
        f"{decision_rate:.2f}%"
    )

    print(
        f"Final-hop groups excluded:   "
        f"{m['final_hop_groups']}"
    )

    print(
        f"Empty/terminated groups:     "
        f"{m['empty_terminated']}"
    )

    print(
        f"Labeled feasible branches:   "
        f"{total}"
    )

    print(
        f"Positive branches:           "
        f"{pos}"
    )

    print(
        f"Negative branches:           "
        f"{neg}"
    )

    print(
        f"Positive rate:               "
        f"{pos_rate:.2f}%"
    )

    print(
        f"Labels SHA256:               "
        f"{m['labels_sha256'][:16]}..."
    )

    print(
        f"Groups SHA256:               "
        f"{m['groups_sha256'][:16]}..."
    )


report_branch_manifest(
    webqsp_branch_manifest
)

report_branch_manifest(
    cwq_branch_manifest
)

# =========================================================
# 13. Verify observed empty-plan counts against frozen-plan diagnostic
# =========================================================
assert (
    webqsp_branch_manifest[
        "empty_plans_skipped"
    ]
    == 5
), (
    "Unexpected WebQSP empty-plan count."
)

assert (
    cwq_branch_manifest[
        "empty_plans_skipped"
    ]
    == 132
), (
    "Unexpected CWQ empty-plan count."
)

print("\n" + "=" * 78)
print(
    "=== RQ2 CELL 3: "
    "TRAIN BRANCH SUPERVISION FROZEN ==="
)
print("=" * 78)

print(
    "TRAIN gold answers only."
)

print(
    "Empty relation plans preserved in planner artifacts "
    "but skipped from supervision."
)

print(
    "Duplicate frozen relation plans preserved."
)

print(
    "Final-hop candidates excluded from pruning supervision."
)

print(
    "All-negative infeasible groups logged separately."
)

print(
    "Only feasible intermediate groups enter scorer labels."
)

print(
    "Gold labels never alter unpruned RoG traversal."
)

print(
    "No validation/test gold answers used."
)

print(
    "Next: label sanity analysis and "
    "class-distribution diagnostics."
)

Suffix-DP signature: (graph, rule, gold_answers)
Suffix-DP SHA256:    5e60f720cc392a9a...
[WebQSP] schema/alignment gate: PASSED | 2826 questions
[CWQ] schema/alignment gate: PASSED | 27639 questions
[WebQSP] suffix-DP terminal gate: PASSED | checked=63
[CWQ] suffix-DP terminal gate: PASSED | checked=63

=== SUFFIX-DP TRAINING-LABEL PREFLIGHT: PASSED ===

WEBQSP: 2826 TRAIN questions | 12 label shards


[webqsp] label shard 1/12: FROZEN | branches=1319 | pos=549 | neg=770 | infeasible=82 | empty_plans=1


[webqsp] label shard 2/12: FROZEN | branches=1683 | pos=739 | neg=944 | infeasible=85 | empty_plans=0


[webqsp] label shard 3/12: FROZEN | branches=1585 | pos=663 | neg=922 | infeasible=80 | empty_plans=0


[webqsp] label shard 4/12: FROZEN | branches=1466 | pos=548 | neg=918 | infeasible=86 | empty_plans=1


...

[cwq] label shard 107/111: FROZEN | branches=2503 | pos=1221 | neg=1282 | infeasible=203 | empty_plans=0


[cwq] label shard 108/111: FROZEN | branches=2175 | pos=744 | neg=1431 | infeasible=210 | empty_plans=0


[cwq] label shard 109/111: FROZEN | branches=1657 | pos=535 | neg=1122 | infeasible=195 | empty_plans=0


[cwq] label shard 110/111: FROZEN | branches=2164 | pos=617 | neg=1547 | infeasible=204 | empty_plans=0


[cwq] label shard 111/111: FROZEN | branches=1815 | pos=345 | neg=1470 | infeasible=122 | empty_plans=0

WEBQSP TRAIN BRANCH SUPERVISION
Questions:                   2826
Empty relation plans skipped: 5
Intermediate groups:         3099
Feasible groups:             2177
Infeasible groups:           922
Decision groups |C_h|>1:     2000
Decision-group rate:         64.54%
Final-hop groups excluded:   6223
Empty/terminated groups:     2369
Labeled feasible branches:   19157
Positive branches:           8021
Negative branches:           11136
Positive rate:               41.87%
Labels SHA256:               428992995f787122...
Groups SHA256:               2cf2515efa647e28...

CWQ TRAIN BRANCH SUPERVISION
Questions:                   27639
Empty relation plans skipped: 132
Intermediate groups:         57866
Feasible groups:             33837
Infeasible groups:           24029
Decision groups |C_h|>1:     24222
Decision-group rate:         41.86%
Final-hop groups excluded:   64839
Empty/term

## Training-label sanity checks and class-balance statistics

In [86]:
# Purpose:
#   1. verify frozen label/group artifacts,
#   2. quantify feasible decision groups vs singleton groups,
#   3. inspect class balance by dataset and hop,
#   4. inspect candidate-set sizes and positive multiplicity,
#   5. determine whether scorer training should use:
#         all feasible groups
#      or feasible decision groups only.
#
# NO model training occurs in this cell.
# NO validation/test gold data are used.

import os
import json
import math
from collections import Counter, defaultdict

CELL4_TAG = "rq2_label_diagnostics_v1"

# =========================================================
# 1. Required artifacts
# =========================================================
required = [
    "webqsp_branch_manifest",
    "cwq_branch_manifest",
]

missing = [
    x for x in required
    if x not in globals()
]

assert not missing, (
    "Run Cell 3 first. Missing: "
    + ", ".join(missing)
)

# =========================================================
# 2. Streaming JSONL readers
# =========================================================
def iter_jsonl(path):
    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        for line in f:
            line = line.strip()

            if line:
                yield json.loads(line)


def load_group_stats(manifest):
    groups = list(
        iter_jsonl(
            manifest["groups_file"]
        )
    )

    labels = list(
        iter_jsonl(
            manifest["labels_file"]
        )
    )

    return groups, labels

# =========================================================
# 3. Dataset diagnostic
# =========================================================
def diagnose_branch_supervision(
    manifest,
    dataset_name
):
    groups, labels = load_group_stats(
        manifest
    )

    # -----------------------------------------------------
    # Group categories
    # -----------------------------------------------------
    feasible = [
        g for g in groups
        if g["status"] == "feasible"
    ]

    infeasible = [
        g for g in groups
        if g["status"]
        == "infeasible_all_negative"
    ]

    final_groups = [
        g for g in groups
        if g["status"]
        == "final_hop_excluded"
    ]

    empty_groups = [
        g for g in groups
        if g["status"]
        == "empty_terminated"
    ]

    feasible_decision = [
        g for g in feasible
        if g["candidate_count"] > 1
    ]

    feasible_singleton = [
        g for g in feasible
        if g["candidate_count"] == 1
    ]

    infeasible_decision = [
        g for g in infeasible
        if g["candidate_count"] > 1
    ]

    infeasible_singleton = [
        g for g in infeasible
        if g["candidate_count"] == 1
    ]

    # -----------------------------------------------------
    # Label rows split by actual decision opportunity
    # -----------------------------------------------------
    decision_labels = [
        r for r in labels
        if r["decision_opportunity"]
    ]

    singleton_labels = [
        r for r in labels
        if not r["decision_opportunity"]
    ]

    def class_stats(rows):
        n = len(rows)
        pos = sum(
            int(r["label"])
            for r in rows
        )

        neg = n - pos

        return {
            "n": n,
            "pos": pos,
            "neg": neg,
            "pos_rate":
                pos / n if n else 0.0,
            "neg_pos_ratio":
                neg / pos
                if pos > 0 else float("inf"),
        }

    all_stats = class_stats(labels)
    decision_stats = class_stats(
        decision_labels
    )
    singleton_stats = class_stats(
        singleton_labels
    )

    # -----------------------------------------------------
    # Candidate-count distribution among feasible groups
    # -----------------------------------------------------
    candidate_counts = [
        int(g["candidate_count"])
        for g in feasible
    ]

    candidate_counter = Counter(
        candidate_counts
    )

    # -----------------------------------------------------
    # Positive-count multiplicity per feasible group
    # -----------------------------------------------------
    positive_counts = [
        int(g["positive_count"])
        for g in feasible
    ]

    positive_counter = Counter(
        positive_counts
    )

    # -----------------------------------------------------
    # Hop-wise statistics over LABEL ROWS
    # -----------------------------------------------------
    hop_rows = defaultdict(list)

    for r in labels:
        hop_rows[
            int(r["hop"])
        ].append(r)

    # -----------------------------------------------------
    # Hop-wise statistics over feasible DECISION rows
    # -----------------------------------------------------
    hop_decision_rows = defaultdict(list)

    for r in decision_labels:
        hop_decision_rows[
            int(r["hop"])
        ].append(r)

    # -----------------------------------------------------
    # Sanity invariants
    # -----------------------------------------------------
    assert (
        len(feasible)
        ==
        manifest["feasible_groups"]
    )

    assert (
        len(infeasible)
        ==
        manifest["infeasible_groups"]
    )

    assert (
        len(final_groups)
        ==
        manifest["final_hop_groups"]
    )

    assert (
        len(empty_groups)
        ==
        manifest["empty_terminated"]
    )

    assert (
        len(labels)
        ==
        manifest["labeled_branches"]
    )

    assert (
        all_stats["pos"]
        ==
        manifest["positive_branches"]
    )

    assert (
        all_stats["neg"]
        ==
        manifest["negative_branches"]
    )

    # Every branch row must come from a feasible group.
    feasible_ids = {
        g["group_id"]
        for g in feasible
    }

    assert all(
        r["group_id"] in feasible_ids
        for r in labels
    )

    # Every singleton feasible group must have exactly
    # one branch row.
    singleton_group_ids = {
        g["group_id"]
        for g in feasible_singleton
    }

    singleton_label_counts = Counter(
        r["group_id"]
        for r in singleton_labels
    )

    assert all(
        singleton_label_counts[g] == 1
        for g in singleton_group_ids
    )

    # All feasible groups must have >=1 positive.
    assert all(
        int(g["positive_count"]) >= 1
        for g in feasible
    )

    # All infeasible groups must have exactly 0 positives.
    assert all(
        int(g["positive_count"]) == 0
        for g in infeasible
    )

    # -----------------------------------------------------
    # Report
    # -----------------------------------------------------
    print("\n" + "=" * 84)
    print(
        f"{dataset_name.upper()} "
        f"TRAIN LABEL DIAGNOSTICS"
    )
    print("=" * 84)

    print("\n[GROUP SPACE]")
    print(
        f"Feasible groups:                 "
        f"{len(feasible)}"
    )
    print(
        f"  feasible decision |C_h|>1:     "
        f"{len(feasible_decision)}"
    )
    print(
        f"  feasible singleton |C_h|=1:    "
        f"{len(feasible_singleton)}"
    )

    if len(feasible) > 0:
        print(
            f"  decision share of feasible:    "
            f"{100 * len(feasible_decision) / len(feasible):.2f}%"
        )

    print(
        f"Infeasible groups:               "
        f"{len(infeasible)}"
    )
    print(
        f"  infeasible decision |C_h|>1:   "
        f"{len(infeasible_decision)}"
    )
    print(
        f"  infeasible singleton |C_h|=1:  "
        f"{len(infeasible_singleton)}"
    )

    print("\n[BRANCH LABEL SPACE]")

    def print_class_block(
        name,
        stats
    ):
        print(
            f"{name:<28}"
            f"n={stats['n']:<8} "
            f"pos={stats['pos']:<8} "
            f"neg={stats['neg']:<8} "
            f"pos%={100 * stats['pos_rate']:.2f} "
            f"neg/pos={stats['neg_pos_ratio']:.3f}"
        )

    print_class_block(
        "All feasible branches",
        all_stats
    )

    print_class_block(
        "Decision-group branches",
        decision_stats
    )

    print_class_block(
        "Singleton-group branches",
        singleton_stats
    )

    print("\n[CANDIDATE COUNT DISTRIBUTION: FEASIBLE GROUPS]")

    for k in sorted(candidate_counter):
        print(
            f"|C_h|={k:<4} "
            f"groups={candidate_counter[k]}"
        )

    print("\n[POSITIVE MULTIPLICITY: FEASIBLE GROUPS]")

    for k in sorted(positive_counter):
        print(
            f"positive_count={k:<4} "
            f"groups={positive_counter[k]}"
        )

    print("\n[CLASS BALANCE BY HOP: ALL FEASIBLE LABELS]")

    for h in sorted(hop_rows):
        s = class_stats(
            hop_rows[h]
        )

        print(
            f"hop={h:<2} "
            f"n={s['n']:<8} "
            f"pos={s['pos']:<8} "
            f"neg={s['neg']:<8} "
            f"pos%={100 * s['pos_rate']:.2f}"
        )

    print("\n[CLASS BALANCE BY HOP: DECISION GROUPS ONLY]")

    for h in sorted(
        hop_decision_rows
    ):
        s = class_stats(
            hop_decision_rows[h]
        )

        print(
            f"hop={h:<2} "
            f"n={s['n']:<8} "
            f"pos={s['pos']:<8} "
            f"neg={s['neg']:<8} "
            f"pos%={100 * s['pos_rate']:.2f}"
        )

    print("\n[SANITY]")
    print(
        "Manifest counts:              PASSED"
    )
    print(
        "Feasible-group linkage:       PASSED"
    )
    print(
        "Feasible groups >=1 positive: PASSED"
    )
    print(
        "Infeasible groups =0 positive:PASSED"
    )
    print(
        "Singleton row cardinality:    PASSED"
    )

    return {
        "dataset":
            dataset_name,

        "feasible_groups":
            len(feasible),

        "feasible_decision_groups":
            len(feasible_decision),

        "feasible_singleton_groups":
            len(feasible_singleton),

        "infeasible_groups":
            len(infeasible),

        "all_stats":
            all_stats,

        "decision_stats":
            decision_stats,

        "singleton_stats":
            singleton_stats,

        "candidate_counter":
            dict(candidate_counter),

        "positive_counter":
            dict(positive_counter),
    }

# =========================================================
# 4. Run diagnostics
# =========================================================
webqsp_label_diag = (
    diagnose_branch_supervision(
        webqsp_branch_manifest,
        "webqsp"
    )
)

cwq_label_diag = (
    diagnose_branch_supervision(
        cwq_branch_manifest,
        "cwq"
    )
)

print("\n" + "=" * 84)
print(
    "=== RQ2 CELL 4: "
    "TRAIN LABEL DIAGNOSTICS COMPLETE ==="
)
print("=" * 84)

print(
    "No scorer-training choice has been made yet."
)

print(
    "Next decision: whether BCE training should use "
    "all feasible groups or feasible decision groups only."
)


WEBQSP TRAIN LABEL DIAGNOSTICS

[GROUP SPACE]
Feasible groups:                 2177
  feasible decision |C_h|>1:     1457
  feasible singleton |C_h|=1:    720
  decision share of feasible:    66.93%
Infeasible groups:               922
  infeasible decision |C_h|>1:   543
  infeasible singleton |C_h|=1:  379

[BRANCH LABEL SPACE]
All feasible branches       n=19157    pos=8021     neg=11136    pos%=41.87 neg/pos=1.388
Decision-group branches     n=18437    pos=7301     neg=11136    pos%=39.60 neg/pos=1.525
Singleton-group branches    n=720      pos=720      neg=0        pos%=100.00 neg/pos=0.000

[CANDIDATE COUNT DISTRIBUTION: FEASIBLE GROUPS]
|C_h|=1    groups=720
|C_h|=2    groups=261
|C_h|=3    groups=167
|C_h|=4    groups=145
|C_h|=5    groups=73
|C_h|=6    groups=80
|C_h|=7    groups=54
|C_h|=8    groups=62
|C_h|=9    groups=34
|C_h|=10   groups=45
|C_h|=11   groups=34
|C_h|=12   groups=33
|C_h|=13   groups=28
|C_h|=14   groups=24
|C_h|=15   groups=27
|C_h|=16   groups=24
|C_h|=1

In [94]:
# ======================================================================
# RECOVER / BIND EXISTING FROZEN VALIDATION PLAN ROWS
# ======================================================================
#
# Purpose:
#   Find the already-existing frozen validation relation plans from RQ1.
#
# We DO NOT regenerate any plans.
# We DO NOT call the planner.
# We only identify and verify existing objects.
# ======================================================================

import json
from collections.abc import Sequence


def looks_like_plan_row(row):
    """
    A frozen plan row should contain:
      - predicted_paths
      - id
    source_index is strongly preferred.
    """
    if not isinstance(row, dict):
        return False

    return (
        "predicted_paths" in row
        and
        "id" in row
    )


def inspect_plan_candidate(
    obj,
    expected_len,
    dataset,
    name
):
    """
    Verify whether obj behaves like the frozen validation-plan rows.
    """

    # Must support len()
    try:
        n = len(obj)
    except Exception:
        return False, None

    if n != expected_len:
        return False, None

    # Must support indexing
    try:
        first = obj[0]
        middle = obj[n // 2]
        last = obj[n - 1]
    except Exception:
        return False, None

    for row in [first, middle, last]:
        if not looks_like_plan_row(row):
            return False, None

    # Verify ID alignment against actual validation dataset
    check_indices = [
        0,
        n // 2,
        n - 1,
    ]

    try:
        for i in check_indices:

            row = obj[i]
            rec = dataset[i]

            if str(row["id"]) != str(rec["id"]):
                return False, None

            if (
                "source_index" in row
                and
                int(row["source_index"]) != i
            ):
                return False, None

    except Exception:
        return False, None

    return True, {
        "name": name,
        "length": n,
        "sample_id": str(first["id"]),
        "sample_plans": first["predicted_paths"],
    }


def find_validation_plan_object(
    dataset,
    expected_len,
    dataset_name
):
    matches = []

    # --------------------------------------------------------------
    # Search all notebook globals
    # --------------------------------------------------------------
    for name, obj in list(globals().items()):

        # Skip obvious irrelevant namespaces / modules
        if name.startswith("_"):
            continue

        ok, info = inspect_plan_candidate(
            obj,
            expected_len,
            dataset,
            name
        )

        if ok:
            matches.append(
                (name, obj, info)
            )

    print("\n" + "=" * 80)
    print(
        f"{dataset_name.upper()} "
        f"VALIDATION PLAN OBJECT SEARCH"
    )
    print("=" * 80)

    if len(matches) == 0:

        print(
            "No matching frozen validation-plan object "
            "was found in current globals."
        )

        return None, None

    for name, _, info in matches:
        print(
            f"Candidate: {name}"
        )
        print(
            f"  rows:       {info['length']}"
        )
        print(
            f"  sample id:  {info['sample_id']}"
        )
        print(
            f"  sample plan:{info['sample_plans']}"
        )

    # Prefer names containing dataset + val terminology
    def preference(item):
        name = item[0].lower()

        score = 0

        if dataset_name.lower() in name:
            score += 10

        if "val" in name:
            score += 5

        if "plan" in name:
            score += 5

        if "frozen" in name:
            score += 2

        if "row" in name:
            score += 1

        return score

    matches.sort(
        key=preference,
        reverse=True
    )

    selected_name, selected_obj, _ = (
        matches[0]
    )

    print(
        f"\nAUTO-SELECTED: {selected_name}"
    )

    return (
        selected_obj,
        selected_name
    )


# ======================================================================
# 1. Find WebQSP frozen validation plans
# ======================================================================

(
    webqsp_val_plan_rows,
    WEBQSP_VAL_PLAN_VAR
) = find_validation_plan_object(
    dataset=webqsp_val,
    expected_len=246,
    dataset_name="webqsp"
)


# ======================================================================
# 2. Find CWQ frozen validation plans
# ======================================================================

(
    cwq_val_plan_rows,
    CWQ_VAL_PLAN_VAR
) = find_validation_plan_object(
    dataset=cwq_val,
    expected_len=3519,
    dataset_name="cwq"
)


# ======================================================================
# 3. Hard gate
# ======================================================================

assert webqsp_val_plan_rows is not None, (
    "WebQSP frozen validation plans were not found "
    "in current notebook memory."
)

assert cwq_val_plan_rows is not None, (
    "CWQ frozen validation plans were not found "
    "in current notebook memory."
)


assert len(
    webqsp_val_plan_rows
) == 246

assert len(
    cwq_val_plan_rows
) == 3519


# ======================================================================
# 4. Full alignment verification
# ======================================================================

def verify_full_plan_alignment(
    dataset,
    plan_rows,
    dataset_name
):

    zero_plan_questions = 0
    total_plans = 0

    for i in range(
        len(dataset)
    ):

        rec = dataset[i]
        row = plan_rows[i]

        assert (
            str(rec["id"])
            ==
            str(row["id"])
        ), (
            f"{dataset_name}: ID mismatch "
            f"at index {i}"
        )

        if "source_index" in row:
            assert (
                int(
                    row[
                        "source_index"
                    ]
                )
                == i
            )

        plans = row[
            "predicted_paths"
        ]

        assert isinstance(
            plans,
            (list, tuple)
        )

        if len(plans) == 0:
            zero_plan_questions += 1

        total_plans += len(plans)

    print(
        f"\n[{dataset_name}] "
        "FULL VALIDATION PLAN ALIGNMENT: PASSED"
    )

    print(
        "Questions:",
        len(dataset)
    )

    print(
        "Total predicted plans:",
        total_plans
    )

    print(
        "Questions with 0 plans:",
        zero_plan_questions
    )


verify_full_plan_alignment(
    webqsp_val,
    webqsp_val_plan_rows,
    "WebQSP"
)

verify_full_plan_alignment(
    cwq_val,
    cwq_val_plan_rows,
    "CWQ"
)


print("\n" + "=" * 80)
print(
    "=== FROZEN VALIDATION PLAN OBJECTS RECOVERED ==="
)
print("=" * 80)

print(
    "WebQSP:",
    WEBQSP_VAL_PLAN_VAR
)

print(
    "CWQ:   ",
    CWQ_VAL_PLAN_VAR
)

print(
    "\nNo relation plans regenerated."
)

print(
    "You can now rerun Cell 6."
)


WEBQSP VALIDATION PLAN OBJECT SEARCH
Candidate: webqsp_val_planning
  rows:       246
  sample id:  WebQTrn-9
  sample plan:[['people.person.nationality'], ['people.person.nationality', 'people.person.nationality'], ['people.person.nationality', 'location.location.containedby']]

AUTO-SELECTED: webqsp_val_planning

CWQ VALIDATION PLAN OBJECT SEARCH
Candidate: cwq_val_planning
  rows:       3519
  sample id:  WebQTrn-1430_ac053cda0a7424c48e4809c71171fbed
  sample plan:[['government.government_position_held.office_position_or_title', 'government.politician.government_positions_held'], ['location.location.containedby', 'people.person.nationality'], ['government.government_position_held.office_position_or_title', 'government.government_position_held.office_holder']]

AUTO-SELECTED: cwq_val_planning

[WebQSP] FULL VALIDATION PLAN ALIGNMENT: PASSED
Questions: 246
Total predicted plans: 721
Questions with 0 plans: 0

[CWQ] FULL VALIDATION PLAN ALIGNMENT: PASSED
Questions: 3519
Total predicte

## Defining AFP Feature Extraction

In [92]:
# ======================================================================
# 3.5 AFP FEATURE EXTRACTION — FINAL SPECIFICATION v2
# ======================================================================
#
# Revision reason:
#   A large fraction of candidate nodes are unresolved Freebase MIDs:
#       WebQSP: 58.44%
#       CWQ:    46.14%
#
# Policy:
#   - NEVER embed raw Freebase MIDs as natural-language text.
#   - Unresolved entity -> zero semantic embedding.
#   - Explicit availability indicators distinguish missing semantics
#     from a genuine cosine similarity near zero.
#   - Prefix semantic mean uses readable entities only.
#   - No external MID->name lookup is introduced.
#
# No model training occurs here.
# No gold-answer / suffix-DP information is available to this extractor.
# No future candidate neighborhood is inspected.
# ======================================================================

import re
import json
import math
import hashlib
from collections import Counter
from typing import Optional, Dict

import numpy as np


# ======================================================================
# 1. Freeze FINAL feature policy
# ======================================================================

AFP_FEATURE_VERSION = "afp_features_v2_masked_entity_semantics"

AFP_TRAIN_DECISION_ONLY = True
AFP_EXCLUDE_SINGLETON_FEASIBLE = True

AFP_USE_FUTURE_NEIGHBORHOOD = False
AFP_USE_GOLD_AS_FEATURE = False
AFP_USE_SUFFIX_REACHABILITY_AS_FEATURE = False
AFP_USE_KGE_CORE = False

AFP_USE_EXTERNAL_ENTITY_RESOLVER = False
AFP_MASK_UNRESOLVED_ENTITY_IDS = True

AFP_SEMANTIC_ENCODER_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

AFP_SEMANTIC_EXPECTED_DIM = 384


# ======================================================================
# 2. Raw Freebase-ID detection
# ======================================================================

FREEBASE_ID_RE = re.compile(
    r"^(?:m|g)\.[A-Za-z0-9_\-]+$"
)


def is_raw_freebase_id(x):
    if x is None:
        return False

    return bool(
        FREEBASE_ID_RE.match(
            str(x).strip()
        )
    )


def has_readable_entity_surface(
    entity,
    entity_name_map: Optional[Dict] = None
):
    # Verified resolver can be supplied later only if one exists.
    if (
        entity_name_map is not None
        and entity in entity_name_map
    ):
        resolved = entity_name_map[entity]

        return (
            resolved is not None
            and str(resolved).strip() != ""
            and not is_raw_freebase_id(resolved)
        )

    return (
        entity is not None
        and str(entity).strip() != ""
        and not is_raw_freebase_id(entity)
    )


# ======================================================================
# 3. Text normalization
# ======================================================================

def normalize_surface_text(x):
    if x is None:
        return ""

    text = str(x).strip()

    text = text.replace("_", " ")
    text = text.replace(".", " ")
    text = text.replace("/", " ")

    text = re.sub(
        r"(?<=[a-z])(?=[A-Z])",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip().lower()


def relation_surface_text(relation):
    return normalize_surface_text(
        relation
    )


def entity_surface_text(
    entity,
    entity_name_map=None
):
    if (
        entity_name_map is not None
        and entity in entity_name_map
    ):
        resolved = entity_name_map[
            entity
        ]

        if (
            resolved is not None
            and not is_raw_freebase_id(
                resolved
            )
        ):
            return normalize_surface_text(
                resolved
            )

    # CRITICAL:
    # never convert raw MID to MiniLM text.
    if is_raw_freebase_id(entity):
        return ""

    return normalize_surface_text(
        entity
    )


def relation_sequence_text(relations):
    if not relations:
        return ""

    return " ; ".join(
        relation_surface_text(r)
        for r in relations
    )


# ======================================================================
# 4. Vector helpers
# ======================================================================

def safe_l2_normalize(
    x,
    eps=1e-12
):
    x = np.asarray(
        x,
        dtype=np.float32
    )

    norm = float(
        np.linalg.norm(x)
    )

    if norm <= eps:
        return np.zeros_like(
            x,
            dtype=np.float32
        )

    return (
        x / norm
    ).astype(
        np.float32
    )


def safe_cosine(a, b):
    a = safe_l2_normalize(a)
    b = safe_l2_normalize(b)

    if (
        not np.any(a)
        or not np.any(b)
    ):
        return 0.0

    return float(
        np.clip(
            np.dot(a, b),
            -1.0,
            1.0
        )
    )


def mean_embedding(
    embeddings,
    dim
):
    if len(embeddings) == 0:
        return np.zeros(
            dim,
            dtype=np.float32
        )

    x = np.mean(
        np.asarray(
            embeddings,
            dtype=np.float32
        ),
        axis=0
    )

    return safe_l2_normalize(x)


# ======================================================================
# 5. Safe entity embedding
# ======================================================================
#
# Raw MID:
#     embedding = 0
#     available = 0
#
# Readable entity:
#     MiniLM embedding
#     available = 1
# ======================================================================

def get_safe_entity_embedding(
    semantic_encoder,
    entity,
    entity_name_map=None
):
    available = (
        has_readable_entity_surface(
            entity,
            entity_name_map
        )
    )

    if not available:
        return (
            np.zeros(
                semantic_encoder.dim,
                dtype=np.float32
            ),
            0.0
        )

    text = entity_surface_text(
        entity,
        entity_name_map
    )

    emb = semantic_encoder.get(
        "entity",
        entity,
        text
    )

    return (
        np.asarray(
            emb,
            dtype=np.float32
        ),
        1.0
    )


# ======================================================================
# 6. FINAL feature schema
# ======================================================================

AFP_SEMANTIC_FEATURE_NAMES = [

    # Candidate semantics
    "sem_q_candidate",
    "sem_candidate_surface_available",

    # Current entity semantics
    "sem_q_current_entity",
    "sem_current_surface_available",

    # Relation/plan semantics
    "sem_q_current_relation",
    "sem_q_full_plan",
    "sem_q_remaining_suffix",

    # Candidate ↔ current relation
    "sem_candidate_current_relation",
]


AFP_PATH_FEATURE_NAMES = [

    # Prefix semantic context
    "path_q_prefix_entity_mean",
    "path_candidate_prefix_entity_mean",

    # NEW: fraction of readable entities in prefix
    "path_prefix_surface_fraction",

    # Symbolic/path-history features
    "path_candidate_repeats_entity",
    "path_candidate_occurrence_fraction",
    "path_unique_entity_ratio",
    "path_relation_repeat_fraction_before",
]


AFP_STRUCTURAL_FEATURE_NAMES = [

    "struct_log_candidate_count",
    "struct_log_unique_candidate_entities",
    "struct_log_contributing_parents",
    "struct_log_parent_fanout",
    "struct_parent_frontier_share",
    "struct_log_endpoint_multiplicity",
    "struct_endpoint_frontier_share",
    "struct_duplicate_endpoint_ratio",
]


AFP_PROGRESS_FEATURE_NAMES = [

    "prog_hop_fraction",
    "prog_remaining_fraction",
    "prog_log_plan_length",
    "prog_penultimate_indicator",
]


AFP_FEATURE_NAMES = (
    AFP_SEMANTIC_FEATURE_NAMES
    + AFP_PATH_FEATURE_NAMES
    + AFP_STRUCTURAL_FEATURE_NAMES
    + AFP_PROGRESS_FEATURE_NAMES
)


AFP_FEATURE_DIM = len(
    AFP_FEATURE_NAMES
)

assert AFP_FEATURE_DIM == 27


# ======================================================================
# 7. Feature-group slices
# ======================================================================

_n_sem = len(
    AFP_SEMANTIC_FEATURE_NAMES
)

_n_path = len(
    AFP_PATH_FEATURE_NAMES
)

_n_struct = len(
    AFP_STRUCTURAL_FEATURE_NAMES
)


AFP_FEATURE_GROUP_SLICES = {

    "semantic": (
        0,
        _n_sem
    ),

    "path": (
        _n_sem,
        _n_sem + _n_path
    ),

    "structural": (
        _n_sem + _n_path,
        _n_sem + _n_path + _n_struct
    ),

    "progress": (
        _n_sem + _n_path + _n_struct,
        AFP_FEATURE_DIM
    ),
}


# ======================================================================
# 8. Feature specification fingerprint
# ======================================================================

AFP_FEATURE_SPEC = {

    "version":
        AFP_FEATURE_VERSION,

    "semantic_encoder":
        AFP_SEMANTIC_ENCODER_NAME,

    "semantic_encoder_dim":
        AFP_SEMANTIC_EXPECTED_DIM,

    "feature_dim":
        AFP_FEATURE_DIM,

    "decision_only_training":
        AFP_TRAIN_DECISION_ONLY,

    "exclude_singleton_feasible":
        AFP_EXCLUDE_SINGLETON_FEASIBLE,

    "mask_raw_freebase_ids":
        AFP_MASK_UNRESOLVED_ENTITY_IDS,

    "external_entity_resolver":
        AFP_USE_EXTERNAL_ENTITY_RESOLVER,

    "future_neighborhood":
        AFP_USE_FUTURE_NEIGHBORHOOD,

    "gold_features":
        AFP_USE_GOLD_AS_FEATURE,

    "suffix_dp_feature":
        AFP_USE_SUFFIX_REACHABILITY_AS_FEATURE,

    "kge_core":
        AFP_USE_KGE_CORE,

    "feature_names":
        AFP_FEATURE_NAMES,
}


AFP_FEATURE_SPEC_SHA256 = hashlib.sha256(
    json.dumps(
        AFP_FEATURE_SPEC,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()


# ======================================================================
# 9. FINAL candidate feature extractor
# ======================================================================

def extract_afp_candidate_features(
    question_id,
    question,
    plan,
    hop,
    candidates,
    candidate_index,
    semantic_encoder,
    entity_name_map=None
):

    plan = list(plan)
    L = len(plan)

    assert L >= 2

    assert (
        0 <= hop < L - 1
    )

    n_candidates = len(
        candidates
    )

    assert n_candidates > 1

    cand = candidates[
        candidate_index
    ]

    prefix_entities = list(
        cand["prefix_entities"]
    )

    assert len(
        prefix_entities
    ) >= 1

    current_entity = (
        prefix_entities[-1]
    )

    candidate_entity = (
        cand["candidate_entity"]
    )

    parent_prefix_index = int(
        cand["parent_prefix_index"]
    )

    current_relation = (
        plan[hop]
    )

    remaining_suffix = (
        plan[hop + 1:]
    )


    # ==================================================================
    # Reusable text embeddings
    # ==================================================================

    q_emb = semantic_encoder.get(
        "question",
        question_id,
        str(question)
    )


    (
        current_entity_emb,
        current_surface_available
    ) = get_safe_entity_embedding(
        semantic_encoder,
        current_entity,
        entity_name_map
    )


    (
        candidate_entity_emb,
        candidate_surface_available
    ) = get_safe_entity_embedding(
        semantic_encoder,
        candidate_entity,
        entity_name_map
    )


    current_relation_text = (
        relation_surface_text(
            current_relation
        )
    )

    current_relation_emb = (
        semantic_encoder.get(
            "relation",
            current_relation,
            current_relation_text
        )
    )


    full_plan_text = (
        relation_sequence_text(
            plan
        )
    )

    full_plan_key = "||".join(
        str(x)
        for x in plan
    )

    full_plan_emb = semantic_encoder.get(
        "plan",
        full_plan_key,
        full_plan_text
    )


    suffix_text = (
        relation_sequence_text(
            remaining_suffix
        )
    )

    suffix_key = "||".join(
        str(x)
        for x in remaining_suffix
    )

    suffix_emb = semantic_encoder.get(
        "suffix",
        suffix_key,
        suffix_text
    )


    # ==================================================================
    # A. SEMANTIC FEATURES (8)
    # ==================================================================

    f_sem = np.asarray(
        [
            safe_cosine(
                q_emb,
                candidate_entity_emb
            ),

            candidate_surface_available,

            safe_cosine(
                q_emb,
                current_entity_emb
            ),

            current_surface_available,

            safe_cosine(
                q_emb,
                current_relation_emb
            ),

            safe_cosine(
                q_emb,
                full_plan_emb
            ),

            safe_cosine(
                q_emb,
                suffix_emb
            ),

            safe_cosine(
                candidate_entity_emb,
                current_relation_emb
            ),
        ],
        dtype=np.float32
    )


    # ==================================================================
    # B. PATH-CONTEXT FEATURES (7)
    # ==================================================================

    readable_prefix_embeddings = []

    readable_prefix_count = 0

    for entity in prefix_entities:

        (
            entity_emb,
            entity_available
        ) = get_safe_entity_embedding(
            semantic_encoder,
            entity,
            entity_name_map
        )

        if entity_available > 0:
            readable_prefix_embeddings.append(
                entity_emb
            )

            readable_prefix_count += 1


    prefix_mean_emb = mean_embedding(
        readable_prefix_embeddings,
        semantic_encoder.dim
    )


    prefix_len = len(
        prefix_entities
    )


    prefix_surface_fraction = (
        readable_prefix_count
        / max(
            1,
            prefix_len
        )
    )


    candidate_occurrences = sum(
        1
        for e in prefix_entities
        if e == candidate_entity
    )


    unique_prefix_entities = len(
        set(
            prefix_entities
        )
    )


    previous_relations = (
        plan[:hop]
    )


    relation_repeat_before = sum(
        1
        for r in previous_relations
        if r == current_relation
    )


    f_path = np.asarray(
        [
            safe_cosine(
                q_emb,
                prefix_mean_emb
            ),

            safe_cosine(
                candidate_entity_emb,
                prefix_mean_emb
            ),

            float(
                prefix_surface_fraction
            ),

            float(
                candidate_occurrences > 0
            ),

            float(
                candidate_occurrences
                / max(
                    1,
                    prefix_len
                )
            ),

            float(
                unique_prefix_entities
                / max(
                    1,
                    prefix_len
                )
            ),

            float(
                relation_repeat_before
                / max(
                    1,
                    hop
                )
            ),
        ],
        dtype=np.float32
    )


    # ==================================================================
    # C. CURRENT-FRONTIER STRUCTURAL FEATURES (8)
    # ==================================================================
    #
    # Only C_h is used.
    # No Adj(candidate), degree, or next-hop expansion.
    # ==================================================================

    endpoint_counter = Counter(
        c["candidate_entity"]
        for c in candidates
    )


    parent_counter = Counter(
        int(
            c[
                "parent_prefix_index"
            ]
        )
        for c in candidates
    )


    unique_candidate_entities = len(
        endpoint_counter
    )


    contributing_parents = len(
        parent_counter
    )


    parent_fanout = int(
        parent_counter[
            parent_prefix_index
        ]
    )


    endpoint_multiplicity = int(
        endpoint_counter[
            candidate_entity
        ]
    )


    duplicate_endpoint_ratio = (
        1.0
        -
        (
            unique_candidate_entities
            / n_candidates
        )
    )


    f_struct = np.asarray(
        [
            math.log1p(
                n_candidates
            ),

            math.log1p(
                unique_candidate_entities
            ),

            math.log1p(
                contributing_parents
            ),

            math.log1p(
                parent_fanout
            ),

            float(
                parent_fanout
                / n_candidates
            ),

            math.log1p(
                endpoint_multiplicity
            ),

            float(
                endpoint_multiplicity
                / n_candidates
            ),

            float(
                duplicate_endpoint_ratio
            ),
        ],
        dtype=np.float32
    )


    # ==================================================================
    # D. PROGRESS FEATURES (4)
    # ==================================================================

    remaining_hops = (
        L - hop - 1
    )


    hop_fraction = (
        hop
        / max(
            1,
            L - 1
        )
    )


    remaining_fraction = (
        remaining_hops
        / L
    )


    penultimate_indicator = float(
        remaining_hops == 1
    )


    f_prog = np.asarray(
        [
            float(
                hop_fraction
            ),

            float(
                remaining_fraction
            ),

            float(
                math.log1p(L)
            ),

            penultimate_indicator,
        ],
        dtype=np.float32
    )


    # ==================================================================
    # FINAL VECTOR
    # ==================================================================

    x = np.concatenate(
        [
            f_sem,
            f_path,
            f_struct,
            f_prog
        ]
    ).astype(
        np.float32
    )


    assert x.shape == (
        AFP_FEATURE_DIM,
    )

    assert np.all(
        np.isfinite(x)
    )

    return x


# ======================================================================
# 10. Group extractor
# ======================================================================

def extract_afp_group_features(
    question_id,
    question,
    plan,
    hop,
    candidate_rows,
    semantic_encoder,
    entity_name_map=None
):

    assert len(
        candidate_rows
    ) > 1


    candidates = []

    for row in candidate_rows:

        candidates.append(
            {
                "prefix_entities":
                    list(
                        row[
                            "prefix_entities"
                        ]
                    ),

                "candidate_entity":
                    row[
                        "candidate_entity"
                    ],

                "parent_prefix_index":
                    int(
                        row[
                            "parent_prefix_index"
                        ]
                    ),
            }
        )


    X = np.vstack(
        [
            extract_afp_candidate_features(
                question_id=
                    question_id,

                question=
                    question,

                plan=
                    plan,

                hop=
                    hop,

                candidates=
                    candidates,

                candidate_index=
                    i,

                semantic_encoder=
                    semantic_encoder,

                entity_name_map=
                    entity_name_map
            )

            for i in range(
                len(candidates)
            )
        ]
    ).astype(
        np.float32
    )


    assert X.shape == (
        len(candidates),
        AFP_FEATURE_DIM
    )

    return X


# ======================================================================
# 11. Feature-group helper
# ======================================================================

def split_afp_feature_groups(X):

    X = np.asarray(
        X,
        dtype=np.float32
    )

    result = {}

    for (
        group_name,
        (start, end)
    ) in (
        AFP_FEATURE_GROUP_SLICES
        .items()
    ):

        result[
            group_name
        ] = X[
            ...,
            start:end
        ]

    return result


# ======================================================================
# 12. SANITY TEST WITH RAW FREEBASE IDs
# ======================================================================

# Reuse MockSemanticEncoder from previous Cell 5.
# If unavailable, define a tiny one.

if "MockSemanticEncoder" not in globals():

    class MockSemanticEncoder:

        def __init__(
            self,
            dim=32
        ):
            self.dim = dim

        def get(
            self,
            namespace,
            identifier,
            raw_text
        ):
            # Raw MID must NEVER arrive here as entity text.
            if (
                namespace == "entity"
                and is_raw_freebase_id(
                    raw_text
                )
            ):
                raise AssertionError(
                    "Raw Freebase MID reached semantic encoder."
                )

            token = (
                f"{namespace}|"
                f"{identifier}|"
                f"{raw_text}"
            )

            digest = hashlib.sha256(
                token.encode(
                    "utf-8"
                )
            ).digest()

            seed = int.from_bytes(
                digest[:8],
                "little"
            )

            rng = (
                np.random.default_rng(
                    seed
                )
            )

            vec = rng.normal(
                size=self.dim
            ).astype(
                np.float32
            )

            return safe_l2_normalize(
                vec
            )


_mock_encoder = (
    MockSemanticEncoder(
        dim=32
    )
)


_mock_rows = [

    {
        "prefix_entities": [
            "Natalie Portman"
        ],
        "candidate_entity":
            "m.0k3qzz",
        "parent_prefix_index":
            0,
    },

    {
        "prefix_entities": [
            "Natalie Portman"
        ],
        "candidate_entity":
            "Canada",
        "parent_prefix_index":
            0,
    },
]


_mock_X = (
    extract_afp_group_features(

        question_id=
            "mock-mid-test",

        question=
            "where was the person born",

        plan=[
            "people.person.place_of_birth",
            "location.location.containedby",
        ],

        hop=0,

        candidate_rows=
            _mock_rows,

        semantic_encoder=
            _mock_encoder
    )
)


assert _mock_X.shape == (
    2,
    27
)


# ----------------------------------------------------------
# Raw-MID candidate:
#
# feature 0 = sem_q_candidate -> must be 0
# feature 1 = candidate_surface_available -> must be 0
# feature 7 = sem_candidate_current_relation -> must be 0
# feature 9 = path_candidate_prefix_entity_mean -> must be 0
# ----------------------------------------------------------

assert np.isclose(
    _mock_X[0, 0],
    0.0
)

assert np.isclose(
    _mock_X[0, 1],
    0.0
)

assert np.isclose(
    _mock_X[0, 7],
    0.0
)

assert np.isclose(
    _mock_X[0, 9],
    0.0
)


# Readable candidate must report availability=1
assert np.isclose(
    _mock_X[1, 1],
    1.0
)


assert np.all(
    np.isfinite(
        _mock_X
    )
)


# ======================================================================
# 13. FINAL REPORT
# ======================================================================

print(
    "\n" + "=" * 84
)

print(
    "=== RQ2 CELL 5: "
    "AFP FEATURE EXTRACTION v2 FROZEN ==="
)

print("=" * 84)

print(
    f"Semantic features:     "
    f"{len(AFP_SEMANTIC_FEATURE_NAMES)}"
)

print(
    f"Path features:         "
    f"{len(AFP_PATH_FEATURE_NAMES)}"
)

print(
    f"Structural features:   "
    f"{len(AFP_STRUCTURAL_FEATURE_NAMES)}"
)

print(
    f"Progress features:     "
    f"{len(AFP_PROGRESS_FEATURE_NAMES)}"
)

print(
    f"TOTAL FEATURE DIM:     "
    f"{AFP_FEATURE_DIM}"
)

print(
    f"Feature-spec SHA256:   "
    f"{AFP_FEATURE_SPEC_SHA256}"
)


print(
    "\nEntity semantic policy:"
)

print(
    "  Readable entity name       -> frozen MiniLM embedding"
)

print(
    "  Raw Freebase MID           -> ZERO embedding"
)

print(
    "  Candidate availability     -> explicit feature"
)

print(
    "  Current availability       -> explicit feature"
)

print(
    "  Prefix semantic mean       -> readable entities only"
)

print(
    "  Prefix readable fraction   -> explicit feature"
)


print(
    "\nMethodological safeguards:"
)

print(
    "  Raw MID sent to MiniLM:          NO"
)

print(
    "  External MID resolver added:     NO"
)

print(
    "  Gold-answer feature:             NO"
)

print(
    "  Suffix-reachability feature:     NO"
)

print(
    "  Future-neighborhood inspection:  NO"
)

print(
    "  Candidate degree lookup:         NO"
)

print(
    "  Singleton scorer invocation:     NO"
)

print(
    "  Training groups: feasible decisions only"
)


print(
    "\nRaw-MID masking synthetic gate: PASSED"
)

print(
    "\nThis v2 specification supersedes "
    "afp_features_v1."
)

print(
    "Next: build/cache TRAIN + VALIDATION "
    "feature datasets."
)


=== RQ2 CELL 5: AFP FEATURE EXTRACTION v2 FROZEN ===
Semantic features:     8
Path features:         7
Structural features:   8
Progress features:     4
TOTAL FEATURE DIM:     27
Feature-spec SHA256:   738985d1232a8ac5935c397ed95eca23377bc59b4a99494547fa7779062ade86

Entity semantic policy:
  Readable entity name       -> frozen MiniLM embedding
  Raw Freebase MID           -> ZERO embedding
  Candidate availability     -> explicit feature
  Current availability       -> explicit feature
  Prefix semantic mean       -> readable entities only
  Prefix readable fraction   -> explicit feature

Methodological safeguards:
  Raw MID sent to MiniLM:          NO
  External MID resolver added:     NO
  Gold-answer feature:             NO
  Suffix-reachability feature:     NO
  Future-neighborhood inspection:  NO
  Candidate degree lookup:         NO
  Singleton scorer invocation:     NO
  Training groups: feasible decisions only

Raw-MID masking synthetic gate: PASSED

This v2 specification su

In [89]:
# =========================================================
# Semantic surface-form sanity check
# =========================================================

def inspect_entity_surface_forms(
    branch_manifest,
    dataset_name,
    n=20
):
    rows = []

    with open(
        branch_manifest["labels_file"],
        "r",
        encoding="utf-8"
    ) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

            if len(rows) >= n:
                break

    print("\n" + "=" * 72)
    print(f"{dataset_name.upper()} ENTITY-SURFACE SANITY")
    print("=" * 72)

    for i, row in enumerate(rows[:n]):
        candidate = row["candidate_entity"]

        prefix_last = (
            row["prefix_entities"][-1]
            if row["prefix_entities"]
            else None
        )

        print(
            f"{i:02d} | "
            f"current={prefix_last} | "
            f"candidate={candidate}"
        )

    print()


inspect_entity_surface_forms(
    webqsp_branch_manifest,
    "webqsp",
    n=20
)

inspect_entity_surface_forms(
    cwq_branch_manifest,
    "cwq",
    n=20
)


WEBQSP ENTITY-SURFACE SANITY
00 | current=Justin Bieber | candidate=Canada
01 | current=Natalie Portman | candidate=m.0k3qzz
02 | current=Natalie Portman | candidate=m.03jt5jq
03 | current=Natalie Portman | candidate=m.0nfnhrj
04 | current=Natalie Portman | candidate=m.040myw2
05 | current=Natalie Portman | candidate=m.04dcjy9
06 | current=Natalie Portman | candidate=m.0k3r0b
07 | current=Natalie Portman | candidate=m.0cs2bt7
08 | current=Natalie Portman | candidate=m.0j_xb7
09 | current=Natalie Portman | candidate=m.0nh4bnk
10 | current=Natalie Portman | candidate=m.02vb_fz
11 | current=Natalie Portman | candidate=m.0k7msn
12 | current=Natalie Portman | candidate=m.0109rjwm
13 | current=Natalie Portman | candidate=m.0jtns5
14 | current=Natalie Portman | candidate=m.0cccx3m
15 | current=Natalie Portman | candidate=m.07zm_x0
16 | current=Natalie Portman | candidate=m.0jwjgq
17 | current=Natalie Portman | candidate=m.0k3qy8
18 | current=Natalie Portman | candidate=g.11b7qqspgv
19 | curr

In [90]:
# ======================================================================
# AUDIT ENTITY SURFACE-FORM COVERAGE
# ======================================================================
#
# Purpose:
#   1. quantify how often candidate/current entities are raw Freebase IDs,
#   2. inspect whether NetworkX graph nodes already contain useful
#      name/label attributes,
#   3. inspect available graph-record structure before deciding how
#      entity surface resolution should be implemented.
#
# No feature cache is constructed here.
# No gold/test information is used.
# ======================================================================

import re
import json
from collections import Counter

FREEBASE_ID_RE = re.compile(
    r"^(?:m|g)\.[A-Za-z0-9_\-]+$"
)


def is_raw_freebase_id(x):
    if x is None:
        return False

    return bool(
        FREEBASE_ID_RE.match(
            str(x).strip()
        )
    )


# ----------------------------------------------------------------------
# 1. Stream decision-group branch rows only
# ----------------------------------------------------------------------
def audit_entity_id_rate(
    manifest,
    dataset_name
):
    total = 0

    candidate_raw = 0
    current_raw = 0

    both_named = 0
    any_raw = 0

    raw_examples = []

    with open(
        manifest["labels_file"],
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            row = json.loads(line)

            # Scorer is trained only on actual decision groups.
            if not row.get(
                "decision_opportunity",
                False
            ):
                continue

            total += 1

            candidate = row[
                "candidate_entity"
            ]

            current = row[
                "prefix_entities"
            ][-1]

            cand_is_raw = (
                is_raw_freebase_id(
                    candidate
                )
            )

            curr_is_raw = (
                is_raw_freebase_id(
                    current
                )
            )

            candidate_raw += int(
                cand_is_raw
            )

            current_raw += int(
                curr_is_raw
            )

            if (
                not cand_is_raw
                and not curr_is_raw
            ):
                both_named += 1

            if (
                cand_is_raw
                or curr_is_raw
            ):
                any_raw += 1

                if len(
                    raw_examples
                ) < 10:

                    raw_examples.append({
                        "source_index":
                            row[
                                "source_index"
                            ],

                        "question_id":
                            row[
                                "question_id"
                            ],

                        "current":
                            current,

                        "candidate":
                            candidate,
                    })

    print(
        "\n" + "=" * 78
    )

    print(
        f"{dataset_name.upper()} "
        f"DECISION-BRANCH ENTITY AUDIT"
    )

    print(
        "=" * 78
    )

    print(
        f"Decision branches:           "
        f"{total}"
    )

    print(
        f"Raw candidate IDs:           "
        f"{candidate_raw} "
        f"({100*candidate_raw/max(1,total):.2f}%)"
    )

    print(
        f"Raw current-entity IDs:      "
        f"{current_raw} "
        f"({100*current_raw/max(1,total):.2f}%)"
    )

    print(
        f"Any raw endpoint:            "
        f"{any_raw} "
        f"({100*any_raw/max(1,total):.2f}%)"
    )

    print(
        f"Both endpoints readable:     "
        f"{both_named} "
        f"({100*both_named/max(1,total):.2f}%)"
    )

    print(
        "\nExamples:"
    )

    for x in raw_examples:
        print(x)

    return raw_examples


webqsp_raw_examples = (
    audit_entity_id_rate(
        webqsp_branch_manifest,
        "webqsp"
    )
)

cwq_raw_examples = (
    audit_entity_id_rate(
        cwq_branch_manifest,
        "cwq"
    )
)


# ======================================================================
# 2. Inspect graph representation around raw-ID examples
# ======================================================================

def inspect_graph_node_metadata(
    dataset,
    examples,
    dataset_name,
    max_examples=5
):
    print(
        "\n" + "=" * 78
    )

    print(
        f"{dataset_name.upper()} "
        f"RAW-ID GRAPH METADATA INSPECTION"
    )

    print(
        "=" * 78
    )

    checked = 0

    for ex in examples:

        if checked >= max_examples:
            break

        idx = int(
            ex["source_index"]
        )

        rec = dataset[idx]

        G = build_graph(
            rec["graph"]
        )

        for role in [
            "current",
            "candidate"
        ]:

            entity = ex[role]

            if not is_raw_freebase_id(
                entity
            ):
                continue

            print(
                f"\nsource_index={idx}"
            )

            print(
                f"role={role}"
            )

            print(
                f"entity={entity}"
            )

            if entity in G:

                attrs = dict(
                    G.nodes[
                        entity
                    ]
                )

                print(
                    "NetworkX node attrs:",
                    attrs
                )

            else:

                print(
                    "Entity not found "
                    "as NetworkX node."
                )

            checked += 1

            if checked >= max_examples:
                break


inspect_graph_node_metadata(
    webqsp_train,
    webqsp_raw_examples,
    "webqsp"
)

inspect_graph_node_metadata(
    cwq_train,
    cwq_raw_examples,
    "cwq"
)


# ======================================================================
# 3. Inspect raw source graph schema for a few affected questions
# ======================================================================

def inspect_source_graph_schema(
    dataset,
    examples,
    dataset_name,
    max_questions=2
):
    print(
        "\n" + "=" * 78
    )

    print(
        f"{dataset_name.upper()} "
        f"SOURCE GRAPH SCHEMA"
    )

    print(
        "=" * 78
    )

    seen = set()

    shown = 0

    for ex in examples:

        idx = int(
            ex["source_index"]
        )

        if idx in seen:
            continue

        seen.add(idx)

        rec = dataset[idx]

        raw_graph = rec[
            "graph"
        ]

        print(
            f"\nsource_index={idx}"
        )

        print(
            "graph python type:",
            type(raw_graph)
        )

        try:
            print(
                "graph sample:",
                raw_graph[:3]
            )
        except Exception:
            print(
                "graph sample:",
                str(raw_graph)[:1500]
            )

        shown += 1

        if shown >= max_questions:
            break


inspect_source_graph_schema(
    webqsp_train,
    webqsp_raw_examples,
    "webqsp"
)

inspect_source_graph_schema(
    cwq_train,
    cwq_raw_examples,
    "cwq"
)


print(
    "\n" + "=" * 78
)

print(
    "=== ENTITY SURFACE-FORM AUDIT COMPLETE ==="
)

print(
    "Do NOT build MiniLM feature cache yet."
)


WEBQSP DECISION-BRANCH ENTITY AUDIT
Decision branches:           18437
Raw candidate IDs:           10774 (58.44%)
Raw current-entity IDs:      0 (0.00%)
Any raw endpoint:            10774 (58.44%)
Both endpoints readable:     7663 (41.56%)

Examples:
{'source_index': 1, 'question_id': 'WebQTrn-1', 'current': 'Natalie Portman', 'candidate': 'm.0k3qzz'}
{'source_index': 1, 'question_id': 'WebQTrn-1', 'current': 'Natalie Portman', 'candidate': 'm.03jt5jq'}
{'source_index': 1, 'question_id': 'WebQTrn-1', 'current': 'Natalie Portman', 'candidate': 'm.0nfnhrj'}
{'source_index': 1, 'question_id': 'WebQTrn-1', 'current': 'Natalie Portman', 'candidate': 'm.040myw2'}
{'source_index': 1, 'question_id': 'WebQTrn-1', 'current': 'Natalie Portman', 'candidate': 'm.04dcjy9'}
{'source_index': 1, 'question_id': 'WebQTrn-1', 'current': 'Natalie Portman', 'candidate': 'm.0k3r0b'}
{'source_index': 1, 'question_id': 'WebQTrn-1', 'current': 'Natalie Portman', 'candidate': 'm.0cs2bt7'}
{'source_index': 1, '

In [91]:
# ======================================================================
# CHECK FOR EXISTING FREEBASE MID -> ENTITY NAME RESOLVER
# ======================================================================
#
# Goal:
#   Find whether the current RoG environment already contains a verified
#   MID-to-readable-name mapping.
#
# IMPORTANT:
#   We do NOT construct semantic features yet.
# ======================================================================

import os
import re
import json

MID_RE = re.compile(r"^(?:m|g)\.[A-Za-z0-9_\-]+$")


def is_mid(x):
    return bool(
        MID_RE.match(
            str(x).strip()
        )
    )


# ======================================================================
# 1. Dataset columns
# ======================================================================

print("=" * 80)
print("DATASET COLUMN AUDIT")
print("=" * 80)

print("\nWebQSP columns:")
print(webqsp_train.column_names)

print("\nCWQ columns:")
print(cwq_train.column_names)


# ======================================================================
# 2. Search notebook globals for possible entity-name dictionaries
# ======================================================================

keywords = (
    "name",
    "label",
    "entity",
    "mid",
    "id2",
    "2id",
    "alias",
)


candidate_globals = []

for var_name, obj in list(globals().items()):

    low = var_name.lower()

    if not any(
        k in low
        for k in keywords
    ):
        continue

    if isinstance(obj, dict):

        # Avoid dumping giant dictionaries
        size = len(obj)

        sample_items = list(
            obj.items()
        )[:5]

        candidate_globals.append(
            (
                var_name,
                size,
                sample_items,
            )
        )


print("\n" + "=" * 80)
print("POSSIBLE MAPPING DICTIONARIES IN NOTEBOOK")
print("=" * 80)

if not candidate_globals:
    print("No obvious dictionary candidates found.")

else:
    for name, size, sample in candidate_globals:

        print(
            f"\n{name} | size={size}"
        )

        for k, v in sample:
            print(
                "   ",
                repr(k),
                "->",
                repr(v)
            )


# ======================================================================
# 3. Test candidate dictionaries against known unresolved MIDs
# ======================================================================

KNOWN_MIDS = [
    "m.0k3qzz",
    "m.03jt5jq",
    "m.0nfnhrj",
    "m.0hqf007",
    "m.0hqf002",
]


print("\n" + "=" * 80)
print("KNOWN MID LOOKUP TEST")
print("=" * 80)


successful_resolvers = []


for name, size, _ in candidate_globals:

    obj = globals()[name]

    hits = {}

    for mid in KNOWN_MIDS:

        if mid in obj:

            value = obj[mid]

            # We only count readable mappings
            if (
                value is not None
                and str(value).strip()
                and not is_mid(value)
            ):
                hits[mid] = value

    if hits:

        successful_resolvers.append(
            name
        )

        print(
            f"\n{name}: "
            f"{len(hits)}/{len(KNOWN_MIDS)} readable hits"
        )

        for mid, value in hits.items():
            print(
                f"  {mid} -> {value}"
            )


if not successful_resolvers:
    print(
        "\nNo existing notebook dictionary resolved "
        "the sampled Freebase IDs."
    )


# ======================================================================
# 4. Search dataset columns for likely entity-name mappings
# ======================================================================

def inspect_mapping_like_columns(
    dataset,
    dataset_name,
    n_rows=3
):

    interesting = []

    for col in dataset.column_names:

        low = col.lower()

        if any(
            k in low
            for k in [
                "name",
                "label",
                "entity",
                "alias",
                "mid",
            ]
        ):
            interesting.append(col)

    print(
        "\n" + "=" * 80
    )

    print(
        f"{dataset_name.upper()} "
        f"MAPPING-LIKE DATASET COLUMNS"
    )

    print("=" * 80)

    print(
        "Candidate columns:",
        interesting
    )

    for col in interesting:

        print(
            f"\nCOLUMN: {col}"
        )

        for i in range(
            min(n_rows, len(dataset))
        ):

            value = dataset[i][col]

            text = repr(value)

            if len(text) > 1000:
                text = (
                    text[:1000]
                    + " ..."
                )

            print(
                f"row {i}: {text}"
            )


inspect_mapping_like_columns(
    webqsp_train,
    "WebQSP"
)

inspect_mapping_like_columns(
    cwq_train,
    "CWQ"
)


# ======================================================================
# 5. Final status
# ======================================================================

print(
    "\n" + "=" * 80
)

print(
    "=== EXISTING ENTITY RESOLVER AUDIT COMPLETE ==="
)

print("=" * 80)

if successful_resolvers:

    print(
        "Potential existing resolver(s):"
    )

    for x in successful_resolvers:
        print("  ", x)

    print(
        "\nDo NOT use automatically yet; "
        "we will verify coverage first."
    )

else:

    print(
        "No verified existing MID->name resolver found yet."
    )

    print(
        "If none exists, we will revise AFP semantic "
        "features rather than embedding raw MIDs."
    )

DATASET COLUMN AUDIT

WebQSP columns:
['id', 'question', 'answer', 'q_entity', 'a_entity', 'graph', 'choices']

CWQ columns:
['id', 'question', 'answer', 'q_entity', 'a_entity', 'graph', 'choices']

POSSIBLE MAPPING DICTIONARIES IN NOTEBOOK

webqsp_label_diag | size=10
    'dataset' -> 'webqsp'
    'feasible_groups' -> 2177
    'feasible_decision_groups' -> 1457
    'feasible_singleton_groups' -> 720
    'infeasible_groups' -> 922

cwq_label_diag | size=10
    'dataset' -> 'cwq'
    'feasible_groups' -> 33837
    'feasible_decision_groups' -> 15937
    'feasible_singleton_groups' -> 17900
    'infeasible_groups' -> 24029

KNOWN MID LOOKUP TEST

No existing notebook dictionary resolved the sampled Freebase IDs.

WEBQSP MAPPING-LIKE DATASET COLUMNS
Candidate columns: ['q_entity', 'a_entity']

COLUMN: q_entity
row 0: ['Justin Bieber']
row 1: ['Natalie Portman']
row 2: ['Grand Bahama']

COLUMN: a_entity
row 0: ['Jaxon Bieber']
row 1: ['Padmé Amidala']
row 2: ['Bahamas']

CWQ MAPPING-LIKE D

In [95]:
# ======================================================================
# CELL 6A — RECOVER / BIND EXISTING FROZEN VALIDATION PLAN ROWS
# ======================================================================
#
# Purpose:
#   Find the already-existing frozen validation relation plans from RQ1.
#
# We DO NOT regenerate any plans.
# We DO NOT call the planner.
# We only identify and verify existing objects.
# ======================================================================

import json
from collections.abc import Sequence


def looks_like_plan_row(row):
    """
    A frozen plan row should contain:
      - predicted_paths
      - id
    source_index is strongly preferred.
    """
    if not isinstance(row, dict):
        return False

    return (
        "predicted_paths" in row
        and
        "id" in row
    )


def inspect_plan_candidate(
    obj,
    expected_len,
    dataset,
    name
):
    """
    Verify whether obj behaves like the frozen validation-plan rows.
    """

    # Must support len()
    try:
        n = len(obj)
    except Exception:
        return False, None

    if n != expected_len:
        return False, None

    # Must support indexing
    try:
        first = obj[0]
        middle = obj[n // 2]
        last = obj[n - 1]
    except Exception:
        return False, None

    for row in [first, middle, last]:
        if not looks_like_plan_row(row):
            return False, None

    # Verify ID alignment against actual validation dataset
    check_indices = [
        0,
        n // 2,
        n - 1,
    ]

    try:
        for i in check_indices:

            row = obj[i]
            rec = dataset[i]

            if str(row["id"]) != str(rec["id"]):
                return False, None

            if (
                "source_index" in row
                and
                int(row["source_index"]) != i
            ):
                return False, None

    except Exception:
        return False, None

    return True, {
        "name": name,
        "length": n,
        "sample_id": str(first["id"]),
        "sample_plans": first["predicted_paths"],
    }


def find_validation_plan_object(
    dataset,
    expected_len,
    dataset_name
):
    matches = []

    # --------------------------------------------------------------
    # Search all notebook globals
    # --------------------------------------------------------------
    for name, obj in list(globals().items()):

        # Skip obvious irrelevant namespaces / modules
        if name.startswith("_"):
            continue

        ok, info = inspect_plan_candidate(
            obj,
            expected_len,
            dataset,
            name
        )

        if ok:
            matches.append(
                (name, obj, info)
            )

    print("\n" + "=" * 80)
    print(
        f"{dataset_name.upper()} "
        f"VALIDATION PLAN OBJECT SEARCH"
    )
    print("=" * 80)

    if len(matches) == 0:

        print(
            "No matching frozen validation-plan object "
            "was found in current globals."
        )

        return None, None

    for name, _, info in matches:
        print(
            f"Candidate: {name}"
        )
        print(
            f"  rows:       {info['length']}"
        )
        print(
            f"  sample id:  {info['sample_id']}"
        )
        print(
            f"  sample plan:{info['sample_plans']}"
        )

    # Prefer names containing dataset + val terminology
    def preference(item):
        name = item[0].lower()

        score = 0

        if dataset_name.lower() in name:
            score += 10

        if "val" in name:
            score += 5

        if "plan" in name:
            score += 5

        if "frozen" in name:
            score += 2

        if "row" in name:
            score += 1

        return score

    matches.sort(
        key=preference,
        reverse=True
    )

    selected_name, selected_obj, _ = (
        matches[0]
    )

    print(
        f"\nAUTO-SELECTED: {selected_name}"
    )

    return (
        selected_obj,
        selected_name
    )


# ======================================================================
# 1. Find WebQSP frozen validation plans
# ======================================================================

(
    webqsp_val_plan_rows,
    WEBQSP_VAL_PLAN_VAR
) = find_validation_plan_object(
    dataset=webqsp_val,
    expected_len=246,
    dataset_name="webqsp"
)


# ======================================================================
# 2. Find CWQ frozen validation plans
# ======================================================================

(
    cwq_val_plan_rows,
    CWQ_VAL_PLAN_VAR
) = find_validation_plan_object(
    dataset=cwq_val,
    expected_len=3519,
    dataset_name="cwq"
)


# ======================================================================
# 3. Hard gate
# ======================================================================

assert webqsp_val_plan_rows is not None, (
    "WebQSP frozen validation plans were not found "
    "in current notebook memory."
)

assert cwq_val_plan_rows is not None, (
    "CWQ frozen validation plans were not found "
    "in current notebook memory."
)


assert len(
    webqsp_val_plan_rows
) == 246

assert len(
    cwq_val_plan_rows
) == 3519


# ======================================================================
# 4. Full alignment verification
# ======================================================================

def verify_full_plan_alignment(
    dataset,
    plan_rows,
    dataset_name
):

    zero_plan_questions = 0
    total_plans = 0

    for i in range(
        len(dataset)
    ):

        rec = dataset[i]
        row = plan_rows[i]

        assert (
            str(rec["id"])
            ==
            str(row["id"])
        ), (
            f"{dataset_name}: ID mismatch "
            f"at index {i}"
        )

        if "source_index" in row:
            assert (
                int(
                    row[
                        "source_index"
                    ]
                )
                == i
            )

        plans = row[
            "predicted_paths"
        ]

        assert isinstance(
            plans,
            (list, tuple)
        )

        if len(plans) == 0:
            zero_plan_questions += 1

        total_plans += len(plans)

    print(
        f"\n[{dataset_name}] "
        "FULL VALIDATION PLAN ALIGNMENT: PASSED"
    )

    print(
        "Questions:",
        len(dataset)
    )

    print(
        "Total predicted plans:",
        total_plans
    )

    print(
        "Questions with 0 plans:",
        zero_plan_questions
    )


verify_full_plan_alignment(
    webqsp_val,
    webqsp_val_plan_rows,
    "WebQSP"
)

verify_full_plan_alignment(
    cwq_val,
    cwq_val_plan_rows,
    "CWQ"
)


print("\n" + "=" * 80)
print(
    "=== FROZEN VALIDATION PLAN OBJECTS RECOVERED ==="
)
print("=" * 80)

print(
    "WebQSP:",
    WEBQSP_VAL_PLAN_VAR
)

print(
    "CWQ:   ",
    CWQ_VAL_PLAN_VAR
)

print(
    "\nNo relation plans regenerated."
)

print(
    "You can now rerun Cell 6."
)


WEBQSP VALIDATION PLAN OBJECT SEARCH
Candidate: webqsp_val_planning
  rows:       246
  sample id:  WebQTrn-9
  sample plan:[['people.person.nationality'], ['people.person.nationality', 'people.person.nationality'], ['people.person.nationality', 'location.location.containedby']]
Candidate: webqsp_val_plan_rows
  rows:       246
  sample id:  WebQTrn-9
  sample plan:[['people.person.nationality'], ['people.person.nationality', 'people.person.nationality'], ['people.person.nationality', 'location.location.containedby']]

AUTO-SELECTED: webqsp_val_plan_rows

CWQ VALIDATION PLAN OBJECT SEARCH
Candidate: cwq_val_planning
  rows:       3519
  sample id:  WebQTrn-1430_ac053cda0a7424c48e4809c71171fbed
  sample plan:[['government.government_position_held.office_position_or_title', 'government.politician.government_positions_held'], ['location.location.containedby', 'people.person.nationality'], ['government.government_position_held.office_position_or_title', 'government.government_position_hel

## Build/cache train + validation feature datasets

In [97]:
# ======================================================================
# 3.6 BUILD + CACHE TRAIN AND VALIDATION AFP FEATURE DATASETS
# ======================================================================
#
# TRAIN:
#   Uses already-frozen Cell-3 supervision.
#
# VALIDATION:
#   Constructs suffix-DP labels using VALIDATION gold answers only.
#
# Scorer dataset:
#   feasible INTERMEDIATE DECISION groups only:
#       |C_h| > 1
#       and at least one positive candidate
#
# IMPORTANT:
#   - NO TEST data.
#   - NO test gold.
#   - Gold is NEVER an AFP feature.
#   - Gold labels NEVER alter unpruned RoG traversal.
#   - Final hop excluded.
#   - Singleton groups excluded from scorer dataset.
#   - Raw Freebase MID -> masked semantic representation.
#
# Validation-plan compatibility:
#   Older frozen RQ1 validation rows may not contain source_index.
#   Their already-verified positional index is used instead.
#
# Output:
#   X              [N_branches, 27]
#   y              [N_branches]
#   group_ptr      [N_groups + 1]
#   group metadata
# ======================================================================

import os
import json
import hashlib
from datetime import datetime, timezone

import numpy as np
from tqdm import tqdm

# ======================================================================
# 1. HARD FEATURE-SPECIFICATION GATES
# ======================================================================

EXPECTED_FEATURE_VERSION = "afp_features_v2_masked_entity_semantics"
EXPECTED_FEATURE_SHA256 = (
    "738985d1232a8ac5935c397ed95eca233"
    "77bc59b4a99494547fa7779062ade86"
)

assert AFP_FEATURE_VERSION == EXPECTED_FEATURE_VERSION
assert AFP_FEATURE_SPEC_SHA256 == EXPECTED_FEATURE_SHA256
assert AFP_FEATURE_DIM == 27

assert AFP_TRAIN_DECISION_ONLY is True
assert AFP_EXCLUDE_SINGLETON_FEASIBLE is True
assert AFP_USE_FUTURE_NEIGHBORHOOD is False
assert AFP_USE_GOLD_AS_FEATURE is False
assert AFP_USE_SUFFIX_REACHABILITY_AS_FEATURE is False
assert AFP_USE_KGE_CORE is False

required_objects = [
    "webqsp_train",
    "cwq_train",
    "webqsp_branch_manifest",
    "cwq_branch_manifest",
    "build_graph",
    "relation_valid_neighbors",
    "call_suffix_dp",
    "dp_is_reachable",
    "extract_afp_group_features",
    "FrozenMiniLMEncoder",
]

missing = [x for x in required_objects if x not in globals()]

assert not missing, (
    "Missing required previous-cell objects: "
    + ", ".join(missing)
)

print("Feature version:", AFP_FEATURE_VERSION)
print("Feature dimension:", AFP_FEATURE_DIM)
print("Feature SHA256:", AFP_FEATURE_SPEC_SHA256[:16] + "...")

# ======================================================================
# 2. RESOLVE VALIDATION DATASETS
# ======================================================================

def resolve_global(names):
    for name in names:
        if name in globals():
            return globals()[name], name

    raise RuntimeError(
        "Could not resolve any of these globals:\n"
        + "\n".join(names)
    )

webqsp_val, WEBQSP_VAL_VAR = resolve_global([
    "webqsp_val",
    "webqsp_validation",
    "webqsp_valid",
])

cwq_val, CWQ_VAL_VAR = resolve_global([
    "cwq_val",
    "cwq_validation",
    "cwq_valid",
])

assert len(webqsp_val) == 246
assert len(cwq_val) == 3519

print("WebQSP validation dataset:", WEBQSP_VAL_VAR)
print("CWQ validation dataset:   ", CWQ_VAL_VAR)

# ======================================================================
# 3. RESOLVE FROZEN VALIDATION PLAN ROWS
# ======================================================================

webqsp_val_plan_rows, WEBQSP_VAL_PLAN_VAR = resolve_global([
    "webqsp_val_plan_rows",
    "webqsp_val_planning",
    "webqsp_validation_plan_rows",
    "webqsp_valid_plan_rows",
    "webqsp_val_plans",
    "webqsp_frozen_val_plan_rows",
])

cwq_val_plan_rows, CWQ_VAL_PLAN_VAR = resolve_global([
    "cwq_val_plan_rows",
    "cwq_val_planning",
    "cwq_validation_plan_rows",
    "cwq_valid_plan_rows",
    "cwq_val_plans",
    "cwq_frozen_val_plan_rows",
])

assert len(webqsp_val_plan_rows) == len(webqsp_val)
assert len(cwq_val_plan_rows) == len(cwq_val)

print("WebQSP validation plans:", WEBQSP_VAL_PLAN_VAR)
print("CWQ validation plans:   ", CWQ_VAL_PLAN_VAR)

# ======================================================================
# 4. FULL VALIDATION PLAN ALIGNMENT GATE
# ======================================================================

def verify_validation_plan_alignment(dataset, plan_rows, dataset_name):
    total_plans = 0
    empty_plans = 0

    for i, (rec, row) in enumerate(zip(dataset, plan_rows)):
        assert "id" in row
        assert "predicted_paths" in row

        assert str(rec["id"]) == str(row["id"]), (
            f"{dataset_name}: ID mismatch at index {i}\n"
            f"dataset={rec['id']}\n"
            f"plan_row={row['id']}"
        )

        # Older RQ1 validation artifacts may not contain this field.
        if "source_index" in row:
            assert int(row["source_index"]) == i, (
                f"{dataset_name}: source_index mismatch at {i}"
            )

        plans = row["predicted_paths"]
        assert isinstance(plans, (list, tuple))

        total_plans += len(plans)
        empty_plans += sum(len(p) == 0 for p in plans)

    print(f"\n[{dataset_name}] validation alignment: PASSED")
    print("Questions:          ", len(dataset))
    print("Predicted plans:    ", total_plans)
    print("Empty plans:        ", empty_plans)

verify_validation_plan_alignment(
    webqsp_val,
    webqsp_val_plan_rows,
    "WebQSP"
)

verify_validation_plan_alignment(
    cwq_val,
    cwq_val_plan_rows,
    "CWQ"
)

# ======================================================================
# 5. OUTPUT DIRECTORY
# ======================================================================

if "RQ2_FEATURE_DIR" not in globals():
    RQ2_FEATURE_DIR = os.path.join(
        os.path.dirname(os.path.abspath(RQ2_LABEL_DIR)),
        "03_features"
    )

os.makedirs(RQ2_FEATURE_DIR, exist_ok=True)

print("\nFeature directory:", RQ2_FEATURE_DIR)

# ======================================================================
# 6. HASH / ATOMIC-WRITE HELPERS
# ======================================================================

def sha256_file_local(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


def atomic_json_local(obj, path):
    tmp = path + ".tmp"

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)


def stable_plan_rows_sha256(plan_rows):
    """
    Deterministic hash for frozen validation planner outputs.

    Older RQ1 validation rows may not contain source_index.
    Their already-verified positional index is used instead.
    """
    h = hashlib.sha256()

    for i, row in enumerate(plan_rows):
        source_index = int(row.get("source_index", i))

        assert source_index == i, (
            f"Validation plan ordering mismatch: "
            f"row={i}, source_index={source_index}"
        )

        payload = {
            "source_index": source_index,
            "id": str(row["id"]),
            "predicted_paths": row["predicted_paths"],
        }

        line = json.dumps(
            payload,
            ensure_ascii=False,
            sort_keys=True
        )

        h.update(line.encode("utf-8"))
        h.update(b"\n")

    return h.hexdigest()


WEBQSP_VAL_PLAN_SHA256 = stable_plan_rows_sha256(
    webqsp_val_plan_rows
)

CWQ_VAL_PLAN_SHA256 = stable_plan_rows_sha256(
    cwq_val_plan_rows
)

print(
    "\nWebQSP VAL-plan SHA256:",
    WEBQSP_VAL_PLAN_SHA256[:16] + "..."
)

print(
    "CWQ VAL-plan SHA256:   ",
    CWQ_VAL_PLAN_SHA256[:16] + "..."
)

# ======================================================================
# 7. VALIDATION SUPERVISION DIRECTORY
# ======================================================================

VAL_LABEL_DIR = os.path.join(
    RQ2_FEATURE_DIR,
    "validation_supervision"
)

os.makedirs(VAL_LABEL_DIR, exist_ok=True)

# ======================================================================
# 8. CONSTRUCT VALIDATION DECISION SUPERVISION
# ======================================================================
#
# Mirrors TRAIN branch-label construction, but:
#   - validation gold only
#   - feasible decision groups written
#   - final hop excluded
#   - singleton groups excluded
#   - traversal always remains unpruned
# ======================================================================

def generate_validation_decision_rows(
    dataset,
    plan_rows,
    dataset_name,
    plan_sha256
):
    out_path = os.path.join(
        VAL_LABEL_DIR,
        f"{dataset_name}_validation_decision_labels.jsonl"
    )

    manifest_path = os.path.join(
        VAL_LABEL_DIR,
        f"{dataset_name}_validation_decision_labels_manifest.json"
    )

    # Reuse already-completed artifact if valid.
    if os.path.exists(out_path) and os.path.exists(manifest_path):
        with open(manifest_path, "r", encoding="utf-8") as f:
            manifest = json.load(f)

        valid = (
            manifest.get("feature_spec_sha256") == AFP_FEATURE_SPEC_SHA256
            and manifest.get("val_plan_sha256") == plan_sha256
            and sha256_file_local(out_path) == manifest.get("labels_sha256")
        )

        if valid:
            print(
                f"[{dataset_name}] validation supervision: "
                "verified frozen"
            )
            return out_path, manifest

    tmp_path = out_path + ".tmp"

    stats = {
        "questions": len(dataset),
        "empty_plans": 0,
        "intermediate_groups": 0,
        "feasible_groups": 0,
        "feasible_decision_groups": 0,
        "feasible_singletons": 0,
        "infeasible_groups": 0,
        "final_hop_groups": 0,
        "empty_terminated": 0,
        "decision_branches": 0,
        "positive_branches": 0,
        "negative_branches": 0,
    }

    with open(tmp_path, "w", encoding="utf-8") as out:
        for source_index in tqdm(
            range(len(dataset)),
            desc=f"{dataset_name} validation labels"
        ):
            rec = dataset[source_index]
            plan_row = plan_rows[source_index]

            assert str(rec["id"]) == str(plan_row["id"])

            # Positional source index is canonical for old RQ1 artifacts.
            if "source_index" in plan_row:
                assert int(plan_row["source_index"]) == source_index

            G = build_graph(rec["graph"])
            gold_answers = set(rec["a_entity"])
            topic_entities = list(rec["q_entity"])

            plans = [
                list(p)
                for p in plan_row["predicted_paths"]
            ]

            for plan_idx, plan in enumerate(plans):
                L = len(plan)

                if L == 0:
                    stats["empty_plans"] += 1
                    continue

                reachable_dp = call_suffix_dp(
                    G,
                    plan,
                    gold_answers
                )

                for topic_idx, topic_entity in enumerate(topic_entities):
                    active_prefixes = [(topic_entity,)]

                    for h in range(L):
                        required_relation = plan[h]
                        candidates = []

                        # Exact relation-valid RoG expansion
                        for parent_idx, prefix in enumerate(active_prefixes):
                            current_entity = prefix[-1]

                            neighbors = relation_valid_neighbors(
                                G,
                                current_entity,
                                required_relation
                            )

                            for nbr in neighbors:
                                candidates.append({
                                    "parent_prefix_index": parent_idx,
                                    "prefix_entities": list(prefix),
                                    "candidate_entity": nbr,
                                    "branch_entities": list(prefix + (nbr,)),
                                })

                        # No valid continuation
                        if not candidates:
                            stats["empty_terminated"] += 1
                            break

                        # Final-hop protection
                        if h == L - 1:
                            stats["final_hop_groups"] += 1
                            break

                        stats["intermediate_groups"] += 1

                        labels = [
                            int(
                                dp_is_reachable(
                                    reachable_dp,
                                    h + 1,
                                    c["candidate_entity"]
                                )
                            )
                            for c in candidates
                        ]

                        n_pos = int(sum(labels))
                        n_neg = len(labels) - n_pos

                        feasible = n_pos > 0
                        decision = len(candidates) > 1

                        if feasible:
                            stats["feasible_groups"] += 1

                            if decision:
                                stats["feasible_decision_groups"] += 1

                                group_id = (
                                    f"{dataset_name}|validation|"
                                    f"{source_index}|"
                                    f"p{plan_idx}|"
                                    f"t{topic_idx}|"
                                    f"h{h}"
                                )

                                for candidate_idx, (cand, y) in enumerate(
                                    zip(candidates, labels)
                                ):
                                    row = {
                                        "dataset": dataset_name,
                                        "split": "validation",
                                        "source_index": source_index,
                                        "question_id": rec["id"],
                                        "group_id": group_id,
                                        "plan_index": plan_idx,
                                        "plan": plan,
                                        "plan_length": L,
                                        "topic_index": topic_idx,
                                        "topic_entity": topic_entity,
                                        "hop": h,
                                        "required_relation": required_relation,
                                        "candidate_index": candidate_idx,
                                        "candidate_count": len(candidates),
                                        "parent_prefix_index":
                                            cand["parent_prefix_index"],
                                        "prefix_entities":
                                            cand["prefix_entities"],
                                        "candidate_entity":
                                            cand["candidate_entity"],
                                        "label": int(y),
                                    }

                                    out.write(
                                        json.dumps(
                                            row,
                                            ensure_ascii=False
                                        ) + "\n"
                                    )

                                stats["decision_branches"] += len(candidates)
                                stats["positive_branches"] += n_pos
                                stats["negative_branches"] += n_neg

                            else:
                                stats["feasible_singletons"] += 1

                        else:
                            stats["infeasible_groups"] += 1

                        # CRITICAL:
                        # labels do NOT modify traversal.
                        active_prefixes = [
                            tuple(c["branch_entities"])
                            for c in candidates
                        ]

        out.flush()
        os.fsync(out.fileno())

    os.replace(tmp_path, out_path)

    manifest = {
        "dataset": dataset_name,
        "split": "validation",
        "val_plan_sha256": plan_sha256,
        "feature_spec_sha256": AFP_FEATURE_SPEC_SHA256,
        **stats,
        "labels_sha256": sha256_file_local(out_path),
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }

    atomic_json_local(
        manifest,
        manifest_path
    )

    print(
        f"\n[{dataset_name}] VALIDATION SUPERVISION FROZEN"
    )
    print(
        "Intermediate groups:      ",
        stats["intermediate_groups"]
    )
    print(
        "Feasible decision groups: ",
        stats["feasible_decision_groups"]
    )
    print(
        "Decision branches:        ",
        stats["decision_branches"]
    )
    print(
        "Positive / Negative:      ",
        stats["positive_branches"],
        "/",
        stats["negative_branches"]
    )
    print(
        "Empty plans skipped:      ",
        stats["empty_plans"]
    )

    return out_path, manifest

# ======================================================================
# 9. BUILD/FREEZE VALIDATION SUPERVISION
# ======================================================================

(
    webqsp_val_labels_file,
    webqsp_val_label_manifest
) = generate_validation_decision_rows(
    webqsp_val,
    webqsp_val_plan_rows,
    "webqsp",
    WEBQSP_VAL_PLAN_SHA256
)

(
    cwq_val_labels_file,
    cwq_val_label_manifest
) = generate_validation_decision_rows(
    cwq_val,
    cwq_val_plan_rows,
    "cwq",
    CWQ_VAL_PLAN_SHA256
)

# ======================================================================
# 10. TRAIN LABEL FILES FROM CELL 3
# ======================================================================

webqsp_train_labels_file = webqsp_branch_manifest["labels_file"]
cwq_train_labels_file = cwq_branch_manifest["labels_file"]

# ======================================================================
# 11. ITERATE FEASIBLE DECISION GROUPS
# ======================================================================

def iter_decision_groups(labels_file):
    """
    Rows are expected to be group-contiguous.

    TRAIN file:
        singleton feasible groups exist but are skipped.

    VALIDATION file:
        only feasible decision groups were written.
    """
    current_id = None
    current_rows = []

    with open(labels_file, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            row = json.loads(line)

            # TRAIN labels contain singleton feasible rows.
            if row.get("decision_opportunity", True) is False:
                continue

            group_id = row["group_id"]

            if current_id is not None and group_id != current_id:
                assert len(current_rows) > 1
                yield current_rows
                current_rows = []

            current_id = group_id
            current_rows.append(row)

    if current_rows:
        assert len(current_rows) > 1
        yield current_rows

# ======================================================================
# 12. COLLECT REUSABLE SEMANTIC TEXTS
# ======================================================================

def collect_semantic_texts(labels_file, dataset):
    questions = {}
    entities = {}
    relations = {}
    plans = {}
    suffixes = {}

    n_groups = 0
    n_branches = 0

    for rows in iter_decision_groups(labels_file):
        n_groups += 1
        n_branches += len(rows)

        first = rows[0]
        source_index = int(first["source_index"])
        rec = dataset[source_index]

        question_id = str(first["question_id"])
        questions[question_id] = rec["question"]

        plan = list(first["plan"])
        hop = int(first["hop"])

        for relation in plan:
            relations[str(relation)] = relation_surface_text(relation)

        plan_key = "||".join(str(x) for x in plan)
        plans[plan_key] = relation_sequence_text(plan)

        suffix = plan[hop + 1:]
        suffix_key = "||".join(str(x) for x in suffix)
        suffixes[suffix_key] = relation_sequence_text(suffix)

        # Readable entity names only.
        # Raw Freebase MIDs are excluded from MiniLM inventory.
        for row in rows:
            candidate = row["candidate_entity"]

            if has_readable_entity_surface(candidate):
                entities[str(candidate)] = entity_surface_text(candidate)

            for entity in row["prefix_entities"]:
                if has_readable_entity_surface(entity):
                    entities[str(entity)] = entity_surface_text(entity)

    return {
        "questions": questions,
        "entities": entities,
        "relations": relations,
        "plans": plans,
        "suffixes": suffixes,
        "n_groups": n_groups,
        "n_branches": n_branches,
    }


def merge_mapping_dicts(*dicts):
    result = {}

    for d in dicts:
        result.update(d)

    return result

print("\nCollecting semantic-text inventory...")

wq_train_text = collect_semantic_texts(
    webqsp_train_labels_file,
    webqsp_train
)

wq_val_text = collect_semantic_texts(
    webqsp_val_labels_file,
    webqsp_val
)

cwq_train_text = collect_semantic_texts(
    cwq_train_labels_file,
    cwq_train
)

cwq_val_text = collect_semantic_texts(
    cwq_val_labels_file,
    cwq_val
)

ALL_QUESTIONS = merge_mapping_dicts(
    wq_train_text["questions"],
    wq_val_text["questions"],
    cwq_train_text["questions"],
    cwq_val_text["questions"],
)

ALL_ENTITIES = merge_mapping_dicts(
    wq_train_text["entities"],
    wq_val_text["entities"],
    cwq_train_text["entities"],
    cwq_val_text["entities"],
)

ALL_RELATIONS = merge_mapping_dicts(
    wq_train_text["relations"],
    wq_val_text["relations"],
    cwq_train_text["relations"],
    cwq_val_text["relations"],
)

ALL_PLANS = merge_mapping_dicts(
    wq_train_text["plans"],
    wq_val_text["plans"],
    cwq_train_text["plans"],
    cwq_val_text["plans"],
)

ALL_SUFFIXES = merge_mapping_dicts(
    wq_train_text["suffixes"],
    wq_val_text["suffixes"],
    cwq_train_text["suffixes"],
    cwq_val_text["suffixes"],
)

print("Unique questions:  ", len(ALL_QUESTIONS))
print("Readable entities: ", len(ALL_ENTITIES))
print("Relations:         ", len(ALL_RELATIONS))
print("Full plans:        ", len(ALL_PLANS))
print("Suffixes:          ", len(ALL_SUFFIXES))

# ======================================================================
# 13. LOAD FROZEN MINILM
# ======================================================================

try:
    import torch
    FEATURE_DEVICE = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )
except Exception:
    FEATURE_DEVICE = "cpu"

print("\nMiniLM device:", FEATURE_DEVICE)

afp_semantic_encoder = FrozenMiniLMEncoder(
    model_name=AFP_SEMANTIC_ENCODER_NAME,
    device=FEATURE_DEVICE,
    batch_size=256
)

# ======================================================================
# 14. BATCH CACHE SEMANTIC EMBEDDINGS
# ======================================================================

print("\nCaching question embeddings...")
afp_semantic_encoder.prefill(
    "question",
    ALL_QUESTIONS
)

print("Caching readable entity embeddings...")
afp_semantic_encoder.prefill(
    "entity",
    ALL_ENTITIES
)

print("Caching relation embeddings...")
afp_semantic_encoder.prefill(
    "relation",
    ALL_RELATIONS
)

print("Caching full-plan embeddings...")
afp_semantic_encoder.prefill(
    "plan",
    ALL_PLANS
)

print("Caching suffix embeddings...")
afp_semantic_encoder.prefill(
    "suffix",
    ALL_SUFFIXES
)

print(
    "Semantic cache entries:",
    len(afp_semantic_encoder.cache)
)

# ======================================================================
# 15. BUILD ONE FEATURE DATASET
# ======================================================================

def build_feature_dataset(
    dataset_name,
    split,
    dataset,
    labels_file
):
    X_blocks = []
    y_blocks = []

    group_ptr = [0]
    group_source_index = []
    group_hop = []
    group_plan_length = []
    group_candidate_count = []

    n_groups = 0

    for rows in tqdm(
        iter_decision_groups(labels_file),
        desc=f"{dataset_name} {split} features"
    ):
        first = rows[0]

        source_index = int(first["source_index"])
        rec = dataset[source_index]

        assert str(rec["id"]) == str(first["question_id"])

        plan = list(first["plan"])
        hop = int(first["hop"])

        X_group = extract_afp_group_features(
            question_id=first["question_id"],
            question=rec["question"],
            plan=plan,
            hop=hop,
            candidate_rows=rows,
            semantic_encoder=afp_semantic_encoder,
            entity_name_map=None
        )

        y_group = np.asarray(
            [int(r["label"]) for r in rows],
            dtype=np.uint8
        )

        assert X_group.shape == (
            len(rows),
            AFP_FEATURE_DIM
        )

        assert len(y_group) == len(rows)
        assert int(y_group.sum()) >= 1
        assert len(rows) > 1
        assert np.all(np.isfinite(X_group))

        X_blocks.append(X_group)
        y_blocks.append(y_group)

        group_ptr.append(
            group_ptr[-1] + len(rows)
        )

        group_source_index.append(source_index)
        group_hop.append(hop)
        group_plan_length.append(len(plan))
        group_candidate_count.append(len(rows))

        n_groups += 1

    assert n_groups > 0

    X = np.concatenate(
        X_blocks,
        axis=0
    ).astype(np.float32)

    y = np.concatenate(
        y_blocks,
        axis=0
    ).astype(np.uint8)

    group_ptr = np.asarray(
        group_ptr,
        dtype=np.int64
    )

    group_source_index = np.asarray(
        group_source_index,
        dtype=np.int32
    )

    group_hop = np.asarray(
        group_hop,
        dtype=np.int16
    )

    group_plan_length = np.asarray(
        group_plan_length,
        dtype=np.int16
    )

    group_candidate_count = np.asarray(
        group_candidate_count,
        dtype=np.int32
    )

    # Final invariants
    assert X.shape[0] == len(y)
    assert X.shape[1] == AFP_FEATURE_DIM
    assert group_ptr[0] == 0
    assert group_ptr[-1] == len(y)
    assert len(group_ptr) == n_groups + 1
    assert np.all(np.diff(group_ptr) > 1)
    assert np.all(np.isfinite(X))
    assert set(np.unique(y)).issubset({0, 1})

    return {
        "X": X,
        "y": y,
        "group_ptr": group_ptr,
        "group_source_index": group_source_index,
        "group_hop": group_hop,
        "group_plan_length": group_plan_length,
        "group_candidate_count": group_candidate_count,
    }

# ======================================================================
# 16. FREEZE ONE FEATURE DATASET
# ======================================================================

def freeze_feature_dataset(
    dataset_name,
    split,
    data
):
    out_dir = os.path.join(
        RQ2_FEATURE_DIR,
        dataset_name
    )

    os.makedirs(
        out_dir,
        exist_ok=True
    )

    npz_path = os.path.join(
        out_dir,
        f"{dataset_name}_{split}_afp_features_v2.npz"
    )

    manifest_path = os.path.join(
        out_dir,
        f"{dataset_name}_{split}_afp_features_v2_manifest.json"
    )

    np.savez(
        npz_path,
        X=data["X"],
        y=data["y"],
        group_ptr=data["group_ptr"],
        group_source_index=data["group_source_index"],
        group_hop=data["group_hop"],
        group_plan_length=data["group_plan_length"],
        group_candidate_count=data["group_candidate_count"],
    )

    y = data["y"]
    group_ptr = data["group_ptr"]

    n_groups = len(group_ptr) - 1
    n_branches = len(y)

    n_pos = int(y.sum())
    n_neg = n_branches - n_pos

    manifest = {
        "dataset": dataset_name,
        "split": split,
        "feature_version": AFP_FEATURE_VERSION,
        "feature_spec_sha256": AFP_FEATURE_SPEC_SHA256,
        "feature_dim": AFP_FEATURE_DIM,
        "semantic_encoder": AFP_SEMANTIC_ENCODER_NAME,
        "raw_mid_policy":
            "zero_embedding_plus_availability",
        "decision_only": True,
        "n_groups": int(n_groups),
        "n_branches": int(n_branches),
        "positive_branches": int(n_pos),
        "negative_branches": int(n_neg),
        "positive_rate": float(
            n_pos / n_branches
        ),
        "npz_sha256":
            sha256_file_local(npz_path),
        "created_utc":
            datetime.now(timezone.utc).isoformat(),
    }

    atomic_json_local(
        manifest,
        manifest_path
    )

    return npz_path, manifest

# ======================================================================
# 17. BUILD WEBQSP TRAIN
# ======================================================================

print("\n" + "=" * 80)
print("BUILDING WEBQSP TRAIN FEATURES")
print("=" * 80)

webqsp_train_features = build_feature_dataset(
    "webqsp",
    "train",
    webqsp_train,
    webqsp_train_labels_file
)

(
    webqsp_train_feature_file,
    webqsp_train_feature_manifest
) = freeze_feature_dataset(
    "webqsp",
    "train",
    webqsp_train_features
)

# ======================================================================
# 18. BUILD WEBQSP VALIDATION
# ======================================================================

print("\n" + "=" * 80)
print("BUILDING WEBQSP VALIDATION FEATURES")
print("=" * 80)

webqsp_val_features = build_feature_dataset(
    "webqsp",
    "validation",
    webqsp_val,
    webqsp_val_labels_file
)

(
    webqsp_val_feature_file,
    webqsp_val_feature_manifest
) = freeze_feature_dataset(
    "webqsp",
    "validation",
    webqsp_val_features
)

# ======================================================================
# 19. BUILD CWQ TRAIN
# ======================================================================

print("\n" + "=" * 80)
print("BUILDING CWQ TRAIN FEATURES")
print("=" * 80)

cwq_train_features = build_feature_dataset(
    "cwq",
    "train",
    cwq_train,
    cwq_train_labels_file
)

(
    cwq_train_feature_file,
    cwq_train_feature_manifest
) = freeze_feature_dataset(
    "cwq",
    "train",
    cwq_train_features
)

# ======================================================================
# 20. BUILD CWQ VALIDATION
# ======================================================================

print("\n" + "=" * 80)
print("BUILDING CWQ VALIDATION FEATURES")
print("=" * 80)

cwq_val_features = build_feature_dataset(
    "cwq",
    "validation",
    cwq_val,
    cwq_val_labels_file
)

(
    cwq_val_feature_file,
    cwq_val_feature_manifest
) = freeze_feature_dataset(
    "cwq",
    "validation",
    cwq_val_features
)

# ======================================================================
# 21. FINAL FEATURE REPORT
# ======================================================================

def report_feature_manifest(m):
    print("\n" + "-" * 72)
    print(
        f"{m['dataset'].upper()} "
        f"{m['split'].upper()}"
    )
    print("-" * 72)

    print("Decision groups:  ", m["n_groups"])
    print("Branches:         ", m["n_branches"])
    print("Positive:         ", m["positive_branches"])
    print("Negative:         ", m["negative_branches"])
    print(
        "Positive rate:    ",
        f"{100*m['positive_rate']:.2f}%"
    )
    print("Feature dimension:", m["feature_dim"])
    print(
        "NPZ SHA256:       ",
        m["npz_sha256"][:16] + "..."
    )

report_feature_manifest(
    webqsp_train_feature_manifest
)

report_feature_manifest(
    webqsp_val_feature_manifest
)

report_feature_manifest(
    cwq_train_feature_manifest
)

report_feature_manifest(
    cwq_val_feature_manifest
)

# ======================================================================
# 22. CROSS-CHECK TRAIN COUNTS AGAINST CELL 4
# ======================================================================

assert (
    webqsp_train_feature_manifest["n_groups"]
    ==
    webqsp_label_diag["feasible_decision_groups"]
)

assert (
    cwq_train_feature_manifest["n_groups"]
    ==
    cwq_label_diag["feasible_decision_groups"]
)

assert (
    webqsp_train_feature_manifest["n_branches"]
    ==
    webqsp_label_diag["decision_stats"]["n"]
)

assert (
    cwq_train_feature_manifest["n_branches"]
    ==
    cwq_label_diag["decision_stats"]["n"]
)

assert (
    webqsp_train_feature_manifest["positive_branches"]
    ==
    webqsp_label_diag["decision_stats"]["pos"]
)

assert (
    cwq_train_feature_manifest["positive_branches"]
    ==
    cwq_label_diag["decision_stats"]["pos"]
)

# ======================================================================
# 23. FINAL FREEZE GATE
# ======================================================================

print("\n" + "=" * 84)
print(
    "=== RQ2 CELL 6: "
    "TRAIN + VALIDATION FEATURE DATASETS FROZEN ==="
)
print("=" * 84)

print("Feature spec:", AFP_FEATURE_SPEC_SHA256)
print("Feature dimension:", AFP_FEATURE_DIM)
print("Raw Freebase MIDs were never passed to MiniLM.")
print("TRAIN supervision: training gold only.")
print("VALIDATION supervision: validation gold only.")
print("TEST data/gold: NOT USED.")
print("All scorer datasets contain feasible decision groups only.")
print("Group boundaries preserved for group-aware loss/ranking.")
print("Frozen validation plans reused; none regenerated.")
print("\nNext: define AFP scorer and validation metrics.")

Feature version: afp_features_v2_masked_entity_semantics
Feature dimension: 27
Feature SHA256: 738985d1232a8ac5...
WebQSP validation dataset: webqsp_val
CWQ validation dataset:    cwq_val
WebQSP validation plans: webqsp_val_plan_rows
CWQ validation plans:    cwq_val_plan_rows

[WebQSP] validation alignment: PASSED
Questions:           246
Predicted plans:     721
Empty plans:         0

[CWQ] validation alignment: PASSED
Questions:           3519
Predicted plans:     10536
Empty plans:         7

Feature directory: /kaggle/working/step3_rq2_dev_v1/03_features

WebQSP VAL-plan SHA256: df44197f9cf9e244...
CWQ VAL-plan SHA256:    1a0590c1cd183860...


webqsp validation labels: 100%|██████████| 246/246 [00:09<00:00, 26.62it/s]



[webqsp] VALIDATION SUPERVISION FROZEN
Intermediate groups:       253
Feasible decision groups:  87
Decision branches:         966
Positive / Negative:       319 / 647
Empty plans skipped:       0


cwq validation labels: 100%|██████████| 3519/3519 [02:40<00:00, 21.96it/s]



[cwq] VALIDATION SUPERVISION FROZEN
Intermediate groups:       6462
Feasible decision groups:  1352
Decision branches:         18688
Positive / Negative:       5604 / 13084
Empty plans skipped:       7

Unique questions:   10133
Readable entities:  21218
Relations:          795
Full plans:         1846
Suffixes:           801

MiniLM device: cuda

Caching question embeddings...


...


BUILDING CWQ TRAIN FEATURES


cwq train features: 15937it [07:08, 37.21it/s]



BUILDING CWQ VALIDATION FEATURES


cwq validation features: 1352it [00:35, 37.68it/s]



------------------------------------------------------------------------
WEBQSP TRAIN
------------------------------------------------------------------------
Decision groups:   1457
Branches:          18437
Positive:          7301
Negative:          11136
Positive rate:     39.60%
Feature dimension: 27
NPZ SHA256:        55db1698520e6c6b...

------------------------------------------------------------------------
WEBQSP VALIDATION
------------------------------------------------------------------------
Decision groups:   87
Branches:          966
Positive:          319
Negative:          647
Positive rate:     33.02%
Feature dimension: 27
NPZ SHA256:        de6b1ed170f93181...

------------------------------------------------------------------------
CWQ TRAIN
------------------------------------------------------------------------
Decision groups:   15937
Branches:          218544
Positive:          52746
Negative:          165798
Positive rate:     24.14%
Feature dimension: 27
NPZ S

## Defining lightweight AFP scorer and group aware validation metrics


In [98]:
# ======================================================================
# DEFINE LIGHTWEIGHT AFP SCORER + GROUP-AWARE VALIDATION METRICS
# ======================================================================
#
# Scorer:
#   h_i = ReLU(W1 x_i + b1)
#   l_i = W2 h_i + b2
#   s_i = sigmoid(l_i)
#
# Input:
#   27-d frozen AFP feature vector.
#
# This cell DEFINES:
#   - train-only feature standardization
#   - lightweight AFP MLP
#   - branch BCE
#   - group-aware ranking metrics
#   - evaluation utilities
#
# This cell DOES NOT train the scorer.
# ======================================================================

import math
import copy
import json
import hashlib
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ======================================================================
# 1. Frozen scorer search space
# ======================================================================

AFP_SCORER_VERSION = "afp_mlp_v1"

AFP_HIDDEN_CANDIDATES = [32, 64, 128]
AFP_DROPOUT = 0.0

AFP_INPUT_DIM = AFP_FEATURE_DIM
assert AFP_INPUT_DIM == 27

# We keep architecture intentionally lightweight:
# 27 -> H -> 1
AFP_SCORER_SPEC = {
    "version": AFP_SCORER_VERSION,
    "input_dim": AFP_INPUT_DIM,
    "hidden_candidates": AFP_HIDDEN_CANDIDATES,
    "activation": "ReLU",
    "output": "single_logit",
    "dropout": AFP_DROPOUT,
    "feature_spec_sha256": AFP_FEATURE_SPEC_SHA256,
}

AFP_SCORER_SPEC_SHA256 = hashlib.sha256(
    json.dumps(AFP_SCORER_SPEC, sort_keys=True).encode("utf-8")
).hexdigest()

print("AFP scorer version:", AFP_SCORER_VERSION)
print("Input dimension:   ", AFP_INPUT_DIM)
print("Hidden candidates: ", AFP_HIDDEN_CANDIDATES)
print("Scorer SHA256:     ", AFP_SCORER_SPEC_SHA256[:16] + "...")

# ======================================================================
# 2. TRAIN-ONLY feature standardizer
# ======================================================================
#
# IMPORTANT:
# Mean/std must be fitted on TRAIN only.
# The same fitted statistics are then applied to validation/test.
#
# Constant features are protected with std = 1.
# ======================================================================

class AFPFeatureStandardizer:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None
        self.fitted = False

    def fit(self, X):
        X = np.asarray(X, dtype=np.float32)

        assert X.ndim == 2
        assert X.shape[1] == AFP_INPUT_DIM
        assert np.all(np.isfinite(X))

        self.mean_ = X.mean(axis=0).astype(np.float32)
        self.std_ = X.std(axis=0).astype(np.float32)

        self.std_[self.std_ < self.eps] = 1.0
        self.fitted = True
        return self

    def transform(self, X):
        assert self.fitted

        X = np.asarray(X, dtype=np.float32)
        Z = (X - self.mean_) / self.std_

        assert np.all(np.isfinite(Z))
        return Z.astype(np.float32)

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def state_dict(self):
        assert self.fitted
        return {
            "mean": self.mean_.tolist(),
            "std": self.std_.tolist(),
            "eps": float(self.eps),
        }

    def load_state_dict(self, state):
        self.mean_ = np.asarray(state["mean"], dtype=np.float32)
        self.std_ = np.asarray(state["std"], dtype=np.float32)
        self.eps = float(state["eps"])
        self.fitted = True
        return self

# ======================================================================
# 3. Lightweight AFP scorer
# ======================================================================

class AFPScorer(nn.Module):
    def __init__(self, input_dim=AFP_INPUT_DIM, hidden_dim=64, dropout=0.0):
        super().__init__()

        assert hidden_dim in AFP_HIDDEN_CANDIDATES

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.dropout_p = dropout

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)

        self.dropout = (
            nn.Dropout(dropout)
            if dropout > 0
            else nn.Identity()
        )

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)

        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        """
        Returns raw logits.

        x:
            [N, 27]

        output:
            [N]
        """
        h = F.relu(self.fc1(x))
        h = self.dropout(h)
        logits = self.fc2(h).squeeze(-1)

        return logits

    @torch.no_grad()
    def predict_proba(self, x):
        return torch.sigmoid(self.forward(x))

# ======================================================================
# 4. Branch-level BCE
# ======================================================================

def branch_bce_from_logits(logits, targets):
    """
    Ordinary branch-averaged BCE.

    This is an evaluation metric and can also be used as a training loss.
    """
    logits = logits.float()
    targets = targets.float()

    assert logits.shape == targets.shape

    return F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="mean"
    )

# ======================================================================
# 5. Group-balanced BCE
# ======================================================================
#
# Important because candidate-frontier sizes are heavy-tailed.
#
# Each decision GROUP gets equal weight:
#
# L = mean_g [ mean_{i in g} BCE_i ]
#
# This prevents a 600-candidate frontier from contributing 300x
# more loss than a 2-candidate frontier merely because of size.
# ======================================================================

def group_balanced_bce_from_logits(logits, targets, group_ptr):
    logits = logits.float()
    targets = targets.float()

    assert logits.shape == targets.shape

    group_ptr = np.asarray(group_ptr, dtype=np.int64)

    losses = []

    for g in range(len(group_ptr) - 1):
        start = int(group_ptr[g])
        end = int(group_ptr[g + 1])

        assert end > start

        group_loss = F.binary_cross_entropy_with_logits(
            logits[start:end],
            targets[start:end],
            reduction="mean"
        )

        losses.append(group_loss)

    assert losses

    return torch.stack(losses).mean()

# ======================================================================
# 6. Ranking helpers
# ======================================================================

def _group_average_precision(labels_sorted):
    """
    Average Precision inside one decision group.

    labels_sorted:
        binary labels ordered from highest score to lowest score.
    """
    labels_sorted = np.asarray(labels_sorted, dtype=np.int32)

    total_pos = int(labels_sorted.sum())

    if total_pos == 0:
        return np.nan

    hit_count = 0
    precision_sum = 0.0

    for rank, y in enumerate(labels_sorted, start=1):
        if y == 1:
            hit_count += 1
            precision_sum += hit_count / rank

    return precision_sum / total_pos

# ======================================================================
# 7. Group-aware ranking metrics
# ======================================================================

def compute_group_ranking_metrics(scores, labels, group_ptr, ks=(1, 2, 4)):
    """
    scores:
        higher = better candidate

    labels:
        1 iff answer-supporting under exact suffix supervision

    group_ptr:
        boundaries of feasible decision groups.

    Returns:
        top1_positive_hit
        mrr_first_positive
        group_average_precision
        hit@k
        positive_recall@k
    """
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int32)
    group_ptr = np.asarray(group_ptr, dtype=np.int64)

    assert len(scores) == len(labels)
    assert group_ptr[0] == 0
    assert group_ptr[-1] == len(labels)

    top1_hits = []
    reciprocal_ranks = []
    average_precisions = []

    hit_at_k = {k: [] for k in ks}
    recall_at_k = {k: [] for k in ks}

    group_sizes = []
    positive_counts = []

    for g in range(len(group_ptr) - 1):
        start = int(group_ptr[g])
        end = int(group_ptr[g + 1])

        g_scores = scores[start:end]
        g_labels = labels[start:end]

        assert len(g_labels) > 1
        assert g_labels.sum() >= 1

        # Stable descending sort.
        order = np.argsort(
            -g_scores,
            kind="mergesort"
        )

        ranked_labels = g_labels[order]

        group_sizes.append(len(g_labels))
        positive_counts.append(int(g_labels.sum()))

        # ----------------------------------------------------------
        # Top-1 positive hit
        # ----------------------------------------------------------
        top1_hits.append(
            float(ranked_labels[0] == 1)
        )

        # ----------------------------------------------------------
        # Reciprocal rank of FIRST positive
        # ----------------------------------------------------------
        first_positive_rank = (
            np.flatnonzero(ranked_labels == 1)[0] + 1
        )

        reciprocal_ranks.append(
            1.0 / first_positive_rank
        )

        # ----------------------------------------------------------
        # Group Average Precision
        # ----------------------------------------------------------
        average_precisions.append(
            _group_average_precision(ranked_labels)
        )

        # ----------------------------------------------------------
        # Hit@k and Positive Recall@k
        # ----------------------------------------------------------
        total_positive = int(ranked_labels.sum())

        for k in ks:
            effective_k = min(k, len(ranked_labels))
            top_k = ranked_labels[:effective_k]

            positive_in_top_k = int(top_k.sum())

            hit_at_k[k].append(
                float(positive_in_top_k > 0)
            )

            recall_at_k[k].append(
                positive_in_top_k / total_positive
            )

    metrics = {
        "n_groups": int(len(group_ptr) - 1),
        "mean_group_size": float(np.mean(group_sizes)),
        "median_group_size": float(np.median(group_sizes)),
        "mean_positive_count": float(np.mean(positive_counts)),

        "top1_positive_hit": float(np.mean(top1_hits)),
        "mrr_first_positive": float(np.mean(reciprocal_ranks)),
        "group_average_precision": float(np.mean(average_precisions)),
    }

    for k in ks:
        metrics[f"hit@{k}"] = float(
            np.mean(hit_at_k[k])
        )

        metrics[f"positive_recall@{k}"] = float(
            np.mean(recall_at_k[k])
        )

    return metrics

# ======================================================================
# 8. Complete validation evaluator
# ======================================================================

@torch.no_grad()
def evaluate_afp_scorer(
    model,
    X,
    y,
    group_ptr,
    device="cpu"
):
    model.eval()

    X_tensor = torch.as_tensor(
        X,
        dtype=torch.float32,
        device=device
    )

    y_tensor = torch.as_tensor(
        y,
        dtype=torch.float32,
        device=device
    )

    logits = model(X_tensor)

    branch_bce = branch_bce_from_logits(
        logits,
        y_tensor
    ).item()

    group_bce = group_balanced_bce_from_logits(
        logits,
        y_tensor,
        group_ptr
    ).item()

    probs = torch.sigmoid(
        logits
    ).detach().cpu().numpy()

    ranking = compute_group_ranking_metrics(
        scores=probs,
        labels=y,
        group_ptr=group_ptr,
        ks=(1, 2, 4)
    )

    result = {
        "branch_bce": float(branch_bce),
        "group_balanced_bce": float(group_bce),
        **ranking,
    }

    return result

# ======================================================================
# 9. Random-ranking sanity baseline
# ======================================================================
#
# This is NOT one of the final retrieval baselines.
# It is only a diagnostic baseline for scorer ranking metrics.
# ======================================================================

def random_ranking_metrics(y, group_ptr, seed=42):
    rng = np.random.default_rng(seed)

    random_scores = rng.random(
        len(y)
    )

    return compute_group_ranking_metrics(
        scores=random_scores,
        labels=y,
        group_ptr=group_ptr,
        ks=(1, 2, 4)
    )

# ======================================================================
# 10. Oracle-ranking sanity ceiling
# ======================================================================
#
# Again, diagnostic only.
# Gold labels are NEVER used by the model.
# ======================================================================

def oracle_ranking_metrics(y, group_ptr):
    # Positive candidates receive a higher synthetic score.
    oracle_scores = np.asarray(
        y,
        dtype=np.float64
    )

    return compute_group_ranking_metrics(
        scores=oracle_scores,
        labels=y,
        group_ptr=group_ptr,
        ks=(1, 2, 4)
    )

# ======================================================================
# 11. Definition sanity gate
# ======================================================================

mock_group_ptr = np.asarray(
    [0, 3, 6],
    dtype=np.int64
)

mock_y = np.asarray(
    [
        0, 1, 0,   # group 1
        1, 0, 1,   # group 2
    ],
    dtype=np.uint8
)

mock_scores = np.asarray(
    [
        0.2, 0.9, 0.1,
        0.8, 0.2, 0.7,
    ],
    dtype=np.float32
)

mock_metrics = compute_group_ranking_metrics(
    scores=mock_scores,
    labels=mock_y,
    group_ptr=mock_group_ptr,
    ks=(1, 2, 4)
)

assert np.isclose(
    mock_metrics["top1_positive_hit"],
    1.0
)

assert np.isclose(
    mock_metrics["mrr_first_positive"],
    1.0
)

mock_model = AFPScorer(
    input_dim=27,
    hidden_dim=64,
    dropout=AFP_DROPOUT
)

mock_X = torch.randn(
    6,
    27
)

mock_logits = mock_model(
    mock_X
)

assert mock_logits.shape == (6,)
assert torch.all(
    torch.isfinite(mock_logits)
)

# ======================================================================
# 12. Inspect random/oracle validation ranking ranges
# ======================================================================

print("\n" + "=" * 80)
print("WEBQSP VALIDATION RANKING SANITY")
print("=" * 80)

webqsp_random_metrics = random_ranking_metrics(
    webqsp_val_features["y"],
    webqsp_val_features["group_ptr"],
    seed=42
)

webqsp_oracle_metrics = oracle_ranking_metrics(
    webqsp_val_features["y"],
    webqsp_val_features["group_ptr"]
)

print("Random ranking:")
for k, v in webqsp_random_metrics.items():
    if isinstance(v, float):
        print(f"  {k:<26} {v:.4f}")

print("\nOracle ranking:")
for k, v in webqsp_oracle_metrics.items():
    if isinstance(v, float):
        print(f"  {k:<26} {v:.4f}")

print("\n" + "=" * 80)
print("CWQ VALIDATION RANKING SANITY")
print("=" * 80)

cwq_random_metrics = random_ranking_metrics(
    cwq_val_features["y"],
    cwq_val_features["group_ptr"],
    seed=42
)

cwq_oracle_metrics = oracle_ranking_metrics(
    cwq_val_features["y"],
    cwq_val_features["group_ptr"]
)

print("Random ranking:")
for k, v in cwq_random_metrics.items():
    if isinstance(v, float):
        print(f"  {k:<26} {v:.4f}")

print("\nOracle ranking:")
for k, v in cwq_oracle_metrics.items():
    if isinstance(v, float):
        print(f"  {k:<26} {v:.4f}")

# ======================================================================
# 13. Final report
# ======================================================================

print("\n" + "=" * 84)
print("=== RQ2 CELL 7: LIGHTWEIGHT AFP SCORER DEFINED ===")
print("=" * 84)

print("Architecture:")
print("  Input -> Linear(H) -> ReLU -> Linear(1)")
print("  Input dimension:", AFP_INPUT_DIM)
print("  Hidden candidates:", AFP_HIDDEN_CANDIDATES)
print("  Dropout:", AFP_DROPOUT)

print("\nDefined loss/statistics:")
print("  Branch BCE")
print("  Group-balanced BCE")

print("\nDefined ranking metrics:")
print("  Top-1 Positive Hit")
print("  MRR of first positive")
print("  Group Average Precision")
print("  Hit@1 / Hit@2 / Hit@4")
print("  Positive Recall@1 / @2 / @4")

print("\nNormalization:")
print("  Feature mean/std fitted on TRAIN only")
print("  Same statistics applied to validation/test")

print("\nImportant:")
print("  No scorer training performed.")
print("  No test data used.")
print("  Gold labels used only for evaluation diagnostics.")
print("\nNext: train AFP scorer on TRAIN and evaluate on VALIDATION.")

AFP scorer version: afp_mlp_v1
Input dimension:    27
Hidden candidates:  [32, 64, 128]
Scorer SHA256:      0d096c8aa07d893a...

WEBQSP VALIDATION RANKING SANITY
Random ranking:
  mean_group_size            11.1034
  median_group_size          5.0000
  mean_positive_count        3.6667
  top1_positive_hit          0.4943
  mrr_first_positive         0.6701
  group_average_precision    0.6319
  hit@1                      0.4943
  positive_recall@1          0.2418
  hit@2                      0.7241
  positive_recall@2          0.4367
  hit@4                      0.8621
  positive_recall@4          0.6903

Oracle ranking:
  mean_group_size            11.1034
  median_group_size          5.0000
  mean_positive_count        3.6667
  top1_positive_hit          1.0000
  mrr_first_positive         1.0000
  group_average_precision    1.0000
  hit@1                      1.0000
  positive_recall@1          0.5708
  hit@2                      1.0000
  positive_recall@2          0.7737
  hit@4    

## Training scorer

In [99]:
# ======================================================================
# 3.8 TRAIN LIGHTWEIGHT AFP SCORER
# ======================================================================
#
# Development grid:
#   Hidden size: {32, 64, 128}
#   Loss:        {branch BCE, group-balanced BCE}
#   Seeds:       {42, 43, 44}
#
# Fixed for all runs:
#   AdamW
#   LR = 1e-3
#   Weight decay = 1e-4
#   Epochs = 80
#   Full-batch optimization
#   No class weighting
#   No dropout
#
# IMPORTANT:
#   - Feature normalization fitted on TRAIN only.
#   - Validation is NEVER used to fit normalization.
#   - No test data/gold used.
#   - This cell does NOT select the winning configuration.
#   - Cell 9 will perform validation-based model selection.
# ======================================================================

import os
import json
import time
import random
import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# ======================================================================
# 1. Hard gates
# ======================================================================

assert AFP_FEATURE_DIM == 27
assert AFP_SCORER_VERSION == "afp_mlp_v1"
assert AFP_HIDDEN_CANDIDATES == [32, 64, 128]
assert AFP_DROPOUT == 0.0

required = [
    "AFPScorer",
    "AFPFeatureStandardizer",
    "evaluate_afp_scorer",
    "webqsp_train_features",
    "webqsp_val_features",
    "cwq_train_features",
    "cwq_val_features",
]

missing = [x for x in required if x not in globals()]
assert not missing, "Missing Cell 6/7 objects: " + ", ".join(missing)

# ======================================================================
# 2. Fixed training protocol
# ======================================================================

AFP_TRAINING_VERSION = "afp_train_v1"

AFP_TRAIN_SEEDS = [42, 43, 44]
AFP_LOSS_CANDIDATES = [
    "branch_bce",
    "group_balanced_bce",
]

AFP_LEARNING_RATE = 1e-3
AFP_WEIGHT_DECAY = 1e-4
AFP_EPOCHS = 80
AFP_GRAD_CLIP = 5.0

AFP_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

AFP_TRAINING_SPEC = {
    "version": AFP_TRAINING_VERSION,
    "scorer_version": AFP_SCORER_VERSION,
    "feature_spec_sha256": AFP_FEATURE_SPEC_SHA256,
    "hidden_candidates": AFP_HIDDEN_CANDIDATES,
    "loss_candidates": AFP_LOSS_CANDIDATES,
    "seeds": AFP_TRAIN_SEEDS,
    "optimizer": "AdamW",
    "learning_rate": AFP_LEARNING_RATE,
    "weight_decay": AFP_WEIGHT_DECAY,
    "epochs": AFP_EPOCHS,
    "gradient_clip": AFP_GRAD_CLIP,
    "dropout": AFP_DROPOUT,
    "class_weighting": False,
    "training_mode": "full_batch",
}

AFP_TRAINING_SPEC_SHA256 = hashlib.sha256(
    json.dumps(
        AFP_TRAINING_SPEC,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

print("Training version:", AFP_TRAINING_VERSION)
print("Device:          ", AFP_DEVICE)
print("Hidden sizes:    ", AFP_HIDDEN_CANDIDATES)
print("Losses:          ", AFP_LOSS_CANDIDATES)
print("Seeds:           ", AFP_TRAIN_SEEDS)
print("Epochs:          ", AFP_EPOCHS)
print("Learning rate:   ", AFP_LEARNING_RATE)
print("Weight decay:    ", AFP_WEIGHT_DECAY)
print("Training SHA256: ", AFP_TRAINING_SPEC_SHA256[:16] + "...")

# ======================================================================
# 3. Output directory
# ======================================================================

RQ2_SCORER_DIR = os.path.join(
    os.path.dirname(RQ2_FEATURE_DIR),
    "04_scorer"
)

os.makedirs(RQ2_SCORER_DIR, exist_ok=True)

print("Scorer directory:", RQ2_SCORER_DIR)

# ======================================================================
# 4. Reproducibility helper
# ======================================================================

def set_afp_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# ======================================================================
# 5. Exact group-balanced branch weights
# ======================================================================
#
# For group g with n_g branches:
#
#       w_i = 1 / n_g
#
# Therefore:
#
#   sum_i w_i BCE_i / G
#
# equals:
#
#   (1/G) sum_g (1/n_g) sum_i BCE_i
#
# which is exact group-balanced BCE.
# ======================================================================

def make_group_balanced_weights(group_ptr, n_branches):
    group_ptr = np.asarray(group_ptr, dtype=np.int64)

    weights = np.zeros(
        n_branches,
        dtype=np.float32
    )

    for g in range(len(group_ptr) - 1):
        start = int(group_ptr[g])
        end = int(group_ptr[g + 1])

        size = end - start
        assert size > 1

        weights[start:end] = 1.0 / size

    assert np.all(weights > 0)

    n_groups = len(group_ptr) - 1

    # Sum of weights must equal number of groups.
    assert np.isclose(
        weights.sum(),
        n_groups,
        rtol=1e-5
    )

    return weights

# ======================================================================
# 6. Vectorized training loss
# ======================================================================

def compute_training_loss(logits, targets, loss_name, group_weights=None):
    per_branch = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none"
    )

    if loss_name == "branch_bce":
        return per_branch.mean()

    if loss_name == "group_balanced_bce":
        assert group_weights is not None

        # Exact group-balanced objective.
        return (
            per_branch * group_weights
        ).sum() / group_weights.sum()

    raise ValueError(
        f"Unknown training loss: {loss_name}"
    )

# ======================================================================
# 7. Save standardizer
# ======================================================================

def save_standardizer(dataset_name, standardizer):
    path = os.path.join(
        RQ2_SCORER_DIR,
        f"{dataset_name}_train_standardizer.json"
    )

    payload = {
        "dataset": dataset_name,
        "feature_spec_sha256": AFP_FEATURE_SPEC_SHA256,
        "fit_split": "train",
        "state": standardizer.state_dict(),
    }

    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            payload,
            f,
            indent=2,
            ensure_ascii=False
        )

    return path

# ======================================================================
# 8. Prepare one dataset
# ======================================================================

def prepare_scorer_dataset(
    dataset_name,
    train_features,
    val_features
):
    X_train = train_features["X"]
    y_train = train_features["y"].astype(np.float32)
    train_group_ptr = train_features["group_ptr"]

    X_val = val_features["X"]
    y_val = val_features["y"].astype(np.float32)
    val_group_ptr = val_features["group_ptr"]

    assert X_train.shape[1] == AFP_FEATURE_DIM
    assert X_val.shape[1] == AFP_FEATURE_DIM

    # --------------------------------------------------------------
    # TRAIN-ONLY normalization
    # --------------------------------------------------------------
    standardizer = AFPFeatureStandardizer()

    X_train_z = standardizer.fit_transform(
        X_train
    )

    X_val_z = standardizer.transform(
        X_val
    )

    standardizer_file = save_standardizer(
        dataset_name,
        standardizer
    )

    group_weights = make_group_balanced_weights(
        train_group_ptr,
        len(y_train)
    )

    print(f"\n[{dataset_name.upper()}] preparation")
    print("Train branches:     ", len(y_train))
    print("Train groups:       ", len(train_group_ptr) - 1)
    print("Validation branches:", len(y_val))
    print("Validation groups:  ", len(val_group_ptr) - 1)
    print("Positive train rate:",
          f"{100*y_train.mean():.2f}%")
    print("Positive val rate:  ",
          f"{100*y_val.mean():.2f}%")
    print("Standardizer:       ", standardizer_file)

    return {
        "X_train": X_train_z,
        "y_train": y_train,
        "train_group_ptr": train_group_ptr,
        "group_weights": group_weights,
        "X_val": X_val_z,
        "y_val": y_val,
        "val_group_ptr": val_group_ptr,
        "standardizer": standardizer,
        "standardizer_file": standardizer_file,
    }

# ======================================================================
# 9. Train ONE run
# ======================================================================

def train_one_afp_run(
    dataset_name,
    prepared,
    hidden_dim,
    loss_name,
    seed
):
    set_afp_seed(seed)

    X_train_t = torch.as_tensor(
        prepared["X_train"],
        dtype=torch.float32,
        device=AFP_DEVICE
    )

    y_train_t = torch.as_tensor(
        prepared["y_train"],
        dtype=torch.float32,
        device=AFP_DEVICE
    )

    group_weights_t = torch.as_tensor(
        prepared["group_weights"],
        dtype=torch.float32,
        device=AFP_DEVICE
    )

    model = AFPScorer(
        input_dim=AFP_FEATURE_DIM,
        hidden_dim=hidden_dim,
        dropout=AFP_DROPOUT
    ).to(AFP_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=AFP_LEARNING_RATE,
        weight_decay=AFP_WEIGHT_DECAY
    )

    start_time = time.time()
    final_train_loss = None

    # --------------------------------------------------------------
    # Fixed-epoch training.
    #
    # No validation-based early stopping here.
    # This avoids introducing another tuning dimension.
    # --------------------------------------------------------------
    for epoch in range(1, AFP_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        logits = model(X_train_t)

        loss = compute_training_loss(
            logits=logits,
            targets=y_train_t,
            loss_name=loss_name,
            group_weights=group_weights_t
        )

        assert torch.isfinite(loss), (
            f"Non-finite training loss: "
            f"{dataset_name}, H={hidden_dim}, "
            f"loss={loss_name}, seed={seed}"
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            AFP_GRAD_CLIP
        )

        optimizer.step()

        final_train_loss = float(
            loss.detach().cpu()
        )

    runtime_sec = time.time() - start_time

    # --------------------------------------------------------------
    # Training-set loss diagnostics
    # --------------------------------------------------------------
    model.eval()

    with torch.no_grad():
        train_logits = model(X_train_t)

        final_train_branch_bce = float(
            F.binary_cross_entropy_with_logits(
                train_logits,
                y_train_t
            ).cpu()
        )

        train_group_loss = compute_training_loss(
            logits=train_logits,
            targets=y_train_t,
            loss_name="group_balanced_bce",
            group_weights=group_weights_t
        )

        final_train_group_bce = float(
            train_group_loss.cpu()
        )

    # --------------------------------------------------------------
    # Validation metrics from frozen Cell-7 evaluator
    # --------------------------------------------------------------
    val_metrics = evaluate_afp_scorer(
        model=model,
        X=prepared["X_val"],
        y=prepared["y_val"],
        group_ptr=prepared["val_group_ptr"],
        device=AFP_DEVICE
    )

    # --------------------------------------------------------------
    # Save final checkpoint.
    #
    # This is NOT yet the selected/frozen deployment model.
    # Cell 9 will select among these runs using validation only.
    # --------------------------------------------------------------
    run_name = (
        f"{dataset_name}"
        f"_h{hidden_dim}"
        f"_{loss_name}"
        f"_seed{seed}"
    )

    checkpoint_path = os.path.join(
        RQ2_SCORER_DIR,
        run_name + ".pt"
    )

    cpu_state = {
        k: v.detach().cpu()
        for k, v in model.state_dict().items()
    }

    checkpoint = {
        "dataset": dataset_name,
        "run_name": run_name,
        "feature_spec_sha256": AFP_FEATURE_SPEC_SHA256,
        "scorer_spec_sha256": AFP_SCORER_SPEC_SHA256,
        "training_spec_sha256": AFP_TRAINING_SPEC_SHA256,
        "hidden_dim": hidden_dim,
        "loss_name": loss_name,
        "seed": seed,
        "epochs": AFP_EPOCHS,
        "learning_rate": AFP_LEARNING_RATE,
        "weight_decay": AFP_WEIGHT_DECAY,
        "model_state_dict": cpu_state,
        "standardizer_state":
            prepared["standardizer"].state_dict(),
        "validation_metrics": val_metrics,
    }

    torch.save(
        checkpoint,
        checkpoint_path
    )

    result = {
        "dataset": dataset_name,
        "hidden_dim": hidden_dim,
        "loss_name": loss_name,
        "seed": seed,
        "epochs": AFP_EPOCHS,

        "final_train_objective":
            final_train_loss,

        "train_branch_bce":
            final_train_branch_bce,

        "train_group_balanced_bce":
            final_train_group_bce,

        "val_branch_bce":
            val_metrics["branch_bce"],

        "val_group_balanced_bce":
            val_metrics["group_balanced_bce"],

        "val_top1_positive_hit":
            val_metrics["top1_positive_hit"],

        "val_mrr_first_positive":
            val_metrics["mrr_first_positive"],

        "val_group_average_precision":
            val_metrics["group_average_precision"],

        "val_hit@1":
            val_metrics["hit@1"],

        "val_hit@2":
            val_metrics["hit@2"],

        "val_hit@4":
            val_metrics["hit@4"],

        "val_positive_recall@1":
            val_metrics["positive_recall@1"],

        "val_positive_recall@2":
            val_metrics["positive_recall@2"],

        "val_positive_recall@4":
            val_metrics["positive_recall@4"],

        "runtime_sec":
            runtime_sec,

        "checkpoint":
            checkpoint_path,
    }

    del model
    del optimizer
    del X_train_t
    del y_train_t
    del group_weights_t

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

# ======================================================================
# 10. Train full predefined grid for one dataset
# ======================================================================

def train_afp_grid(
    dataset_name,
    train_features,
    val_features
):
    prepared = prepare_scorer_dataset(
        dataset_name,
        train_features,
        val_features
    )

    results = []

    total_runs = (
        len(AFP_HIDDEN_CANDIDATES)
        * len(AFP_LOSS_CANDIDATES)
        * len(AFP_TRAIN_SEEDS)
    )

    run_no = 0

    print("\n" + "=" * 84)
    print(f"TRAINING {dataset_name.upper()} AFP SCORER GRID")
    print(f"Total runs: {total_runs}")
    print("=" * 84)

    for hidden_dim in AFP_HIDDEN_CANDIDATES:
        for loss_name in AFP_LOSS_CANDIDATES:
            for seed in AFP_TRAIN_SEEDS:
                run_no += 1

                print(
                    f"\n[{run_no:02d}/{total_runs}] "
                    f"H={hidden_dim} | "
                    f"loss={loss_name} | "
                    f"seed={seed}"
                )

                result = train_one_afp_run(
                    dataset_name=dataset_name,
                    prepared=prepared,
                    hidden_dim=hidden_dim,
                    loss_name=loss_name,
                    seed=seed
                )

                results.append(result)

                print(
                    f"  train objective = "
                    f"{result['final_train_objective']:.5f}"
                )
                print(
                    f"  val Group AP    = "
                    f"{result['val_group_average_precision']:.4f}"
                )
                print(
                    f"  val MRR         = "
                    f"{result['val_mrr_first_positive']:.4f}"
                )
                print(
                    f"  val Top-1 Hit   = "
                    f"{result['val_top1_positive_hit']:.4f}"
                )
                print(
                    f"  val Group BCE   = "
                    f"{result['val_group_balanced_bce']:.4f}"
                )
                print(
                    f"  runtime         = "
                    f"{result['runtime_sec']:.2f}s"
                )

    return prepared, results

# ======================================================================
# 11. WEBQSP training grid
# ======================================================================

webqsp_scorer_prepared, webqsp_scorer_results = train_afp_grid(
    dataset_name="webqsp",
    train_features=webqsp_train_features,
    val_features=webqsp_val_features
)

# ======================================================================
# 12. CWQ training grid
# ======================================================================

cwq_scorer_prepared, cwq_scorer_results = train_afp_grid(
    dataset_name="cwq",
    train_features=cwq_train_features,
    val_features=cwq_val_features
)

# ======================================================================
# 13. Combine and save run-level results
# ======================================================================

afp_scorer_results = (
    webqsp_scorer_results
    + cwq_scorer_results
)

afp_scorer_results_df = pd.DataFrame(
    afp_scorer_results
)

results_csv = os.path.join(
    RQ2_SCORER_DIR,
    "afp_scorer_training_runs.csv"
)

afp_scorer_results_df.to_csv(
    results_csv,
    index=False
)

results_json = os.path.join(
    RQ2_SCORER_DIR,
    "afp_scorer_training_runs.json"
)

with open(
    results_json,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        afp_scorer_results,
        f,
        indent=2,
        ensure_ascii=False
    )

# ======================================================================
# 14. Aggregate across seeds
# ======================================================================
#
# IMPORTANT:
# This is descriptive only.
# We are NOT selecting the winner in this cell.
# ======================================================================

metric_columns = [
    "val_group_average_precision",
    "val_mrr_first_positive",
    "val_top1_positive_hit",
    "val_group_balanced_bce",
    "val_branch_bce",
    "val_positive_recall@1",
    "val_positive_recall@2",
    "val_positive_recall@4",
]

aggregate = (
    afp_scorer_results_df
    .groupby(
        ["dataset", "hidden_dim", "loss_name"]
    )[metric_columns]
    .agg(["mean", "std"])
    .reset_index()
)

aggregate_csv = os.path.join(
    RQ2_SCORER_DIR,
    "afp_scorer_training_aggregate.csv"
)

aggregate.to_csv(
    aggregate_csv,
    index=False
)

# ======================================================================
# 15. Compact validation summary
# ======================================================================

summary = (
    afp_scorer_results_df
    .groupby(
        ["dataset", "hidden_dim", "loss_name"]
    )
    .agg(
        group_ap_mean=(
            "val_group_average_precision",
            "mean"
        ),
        group_ap_std=(
            "val_group_average_precision",
            "std"
        ),
        mrr_mean=(
            "val_mrr_first_positive",
            "mean"
        ),
        top1_mean=(
            "val_top1_positive_hit",
            "mean"
        ),
        group_bce_mean=(
            "val_group_balanced_bce",
            "mean"
        ),
        branch_bce_mean=(
            "val_branch_bce",
            "mean"
        ),
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("VALIDATION SUMMARY ACROSS 3 SEEDS — DESCRIPTIVE ONLY")
print("=" * 100)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    subset = summary[
        summary["dataset"] == dataset_name
    ]

    print(
        subset[
            [
                "hidden_dim",
                "loss_name",
                "group_ap_mean",
                "group_ap_std",
                "mrr_mean",
                "top1_mean",
                "group_bce_mean",
                "branch_bce_mean",
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

# ======================================================================
# 16. Final training manifest
# ======================================================================

training_manifest = {
    "training_version": AFP_TRAINING_VERSION,
    "training_spec_sha256": AFP_TRAINING_SPEC_SHA256,
    "feature_spec_sha256": AFP_FEATURE_SPEC_SHA256,
    "scorer_spec_sha256": AFP_SCORER_SPEC_SHA256,
    "datasets": ["webqsp", "cwq"],
    "hidden_candidates": AFP_HIDDEN_CANDIDATES,
    "loss_candidates": AFP_LOSS_CANDIDATES,
    "seeds": AFP_TRAIN_SEEDS,
    "epochs": AFP_EPOCHS,
    "learning_rate": AFP_LEARNING_RATE,
    "weight_decay": AFP_WEIGHT_DECAY,
    "class_weighting": False,
    "validation_early_stopping": False,
    "test_used": False,
    "n_runs_total": len(afp_scorer_results),
    "results_csv": results_csv,
    "aggregate_csv": aggregate_csv,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}

manifest_path = os.path.join(
    RQ2_SCORER_DIR,
    "afp_scorer_training_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        training_manifest,
        f,
        indent=2
    )

print("\n" + "=" * 84)
print("=== RQ2 CELL 8: AFP SCORER TRAINING COMPLETE ===")
print("=" * 84)
print("Total runs:", len(afp_scorer_results))
print("Runs per dataset: 18")
print("Hidden sizes:", AFP_HIDDEN_CANDIDATES)
print("Losses:", AFP_LOSS_CANDIDATES)
print("Seeds:", AFP_TRAIN_SEEDS)
print("Validation early stopping: NO")
print("Class weighting: NO")
print("TEST data/gold used: NO")
print("Winner selected: NO")
print("Results:", results_csv)
print("\nNext: validation comparison + scorer selection/freeze.")

Training version: afp_train_v1
Device:           cuda
Hidden sizes:     [32, 64, 128]
Losses:           ['branch_bce', 'group_balanced_bce']
Seeds:            [42, 43, 44]
Epochs:           80
Learning rate:    0.001
Weight decay:     0.0001
Training SHA256:  c91adb1c90f9e75d...
Scorer directory: /kaggle/working/step3_rq2_dev_v1/04_scorer

[WEBQSP] preparation
Train branches:      18437
Train groups:        1457
Validation branches: 966
Validation groups:   87
Positive train rate: 39.60%
Positive val rate:   33.02%
Standardizer:        /kaggle/working/step3_rq2_dev_v1/04_scorer/webqsp_train_standardizer.json

TRAINING WEBQSP AFP SCORER GRID
Total runs: 18

[01/18] H=32 | loss=branch_bce | seed=42
  train objective = 0.57420
  val Group AP    = 0.6104
  val MRR         = 0.6512
  val Top-1 Hit   = 0.5057
  val Group BCE   = 0.6264
  runtime         = 0.39s

[02/18] H=32 | loss=branch_bce | seed=43
  train objective = 0.57767
  val Group AP    = 0.6059
  val MRR         = 0.6436
  val To

## AFP Scorer Diagnostic Gate

In [100]:
# ======================================================================
# 3.9A AFP SCORER DIAGNOSTIC GATE
# ======================================================================
#
# Purpose:
#   Diagnose scorer v1 BEFORE validation selection/final freeze.
#
# Tests:
#   1. Proper random-ranking null distribution
#   2. TRAIN vs VALIDATION ranking/generalization
#   3. Within-group feature variation
#   4. Candidate-level feature signal after removing group-level effects
#   5. Diagnostic subgroup performance
#
# IMPORTANT:
#   - TRAIN + VALIDATION only
#   - NO TEST data/gold
#   - NO new training
#   - NO winner frozen
#   - NO architecture changed
# ======================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import roc_auc_score
from tqdm import tqdm

# ======================================================================
# 1. Hard gates
# ======================================================================

assert AFP_SCORER_VERSION == "afp_mlp_v1"
assert AFP_TRAINING_VERSION == "afp_train_v1"
assert AFP_FEATURE_DIM == 27
assert len(afp_scorer_results_df) == 36

required = [
    "compute_group_ranking_metrics",
    "AFPScorer",
    "webqsp_train_features",
    "webqsp_val_features",
    "cwq_train_features",
    "cwq_val_features",
    "webqsp_scorer_prepared",
    "cwq_scorer_prepared",
]

missing = [x for x in required if x not in globals()]
assert not missing, "Missing required objects: " + ", ".join(missing)

DIAG_DIR = os.path.join(
    os.path.dirname(RQ2_SCORER_DIR),
    "05_scorer_diagnostics"
)
os.makedirs(DIAG_DIR, exist_ok=True)

print("Diagnostic directory:", DIAG_DIR)

# ======================================================================
# 2. Frozen Feature-v2 names
# ======================================================================

AFP_FEATURE_NAMES_V2 = [
    "sem_q_candidate",                    # 0
    "sem_candidate_surface_available",    # 1
    "sem_q_current_entity",               # 2
    "sem_current_surface_available",      # 3
    "sem_q_current_relation",             # 4
    "sem_q_full_plan",                    # 5
    "sem_q_remaining_suffix",             # 6
    "sem_candidate_current_relation",     # 7

    "path_q_prefix_entity_mean",           # 8
    "path_candidate_prefix_entity_mean",   # 9
    "path_prefix_surface_fraction",        # 10
    "path_candidate_repeats_entity",       # 11
    "path_candidate_occurrence_fraction",  # 12
    "path_unique_entity_ratio",            # 13
    "path_relation_repeat_fraction_before",# 14

    "struct_log_candidate_count",          # 15
    "struct_log_unique_candidate_entities",# 16
    "struct_log_contributing_parents",     # 17
    "struct_log_parent_fanout",            # 18
    "struct_parent_frontier_share",        # 19
    "struct_log_endpoint_multiplicity",    # 20
    "struct_endpoint_frontier_share",      # 21
    "struct_duplicate_endpoint_ratio",     # 22

    "prog_hop_fraction",                   # 23
    "prog_remaining_fraction",             # 24
    "prog_log_plan_length",                # 25
    "prog_penultimate_indicator",          # 26
]

assert len(AFP_FEATURE_NAMES_V2) == 27

# ======================================================================
# 3. Proper random-ranking null distribution
# ======================================================================
#
# Cell 7 used only one random seed.
# Here we build an empirical null distribution over 1000 random rankings.
# ======================================================================

RANDOM_NULL_TRIALS = 1000
RANDOM_NULL_BASE_SEED = 20260831

def build_random_null(y, group_ptr, n_trials=1000, base_seed=20260831):
    y = np.asarray(y, dtype=np.uint8)
    group_ptr = np.asarray(group_ptr, dtype=np.int64)

    rows = []

    for trial in tqdm(
        range(n_trials),
        desc="Random null",
        leave=False
    ):
        rng = np.random.default_rng(base_seed + trial)
        scores = rng.random(len(y))

        m = compute_group_ranking_metrics(
            scores=scores,
            labels=y,
            group_ptr=group_ptr,
            ks=(1, 2, 4)
        )

        rows.append({
            "trial": trial,
            "group_ap": m["group_average_precision"],
            "mrr": m["mrr_first_positive"],
            "top1": m["top1_positive_hit"],
            "hit2": m["hit@2"],
            "recall2": m["positive_recall@2"],
        })

    return pd.DataFrame(rows)

def summarize_null(df):
    rows = []

    for metric in ["group_ap", "mrr", "top1", "hit2", "recall2"]:
        values = df[metric].values

        rows.append({
            "metric": metric,
            "mean": values.mean(),
            "std": values.std(ddof=1),
            "p2.5": np.percentile(values, 2.5),
            "p50": np.percentile(values, 50),
            "p97.5": np.percentile(values, 97.5),
        })

    return pd.DataFrame(rows)

print("\nBuilding WebQSP random null...")
webqsp_random_null = build_random_null(
    webqsp_val_features["y"],
    webqsp_val_features["group_ptr"],
    RANDOM_NULL_TRIALS,
    RANDOM_NULL_BASE_SEED
)

print("Building CWQ random null...")
cwq_random_null = build_random_null(
    cwq_val_features["y"],
    cwq_val_features["group_ptr"],
    RANDOM_NULL_TRIALS,
    RANDOM_NULL_BASE_SEED + 10000
)

webqsp_null_summary = summarize_null(webqsp_random_null)
cwq_null_summary = summarize_null(cwq_random_null)

print("\n" + "="*90)
print("RANDOM-RANKING NULL — WEBQSP")
print("="*90)
print(webqsp_null_summary.to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))

print("\n" + "="*90)
print("RANDOM-RANKING NULL — CWQ")
print("="*90)
print(cwq_null_summary.to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))

# ======================================================================
# 4. Checkpoint loader
# ======================================================================

def load_afp_checkpoint(path, device="cpu"):
    try:
        ckpt = torch.load(
            path,
            map_location=device,
            weights_only=False
        )
    except TypeError:
        ckpt = torch.load(
            path,
            map_location=device
        )

    model = AFPScorer(
        input_dim=AFP_FEATURE_DIM,
        hidden_dim=int(ckpt["hidden_dim"]),
        dropout=AFP_DROPOUT
    ).to(device)

    model.load_state_dict(
        ckpt["model_state_dict"]
    )
    model.eval()

    return model, ckpt

@torch.no_grad()
def predict_afp_scores(model, X, device):
    X_t = torch.as_tensor(
        X,
        dtype=torch.float32,
        device=device
    )

    logits = model(X_t)
    probs = torch.sigmoid(logits)

    return probs.detach().cpu().numpy()

# ======================================================================
# 5. TRAIN vs VALIDATION ranking for every saved run
# ======================================================================

def get_prepared(dataset_name):
    if dataset_name == "webqsp":
        return webqsp_scorer_prepared
    if dataset_name == "cwq":
        return cwq_scorer_prepared
    raise ValueError(dataset_name)

generalization_rows = []

print("\nEvaluating TRAIN vs VALIDATION ranking...")

for _, row in tqdm(
    afp_scorer_results_df.iterrows(),
    total=len(afp_scorer_results_df)
):
    dataset_name = row["dataset"]
    prepared = get_prepared(dataset_name)

    model, ckpt = load_afp_checkpoint(
        row["checkpoint"],
        AFP_DEVICE
    )

    train_scores = predict_afp_scores(
        model,
        prepared["X_train"],
        AFP_DEVICE
    )

    val_scores = predict_afp_scores(
        model,
        prepared["X_val"],
        AFP_DEVICE
    )

    train_rank = compute_group_ranking_metrics(
        train_scores,
        prepared["y_train"],
        prepared["train_group_ptr"],
        ks=(1, 2, 4)
    )

    val_rank = compute_group_ranking_metrics(
        val_scores,
        prepared["y_val"],
        prepared["val_group_ptr"],
        ks=(1, 2, 4)
    )

    generalization_rows.append({
        "dataset": dataset_name,
        "hidden_dim": int(row["hidden_dim"]),
        "loss_name": row["loss_name"],
        "seed": int(row["seed"]),

        "train_group_ap":
            train_rank["group_average_precision"],
        "val_group_ap":
            val_rank["group_average_precision"],
        "gap_group_ap":
            train_rank["group_average_precision"]
            - val_rank["group_average_precision"],

        "train_mrr":
            train_rank["mrr_first_positive"],
        "val_mrr":
            val_rank["mrr_first_positive"],
        "gap_mrr":
            train_rank["mrr_first_positive"]
            - val_rank["mrr_first_positive"],

        "train_top1":
            train_rank["top1_positive_hit"],
        "val_top1":
            val_rank["top1_positive_hit"],
        "gap_top1":
            train_rank["top1_positive_hit"]
            - val_rank["top1_positive_hit"],
    })

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

generalization_df = pd.DataFrame(generalization_rows)

generalization_agg = (
    generalization_df
    .groupby(["dataset", "hidden_dim", "loss_name"])
    .agg(
        train_group_ap=("train_group_ap", "mean"),
        val_group_ap=("val_group_ap", "mean"),
        gap_group_ap=("gap_group_ap", "mean"),
        train_mrr=("train_mrr", "mean"),
        val_mrr=("val_mrr", "mean"),
        gap_mrr=("gap_mrr", "mean"),
        train_top1=("train_top1", "mean"),
        val_top1=("val_top1", "mean"),
        gap_top1=("gap_top1", "mean"),
    )
    .reset_index()
)

print("\n" + "="*105)
print("TRAIN → VALIDATION GENERALIZATION")
print("="*105)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    sub = generalization_agg[
        generalization_agg["dataset"] == dataset_name
    ]

    print(sub.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    ))

# ======================================================================
# 6. Compare validation configuration means against random null
# ======================================================================

config_mean = (
    generalization_df
    .groupby(["dataset", "hidden_dim", "loss_name"])
    .agg(
        group_ap=("val_group_ap", "mean"),
        mrr=("val_mrr", "mean"),
        top1=("val_top1", "mean"),
    )
    .reset_index()
)

def empirical_upper_p(null_values, observed):
    null_values = np.asarray(null_values)
    return float(
        (1 + np.sum(null_values >= observed))
        / (len(null_values) + 1)
    )

null_comparison_rows = []

for _, row in config_mean.iterrows():
    dataset_name = row["dataset"]

    null_df = (
        webqsp_random_null
        if dataset_name == "webqsp"
        else cwq_random_null
    )

    null_comparison_rows.append({
        "dataset": dataset_name,
        "hidden_dim": int(row["hidden_dim"]),
        "loss_name": row["loss_name"],

        "group_ap": row["group_ap"],
        "random_group_ap_mean":
            null_df["group_ap"].mean(),
        "group_ap_delta":
            row["group_ap"] - null_df["group_ap"].mean(),
        "group_ap_null_p":
            empirical_upper_p(
                null_df["group_ap"],
                row["group_ap"]
            ),

        "mrr": row["mrr"],
        "random_mrr_mean":
            null_df["mrr"].mean(),
        "mrr_delta":
            row["mrr"] - null_df["mrr"].mean(),
        "mrr_null_p":
            empirical_upper_p(
                null_df["mrr"],
                row["mrr"]
            ),

        "top1": row["top1"],
        "random_top1_mean":
            null_df["top1"].mean(),
        "top1_delta":
            row["top1"] - null_df["top1"].mean(),
        "top1_null_p":
            empirical_upper_p(
                null_df["top1"],
                row["top1"]
            ),
    })

null_comparison_df = pd.DataFrame(
    null_comparison_rows
)

print("\n" + "="*110)
print("VALIDATION SCORER vs RANDOM-RANKING NULL")
print("="*110)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    sub = null_comparison_df[
        null_comparison_df["dataset"] == dataset_name
    ]

    cols = [
        "hidden_dim",
        "loss_name",
        "group_ap",
        "random_group_ap_mean",
        "group_ap_delta",
        "group_ap_null_p",
        "mrr_delta",
        "mrr_null_p",
        "top1_delta",
        "top1_null_p",
    ]

    print(sub[cols].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    ))

# ======================================================================
# 7. Within-group feature variation
# ======================================================================
#
# A feature that is constant across every candidate in a frontier
# cannot rank those candidates.
# ======================================================================

def feature_variation_profile(X, group_ptr, feature_names, tol=1e-8):
    X = np.asarray(X, dtype=np.float32)
    group_ptr = np.asarray(group_ptr, dtype=np.int64)

    n_features = X.shape[1]
    varying_count = np.zeros(n_features, dtype=np.int64)
    std_sum = np.zeros(n_features, dtype=np.float64)

    n_groups = len(group_ptr) - 1

    for g in range(n_groups):
        s = int(group_ptr[g])
        e = int(group_ptr[g + 1])

        Xg = X[s:e]
        ranges = Xg.max(axis=0) - Xg.min(axis=0)
        stds = Xg.std(axis=0)

        varying_count += (ranges > tol)
        std_sum += stds

    return pd.DataFrame({
        "feature_index": np.arange(n_features),
        "feature": feature_names,
        "varying_groups": varying_count,
        "n_groups": n_groups,
        "varying_group_rate":
            varying_count / n_groups,
        "mean_within_group_std":
            std_sum / n_groups,
    })

variation_frames = []

for dataset_name, split, data in [
    ("webqsp", "train", webqsp_train_features),
    ("webqsp", "validation", webqsp_val_features),
    ("cwq", "train", cwq_train_features),
    ("cwq", "validation", cwq_val_features),
]:
    v = feature_variation_profile(
        data["X"],
        data["group_ptr"],
        AFP_FEATURE_NAMES_V2
    )

    v.insert(0, "split", split)
    v.insert(0, "dataset", dataset_name)
    variation_frames.append(v)

feature_variation_df = pd.concat(
    variation_frames,
    ignore_index=True
)

print("\n" + "="*100)
print("WITHIN-GROUP FEATURE VARIATION — VALIDATION")
print("="*100)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    sub = feature_variation_df[
        (feature_variation_df["dataset"] == dataset_name)
        &
        (feature_variation_df["split"] == "validation")
    ].sort_values(
        "varying_group_rate",
        ascending=False
    )

    print(sub[
        [
            "feature_index",
            "feature",
            "varying_group_rate",
            "mean_within_group_std",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    ))

# ======================================================================
# 8. Group-center features
# ======================================================================
#
# Removing each frontier's feature mean eliminates pure group-level
# offsets. What remains is candidate-relative variation.
# ======================================================================

def group_center_features(X, group_ptr):
    X = np.asarray(X, dtype=np.float32)
    group_ptr = np.asarray(group_ptr, dtype=np.int64)

    Z = np.empty_like(X)

    for g in range(len(group_ptr) - 1):
        s = int(group_ptr[g])
        e = int(group_ptr[g + 1])

        Xg = X[s:e]
        Z[s:e] = Xg - Xg.mean(axis=0, keepdims=True)

    return Z

# ======================================================================
# 9. Candidate-relative feature signal
# ======================================================================
#
# AUC = 0.5  -> no directionally useful signal
# AUC > 0.5  -> higher feature tends to indicate positive branch
# AUC < 0.5  -> lower feature tends to indicate positive branch
#
# best_auc = max(AUC, 1-AUC)
#
# This is diagnostic only; it is NOT feature selection.
# ======================================================================

def feature_signal_profile(X, y, group_ptr, feature_names):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.uint8)

    centered = group_center_features(
        X,
        group_ptr
    )

    rows = []

    for j, name in enumerate(feature_names):
        x = centered[:, j]

        pos = x[y == 1]
        neg = x[y == 0]

        if np.allclose(x, x[0]):
            auc = 0.5
        else:
            auc = roc_auc_score(y, x)

        rows.append({
            "feature_index": j,
            "feature": name,
            "positive_mean_centered":
                float(pos.mean()),
            "negative_mean_centered":
                float(neg.mean()),
            "centered_delta":
                float(pos.mean() - neg.mean()),
            "auc_higher_is_positive":
                float(auc),
            "best_orientation_auc":
                float(max(auc, 1.0 - auc)),
            "preferred_direction":
                "higher"
                if auc >= 0.5
                else "lower",
        })

    return pd.DataFrame(rows)

signal_frames = []

for dataset_name, split, data in [
    ("webqsp", "train", webqsp_train_features),
    ("webqsp", "validation", webqsp_val_features),
    ("cwq", "train", cwq_train_features),
    ("cwq", "validation", cwq_val_features),
]:
    sig = feature_signal_profile(
        data["X"],
        data["y"],
        data["group_ptr"],
        AFP_FEATURE_NAMES_V2
    )

    sig.insert(0, "split", split)
    sig.insert(0, "dataset", dataset_name)
    signal_frames.append(sig)

feature_signal_df = pd.concat(
    signal_frames,
    ignore_index=True
)

print("\n" + "="*100)
print("CANDIDATE-RELATIVE FEATURE SIGNAL — VALIDATION")
print("="*100)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    sub = feature_signal_df[
        (feature_signal_df["dataset"] == dataset_name)
        &
        (feature_signal_df["split"] == "validation")
    ].sort_values(
        "best_orientation_auc",
        ascending=False
    )

    print(sub[
        [
            "feature_index",
            "feature",
            "centered_delta",
            "auc_higher_is_positive",
            "best_orientation_auc",
            "preferred_direction",
        ]
    ].head(15).to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    ))

# ======================================================================
# 10. Diagnostic reference configuration
# ======================================================================
#
# For subgroup inspection ONLY:
#   choose the configuration with best MEAN validation Group AP.
#
# This is NOT final model selection/freeze.
# The three seeds are ensembled to reduce initialization noise.
# ======================================================================

def get_diagnostic_reference(dataset_name):
    sub = (
        config_mean[
            config_mean["dataset"] == dataset_name
        ]
        .sort_values(
            ["group_ap", "mrr", "top1"],
            ascending=[False, False, False]
        )
        .iloc[0]
    )

    return (
        int(sub["hidden_dim"]),
        sub["loss_name"]
    )

def ensemble_scores_for_config(
    dataset_name,
    hidden_dim,
    loss_name,
    prepared
):
    rows = afp_scorer_results_df[
        (afp_scorer_results_df["dataset"] == dataset_name)
        &
        (afp_scorer_results_df["hidden_dim"] == hidden_dim)
        &
        (afp_scorer_results_df["loss_name"] == loss_name)
    ]

    assert len(rows) == 3

    scores = []

    for _, row in rows.iterrows():
        model, _ = load_afp_checkpoint(
            row["checkpoint"],
            AFP_DEVICE
        )

        scores.append(
            predict_afp_scores(
                model,
                prepared["X_val"],
                AFP_DEVICE
            )
        )

        del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.mean(
        np.stack(scores, axis=0),
        axis=0
    )

# ======================================================================
# 11. Subgroup-ranking helper
# ======================================================================

def subset_groups_metrics(
    scores,
    labels,
    group_ptr,
    selected_group_indices
):
    new_scores = []
    new_labels = []
    new_ptr = [0]

    for g in selected_group_indices:
        s = int(group_ptr[g])
        e = int(group_ptr[g + 1])

        new_scores.extend(scores[s:e])
        new_labels.extend(labels[s:e])
        new_ptr.append(
            new_ptr[-1] + (e - s)
        )

    if len(selected_group_indices) == 0:
        return None

    return compute_group_ranking_metrics(
        np.asarray(new_scores),
        np.asarray(new_labels),
        np.asarray(new_ptr),
        ks=(1, 2, 4)
    )

def diagnostic_subgroups(dataset_name, features, scores):
    y = features["y"]
    ptr = features["group_ptr"]
    hops = features["group_hop"]
    sizes = features["group_candidate_count"]
    X = features["X"]

    rows = []
    n_groups = len(ptr) - 1

    # --------------------------------------------------------------
    # Hop
    # --------------------------------------------------------------
    for hop in sorted(np.unique(hops)):
        idx = np.flatnonzero(hops == hop)

        m = subset_groups_metrics(
            scores,
            y,
            ptr,
            idx
        )

        rows.append({
            "dataset": dataset_name,
            "subgroup_type": "hop",
            "subgroup": f"hop_{int(hop)}",
            "n_groups": len(idx),
            "group_ap": m["group_average_precision"],
            "mrr": m["mrr_first_positive"],
            "top1": m["top1_positive_hit"],
        })

    # --------------------------------------------------------------
    # Candidate-set size
    # --------------------------------------------------------------
    size_bins = [
        ("2-4", 2, 4),
        ("5-8", 5, 8),
        ("9-16", 9, 16),
        ("17+", 17, np.inf),
    ]

    for label, lo, hi in size_bins:
        mask = (sizes >= lo) & (sizes <= hi)
        idx = np.flatnonzero(mask)

        if len(idx) == 0:
            continue

        m = subset_groups_metrics(
            scores,
            y,
            ptr,
            idx
        )

        rows.append({
            "dataset": dataset_name,
            "subgroup_type": "candidate_size",
            "subgroup": label,
            "n_groups": len(idx),
            "group_ap": m["group_average_precision"],
            "mrr": m["mrr_first_positive"],
            "top1": m["top1_positive_hit"],
        })

    # --------------------------------------------------------------
    # Candidate entity-surface availability
    # --------------------------------------------------------------
    availability = []

    for g in range(n_groups):
        s = int(ptr[g])
        e = int(ptr[g + 1])

        a = X[
            s:e,
            1  # candidate-surface availability
        ]

        if np.all(a == 0):
            availability.append("all_raw_mid")
        elif np.all(a == 1):
            availability.append("all_readable")
        else:
            availability.append("mixed")

    availability = np.asarray(availability)

    for category in [
        "all_raw_mid",
        "mixed",
        "all_readable"
    ]:
        idx = np.flatnonzero(
            availability == category
        )

        if len(idx) == 0:
            continue

        m = subset_groups_metrics(
            scores,
            y,
            ptr,
            idx
        )

        rows.append({
            "dataset": dataset_name,
            "subgroup_type": "candidate_surface",
            "subgroup": category,
            "n_groups": len(idx),
            "group_ap": m["group_average_precision"],
            "mrr": m["mrr_first_positive"],
            "top1": m["top1_positive_hit"],
        })

    return pd.DataFrame(rows)

diag_reference = {}
subgroup_frames = []

for dataset_name, features, prepared in [
    (
        "webqsp",
        webqsp_val_features,
        webqsp_scorer_prepared
    ),
    (
        "cwq",
        cwq_val_features,
        cwq_scorer_prepared
    ),
]:
    hidden_dim, loss_name = get_diagnostic_reference(
        dataset_name
    )

    diag_reference[dataset_name] = {
        "hidden_dim": hidden_dim,
        "loss_name": loss_name,
    }

    scores = ensemble_scores_for_config(
        dataset_name,
        hidden_dim,
        loss_name,
        prepared
    )

    sg = diagnostic_subgroups(
        dataset_name,
        features,
        scores
    )

    subgroup_frames.append(sg)

    print(
        f"\nDiagnostic reference {dataset_name.upper()}: "
        f"H={hidden_dim}, loss={loss_name}"
    )

subgroup_df = pd.concat(
    subgroup_frames,
    ignore_index=True
)

print("\n" + "="*100)
print("DIAGNOSTIC REFERENCE — VALIDATION SUBGROUP PERFORMANCE")
print("="*100)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    print(
        subgroup_df[
            subgroup_df["dataset"] == dataset_name
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

# ======================================================================
# 12. Save diagnostics
# ======================================================================

webqsp_random_null.to_csv(
    os.path.join(
        DIAG_DIR,
        "webqsp_random_null.csv"
    ),
    index=False
)

cwq_random_null.to_csv(
    os.path.join(
        DIAG_DIR,
        "cwq_random_null.csv"
    ),
    index=False
)

null_comparison_df.to_csv(
    os.path.join(
        DIAG_DIR,
        "scorer_vs_random_null.csv"
    ),
    index=False
)

generalization_df.to_csv(
    os.path.join(
        DIAG_DIR,
        "train_validation_generalization_runs.csv"
    ),
    index=False
)

generalization_agg.to_csv(
    os.path.join(
        DIAG_DIR,
        "train_validation_generalization_aggregate.csv"
    ),
    index=False
)

feature_variation_df.to_csv(
    os.path.join(
        DIAG_DIR,
        "within_group_feature_variation.csv"
    ),
    index=False
)

feature_signal_df.to_csv(
    os.path.join(
        DIAG_DIR,
        "candidate_relative_feature_signal.csv"
    ),
    index=False
)

subgroup_df.to_csv(
    os.path.join(
        DIAG_DIR,
        "validation_subgroup_diagnostics.csv"
    ),
    index=False
)

diagnostic_manifest = {
    "diagnostic_version":
        "afp_scorer_diagnostic_v1",

    "feature_spec_sha256":
        AFP_FEATURE_SPEC_SHA256,

    "scorer_spec_sha256":
        AFP_SCORER_SPEC_SHA256,

    "training_spec_sha256":
        AFP_TRAINING_SPEC_SHA256,

    "random_null_trials":
        RANDOM_NULL_TRIALS,

    "diagnostic_reference":
        diag_reference,

    "train_used":
        True,

    "validation_used":
        True,

    "test_used":
        False,

    "new_training_performed":
        False,

    "winner_selected":
        False,

    "final_scorer_frozen":
        False,
}

with open(
    os.path.join(
        DIAG_DIR,
        "scorer_diagnostic_manifest.json"
    ),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        diagnostic_manifest,
        f,
        indent=2
    )

# ======================================================================
# 13. Final diagnostic gate report
# ======================================================================

print("\n" + "="*90)
print("=== RQ2 CELL 9A: AFP SCORER DIAGNOSTIC GATE COMPLETE ===")
print("="*90)

print("Random null trials:", RANDOM_NULL_TRIALS)
print("TRAIN ranking evaluated: YES")
print("VALIDATION ranking evaluated: YES")
print("Within-group feature variation evaluated: YES")
print("Candidate-relative feature signal evaluated: YES")
print("Subgroup diagnostics evaluated: YES")
print("New scorer training: NO")
print("TEST data/gold used: NO")
print("Winner selected: NO")
print("Final scorer frozen: NO")
print("\nSTOP HERE.")
print("Send the diagnostic output before any scorer selection/freeze.")

Diagnostic directory: /kaggle/working/step3_rq2_dev_v1/05_scorer_diagnostics

Building WebQSP random null...


Building CWQ random null...



RANDOM-RANKING NULL — WEBQSP
  metric   mean    std   p2.5    p50  p97.5
group_ap 0.6098 0.0202 0.5725 0.6095 0.6523
     mrr 0.6458 0.0274 0.5942 0.6459 0.7015
    top1 0.4754 0.0431 0.3908 0.4713 0.5632
    hit2 0.6917 0.0361 0.6207 0.6897 0.7586
 recall2 0.4328 0.0269 0.3823 0.4320 0.4850

RANDOM-RANKING NULL — CWQ
  metric   mean    std   p2.5    p50  p97.5
group_ap 0.5257 0.0058 0.5144 0.5257 0.5370
     mrr 0.5496 0.0069 0.5357 0.5496 0.5629
    top1 0.3617 0.0103 0.3402 0.3617 0.3809
    hit2 0.5669 0.0098 0.5481 0.5666 0.5858
 recall2 0.3987 0.0083 0.3830 0.3987 0.4148

Evaluating TRAIN vs VALIDATION ranking...


100%|██████████| 36/36 [00:11<00:00,  3.06it/s]



TRAIN → VALIDATION GENERALIZATION

WEBQSP
dataset  hidden_dim          loss_name  train_group_ap  val_group_ap  gap_group_ap  train_mrr  val_mrr  gap_mrr  train_top1  val_top1  gap_top1
 webqsp          32         branch_bce          0.6353        0.6065        0.0288     0.6728   0.6448   0.0280      0.5109    0.4904    0.0204
 webqsp          32 group_balanced_bce          0.6372        0.6052        0.0320     0.6729   0.6462   0.0266      0.5102    0.4943    0.0159
 webqsp          64         branch_bce          0.6369        0.5991        0.0378     0.6748   0.6335   0.0413      0.5125    0.4751    0.0374
 webqsp          64 group_balanced_bce          0.6416        0.6064        0.0351     0.6775   0.6475   0.0300      0.5164    0.4981    0.0183
 webqsp         128         branch_bce          0.6383        0.5930        0.0453     0.6722   0.6275   0.0447      0.5081    0.4598    0.0484
 webqsp         128 group_balanced_bce          0.6412        0.6041        0.0371     0.6771

## Candidate Distinguishability & Feature Revision Gate

In [101]:
# ======================================================================
# CANDIDATE DISTINGUISHABILITY and FEATURE REVISION GATE
# ======================================================================
#
# Purpose:
#   Determine whether AFP scorer v1 is limited primarily by:
#     (A) candidate representation / feature indistinguishability, or
#     (B) training objective / scorer capacity.
#
# Diagnostics:
#   1. Exact feature-vector collisions within each decision group
#   2. All-identical frontier rate
#   3. Unique representation ratio
#   4. Positive/negative conflicts inside identical feature classes
#   5. Unavoidable positive-negative tie fraction
#   6. Representation-aware optimistic Top-1 ceiling
#   7. Raw-MID vs readable candidate groups
#   8. Candidate-wise pairwise feature discrimination
#
# IMPORTANT:
#   - TRAIN + VALIDATION only
#   - NO TEST data/gold
#   - NO scorer training
#   - NO feature modification
#   - NO final scorer selection/freeze
# ======================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd
from datetime import datetime, timezone

# ======================================================================
# 1. Hard gates
# ======================================================================

assert AFP_FEATURE_VERSION == "afp_features_v2_masked_entity_semantics"
assert AFP_FEATURE_DIM == 27
assert AFP_SCORER_VERSION == "afp_mlp_v1"
assert AFP_TRAINING_VERSION == "afp_train_v1"

required = [
    "webqsp_train_features",
    "webqsp_val_features",
    "cwq_train_features",
    "cwq_val_features",
    "webqsp_scorer_prepared",
    "cwq_scorer_prepared",
    "feature_variation_df",
    "null_comparison_df",
]

missing = [x for x in required if x not in globals()]
assert not missing, "Missing Cell 6/8/9A objects: " + ", ".join(missing)

FEATURE_REVISION_DIR = os.path.join(
    os.path.dirname(RQ2_SCORER_DIR),
    "06_feature_revision_gate"
)
os.makedirs(FEATURE_REVISION_DIR, exist_ok=True)

print("Feature revision gate directory:", FEATURE_REVISION_DIR)

# ======================================================================
# 2. Feature names
# ======================================================================

AFP_FEATURE_NAMES_V2 = [
    "sem_q_candidate",
    "sem_candidate_surface_available",
    "sem_q_current_entity",
    "sem_current_surface_available",
    "sem_q_current_relation",
    "sem_q_full_plan",
    "sem_q_remaining_suffix",
    "sem_candidate_current_relation",

    "path_q_prefix_entity_mean",
    "path_candidate_prefix_entity_mean",
    "path_prefix_surface_fraction",
    "path_candidate_repeats_entity",
    "path_candidate_occurrence_fraction",
    "path_unique_entity_ratio",
    "path_relation_repeat_fraction_before",

    "struct_log_candidate_count",
    "struct_log_unique_candidate_entities",
    "struct_log_contributing_parents",
    "struct_log_parent_fanout",
    "struct_parent_frontier_share",
    "struct_log_endpoint_multiplicity",
    "struct_endpoint_frontier_share",
    "struct_duplicate_endpoint_ratio",

    "prog_hop_fraction",
    "prog_remaining_fraction",
    "prog_log_plan_length",
    "prog_penultimate_indicator",
]

assert len(AFP_FEATURE_NAMES_V2) == 27

# Candidate-surface availability feature
CANDIDATE_SURFACE_INDEX = 1

# ======================================================================
# 3. Stable feature signature
# ======================================================================
#
# Features are deterministic float32 values.
# We round to 7 decimals only to avoid meaningless floating-point noise.
#
# This is NOT approximate clustering.
# It is used only to identify effectively identical Feature-v2 vectors.
# ======================================================================

FEATURE_SIGNATURE_DECIMALS = 7

def feature_signature(x):
    x = np.asarray(x, dtype=np.float64)
    return tuple(
        np.round(
            x,
            FEATURE_SIGNATURE_DECIMALS
        ).tolist()
    )

# ======================================================================
# 4. Analyze one decision group
# ======================================================================

def analyze_group_representation(Xg, yg):
    Xg = np.asarray(Xg, dtype=np.float32)
    yg = np.asarray(yg, dtype=np.uint8)

    n = len(yg)
    n_pos = int(yg.sum())
    n_neg = n - n_pos

    assert n > 1
    assert n_pos >= 1

    classes = {}

    for i in range(n):
        sig = feature_signature(Xg[i])

        if sig not in classes:
            classes[sig] = []

        classes[sig].append(i)

    n_unique = len(classes)
    all_identical = (n_unique == 1)

    mixed_classes = 0
    mixed_candidates = 0

    unavoidable_tie_pairs = 0

    class_positive_rates = []

    for indices in classes.values():
        labels = yg[indices]

        p = int(labels.sum())
        q = len(labels) - p

        class_positive_rates.append(
            p / len(labels)
        )

        # Identical feature vector contains BOTH labels.
        if p > 0 and q > 0:
            mixed_classes += 1
            mixed_candidates += len(labels)

        # Every positive-negative pair inside the same exact feature
        # class is impossible for a deterministic scorer to order.
        unavoidable_tie_pairs += p * q

    total_pos_neg_pairs = n_pos * n_neg

    if total_pos_neg_pairs > 0:
        unavoidable_pair_tie_fraction = (
            unavoidable_tie_pairs
            / total_pos_neg_pairs
        )
    else:
        unavoidable_pair_tie_fraction = 0.0

    # --------------------------------------------------------------
    # Optimistic representation-aware Top-1 ceiling
    # --------------------------------------------------------------
    #
    # Suppose an oracle knew which REPRESENTATION CLASS was best,
    # but still could not distinguish candidates sharing the exact
    # same vector.
    #
    # Within the selected class, expected Top-1 success is its
    # positive fraction.
    #
    # This therefore quantifies ambiguity induced by the representation.
    # --------------------------------------------------------------
    optimistic_top1_ceiling = max(
        class_positive_rates
    )

    return {
        "n_candidates": n,
        "n_positive": n_pos,
        "n_negative": n_neg,
        "n_unique_vectors": n_unique,
        "unique_vector_ratio": n_unique / n,
        "all_identical": int(all_identical),
        "mixed_label_classes": mixed_classes,
        "has_label_conflict": int(mixed_classes > 0),
        "mixed_class_candidates": mixed_candidates,
        "mixed_candidate_fraction": mixed_candidates / n,
        "unavoidable_tie_pairs": unavoidable_tie_pairs,
        "total_pos_neg_pairs": total_pos_neg_pairs,
        "unavoidable_pair_tie_fraction":
            unavoidable_pair_tie_fraction,
        "optimistic_top1_representation_ceiling":
            optimistic_top1_ceiling,
    }

# ======================================================================
# 5. Candidate-surface subgroup
# ======================================================================

def candidate_surface_group(Xg):
    availability = Xg[:, CANDIDATE_SURFACE_INDEX]

    if np.all(availability < 0.5):
        return "all_raw_mid"

    if np.all(availability >= 0.5):
        return "all_readable"

    return "mixed"

# ======================================================================
# 6. Analyze complete feature dataset
# ======================================================================

def analyze_dataset_representation(
    dataset_name,
    split,
    features
):
    X = np.asarray(
        features["X"],
        dtype=np.float32
    )

    y = np.asarray(
        features["y"],
        dtype=np.uint8
    )

    ptr = np.asarray(
        features["group_ptr"],
        dtype=np.int64
    )

    hops = np.asarray(
        features["group_hop"],
        dtype=np.int64
    )

    sizes = np.asarray(
        features["group_candidate_count"],
        dtype=np.int64
    )

    rows = []

    for g in range(len(ptr) - 1):
        s = int(ptr[g])
        e = int(ptr[g + 1])

        Xg = X[s:e]
        yg = y[s:e]

        r = analyze_group_representation(
            Xg,
            yg
        )

        r.update({
            "dataset": dataset_name,
            "split": split,
            "group_index": g,
            "hop": int(hops[g]),
            "candidate_size": int(sizes[g]),
            "surface_group":
                candidate_surface_group(Xg),
        })

        rows.append(r)

    return pd.DataFrame(rows)

# ======================================================================
# 7. Run representation diagnostics
# ======================================================================

representation_frames = []

for dataset_name, split, data in [
    ("webqsp", "train", webqsp_train_features),
    ("webqsp", "validation", webqsp_val_features),
    ("cwq", "train", cwq_train_features),
    ("cwq", "validation", cwq_val_features),
]:
    print(
        f"Analyzing {dataset_name.upper()} {split}..."
    )

    df = analyze_dataset_representation(
        dataset_name,
        split,
        data
    )

    representation_frames.append(df)

representation_df = pd.concat(
    representation_frames,
    ignore_index=True
)

# ======================================================================
# 8. Aggregate representation diagnostics
# ======================================================================

def aggregate_representation(df):
    total_pos_neg_pairs = df[
        "total_pos_neg_pairs"
    ].sum()

    total_unavoidable = df[
        "unavoidable_tie_pairs"
    ].sum()

    weighted_pair_tie_fraction = (
        total_unavoidable / total_pos_neg_pairs
        if total_pos_neg_pairs > 0
        else 0.0
    )

    return {
        "n_groups": len(df),

        "all_identical_groups":
            int(df["all_identical"].sum()),

        "all_identical_rate":
            float(df["all_identical"].mean()),

        "groups_with_label_conflict":
            int(df["has_label_conflict"].sum()),

        "label_conflict_rate":
            float(df["has_label_conflict"].mean()),

        "mean_unique_vector_ratio":
            float(df["unique_vector_ratio"].mean()),

        "median_unique_vector_ratio":
            float(df["unique_vector_ratio"].median()),

        "mean_mixed_candidate_fraction":
            float(df["mixed_candidate_fraction"].mean()),

        "weighted_unavoidable_pair_tie_fraction":
            float(weighted_pair_tie_fraction),

        "mean_optimistic_top1_ceiling":
            float(
                df[
                    "optimistic_top1_representation_ceiling"
                ].mean()
            ),
    }

representation_summary_rows = []

for (dataset_name, split), sub in representation_df.groupby(
    ["dataset", "split"]
):
    stats = aggregate_representation(sub)

    stats.update({
        "dataset": dataset_name,
        "split": split,
    })

    representation_summary_rows.append(stats)

representation_summary_df = pd.DataFrame(
    representation_summary_rows
)

print("\n" + "="*110)
print("FEATURE-v2 REPRESENTATION DISTINGUISHABILITY")
print("="*110)

print(
    representation_summary_df[
        [
            "dataset",
            "split",
            "n_groups",
            "all_identical_rate",
            "label_conflict_rate",
            "mean_unique_vector_ratio",
            "weighted_unavoidable_pair_tie_fraction",
            "mean_optimistic_top1_ceiling",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ======================================================================
# 9. Representation diagnostics by candidate-surface availability
# ======================================================================

surface_summary_rows = []

for (
    dataset_name,
    split,
    surface_group
), sub in representation_df.groupby(
    [
        "dataset",
        "split",
        "surface_group"
    ]
):
    stats = aggregate_representation(sub)

    stats.update({
        "dataset": dataset_name,
        "split": split,
        "surface_group": surface_group,
    })

    surface_summary_rows.append(stats)

surface_summary_df = pd.DataFrame(
    surface_summary_rows
)

print("\n" + "="*110)
print("REPRESENTATION DISTINGUISHABILITY BY CANDIDATE SURFACE")
print("="*110)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()} VALIDATION")

    sub = surface_summary_df[
        (surface_summary_df["dataset"] == dataset_name)
        &
        (surface_summary_df["split"] == "validation")
    ]

    cols = [
        "surface_group",
        "n_groups",
        "all_identical_rate",
        "label_conflict_rate",
        "mean_unique_vector_ratio",
        "weighted_unavoidable_pair_tie_fraction",
        "mean_optimistic_top1_ceiling",
    ]

    print(
        sub[cols].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

# ======================================================================
# 10. Candidate-wise pairwise feature discrimination
# ======================================================================
#
# This replaces the misleading centered-AUC interpretation for
# group-constant features.
#
# For each feature and each feasible decision group:
#
#   compare every positive candidate with every negative candidate.
#
# We count:
#   positive feature > negative feature
#   positive feature < negative feature
#   exact tie
#
# Then report:
#
#   best_orientation_accuracy
#
# A feature that never varies within groups will have:
#   tie_rate = 1.0
#
# and cannot be mistaken for useful ranking signal.
# ======================================================================

def pairwise_feature_signal(
    X,
    y,
    group_ptr,
    feature_names,
    tol=1e-8
):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.uint8)
    ptr = np.asarray(group_ptr, dtype=np.int64)

    rows = []

    for j, feature_name in enumerate(feature_names):
        higher = 0
        lower = 0
        ties = 0
        varying_groups = 0
        total_groups = len(ptr) - 1

        for g in range(total_groups):
            s = int(ptr[g])
            e = int(ptr[g + 1])

            xg = X[s:e, j]
            yg = y[s:e]

            if (
                float(xg.max() - xg.min())
                > tol
            ):
                varying_groups += 1

            pos = xg[yg == 1]
            neg = xg[yg == 0]

            if len(pos) == 0 or len(neg) == 0:
                continue

            diff = (
                pos[:, None]
                -
                neg[None, :]
            )

            higher += int(
                np.sum(diff > tol)
            )

            lower += int(
                np.sum(diff < -tol)
            )

            ties += int(
                np.sum(np.abs(diff) <= tol)
            )

        total_pairs = higher + lower + ties

        if total_pairs == 0:
            best_accuracy = np.nan
            tie_rate = np.nan
            direction = "none"
        else:
            # Ties count as 0.5 because the feature cannot order them.
            higher_acc = (
                higher + 0.5 * ties
            ) / total_pairs

            lower_acc = (
                lower + 0.5 * ties
            ) / total_pairs

            best_accuracy = max(
                higher_acc,
                lower_acc
            )

            tie_rate = (
                ties / total_pairs
            )

            if higher_acc > lower_acc:
                direction = "higher"
            elif lower_acc > higher_acc:
                direction = "lower"
            else:
                direction = "tie"

        rows.append({
            "feature_index": j,
            "feature": feature_name,
            "varying_groups": varying_groups,
            "n_groups": total_groups,
            "varying_group_rate":
                varying_groups / total_groups,
            "positive_gt_negative_pairs": higher,
            "positive_lt_negative_pairs": lower,
            "tie_pairs": ties,
            "total_pairs": total_pairs,
            "pairwise_tie_rate": tie_rate,
            "best_orientation_accuracy":
                best_accuracy,
            "preferred_direction": direction,
        })

    return pd.DataFrame(rows)

pairwise_signal_frames = []

for dataset_name, split, data in [
    ("webqsp", "train", webqsp_train_features),
    ("webqsp", "validation", webqsp_val_features),
    ("cwq", "train", cwq_train_features),
    ("cwq", "validation", cwq_val_features),
]:
    sig = pairwise_feature_signal(
        data["X"],
        data["y"],
        data["group_ptr"],
        AFP_FEATURE_NAMES_V2
    )

    sig.insert(
        0,
        "split",
        split
    )

    sig.insert(
        0,
        "dataset",
        dataset_name
    )

    pairwise_signal_frames.append(sig)

pairwise_signal_df = pd.concat(
    pairwise_signal_frames,
    ignore_index=True
)

print("\n" + "="*110)
print("CANDIDATE-WISE PAIRWISE FEATURE SIGNAL — VALIDATION")
print("="*110)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    sub = pairwise_signal_df[
        (pairwise_signal_df["dataset"] == dataset_name)
        &
        (pairwise_signal_df["split"] == "validation")
    ].sort_values(
        [
            "best_orientation_accuracy",
            "varying_group_rate"
        ],
        ascending=False
    )

    print(
        sub[
            [
                "feature_index",
                "feature",
                "varying_group_rate",
                "pairwise_tie_rate",
                "best_orientation_accuracy",
                "preferred_direction",
            ]
        ].head(15).to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

# ======================================================================
# 11. Candidate-size representation breakdown
# ======================================================================

def size_bin(x):
    if x <= 4:
        return "2-4"
    if x <= 8:
        return "5-8"
    if x <= 16:
        return "9-16"
    return "17+"

validation_repr = representation_df[
    representation_df["split"] == "validation"
].copy()

validation_repr["size_bin"] = (
    validation_repr[
        "candidate_size"
    ].map(size_bin)
)

size_summary_rows = []

for (
    dataset_name,
    bin_name
), sub in validation_repr.groupby(
    ["dataset", "size_bin"]
):
    stats = aggregate_representation(sub)

    stats.update({
        "dataset": dataset_name,
        "size_bin": bin_name,
    })

    size_summary_rows.append(stats)

size_summary_df = pd.DataFrame(
    size_summary_rows
)

print("\n" + "="*110)
print("REPRESENTATION DISTINGUISHABILITY BY FRONTIER SIZE")
print("="*110)

for dataset_name in ["webqsp", "cwq"]:
    print(f"\n{dataset_name.upper()}")

    sub = size_summary_df[
        size_summary_df["dataset"]
        == dataset_name
    ]

    print(
        sub[
            [
                "size_bin",
                "n_groups",
                "all_identical_rate",
                "label_conflict_rate",
                "mean_unique_vector_ratio",
                "weighted_unavoidable_pair_tie_fraction",
                "mean_optimistic_top1_ceiling",
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

# ======================================================================
# 12. Representation-limited frontier counts
# ======================================================================

print("\n" + "="*100)
print("VALIDATION REPRESENTATION-LIMITED FRONTIERS")
print("="*100)

revision_gate_summary = {}

for dataset_name in ["webqsp", "cwq"]:
    sub = validation_repr[
        validation_repr["dataset"] == dataset_name
    ]

    n = len(sub)

    all_identical = int(
        sub["all_identical"].sum()
    )

    conflicting = int(
        sub["has_label_conflict"].sum()
    )

    raw = sub[
        sub["surface_group"] == "all_raw_mid"
    ]

    raw_identical = int(
        raw["all_identical"].sum()
    )

    revision_gate_summary[dataset_name] = {
        "n_validation_groups": n,

        "all_identical_groups":
            all_identical,

        "all_identical_rate":
            all_identical / n,

        "label_conflict_groups":
            conflicting,

        "label_conflict_rate":
            conflicting / n,

        "all_raw_mid_groups":
            len(raw),

        "all_raw_mid_rate":
            len(raw) / n,

        "all_raw_mid_identical_groups":
            raw_identical,

        "all_raw_mid_identical_rate":
            (
                raw_identical / len(raw)
                if len(raw) > 0
                else 0.0
            ),

        "weighted_unavoidable_pair_tie_fraction":
            aggregate_representation(
                sub
            )[
                "weighted_unavoidable_pair_tie_fraction"
            ],
    }

    print(f"\n{dataset_name.upper()}")
    print(
        "Validation decision groups:       ",
        n
    )
    print(
        "All-identical Feature-v2 groups:  ",
        f"{all_identical}/{n} "
        f"({100*all_identical/n:.2f}%)"
    )
    print(
        "Groups with +/- feature conflict: ",
        f"{conflicting}/{n} "
        f"({100*conflicting/n:.2f}%)"
    )
    print(
        "All-raw-MID groups:               ",
        f"{len(raw)}/{n} "
        f"({100*len(raw)/n:.2f}%)"
    )

    if len(raw) > 0:
        print(
            "Raw-MID groups all-identical:    ",
            f"{raw_identical}/{len(raw)} "
            f"({100*raw_identical/len(raw):.2f}%)"
        )

# ======================================================================
# 13. Scientific interpretation gate
# ======================================================================
#
# This is deliberately descriptive.
# We DO NOT automatically redesign AFP from arbitrary thresholds.
#
# Instead we identify which of three situations the evidence supports:
#
#   REPRESENTATION-LIMITED
#   MIXED
#   OBJECTIVE/MODEL-LIMITED
#
# The printed label is a diagnostic heuristic, not a statistical test.
# ======================================================================

def diagnostic_gate_label(stats):
    identical = stats["all_identical_rate"]
    raw = stats["all_raw_mid_rate"]
    ties = stats[
        "weighted_unavoidable_pair_tie_fraction"
    ]

    if identical >= 0.50 and raw >= 0.50:
        return "REPRESENTATION-LIMITED"

    if identical <= 0.20 and ties <= 0.20:
        return "OBJECTIVE/MODEL-LIMITED OR MIXED"

    return "MIXED / PARTLY REPRESENTATION-LIMITED"

gate_labels = {}

print("\n" + "="*100)
print("FEATURE REVISION GATE")
print("="*100)

for dataset_name in ["webqsp", "cwq"]:
    label = diagnostic_gate_label(
        revision_gate_summary[
            dataset_name
        ]
    )

    gate_labels[dataset_name] = label

    print(
        f"{dataset_name.upper()}: {label}"
    )

print(
    "\nNOTE: These labels are diagnostic heuristics, "
    "not hypothesis-test results."
)

# ======================================================================
# 14. Save diagnostics
# ======================================================================

representation_df.to_csv(
    os.path.join(
        FEATURE_REVISION_DIR,
        "group_representation_diagnostics.csv"
    ),
    index=False
)

representation_summary_df.to_csv(
    os.path.join(
        FEATURE_REVISION_DIR,
        "representation_summary.csv"
    ),
    index=False
)

surface_summary_df.to_csv(
    os.path.join(
        FEATURE_REVISION_DIR,
        "representation_by_candidate_surface.csv"
    ),
    index=False
)

pairwise_signal_df.to_csv(
    os.path.join(
        FEATURE_REVISION_DIR,
        "pairwise_feature_signal.csv"
    ),
    index=False
)

size_summary_df.to_csv(
    os.path.join(
        FEATURE_REVISION_DIR,
        "representation_by_frontier_size.csv"
    ),
    index=False
)

# ======================================================================
# 15. Hash helper
# ======================================================================

def sha256_file_gate(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()

summary_path = os.path.join(
    FEATURE_REVISION_DIR,
    "representation_summary.csv"
)

signal_path = os.path.join(
    FEATURE_REVISION_DIR,
    "pairwise_feature_signal.csv"
)

# ======================================================================
# 16. Manifest
# ======================================================================

gate_manifest = {
    "gate_version":
        "afp_feature_revision_gate_v1",

    "feature_version":
        AFP_FEATURE_VERSION,

    "feature_spec_sha256":
        AFP_FEATURE_SPEC_SHA256,

    "scorer_version":
        AFP_SCORER_VERSION,

    "training_version":
        AFP_TRAINING_VERSION,

    "signature_decimals":
        FEATURE_SIGNATURE_DECIMALS,

    "datasets":
        ["webqsp", "cwq"],

    "splits_used":
        ["train", "validation"],

    "test_used":
        False,

    "new_training_performed":
        False,

    "feature_spec_modified":
        False,

    "winner_selected":
        False,

    "final_scorer_frozen":
        False,

    "gate_labels":
        gate_labels,

    "validation_summary":
        revision_gate_summary,

    "representation_summary_sha256":
        sha256_file_gate(summary_path),

    "pairwise_signal_sha256":
        sha256_file_gate(signal_path),

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

manifest_path = os.path.join(
    FEATURE_REVISION_DIR,
    "feature_revision_gate_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        gate_manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ======================================================================
# 17. Final gate report
# ======================================================================

print("\n" + "="*92)
print("=== RQ2 CELL 9B: CANDIDATE DISTINGUISHABILITY GATE COMPLETE ===")
print("="*92)

print("Feature version examined:", AFP_FEATURE_VERSION)
print("Exact/effective feature collisions examined: YES")
print("Positive-negative representation conflicts examined: YES")
print("Unavoidable pairwise ties examined: YES")
print("Raw MID vs readable groups examined: YES")
print("Pairwise candidate-specific feature signal examined: YES")
print("New features added: NO")
print("New scorer training: NO")
print("TEST data/gold used: NO")
print("Winner selected: NO")
print("Final scorer frozen: NO")

print("\nDiagnostic gate:")
print("  WebQSP:", gate_labels["webqsp"])
print("  CWQ:   ", gate_labels["cwq"])

print("\nSTOP HERE.")
print(
    "Send the full Cell 9B output before changing Feature-v2, "
    "retraining, or freezing a scorer."
)

Feature revision gate directory: /kaggle/working/step3_rq2_dev_v1/06_feature_revision_gate
Analyzing WEBQSP train...
Analyzing WEBQSP validation...
Analyzing CWQ train...
Analyzing CWQ validation...

FEATURE-v2 REPRESENTATION DISTINGUISHABILITY
dataset      split  n_groups  all_identical_rate  label_conflict_rate  mean_unique_vector_ratio  weighted_unavoidable_pair_tie_fraction  mean_optimistic_top1_ceiling
    cwq      train     15937              0.5022               0.4873                    0.5897                                  0.3320                        0.6387
    cwq validation      1352              0.4771               0.4453                    0.6254                                  0.2233                        0.6798
 webqsp      train      1457              0.6205               0.4935                    0.5132                                  0.6511                        0.6870
 webqsp validation        87              0.7701               0.6437                    0.

In [1]:
# ======================================================================
# RQ2 ARTIFACT RELOAD AFTER KAGGLE SESSION / ACCELERATOR RESTART
# ======================================================================
#
# Restores:
#   - Feature-v2 TRAIN + VALIDATION NPZ files
#   - Feature manifests
#   - Train-only standardizers
#   - All scorer-v1 checkpoints/results
#   - Cell 9A scorer diagnostics
#   - Cell 9B representation diagnostics
#   - Core AFP constants/classes
#
# Does NOT:
#   - regenerate plans
#   - rebuild features
#   - retrain scorers
#   - use test data
# ======================================================================

from pathlib import Path
import os
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_NAME = "step3_rq2_dev_v1"
WORK_ROOT = Path("/kaggle/working") / PROJECT_NAME

# ======================================================================
# 1. Find saved artifact root
# ======================================================================

def find_saved_project():
    # Best case: /kaggle/working survived restart.
    if WORK_ROOT.exists():
        return WORK_ROOT

    # Otherwise look inside mounted Kaggle inputs.
    input_root = Path("/kaggle/input")

    if input_root.exists():
        matches = [
            p for p in input_root.rglob(PROJECT_NAME)
            if p.is_dir()
        ]

        if matches:
            print("Found saved project under Kaggle input:")
            for p in matches:
                print(" ", p)
            return matches[0]

    raise FileNotFoundError(
        "\nCould not find the saved RQ2 artifacts.\n\n"
        "If you changed accelerator and /kaggle/working was cleared:\n"
        "1. Add your latest saved Kaggle notebook VERSION/OUTPUT as an Input.\n"
        "2. Then rerun this cell.\n\n"
        f"Expected folder: {PROJECT_NAME}"
    )

SOURCE_ROOT = find_saved_project()

print("Artifact source:", SOURCE_ROOT)

# ======================================================================
# 2. If source is read-only /kaggle/input, copy back to /kaggle/working
# ======================================================================

if SOURCE_ROOT.resolve() != WORK_ROOT.resolve():
    print("\nRestoring saved artifacts to /kaggle/working...")

    WORK_ROOT.mkdir(parents=True, exist_ok=True)

    shutil.copytree(
        SOURCE_ROOT,
        WORK_ROOT,
        dirs_exist_ok=True
    )

    RQ2_ROOT = WORK_ROOT
else:
    RQ2_ROOT = SOURCE_ROOT

print("Active project root:", RQ2_ROOT)

# ======================================================================
# 3. Restore directory variables
# ======================================================================

RQ2_FEATURE_DIR = str(RQ2_ROOT / "03_features")
RQ2_SCORER_DIR = str(RQ2_ROOT / "04_scorer")
DIAG_DIR = str(RQ2_ROOT / "05_scorer_diagnostics")
FEATURE_REVISION_DIR = str(RQ2_ROOT / "06_feature_revision_gate")

assert Path(RQ2_FEATURE_DIR).exists()
assert Path(RQ2_SCORER_DIR).exists()
assert Path(DIAG_DIR).exists()
assert Path(FEATURE_REVISION_DIR).exists()

# ======================================================================
# 4. Helpers
# ======================================================================

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()

def load_feature_npz(path):
    z = np.load(path, allow_pickle=False)

    required = [
        "X",
        "y",
        "group_ptr",
        "group_source_index",
        "group_hop",
        "group_plan_length",
        "group_candidate_count",
    ]

    missing = [k for k in required if k not in z.files]
    assert not missing, f"Missing NPZ arrays: {missing}"

    return {
        k: z[k]
        for k in required
    }

# ======================================================================
# 5. Locate feature files
# ======================================================================

FEATURE_FILES = {
    "webqsp_train":
        Path(RQ2_FEATURE_DIR) /
        "webqsp/webqsp_train_afp_features_v2.npz",

    "webqsp_validation":
        Path(RQ2_FEATURE_DIR) /
        "webqsp/webqsp_validation_afp_features_v2.npz",

    "cwq_train":
        Path(RQ2_FEATURE_DIR) /
        "cwq/cwq_train_afp_features_v2.npz",

    "cwq_validation":
        Path(RQ2_FEATURE_DIR) /
        "cwq/cwq_validation_afp_features_v2.npz",
}

FEATURE_MANIFEST_FILES = {
    "webqsp_train":
        Path(RQ2_FEATURE_DIR) /
        "webqsp/webqsp_train_afp_features_v2_manifest.json",

    "webqsp_validation":
        Path(RQ2_FEATURE_DIR) /
        "webqsp/webqsp_validation_afp_features_v2_manifest.json",

    "cwq_train":
        Path(RQ2_FEATURE_DIR) /
        "cwq/cwq_train_afp_features_v2_manifest.json",

    "cwq_validation":
        Path(RQ2_FEATURE_DIR) /
        "cwq/cwq_validation_afp_features_v2_manifest.json",
}

for path in list(FEATURE_FILES.values()) + list(FEATURE_MANIFEST_FILES.values()):
    assert path.exists(), f"Missing artifact: {path}"

# ======================================================================
# 6. Reload feature manifests
# ======================================================================

webqsp_train_feature_manifest = load_json(
    FEATURE_MANIFEST_FILES["webqsp_train"]
)

webqsp_val_feature_manifest = load_json(
    FEATURE_MANIFEST_FILES["webqsp_validation"]
)

cwq_train_feature_manifest = load_json(
    FEATURE_MANIFEST_FILES["cwq_train"]
)

cwq_val_feature_manifest = load_json(
    FEATURE_MANIFEST_FILES["cwq_validation"]
)

# ======================================================================
# 7. Restore frozen feature constants
# ======================================================================

AFP_FEATURE_VERSION = webqsp_train_feature_manifest["feature_version"]
AFP_FEATURE_SPEC_SHA256 = webqsp_train_feature_manifest["feature_spec_sha256"]
AFP_FEATURE_DIM = int(webqsp_train_feature_manifest["feature_dim"])
AFP_SEMANTIC_ENCODER_NAME = webqsp_train_feature_manifest["semantic_encoder"]

assert AFP_FEATURE_VERSION == "afp_features_v2_masked_entity_semantics"
assert AFP_FEATURE_DIM == 27

EXPECTED_FEATURE_SHA = (
    "738985d1232a8ac5935c397ed95eca233"
    "77bc59b4a99494547fa7779062ade86"
)

assert AFP_FEATURE_SPEC_SHA256 == EXPECTED_FEATURE_SHA

# ======================================================================
# 8. Verify feature-file hashes BEFORE loading
# ======================================================================

for key in FEATURE_FILES:
    expected = {
        "webqsp_train": webqsp_train_feature_manifest,
        "webqsp_validation": webqsp_val_feature_manifest,
        "cwq_train": cwq_train_feature_manifest,
        "cwq_validation": cwq_val_feature_manifest,
    }[key]["npz_sha256"]

    actual = sha256_file(FEATURE_FILES[key])

    assert actual == expected, (
        f"{key}: NPZ SHA256 mismatch!\n"
        f"expected={expected}\n"
        f"actual={actual}"
    )

print("\nFeature artifact SHA256 gates: PASSED")

# ======================================================================
# 9. Reload feature datasets
# ======================================================================

webqsp_train_features = load_feature_npz(
    FEATURE_FILES["webqsp_train"]
)

webqsp_val_features = load_feature_npz(
    FEATURE_FILES["webqsp_validation"]
)

cwq_train_features = load_feature_npz(
    FEATURE_FILES["cwq_train"]
)

cwq_val_features = load_feature_npz(
    FEATURE_FILES["cwq_validation"]
)

# ======================================================================
# 10. Restore lightweight scorer definition
# ======================================================================

AFP_SCORER_VERSION = "afp_mlp_v1"
AFP_INPUT_DIM = AFP_FEATURE_DIM
AFP_HIDDEN_CANDIDATES = [32, 64, 128]
AFP_DROPOUT = 0.0

class AFPScorer(nn.Module):
    def __init__(
        self,
        input_dim=27,
        hidden_dim=64,
        dropout=0.0
    ):
        super().__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.dropout_p = dropout

        self.fc1 = nn.Linear(
            input_dim,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            1
        )

        self.dropout = (
            nn.Dropout(dropout)
            if dropout > 0
            else nn.Identity()
        )

    def forward(self, x):
        h = F.relu(
            self.fc1(x)
        )

        h = self.dropout(h)

        return self.fc2(
            h
        ).squeeze(-1)

    @torch.no_grad()
    def predict_proba(self, x):
        return torch.sigmoid(
            self.forward(x)
        )

# ======================================================================
# 11. Restore standardizer definition
# ======================================================================

class AFPFeatureStandardizer:
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mean_ = None
        self.std_ = None
        self.fitted = False

    def transform(self, X):
        assert self.fitted

        X = np.asarray(
            X,
            dtype=np.float32
        )

        Z = (
            X - self.mean_
        ) / self.std_

        assert np.all(np.isfinite(Z))

        return Z.astype(np.float32)

    def load_state_dict(self, state):
        self.mean_ = np.asarray(
            state["mean"],
            dtype=np.float32
        )

        self.std_ = np.asarray(
            state["std"],
            dtype=np.float32
        )

        self.eps = float(
            state["eps"]
        )

        self.fitted = True
        return self

# ======================================================================
# 12. Reload train-only standardizers
# ======================================================================

def load_standardizer(path):
    payload = load_json(path)

    assert payload["fit_split"] == "train"
    assert payload["feature_spec_sha256"] == AFP_FEATURE_SPEC_SHA256

    obj = AFPFeatureStandardizer()
    obj.load_state_dict(
        payload["state"]
    )

    return obj, payload

webqsp_standardizer, webqsp_standardizer_manifest = load_standardizer(
    Path(RQ2_SCORER_DIR) /
    "webqsp_train_standardizer.json"
)

cwq_standardizer, cwq_standardizer_manifest = load_standardizer(
    Path(RQ2_SCORER_DIR) /
    "cwq_train_standardizer.json"
)

# ======================================================================
# 13. Reconstruct standardized scorer datasets
# ======================================================================

webqsp_scorer_prepared = {
    "X_train":
        webqsp_standardizer.transform(
            webqsp_train_features["X"]
        ),

    "y_train":
        webqsp_train_features["y"].astype(np.float32),

    "train_group_ptr":
        webqsp_train_features["group_ptr"],

    "X_val":
        webqsp_standardizer.transform(
            webqsp_val_features["X"]
        ),

    "y_val":
        webqsp_val_features["y"].astype(np.float32),

    "val_group_ptr":
        webqsp_val_features["group_ptr"],

    "standardizer":
        webqsp_standardizer,
}

cwq_scorer_prepared = {
    "X_train":
        cwq_standardizer.transform(
            cwq_train_features["X"]
        ),

    "y_train":
        cwq_train_features["y"].astype(np.float32),

    "train_group_ptr":
        cwq_train_features["group_ptr"],

    "X_val":
        cwq_standardizer.transform(
            cwq_val_features["X"]
        ),

    "y_val":
        cwq_val_features["y"].astype(np.float32),

    "val_group_ptr":
        cwq_val_features["group_ptr"],

    "standardizer":
        cwq_standardizer,
}

# ======================================================================
# 14. Reload scorer training results / manifest
# ======================================================================

training_manifest_path = (
    Path(RQ2_SCORER_DIR) /
    "afp_scorer_training_manifest.json"
)

training_results_path = (
    Path(RQ2_SCORER_DIR) /
    "afp_scorer_training_runs.csv"
)

assert training_manifest_path.exists()
assert training_results_path.exists()

afp_training_manifest = load_json(
    training_manifest_path
)

afp_scorer_results_df = pd.read_csv(
    training_results_path
)

AFP_TRAINING_VERSION = (
    afp_training_manifest[
        "training_version"
    ]
)

AFP_TRAINING_SPEC_SHA256 = (
    afp_training_manifest[
        "training_spec_sha256"
    ]
)

AFP_SCORER_SPEC_SHA256 = (
    afp_training_manifest[
        "scorer_spec_sha256"
    ]
)

assert len(
    afp_scorer_results_df
) == 36

# ======================================================================
# 15. Repair checkpoint paths after restore
# ======================================================================
#
# CSV contains old /kaggle/working paths.
# Rebuild paths by checkpoint filename so this also works when restored
# from a saved Kaggle version.
# ======================================================================

def repair_checkpoint_path(old_path):
    filename = Path(
        str(old_path)
    ).name

    new_path = (
        Path(RQ2_SCORER_DIR) /
        filename
    )

    assert new_path.exists(), (
        f"Missing checkpoint: {new_path}"
    )

    return str(new_path)

afp_scorer_results_df[
    "checkpoint"
] = afp_scorer_results_df[
    "checkpoint"
].apply(
    repair_checkpoint_path
)

checkpoint_files = list(
    Path(RQ2_SCORER_DIR).glob(
        "*.pt"
    )
)

print(
    "Scorer checkpoints found:",
    len(checkpoint_files)
)

assert len(checkpoint_files) >= 36

# ======================================================================
# 16. Safe checkpoint loader
# ======================================================================

AFP_DEVICE = "cpu"

def load_afp_checkpoint(
    path,
    device="cpu"
):
    try:
        ckpt = torch.load(
            path,
            map_location=device,
            weights_only=False
        )
    except TypeError:
        ckpt = torch.load(
            path,
            map_location=device
        )

    assert (
        ckpt["feature_spec_sha256"]
        == AFP_FEATURE_SPEC_SHA256
    )

    model = AFPScorer(
        input_dim=AFP_FEATURE_DIM,
        hidden_dim=int(
            ckpt["hidden_dim"]
        ),
        dropout=AFP_DROPOUT
    ).to(device)

    model.load_state_dict(
        ckpt["model_state_dict"]
    )

    model.eval()

    return model, ckpt

# ======================================================================
# 17. Reload Cell 9A diagnostics
# ======================================================================

diagnostic_files = {
    "null_comparison":
        Path(DIAG_DIR) /
        "scorer_vs_random_null.csv",

    "generalization_runs":
        Path(DIAG_DIR) /
        "train_validation_generalization_runs.csv",

    "generalization_aggregate":
        Path(DIAG_DIR) /
        "train_validation_generalization_aggregate.csv",

    "feature_variation":
        Path(DIAG_DIR) /
        "within_group_feature_variation.csv",

    "feature_signal":
        Path(DIAG_DIR) /
        "candidate_relative_feature_signal.csv",

    "subgroups":
        Path(DIAG_DIR) /
        "validation_subgroup_diagnostics.csv",
}

for path in diagnostic_files.values():
    assert path.exists(), f"Missing Cell 9A artifact: {path}"

null_comparison_df = pd.read_csv(
    diagnostic_files["null_comparison"]
)

generalization_df = pd.read_csv(
    diagnostic_files["generalization_runs"]
)

generalization_agg = pd.read_csv(
    diagnostic_files["generalization_aggregate"]
)

feature_variation_df = pd.read_csv(
    diagnostic_files["feature_variation"]
)

feature_signal_df = pd.read_csv(
    diagnostic_files["feature_signal"]
)

subgroup_df = pd.read_csv(
    diagnostic_files["subgroups"]
)

# ======================================================================
# 18. Reload Cell 9B diagnostics
# ======================================================================

revision_files = {
    "representation":
        Path(FEATURE_REVISION_DIR) /
        "group_representation_diagnostics.csv",

    "summary":
        Path(FEATURE_REVISION_DIR) /
        "representation_summary.csv",

    "surface":
        Path(FEATURE_REVISION_DIR) /
        "representation_by_candidate_surface.csv",

    "pairwise":
        Path(FEATURE_REVISION_DIR) /
        "pairwise_feature_signal.csv",

    "size":
        Path(FEATURE_REVISION_DIR) /
        "representation_by_frontier_size.csv",

    "manifest":
        Path(FEATURE_REVISION_DIR) /
        "feature_revision_gate_manifest.json",
}

for path in revision_files.values():
    assert path.exists(), f"Missing Cell 9B artifact: {path}"

representation_df = pd.read_csv(
    revision_files["representation"]
)

representation_summary_df = pd.read_csv(
    revision_files["summary"]
)

surface_summary_df = pd.read_csv(
    revision_files["surface"]
)

pairwise_signal_df = pd.read_csv(
    revision_files["pairwise"]
)

size_summary_df = pd.read_csv(
    revision_files["size"]
)

feature_revision_gate_manifest = load_json(
    revision_files["manifest"]
)

# ======================================================================
# 19. Structural sanity gates
# ======================================================================

assert webqsp_train_features["X"].shape == (18437, 27)
assert webqsp_val_features["X"].shape == (966, 27)

assert cwq_train_features["X"].shape == (218544, 27)
assert cwq_val_features["X"].shape == (18688, 27)

assert len(webqsp_train_features["group_ptr"]) - 1 == 1457
assert len(webqsp_val_features["group_ptr"]) - 1 == 87

assert len(cwq_train_features["group_ptr"]) - 1 == 15937
assert len(cwq_val_features["group_ptr"]) - 1 == 1352

assert feature_revision_gate_manifest["test_used"] is False
assert feature_revision_gate_manifest["winner_selected"] is False

# ======================================================================
# 20. Final report
# ======================================================================

print("\n" + "=" * 86)
print("=== RQ2 DEVELOPMENT ARTIFACTS SUCCESSFULLY RELOADED ===")
print("=" * 86)

print("Project root:", RQ2_ROOT)
print("Device:", AFP_DEVICE)

print("\nFeature-v2")
print("  dimension:      ", AFP_FEATURE_DIM)
print("  SHA256:         ", AFP_FEATURE_SPEC_SHA256[:16] + "...")

print("\nWEBQSP")
print("  train branches: ", len(webqsp_train_features["y"]))
print("  train groups:   ", len(webqsp_train_features["group_ptr"]) - 1)
print("  val branches:   ", len(webqsp_val_features["y"]))
print("  val groups:     ", len(webqsp_val_features["group_ptr"]) - 1)

print("\nCWQ")
print("  train branches: ", len(cwq_train_features["y"]))
print("  train groups:   ", len(cwq_train_features["group_ptr"]) - 1)
print("  val branches:   ", len(cwq_val_features["y"]))
print("  val groups:     ", len(cwq_val_features["group_ptr"]) - 1)

print("\nScorer")
print("  training runs:  ", len(afp_scorer_results_df))
print("  checkpoints:    ", len(checkpoint_files))

print("\nDiagnostics")
print("  Cell 9A: LOADED")
print("  Cell 9B: LOADED")
print(
    "  WebQSP gate:",
    feature_revision_gate_manifest["gate_labels"]["webqsp"]
)
print(
    "  CWQ gate:   ",
    feature_revision_gate_manifest["gate_labels"]["cwq"]
)

print("\nTEST DATA/GOLD LOADED: NO")
print("FINAL SCORER FROZEN: NO")
print("\nReady for the adaptive-selector stage.")

Artifact source: /kaggle/working/step3_rq2_dev_v1
Active project root: /kaggle/working/step3_rq2_dev_v1

Feature artifact SHA256 gates: PASSED
Scorer checkpoints found: 36

=== RQ2 DEVELOPMENT ARTIFACTS SUCCESSFULLY RELOADED ===
Project root: /kaggle/working/step3_rq2_dev_v1
Device: cpu

Feature-v2
  dimension:       27
  SHA256:          738985d1232a8ac5...

WEBQSP
  train branches:  18437
  train groups:    1457
  val branches:    966
  val groups:      87

CWQ
  train branches:  218544
  train groups:    15937
  val branches:    18688
  val groups:      1352

Scorer
  training runs:   36
  checkpoints:     36

Diagnostics
  Cell 9A: LOADED
  Cell 9B: LOADED
  WebQSP gate: REPRESENTATION-LIMITED
  CWQ gate:    MIXED / PARTLY REPRESENTATION-LIMITED

TEST DATA/GOLD LOADED: NO
FINAL SCORER FROZEN: NO

Ready for the adaptive-selector stage.


## Tie-Aware Scorer Validation and checkpoint selection/freeze

In [3]:

# WHY TIE-AWARE:
#   Cell 9B established extensive representation collisions.
#   Ordinary stable-sort AP/MRR/Top1 can depend on arbitrary candidate
#   ordering when logits are tied.
#
# Therefore config selection now uses EXPECTED ranking performance under
# random ordering inside tied-score blocks.
#
# Selection:
#   1. Evaluate all 36 SAVED checkpoints on VALIDATION only.
#   2. Aggregate each (hidden_dim, loss) across seeds 42/43/44.
#   3. Select by:
#        a) highest mean tie-aware Group AP
#        b) highest mean tie-aware MRR
#        c) highest mean tie-aware Top-1 Hit
#        d) lowest mean Group-Balanced BCE
#   4. Deployment seed remains FIXED at 42.
#
# IMPORTANT:
#   - NO new training.
#   - NO test data/gold.
#   - NO seed cherry-picking.
#   - Feature-v2 remains frozen.
# ======================================================================

import os
import json
import math
import shutil
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# ======================================================================
# 1. Hard gates
# ======================================================================

assert AFP_FEATURE_VERSION == "afp_features_v2_masked_entity_semantics"
assert AFP_FEATURE_DIM == 27
assert AFP_SCORER_VERSION == "afp_mlp_v1"
assert AFP_TRAINING_VERSION == "afp_train_v1"
assert len(afp_scorer_results_df) == 36

assert feature_revision_gate_manifest["test_used"] is False
assert feature_revision_gate_manifest["winner_selected"] is False

DEPLOYMENT_SEED = 42

# Treat only numerically indistinguishable logits as tied.
TIE_ATOL = 1e-8

print("Deployment seed:", DEPLOYMENT_SEED)
print("Tie tolerance:  ", TIE_ATOL)

# ======================================================================
# 2. Checkpoint-specific standardization
# ======================================================================

def transform_with_checkpoint_standardizer(X, ckpt):
    state = ckpt["standardizer_state"]

    mean = np.asarray(
        state["mean"],
        dtype=np.float32
    )

    std = np.asarray(
        state["std"],
        dtype=np.float32
    )

    X = np.asarray(
        X,
        dtype=np.float32
    )

    Z = (X - mean) / std

    assert Z.shape[1] == AFP_FEATURE_DIM
    assert np.all(np.isfinite(Z))

    return Z.astype(np.float32)

# ======================================================================
# 3. Build score-tie blocks
# ======================================================================

def make_tie_blocks(logits, atol=TIE_ATOL):
    """
    Sort candidates by descending logit and group numerically tied scores.

    Returns list of arrays containing original candidate indices.
    """
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    order = np.argsort(
        -logits,
        kind="mergesort"
    )

    blocks = []
    current = [int(order[0])]
    reference = float(logits[order[0]])

    for idx in order[1:]:
        value = float(logits[idx])

        if abs(value - reference) <= atol:
            current.append(int(idx))
        else:
            blocks.append(
                np.asarray(current, dtype=np.int64)
            )

            current = [int(idx)]
            reference = value

    blocks.append(
        np.asarray(current, dtype=np.int64)
    )

    return blocks

# ======================================================================
# 4. Expected Top-1 under random tie-breaking
# ======================================================================

def expected_top1_from_blocks(blocks, labels):
    labels = np.asarray(
        labels,
        dtype=np.uint8
    )

    top_block = blocks[0]

    return float(
        labels[top_block].mean()
    )

# ======================================================================
# 5. Expected reciprocal rank of first positive
# ======================================================================

def expected_mrr_from_blocks(blocks, labels):
    """
    Exact expectation under uniform random ordering within each tie block.
    """
    labels = np.asarray(
        labels,
        dtype=np.uint8
    )

    offset = 0

    for block in blocks:
        block_y = labels[block]

        n = len(block_y)
        p = int(block_y.sum())

        if p == 0:
            offset += n
            continue

        # First block containing at least one positive.
        denominator = math.comb(n, p)

        expected_rr = 0.0

        # If p positives are randomly placed among n positions,
        # probability first positive occurs at local rank k:
        #
        # C(n-k, p-1) / C(n, p)
        #
        max_first_rank = n - p + 1

        for k in range(1, max_first_rank + 1):
            probability = (
                math.comb(n - k, p - 1)
                / denominator
            )

            expected_rr += (
                probability
                / (offset + k)
            )

        return float(expected_rr)

    raise AssertionError(
        "Feasible decision group has no positive candidate."
    )

# ======================================================================
# 6. Expected AP under random tie-breaking
# ======================================================================

def expected_ap_from_blocks(blocks, labels):
    """
    Exact expected Average Precision under uniform random ordering
    inside tied-score blocks.

    No Monte-Carlo approximation is used.
    """
    labels = np.asarray(
        labels,
        dtype=np.uint8
    )

    total_positive = int(
        labels.sum()
    )

    assert total_positive > 0

    offset = 0
    positives_before = 0
    expected_precision_sum = 0.0

    for block in blocks:
        block_y = labels[block]

        n = len(block_y)
        p = int(block_y.sum())

        if p == 0:
            offset += n
            continue

        for local_rank in range(1, n + 1):
            # P(position local_rank is positive)
            py = p / n

            # E[Y_r * K_r]
            #
            # K_r = number of positives in this block up to position r.
            if n == 1:
                ey_times_k = 1.0
            else:
                ey_times_k = (
                    py
                    +
                    (local_rank - 1)
                    * p
                    * (p - 1)
                    / (n * (n - 1))
                )

            expected_numerator = (
                positives_before * py
                + ey_times_k
            )

            global_rank = (
                offset + local_rank
            )

            expected_precision_sum += (
                expected_numerator
                / global_rank
            )

        offset += n
        positives_before += p

    return float(
        expected_precision_sum
        / total_positive
    )

# ======================================================================
# 7. Complete tie-aware group ranking evaluator
# ======================================================================

def tie_aware_group_ranking_metrics(
    logits,
    labels,
    group_ptr,
    atol=TIE_ATOL
):
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    labels = np.asarray(
        labels,
        dtype=np.uint8
    )

    ptr = np.asarray(
        group_ptr,
        dtype=np.int64
    )

    assert len(logits) == len(labels)
    assert ptr[0] == 0
    assert ptr[-1] == len(labels)

    aps = []
    mrrs = []
    top1s = []

    groups_with_ties = 0
    fully_tied_groups = 0

    for g in range(len(ptr) - 1):
        s = int(ptr[g])
        e = int(ptr[g + 1])

        group_logits = logits[s:e]
        group_labels = labels[s:e]

        assert len(group_labels) > 1
        assert group_labels.sum() >= 1

        blocks = make_tie_blocks(
            group_logits,
            atol=atol
        )

        if any(len(b) > 1 for b in blocks):
            groups_with_ties += 1

        if len(blocks) == 1:
            fully_tied_groups += 1

        aps.append(
            expected_ap_from_blocks(
                blocks,
                group_labels
            )
        )

        mrrs.append(
            expected_mrr_from_blocks(
                blocks,
                group_labels
            )
        )

        top1s.append(
            expected_top1_from_blocks(
                blocks,
                group_labels
            )
        )

    n_groups = len(ptr) - 1

    return {
        "tie_aware_group_ap":
            float(np.mean(aps)),

        "tie_aware_mrr":
            float(np.mean(mrrs)),

        "tie_aware_top1":
            float(np.mean(top1s)),

        "groups_with_score_ties":
            int(groups_with_ties),

        "score_tie_group_rate":
            float(groups_with_ties / n_groups),

        "fully_tied_score_groups":
            int(fully_tied_groups),

        "fully_tied_score_group_rate":
            float(fully_tied_groups / n_groups),
    }

# ======================================================================
# 8. BCE metrics
# ======================================================================

def bce_metrics(logits, y, group_ptr):
    logits_t = torch.as_tensor(
        logits,
        dtype=torch.float32
    )

    y_t = torch.as_tensor(
        y,
        dtype=torch.float32
    )

    branch_bce = float(
        F.binary_cross_entropy_with_logits(
            logits_t,
            y_t,
            reduction="mean"
        )
    )

    ptr = np.asarray(
        group_ptr,
        dtype=np.int64
    )

    group_losses = []

    for g in range(len(ptr) - 1):
        s = int(ptr[g])
        e = int(ptr[g + 1])

        group_losses.append(
            F.binary_cross_entropy_with_logits(
                logits_t[s:e],
                y_t[s:e],
                reduction="mean"
            )
        )

    group_bce = float(
        torch.stack(
            group_losses
        ).mean()
    )

    return branch_bce, group_bce

# ======================================================================
# 9. Evaluate one saved checkpoint
# ======================================================================

@torch.no_grad()
def evaluate_saved_run(row):
    dataset_name = row["dataset"]

    if dataset_name == "webqsp":
        raw_val = webqsp_val_features
    elif dataset_name == "cwq":
        raw_val = cwq_val_features
    else:
        raise ValueError(dataset_name)

    model, ckpt = load_afp_checkpoint(
        row["checkpoint"],
        device="cpu"
    )

    assert ckpt["dataset"] == dataset_name
    assert int(ckpt["seed"]) == int(row["seed"])
    assert (
        ckpt["feature_spec_sha256"]
        == AFP_FEATURE_SPEC_SHA256
    )

    X_val = transform_with_checkpoint_standardizer(
        raw_val["X"],
        ckpt
    )

    X_t = torch.as_tensor(
        X_val,
        dtype=torch.float32
    )

    logits = model(
        X_t
    ).cpu().numpy()

    y = raw_val[
        "y"
    ].astype(np.uint8)

    ptr = raw_val[
        "group_ptr"
    ]

    branch_bce, group_bce = bce_metrics(
        logits,
        y,
        ptr
    )

    ranking = tie_aware_group_ranking_metrics(
        logits,
        y,
        ptr,
        atol=TIE_ATOL
    )

    return {
        "dataset":
            dataset_name,

        "hidden_dim":
            int(row["hidden_dim"]),

        "loss_name":
            row["loss_name"],

        "seed":
            int(row["seed"]),

        "branch_bce":
            branch_bce,

        "group_bce":
            group_bce,

        **ranking,

        "checkpoint":
            row["checkpoint"],
    }

# ======================================================================
# 10. Re-evaluate all 36 checkpoints
# ======================================================================

print("\nRe-evaluating all 36 checkpoints with tie-aware metrics...")

tie_aware_rows = []

for i, row in afp_scorer_results_df.iterrows():
    result = evaluate_saved_run(row)

    tie_aware_rows.append(
        result
    )

    print(
        f"[{i+1:02d}/36] "
        f"{result['dataset']} "
        f"H={result['hidden_dim']} "
        f"{result['loss_name']} "
        f"seed={result['seed']} | "
        f"AP={result['tie_aware_group_ap']:.4f} "
        f"MRR={result['tie_aware_mrr']:.4f} "
        f"Top1={result['tie_aware_top1']:.4f}"
    )

tie_aware_results_df = pd.DataFrame(
    tie_aware_rows
)

# ======================================================================
# 11. Aggregate configurations over fixed seeds
# ======================================================================

tie_aware_config_summary = (
    tie_aware_results_df
    .groupby(
        [
            "dataset",
            "hidden_dim",
            "loss_name",
        ]
    )
    .agg(
        n_seeds=(
            "seed",
            "count"
        ),

        group_ap_mean=(
            "tie_aware_group_ap",
            "mean"
        ),

        group_ap_std=(
            "tie_aware_group_ap",
            "std"
        ),

        mrr_mean=(
            "tie_aware_mrr",
            "mean"
        ),

        mrr_std=(
            "tie_aware_mrr",
            "std"
        ),

        top1_mean=(
            "tie_aware_top1",
            "mean"
        ),

        top1_std=(
            "tie_aware_top1",
            "std"
        ),

        group_bce_mean=(
            "group_bce",
            "mean"
        ),

        branch_bce_mean=(
            "branch_bce",
            "mean"
        ),

        score_tie_rate_mean=(
            "score_tie_group_rate",
            "mean"
        ),
    )
    .reset_index()
)

assert np.all(
    tie_aware_config_summary[
        "n_seeds"
    ] == 3
)

# ======================================================================
# 12. Deterministic configuration selection
# ======================================================================

def rank_configs(dataset_name):
    sub = tie_aware_config_summary[
        tie_aware_config_summary[
            "dataset"
        ] == dataset_name
    ].copy()

    assert len(sub) == 6

    sub = sub.sort_values(
        by=[
            "group_ap_mean",
            "mrr_mean",
            "top1_mean",
            "group_bce_mean",
            "hidden_dim",
        ],
        ascending=[
            False,
            False,
            False,
            True,
            True,
        ],
        kind="mergesort"
    ).reset_index(
        drop=True
    )

    return sub

webqsp_ranked_configs = rank_configs(
    "webqsp"
)

cwq_ranked_configs = rank_configs(
    "cwq"
)

# ======================================================================
# 13. Print tie-aware validation rankings
# ======================================================================

def print_ranked(dataset_name, ranked):
    print("\n" + "=" * 110)
    print(
        f"{dataset_name.upper()} "
        "TIE-AWARE VALIDATION CONFIGURATION RANKING"
    )
    print("=" * 110)

    cols = [
        "hidden_dim",
        "loss_name",
        "group_ap_mean",
        "group_ap_std",
        "mrr_mean",
        "top1_mean",
        "group_bce_mean",
        "score_tie_rate_mean",
    ]

    print(
        ranked[cols].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

print_ranked(
    "webqsp",
    webqsp_ranked_configs
)

print_ranked(
    "cwq",
    cwq_ranked_configs
)

# ======================================================================
# 14. Select configuration, NOT seed
# ======================================================================

webqsp_selected_config = (
    webqsp_ranked_configs.iloc[0]
)

cwq_selected_config = (
    cwq_ranked_configs.iloc[0]
)

def fixed_seed_run(
    dataset_name,
    selected_config
):
    sub = tie_aware_results_df[
        (tie_aware_results_df["dataset"] == dataset_name)
        &
        (
            tie_aware_results_df["hidden_dim"]
            ==
            int(
                selected_config["hidden_dim"]
            )
        )
        &
        (
            tie_aware_results_df["loss_name"]
            ==
            selected_config["loss_name"]
        )
        &
        (
            tie_aware_results_df["seed"]
            ==
            DEPLOYMENT_SEED
        )
    ]

    assert len(sub) == 1

    return sub.iloc[0]

webqsp_deployment_run = fixed_seed_run(
    "webqsp",
    webqsp_selected_config
)

cwq_deployment_run = fixed_seed_run(
    "cwq",
    cwq_selected_config
)

# ======================================================================
# 15. Save selected development checkpoints
# ======================================================================

FINAL_SCORER_DIR = (
    Path(RQ2_ROOT)
    / "07_final_scorer"
)

FINAL_SCORER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

def sha256_file_final(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()

def copy_selected_checkpoint(
    dataset_name,
    deployment_run
):
    source = Path(
        deployment_run[
            "checkpoint"
        ]
    )

    assert source.exists()

    destination = (
        FINAL_SCORER_DIR
        / f"{dataset_name}_afp_scorer_selected.pt"
    )

    shutil.copy2(
        source,
        destination
    )

    return (
        str(destination),
        sha256_file_final(destination)
    )

(
    webqsp_best_checkpoint,
    webqsp_best_checkpoint_sha256
) = copy_selected_checkpoint(
    "webqsp",
    webqsp_deployment_run
)

(
    cwq_best_checkpoint,
    cwq_best_checkpoint_sha256
) = copy_selected_checkpoint(
    "cwq",
    cwq_deployment_run
)

# ======================================================================
# 16. Save tie-aware evaluation tables
# ======================================================================

tie_aware_runs_file = (
    FINAL_SCORER_DIR
    / "tie_aware_validation_runs.csv"
)

tie_aware_configs_file = (
    FINAL_SCORER_DIR
    / "tie_aware_validation_configurations.csv"
)

tie_aware_results_df.to_csv(
    tie_aware_runs_file,
    index=False
)

tie_aware_config_summary.to_csv(
    tie_aware_configs_file,
    index=False
)

# ======================================================================
# 17. Build selection record
# ======================================================================

def build_selection_record(
    dataset_name,
    selected_config,
    deployment_run,
    checkpoint_path,
    checkpoint_sha
):
    return {
        "dataset":
            dataset_name,

        "selected_hidden_dim":
            int(
                selected_config[
                    "hidden_dim"
                ]
            ),

        "selected_loss":
            selected_config[
                "loss_name"
            ],

        "selection_seeds":
            [42, 43, 44],

        "deployment_seed":
            DEPLOYMENT_SEED,

        "mean_tie_aware_group_ap":
            float(
                selected_config[
                    "group_ap_mean"
                ]
            ),

        "std_tie_aware_group_ap":
            float(
                selected_config[
                    "group_ap_std"
                ]
            ),

        "mean_tie_aware_mrr":
            float(
                selected_config[
                    "mrr_mean"
                ]
            ),

        "mean_tie_aware_top1":
            float(
                selected_config[
                    "top1_mean"
                ]
            ),

        "mean_group_bce":
            float(
                selected_config[
                    "group_bce_mean"
                ]
            ),

        "deployment_tie_aware_group_ap":
            float(
                deployment_run[
                    "tie_aware_group_ap"
                ]
            ),

        "deployment_tie_aware_mrr":
            float(
                deployment_run[
                    "tie_aware_mrr"
                ]
            ),

        "deployment_tie_aware_top1":
            float(
                deployment_run[
                    "tie_aware_top1"
                ]
            ),

        "deployment_group_bce":
            float(
                deployment_run[
                    "group_bce"
                ]
            ),

        "checkpoint":
            checkpoint_path,

        "checkpoint_sha256":
            checkpoint_sha,
    }

webqsp_scorer_selection = build_selection_record(
    "webqsp",
    webqsp_selected_config,
    webqsp_deployment_run,
    webqsp_best_checkpoint,
    webqsp_best_checkpoint_sha256
)

cwq_scorer_selection = build_selection_record(
    "cwq",
    cwq_selected_config,
    cwq_deployment_run,
    cwq_best_checkpoint,
    cwq_best_checkpoint_sha256
)

# ======================================================================
# 18. Selection manifest
# ======================================================================

selection_manifest = {
    "selection_version":
        "afp_scorer_selection_v2_tie_aware",

    "feature_version":
        AFP_FEATURE_VERSION,

    "feature_spec_sha256":
        AFP_FEATURE_SPEC_SHA256,

    "scorer_version":
        AFP_SCORER_VERSION,

    "training_version":
        AFP_TRAINING_VERSION,

    "selection_metric_revision": {
        "reason":
            "Extensive representation/score ties discovered by "
            "Cells 9A-9B make ordinary stable-sort ranking metrics "
            "candidate-order dependent.",

        "tie_handling":
            "Exact expected metric under uniform random ordering "
            "within numerically tied score blocks.",

        "tie_atol":
            TIE_ATOL,

        "new_training":
            False,
    },

    "selection_policy": [
        "highest mean tie-aware Group AP",
        "highest mean tie-aware MRR",
        "highest mean tie-aware Top-1",
        "lowest mean Group-Balanced BCE",
    ],

    "configuration_selection_seeds":
        [42, 43, 44],

    "deployment_seed":
        DEPLOYMENT_SEED,

    "webqsp":
        webqsp_scorer_selection,

    "cwq":
        cwq_scorer_selection,

    "feature_revision":
        False,

    "test_used_for_selection":
        False,

    "test_metrics_observed":
        False,

    "complete_afp_frozen":
        False,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

selection_manifest_path = (
    FINAL_SCORER_DIR
    / "afp_scorer_selection_manifest.json"
)

with open(
    selection_manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        selection_manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ======================================================================
# 19. Final report
# ======================================================================

def print_selection(dataset_name, selection):
    print("\n" + "-" * 78)
    print(dataset_name.upper())
    print("-" * 78)

    print(
        "Selected H:              ",
        selection[
            "selected_hidden_dim"
        ]
    )

    print(
        "Selected loss:           ",
        selection[
            "selected_loss"
        ]
    )

    print(
        "Mean tie-aware Group AP: ",
        f"{selection['mean_tie_aware_group_ap']:.4f}"
    )

    print(
        "Mean tie-aware MRR:      ",
        f"{selection['mean_tie_aware_mrr']:.4f}"
    )

    print(
        "Mean tie-aware Top-1:    ",
        f"{selection['mean_tie_aware_top1']:.4f}"
    )

    print(
        "Deployment seed:         ",
        selection[
            "deployment_seed"
        ]
    )

    print(
        "Seed-42 tie-aware AP:    ",
        f"{selection['deployment_tie_aware_group_ap']:.4f}"
    )

    print(
        "Checkpoint SHA256:       ",
        selection[
            "checkpoint_sha256"
        ][:16] + "..."
    )

print_selection(
    "WebQSP",
    webqsp_scorer_selection
)

print_selection(
    "CWQ",
    cwq_scorer_selection
)

print("\n" + "=" * 92)
print("=== RQ2 CELL 9C: TIE-AWARE SCORER VALIDATION + SELECTION COMPLETE ===")
print("=" * 92)

print("All 36 saved checkpoints revalidated: YES")
print("Tie-aware ranking used for selection: YES")
print("Configuration aggregated over 3 seeds: YES")
print("Validation-selected seed:             NO")
print("Fixed deployment seed:                42")
print("New scorer training:                  NO")
print("Feature-v2 modified:                  NO")
print("TEST data/gold used:                  NO")
print("TEST metrics observed:                NO")
print("Selected scorer checkpoints saved:    YES")
print("Complete AFP configuration frozen:    NO")
print()
print("Next: adaptive selector definition + validation tuning.")

Deployment seed: 42
Tie tolerance:   1e-08

Re-evaluating all 36 checkpoints with tie-aware metrics...
[01/36] webqsp H=32 branch_bce seed=42 | AP=0.6122 MRR=0.6480 Top1=0.4841
[02/36] webqsp H=32 branch_bce seed=43 | AP=0.6071 MRR=0.6386 Top1=0.4713
[03/36] webqsp H=32 branch_bce seed=44 | AP=0.6048 MRR=0.6364 Top1=0.4495
[04/36] webqsp H=32 group_balanced_bce seed=42 | AP=0.6024 MRR=0.6359 Top1=0.4604
[05/36] webqsp H=32 group_balanced_bce seed=43 | AP=0.6178 MRR=0.6557 Top1=0.4956
[06/36] webqsp H=32 group_balanced_bce seed=44 | AP=0.6004 MRR=0.6369 Top1=0.4611
[07/36] webqsp H=64 branch_bce seed=42 | AP=0.6069 MRR=0.6415 Top1=0.4725
[08/36] webqsp H=64 branch_bce seed=43 | AP=0.5941 MRR=0.6162 Top1=0.4259
[09/36] webqsp H=64 branch_bce seed=44 | AP=0.6003 MRR=0.6307 Top1=0.4598
[10/36] webqsp H=64 group_balanced_bce seed=42 | AP=0.6135 MRR=0.6501 Top1=0.4833
[11/36] webqsp H=64 group_balanced_bce seed=43 | AP=0.5952 MRR=0.6314 Top1=0.4603
[12/36] webqsp H=64 group_balanced_bce seed

## Implement AFP confidence-aware adaptive selector

In [4]:
# ======================================================================
# 3.10 DEFINE ADAPTIVE AFP SELECTOR
# ======================================================================
#
# pi_i    = softmax(logit_i / T)
# u_h     = normalized entropy(pi)
# gamma_h = gamma_min + u_h * (1 - gamma_min)
#
# B_h = smallest cumulative top-B probability mass reaching gamma_h.
#
# Safeguards:
#   - singleton -> retain all
#   - final hop -> retain all
#   - fully tied logits -> retain all
#   - score tie at pruning boundary -> preserve whole tied class
#
# NO tuning here.
# NO graph traversal here.
# NO test data.
# ======================================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np

# ======================================================================
# 1. Selector specification
# ======================================================================

AFP_SELECTOR_VERSION = "afp_adaptive_selector_v1"

AFP_SELECTOR_SPEC = {
    "version": AFP_SELECTOR_VERSION,
    "probability": "softmax(logits/T)",
    "uncertainty": "normalized_entropy",
    "gamma": "gamma_min + u*(1-gamma_min)",
    "budget": "minimum cumulative probability mass",
    "singleton_bypass": True,
    "final_hop_protection": True,
    "all_score_ties_retain_all": True,
    "cutoff_tie_expansion": True,
    "temperature_validation_tuned": True,
    "gamma_min_validation_tuned": True,
    "test_used_for_tuning": False,
}

AFP_SELECTOR_SPEC_SHA256 = hashlib.sha256(
    json.dumps(
        AFP_SELECTOR_SPEC,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

print("Selector version:", AFP_SELECTOR_VERSION)
print("Selector SHA256:", AFP_SELECTOR_SPEC_SHA256[:16] + "...")

# ======================================================================
# 2. Scorer invocation policy
# ======================================================================

def afp_should_score(candidate_count, hop, plan_length):
    assert candidate_count >= 0
    assert plan_length >= 1
    assert 0 <= hop < plan_length

    if candidate_count <= 1:
        return False

    if hop == plan_length - 1:
        return False

    return True

# ======================================================================
# 3. Stable temperature-scaled softmax
# ======================================================================

def afp_softmax(logits, temperature):
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    assert logits.ndim == 1
    assert len(logits) >= 1
    assert np.all(np.isfinite(logits))
    assert temperature > 0

    z = logits / float(temperature)
    z -= np.max(z)

    exp_z = np.exp(z)
    probs = exp_z / exp_z.sum()

    assert np.all(np.isfinite(probs))
    assert np.all(probs >= 0)
    assert np.isclose(probs.sum(), 1.0)

    return probs

# ======================================================================
# 4. Normalized entropy
# ======================================================================

def afp_normalized_entropy(probs):
    probs = np.asarray(
        probs,
        dtype=np.float64
    )

    n = len(probs)

    if n <= 1:
        return 0.0

    assert np.all(probs >= 0)
    assert np.isclose(probs.sum(), 1.0)

    nz = probs > 0

    entropy = -np.sum(
        probs[nz] * np.log(probs[nz])
    )

    u = entropy / np.log(n)

    return float(
        np.clip(u, 0.0, 1.0)
    )

# ======================================================================
# 5. Adaptive gamma
# ======================================================================

def afp_gamma(uncertainty, gamma_min):
    assert 0.0 <= uncertainty <= 1.0
    assert 0.0 < gamma_min <= 1.0

    gamma = (
        gamma_min
        + uncertainty
        * (1.0 - gamma_min)
    )

    return float(
        np.clip(
            gamma,
            gamma_min,
            1.0
        )
    )

# ======================================================================
# 6. Adaptive selector for an intermediate decision frontier
# ======================================================================

def afp_select_from_logits(
    logits,
    temperature,
    gamma_min,
    tie_tolerance=1e-8
):
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    assert logits.ndim == 1
    assert len(logits) >= 1
    assert np.all(np.isfinite(logits))

    n = len(logits)

    # Singleton safety
    if n == 1:
        return {
            "selected_indices":
                np.asarray([0], dtype=np.int64),
            "probabilities":
                np.asarray([1.0], dtype=np.float64),
            "uncertainty": 0.0,
            "gamma": 1.0,
            "requested_B": 1,
            "retained_B": 1,
            "retained_mass": 1.0,
            "pruned_count": 0,
            "pruning_fraction": 0.0,
            "reason": "singleton_retain_all",
        }

    probs = afp_softmax(
        logits,
        temperature
    )

    uncertainty = afp_normalized_entropy(
        probs
    )

    gamma = afp_gamma(
        uncertainty,
        gamma_min
    )

    # --------------------------------------------------------------
    # Fully tied logits:
    # scorer cannot distinguish candidates -> abstain from pruning.
    # --------------------------------------------------------------
    if (
        float(logits.max() - logits.min())
        <= tie_tolerance
    ):
        return {
            "selected_indices":
                np.arange(n, dtype=np.int64),
            "probabilities": probs,
            "uncertainty": uncertainty,
            "gamma": gamma,
            "requested_B": n,
            "retained_B": n,
            "retained_mass": 1.0,
            "pruned_count": 0,
            "pruning_fraction": 0.0,
            "reason": "all_scores_tied_retain_all",
        }

    # Stable descending score order
    order = np.argsort(
        -logits,
        kind="mergesort"
    )

    ranked_probs = probs[order]
    cumulative = np.cumsum(
        ranked_probs
    )

    # Numerical safety
    cumulative[-1] = 1.0

    requested_B = int(
        np.searchsorted(
            cumulative,
            gamma,
            side="left"
        ) + 1
    )

    requested_B = min(
        requested_B,
        n
    )

    # --------------------------------------------------------------
    # Preserve score ties at the cutoff.
    # --------------------------------------------------------------
    cutoff_score = float(
        logits[
            order[
                requested_B - 1
            ]
        ]
    )

    retained_B = requested_B

    while retained_B < n:
        next_score = float(
            logits[
                order[retained_B]
            ]
        )

        if (
            abs(next_score - cutoff_score)
            <= tie_tolerance
        ):
            retained_B += 1
        else:
            break

    selected = order[
        :retained_B
    ].astype(np.int64)

    retained_mass = float(
        probs[selected].sum()
    )

    assert 1 <= requested_B <= retained_B <= n
    assert retained_mass + 1e-12 >= gamma

    return {
        "selected_indices": selected,
        "probabilities": probs,
        "uncertainty": uncertainty,
        "gamma": gamma,
        "requested_B": requested_B,
        "retained_B": retained_B,
        "retained_mass": retained_mass,
        "pruned_count": n - retained_B,
        "pruning_fraction":
            (n - retained_B) / n,
        "reason": (
            "adaptive_with_tie_expansion"
            if retained_B > requested_B
            else "adaptive"
        ),
    }

# ======================================================================
# 7. Traversal-level wrapper
# ======================================================================

def afp_select_frontier(
    logits,
    candidate_count,
    hop,
    plan_length,
    temperature,
    gamma_min,
    tie_tolerance=1e-8
):
    assert candidate_count >= 1
    assert plan_length >= 1
    assert 0 <= hop < plan_length

    # Final-hop protection
    if hop == plan_length - 1:
        return {
            "selected_indices":
                np.arange(
                    candidate_count,
                    dtype=np.int64
                ),
            "retained_B":
                candidate_count,
            "requested_B":
                candidate_count,
            "pruned_count": 0,
            "pruning_fraction": 0.0,
            "reason": "final_hop_protection",
            "scorer_invoked": False,
        }

    # Singleton bypass
    if candidate_count == 1:
        return {
            "selected_indices":
                np.asarray(
                    [0],
                    dtype=np.int64
                ),
            "retained_B": 1,
            "requested_B": 1,
            "pruned_count": 0,
            "pruning_fraction": 0.0,
            "reason": "singleton_bypass",
            "scorer_invoked": False,
        }

    assert logits is not None
    assert len(logits) == candidate_count

    result = afp_select_from_logits(
        logits=logits,
        temperature=temperature,
        gamma_min=gamma_min,
        tie_tolerance=tie_tolerance
    )

    result["scorer_invoked"] = True

    return result

# ======================================================================
# 8. Sanity A — invocation policy
# ======================================================================

assert not afp_should_score(
    candidate_count=1,
    hop=0,
    plan_length=3
)

assert not afp_should_score(
    candidate_count=5,
    hop=2,
    plan_length=3
)

assert afp_should_score(
    candidate_count=5,
    hop=1,
    plan_length=3
)

print("\nScorer invocation gate: PASSED")

# ======================================================================
# 9. Sanity B — uniform logits -> maximal uncertainty -> retain all
# ======================================================================

r = afp_select_from_logits(
    logits=[0.0, 0.0, 0.0, 0.0],
    temperature=1.0,
    gamma_min=0.70
)

assert np.isclose(
    r["uncertainty"],
    1.0
)

assert np.isclose(
    r["gamma"],
    1.0
)

assert r["retained_B"] == 4
assert r["pruned_count"] == 0

print("Uniform-score abstention gate: PASSED")

# ======================================================================
# 10. Sanity C — confident frontier can prune
# ======================================================================

r = afp_select_from_logits(
    logits=[10.0, 0.0, -1.0, -2.0],
    temperature=1.0,
    gamma_min=0.70
)

assert r["retained_B"] < 4
assert r["pruned_count"] > 0
assert 0 in r["selected_indices"]

print("Confident-pruning gate: PASSED")

# ======================================================================
# 11. Sanity D — final hop is fully protected
# ======================================================================

r = afp_select_frontier(
    logits=None,
    candidate_count=7,
    hop=2,
    plan_length=3,
    temperature=1.0,
    gamma_min=0.70
)

assert r["retained_B"] == 7
assert r["scorer_invoked"] is False
assert r["reason"] == "final_hop_protection"

print("Final-hop protection gate: PASSED")

# ======================================================================
# 12. Sanity E — cutoff score ties are not split
# ======================================================================

r = afp_select_from_logits(
    logits=[3.0, 1.0, 1.0, 1.0],
    temperature=1.0,
    gamma_min=0.50
)

if r["requested_B"] > 1:
    assert r["retained_B"] == 4

print("Cutoff-tie preservation gate: PASSED")

# ======================================================================
# 13. Sanity F — permutation equivariance
# ======================================================================

base_logits = np.asarray(
    [4.0, 2.5, 1.0, -1.0]
)

base = afp_select_from_logits(
    base_logits,
    temperature=1.0,
    gamma_min=0.70
)

perm = np.asarray(
    [2, 0, 3, 1]
)

permuted = afp_select_from_logits(
    base_logits[perm],
    temperature=1.0,
    gamma_min=0.70
)

selected_original = set(
    base["selected_indices"].tolist()
)

selected_after_permutation = set(
    perm[
        permuted["selected_indices"]
    ].tolist()
)

assert (
    selected_original
    ==
    selected_after_permutation
)

print("Permutation-equivariance gate: PASSED")

# ======================================================================
# 14. Illustrative behavior only
# ======================================================================
#
# T=1 and gamma_min=0.70 are NOT selected values here.
# ======================================================================

examples = {
    "uniform":
        [0.0, 0.0, 0.0, 0.0],

    "weak":
        [1.0, 0.9, 0.8, 0.7],

    "moderate":
        [2.0, 1.0, 0.5, 0.0],

    "strong":
        [8.0, 1.0, 0.0, -1.0],
}

print("\n" + "=" * 86)
print("ILLUSTRATIVE SELECTOR BEHAVIOR — NOT TUNED")
print("=" * 86)

for name, logits in examples.items():
    r = afp_select_from_logits(
        logits=logits,
        temperature=1.0,
        gamma_min=0.70
    )

    print(
        f"{name:<10} "
        f"u={r['uncertainty']:.4f}  "
        f"gamma={r['gamma']:.4f}  "
        f"B={r['retained_B']}/{len(logits)}  "
        f"{r['reason']}"
    )

# ======================================================================
# 15. Save definition manifest
# ======================================================================

SELECTOR_DIR = (
    Path(RQ2_ROOT)
    / "08_adaptive_selector"
)

SELECTOR_DIR.mkdir(
    parents=True,
    exist_ok=True
)

selector_manifest = {
    "selector_version":
        AFP_SELECTOR_VERSION,

    "selector_spec_sha256":
        AFP_SELECTOR_SPEC_SHA256,

    "feature_spec_sha256":
        AFP_FEATURE_SPEC_SHA256,

    "selected_scorers": {
        "webqsp":
            webqsp_best_checkpoint,

        "cwq":
            cwq_best_checkpoint,
    },

    "formula": {
        "probability":
            "softmax(logits/T)",

        "uncertainty":
            "-sum(pi*log(pi))/log(n)",

        "gamma":
            "gamma_min + u*(1-gamma_min)",

        "budget":
            "smallest cumulative top-B probability mass reaching gamma",
    },

    "safeguards": {
        "singleton_bypass": True,
        "final_hop_protection": True,
        "fully_tied_scores_retain_all": True,
        "cutoff_score_ties_preserved": True,
    },

    "temperature_selected":
        False,

    "gamma_min_selected":
        False,

    "validation_traversal_tuning_pending":
        True,

    "test_used":
        False,

    "complete_afp_frozen":
        False,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

selector_manifest_path = (
    SELECTOR_DIR
    / "adaptive_selector_definition_manifest.json"
)

with open(
    selector_manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        selector_manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ======================================================================
# 16. Final report
# ======================================================================

print("\n" + "=" * 90)
print("=== RQ2 CELL 10: ADAPTIVE AFP SELECTOR DEFINED ===")
print("=" * 90)

print("Temperature-scaled softmax:    YES")
print("Normalized entropy:            YES")
print("Adaptive gamma:                YES")
print("Cumulative-mass budget:        YES")
print("Singleton bypass:              YES")
print("Final-hop protection:          YES")
print("Fully tied scores retain all:  YES")
print("Cutoff score ties preserved:   YES")
print()
print("Temperature tuned:             NO")
print("gamma_min tuned:               NO")
print("New training:                  NO")
print("TEST data/gold used:           NO")
print("Complete AFP frozen:           NO")
print()
print("Next: validation traversal integration + selector tuning.")

Selector version: afp_adaptive_selector_v1
Selector SHA256: 62fad1e1f5869a54...

Scorer invocation gate: PASSED
Uniform-score abstention gate: PASSED
Confident-pruning gate: PASSED
Final-hop protection gate: PASSED
Cutoff-tie preservation gate: PASSED
Permutation-equivariance gate: PASSED

ILLUSTRATIVE SELECTOR BEHAVIOR — NOT TUNED
uniform    u=1.0000  gamma=1.0000  B=4/4  all_scores_tied_retain_all
weak       u=0.9955  gamma=0.9987  B=4/4  adaptive
moderate   u=0.8005  gamma=0.9402  B=4/4  adaptive
strong     u=0.0083  gamma=0.7025  B=1/4  adaptive

=== RQ2 CELL 10: ADAPTIVE AFP SELECTOR DEFINED ===
Temperature-scaled softmax:    YES
Normalized entropy:            YES
Adaptive gamma:                YES
Cumulative-mass budget:        YES
Singleton bypass:              YES
Final-hop protection:          YES
Fully tied scores retain all:  YES
Cutoff score ties preserved:   YES

Temperature tuned:             NO
gamma_min tuned:               NO
New training:                  NO
TEST data

## Implement controlled baselines — RoG, Top-\(B\), threshold, Random-\(B\), adaptive-budget random

In [5]:
# ======================================================================
# 3.11 IMPLEMENT CONTROLLED PRUNING BASELINES
# ======================================================================
#
# Methods:
#   1. RoG
#   2. Fixed Top-B
#   3. Fixed Threshold
#   4. Random-B
#   5. Adaptive-Budget Random
#   6. AFP
#
# Controlled comparison principle:
#   Everything except frontier-selection policy remains fixed.
#
# Shared safeguards:
#   - singleton frontier -> retain all
#   - final hop -> retain all
#
# IMPORTANT:
#   - No hyperparameter tuning here.
#   - No graph traversal here.
#   - No test data.
#   - Random methods use deterministic per-group seeds.
# ======================================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np

# ======================================================================
# 1. Hard gates
# ======================================================================

assert AFP_SELECTOR_VERSION == "afp_adaptive_selector_v1"
assert "afp_select_from_logits" in globals()
assert "afp_select_frontier" in globals()

CONTROLLED_METHODS = [
    "rog",
    "fixed_top_b",
    "fixed_threshold",
    "random_b",
    "adaptive_budget_random",
    "afp",
]

RANDOM_BASELINE_SEEDS = [42, 43, 44]

print("Controlled methods:", CONTROLLED_METHODS)
print("Random seeds:", RANDOM_BASELINE_SEEDS)

# ======================================================================
# 2. Baseline specification
# ======================================================================

CONTROLLED_BASELINE_VERSION = "afp_controlled_baselines_v1"

CONTROLLED_BASELINE_SPEC = {
    "version": CONTROLLED_BASELINE_VERSION,
    "methods": CONTROLLED_METHODS,
    "shared_final_hop_protection": True,
    "shared_singleton_bypass": True,

    "rog": "retain_all",

    "fixed_top_b": {
        "ranking": "AFP scorer logits",
        "budget": "fixed_B",
        "cutoff_score_ties": "expand_full_tied_class",
    },

    "fixed_threshold": {
        "score": "sigmoid(AFP scorer logit)",
        "rule": "retain score >= tau",
        "empty_frontier_allowed": True,
    },

    "random_b": {
        "budget": "fixed_B",
        "selection": "uniform_without_replacement",
        "seeds": RANDOM_BASELINE_SEEDS,
    },

    "adaptive_budget_random": {
        "budget": "AFP adaptive retained_B",
        "selection": "uniform_without_replacement",
        "seeds": RANDOM_BASELINE_SEEDS,
    },

    "afp": {
        "budget": "confidence-aware adaptive",
        "selection": "AFP scorer ranking",
    },

    "test_used": False,
}

CONTROLLED_BASELINE_SPEC_SHA256 = hashlib.sha256(
    json.dumps(
        CONTROLLED_BASELINE_SPEC,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

print("Baseline version:", CONTROLLED_BASELINE_VERSION)
print("Baseline SHA256:", CONTROLLED_BASELINE_SPEC_SHA256[:16] + "...")

# ======================================================================
# 3. Helpers
# ======================================================================

def sigmoid_np(logits):
    logits = np.asarray(logits, dtype=np.float64)

    out = np.empty_like(logits)

    positive = logits >= 0
    negative = ~positive

    out[positive] = (
        1.0 /
        (1.0 + np.exp(-logits[positive]))
    )

    exp_x = np.exp(logits[negative])
    out[negative] = exp_x / (1.0 + exp_x)

    return out


def preserve_original_candidate_order(indices):
    """
    Membership may be determined by ranking/random selection,
    but surviving candidates propagate in their ORIGINAL RoG order.

    This prevents traversal/path-order changes becoming a confound.
    """
    return np.asarray(
        sorted(
            int(i) for i in indices
        ),
        dtype=np.int64
    )


def deterministic_group_rng(base_seed, group_key):
    """
    Stable per-group RNG.

    Python's built-in hash() is intentionally NOT used because its
    value may differ across interpreter sessions.
    """
    payload = (
        f"{int(base_seed)}|{str(group_key)}"
    ).encode("utf-8")

    digest = hashlib.sha256(payload).digest()

    group_seed = int.from_bytes(
        digest[:8],
        byteorder="big",
        signed=False
    )

    return np.random.default_rng(group_seed)

# ======================================================================
# 4. RoG — no pruning
# ======================================================================

def select_rog(candidate_count):
    assert candidate_count >= 1

    selected = np.arange(
        candidate_count,
        dtype=np.int64
    )

    return {
        "selected_indices": selected,
        "requested_B": candidate_count,
        "retained_B": candidate_count,
        "pruned_count": 0,
        "pruning_fraction": 0.0,
        "reason": "rog_retain_all",
        "scorer_invoked": False,
        "budget_source": "none",
        "selection_source": "none",
    }

# ======================================================================
# 5. Fixed Top-B
# ======================================================================
#
# Top-B is determined by AFP scorer logits.
#
# IMPORTANT:
# If the B-th score is tied with later candidates, the whole tied
# score class is retained. We do not use arbitrary list order to
# decide between representation-identical candidates.
# ======================================================================

def select_fixed_top_b(
    logits,
    B,
    tie_tolerance=1e-8
):
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    assert logits.ndim == 1
    assert len(logits) >= 1
    assert np.all(np.isfinite(logits))
    assert int(B) >= 1

    n = len(logits)
    requested_B = min(int(B), n)

    if requested_B == n:
        selected = np.arange(
            n,
            dtype=np.int64
        )

        return {
            "selected_indices": selected,
            "requested_B": requested_B,
            "retained_B": n,
            "pruned_count": 0,
            "pruning_fraction": 0.0,
            "reason": "fixed_top_b_retain_all",
            "scorer_invoked": True,
            "budget_source": "fixed",
            "selection_source": "scorer",
        }

    order = np.argsort(
        -logits,
        kind="mergesort"
    )

    cutoff_score = float(
        logits[
            order[requested_B - 1]
        ]
    )

    retained_B = requested_B

    while retained_B < n:
        next_score = float(
            logits[
                order[retained_B]
            ]
        )

        if (
            abs(next_score - cutoff_score)
            <= tie_tolerance
        ):
            retained_B += 1
        else:
            break

    ranked_selected = order[:retained_B]

    # Propagate in original candidate order.
    selected = preserve_original_candidate_order(
        ranked_selected
    )

    return {
        "selected_indices": selected,
        "requested_B": requested_B,
        "retained_B": retained_B,
        "pruned_count": n - retained_B,
        "pruning_fraction": (n - retained_B) / n,
        "reason": (
            "fixed_top_b_with_tie_expansion"
            if retained_B > requested_B
            else "fixed_top_b"
        ),
        "scorer_invoked": True,
        "budget_source": "fixed",
        "selection_source": "scorer",
    }

# ======================================================================
# 6. Fixed probability threshold
# ======================================================================
#
# s_i = sigmoid(logit_i)
#
# retain iff:
#       s_i >= tau
#
# No forced Top-1 fallback is introduced.
# If no branch passes tau, the intermediate traversal terminates.
# That behavior is part of the threshold baseline itself.
# ======================================================================

def select_fixed_threshold(
    logits,
    threshold
):
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    assert logits.ndim == 1
    assert len(logits) >= 1
    assert np.all(np.isfinite(logits))
    assert 0.0 <= threshold <= 1.0

    probs = sigmoid_np(
        logits
    )

    selected = np.flatnonzero(
        probs >= float(threshold)
    ).astype(np.int64)

    n = len(logits)
    retained_B = len(selected)

    return {
        "selected_indices": selected,
        "probabilities": probs,
        "requested_B": retained_B,
        "retained_B": retained_B,
        "pruned_count": n - retained_B,
        "pruning_fraction": (n - retained_B) / n,
        "reason": (
            "fixed_threshold"
            if retained_B > 0
            else "fixed_threshold_empty"
        ),
        "scorer_invoked": True,
        "budget_source": "threshold",
        "selection_source": "scorer",
    }

# ======================================================================
# 7. Fixed Random-B
# ======================================================================

def select_random_b(
    candidate_count,
    B,
    seed,
    group_key
):
    assert candidate_count >= 1
    assert int(B) >= 1
    assert int(seed) in RANDOM_BASELINE_SEEDS

    n = candidate_count
    retained_B = min(
        int(B),
        n
    )

    if retained_B == n:
        selected = np.arange(
            n,
            dtype=np.int64
        )
    else:
        rng = deterministic_group_rng(
            seed,
            group_key
        )

        selected = rng.choice(
            n,
            size=retained_B,
            replace=False
        )

        selected = (
            preserve_original_candidate_order(
                selected
            )
        )

    return {
        "selected_indices": selected,
        "requested_B": retained_B,
        "retained_B": retained_B,
        "pruned_count": n - retained_B,
        "pruning_fraction": (n - retained_B) / n,
        "reason": "random_b",
        "scorer_invoked": False,
        "budget_source": "fixed",
        "selection_source": "random",
        "random_seed": int(seed),
    }

# ======================================================================
# 8. Adaptive-Budget Random
# ======================================================================
#
# This baseline uses AFP ONLY to determine HOW MANY candidates should
# survive.
#
# It deliberately ignores AFP's ranking when deciding WHICH candidates
# survive.
#
# Therefore:
#
#   AFP vs Adaptive-Budget Random
#
# isolates the value of learned ranking while holding adaptive budget
# behavior approximately fixed.
# ======================================================================

def select_adaptive_budget_random(
    logits,
    temperature,
    gamma_min,
    seed,
    group_key,
    tie_tolerance=1e-8
):
    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    assert logits.ndim == 1
    assert len(logits) >= 1
    assert int(seed) in RANDOM_BASELINE_SEEDS

    n = len(logits)

    # Compute EXACT AFP budget.
    afp_reference = afp_select_from_logits(
        logits=logits,
        temperature=temperature,
        gamma_min=gamma_min,
        tie_tolerance=tie_tolerance
    )

    retained_B = int(
        afp_reference[
            "retained_B"
        ]
    )

    assert 1 <= retained_B <= n

    if retained_B == n:
        selected = np.arange(
            n,
            dtype=np.int64
        )
    else:
        rng = deterministic_group_rng(
            seed,
            group_key
        )

        selected = rng.choice(
            n,
            size=retained_B,
            replace=False
        )

        selected = (
            preserve_original_candidate_order(
                selected
            )
        )

    return {
        "selected_indices": selected,

        "requested_B":
            int(
                afp_reference[
                    "requested_B"
                ]
            ),

        "retained_B":
            retained_B,

        "pruned_count":
            n - retained_B,

        "pruning_fraction":
            (n - retained_B) / n,

        "uncertainty":
            afp_reference[
                "uncertainty"
            ],

        "gamma":
            afp_reference[
                "gamma"
            ],

        "retained_mass":
            afp_reference[
                "retained_mass"
            ],

        "afp_budget_reason":
            afp_reference[
                "reason"
            ],

        "reason":
            "adaptive_budget_random",

        "scorer_invoked":
            True,

        "budget_source":
            "afp_adaptive",

        "selection_source":
            "random",

        "random_seed":
            int(seed),
    }

# ======================================================================
# 9. AFP policy wrapper
# ======================================================================

def select_afp_policy(
    logits,
    temperature,
    gamma_min,
    tie_tolerance=1e-8
):
    result = afp_select_from_logits(
        logits=logits,
        temperature=temperature,
        gamma_min=gamma_min,
        tie_tolerance=tie_tolerance
    )

    # Preserve original RoG candidate ordering after membership selection.
    result = dict(result)

    result["selected_indices"] = (
        preserve_original_candidate_order(
            result["selected_indices"]
        )
    )

    result["scorer_invoked"] = True
    result["budget_source"] = "afp_adaptive"
    result["selection_source"] = "scorer"

    return result

# ======================================================================
# 10. Unified controlled-selection interface
# ======================================================================

def controlled_select_frontier(
    method,
    candidate_count,
    hop,
    plan_length,
    logits=None,

    # Fixed Top-B / Random-B
    B=None,

    # Fixed threshold
    threshold=None,

    # AFP / Adaptive-Budget Random
    temperature=None,
    gamma_min=None,

    # Random baselines
    seed=None,
    group_key=None,

    tie_tolerance=1e-8
):
    """
    Common selection interface for controlled validation/test traversal.

    All methods receive the SAME candidate frontier.

    The only difference is the selection policy.
    """
    assert method in CONTROLLED_METHODS
    assert candidate_count >= 1
    assert plan_length >= 1
    assert 0 <= hop < plan_length

    # --------------------------------------------------------------
    # RoG never prunes.
    # --------------------------------------------------------------
    if method == "rog":
        return select_rog(
            candidate_count
        )

    # --------------------------------------------------------------
    # Shared FINAL-HOP protection for ALL pruning baselines.
    # --------------------------------------------------------------
    if hop == plan_length - 1:
        selected = np.arange(
            candidate_count,
            dtype=np.int64
        )

        return {
            "selected_indices": selected,
            "requested_B": candidate_count,
            "retained_B": candidate_count,
            "pruned_count": 0,
            "pruning_fraction": 0.0,
            "reason": "final_hop_protection",
            "scorer_invoked": False,
            "budget_source": "protected",
            "selection_source": "protected",
        }

    # --------------------------------------------------------------
    # Shared singleton bypass.
    # --------------------------------------------------------------
    if candidate_count == 1:
        return {
            "selected_indices":
                np.asarray(
                    [0],
                    dtype=np.int64
                ),
            "requested_B": 1,
            "retained_B": 1,
            "pruned_count": 0,
            "pruning_fraction": 0.0,
            "reason": "singleton_bypass",
            "scorer_invoked": False,
            "budget_source": "bypass",
            "selection_source": "bypass",
        }

    # --------------------------------------------------------------
    # Fixed Top-B
    # --------------------------------------------------------------
    if method == "fixed_top_b":
        assert logits is not None
        assert len(logits) == candidate_count
        assert B is not None

        return select_fixed_top_b(
            logits=logits,
            B=B,
            tie_tolerance=tie_tolerance
        )

    # --------------------------------------------------------------
    # Fixed Threshold
    # --------------------------------------------------------------
    if method == "fixed_threshold":
        assert logits is not None
        assert len(logits) == candidate_count
        assert threshold is not None

        return select_fixed_threshold(
            logits=logits,
            threshold=threshold
        )

    # --------------------------------------------------------------
    # Fixed Random-B
    # --------------------------------------------------------------
    if method == "random_b":
        assert B is not None
        assert seed is not None
        assert group_key is not None

        return select_random_b(
            candidate_count=candidate_count,
            B=B,
            seed=seed,
            group_key=group_key
        )

    # --------------------------------------------------------------
    # Adaptive-Budget Random
    # --------------------------------------------------------------
    if method == "adaptive_budget_random":
        assert logits is not None
        assert len(logits) == candidate_count
        assert temperature is not None
        assert gamma_min is not None
        assert seed is not None
        assert group_key is not None

        return select_adaptive_budget_random(
            logits=logits,
            temperature=temperature,
            gamma_min=gamma_min,
            seed=seed,
            group_key=group_key,
            tie_tolerance=tie_tolerance
        )

    # --------------------------------------------------------------
    # AFP
    # --------------------------------------------------------------
    if method == "afp":
        assert logits is not None
        assert len(logits) == candidate_count
        assert temperature is not None
        assert gamma_min is not None

        return select_afp_policy(
            logits=logits,
            temperature=temperature,
            gamma_min=gamma_min,
            tie_tolerance=tie_tolerance
        )

    raise RuntimeError(
        f"Unhandled method: {method}"
    )

# ======================================================================
# 11. Sanity Gate A — RoG retains all
# ======================================================================

r = controlled_select_frontier(
    method="rog",
    candidate_count=5,
    hop=0,
    plan_length=3
)

assert r["retained_B"] == 5
assert r["pruned_count"] == 0
assert np.array_equal(
    r["selected_indices"],
    np.arange(5)
)

print("\nRoG baseline gate: PASSED")

# ======================================================================
# 12. Sanity Gate B — Fixed Top-B
# ======================================================================

r = controlled_select_frontier(
    method="fixed_top_b",
    candidate_count=5,
    hop=0,
    plan_length=3,
    logits=[5.0, 4.0, 3.0, 2.0, 1.0],
    B=2
)

assert r["requested_B"] == 2
assert r["retained_B"] == 2
assert set(
    r["selected_indices"].tolist()
) == {0, 1}

print("Fixed Top-B gate: PASSED")

# ======================================================================
# 13. Sanity Gate C — Top-B does not split score ties
# ======================================================================

r = controlled_select_frontier(
    method="fixed_top_b",
    candidate_count=4,
    hop=0,
    plan_length=3,
    logits=[4.0, 2.0, 2.0, 2.0],
    B=2
)

assert r["requested_B"] == 2
assert r["retained_B"] == 4

print("Fixed Top-B tie-preservation gate: PASSED")

# ======================================================================
# 14. Sanity Gate D — Threshold
# ======================================================================

r = controlled_select_frontier(
    method="fixed_threshold",
    candidate_count=4,
    hop=0,
    plan_length=3,
    logits=[2.0, 0.5, -1.0, -3.0],
    threshold=0.50
)

# sigmoid(2), sigmoid(.5) >= .5
# sigmoid(-1), sigmoid(-3) < .5
assert set(
    r["selected_indices"].tolist()
) == {0, 1}

print("Fixed-threshold gate: PASSED")

# ======================================================================
# 15. Sanity Gate E — Threshold may terminate frontier
# ======================================================================

r = controlled_select_frontier(
    method="fixed_threshold",
    candidate_count=3,
    hop=0,
    plan_length=3,
    logits=[-5.0, -4.0, -3.0],
    threshold=0.95
)

assert r["retained_B"] == 0
assert r["reason"] == "fixed_threshold_empty"

print("Threshold-empty gate: PASSED")

# ======================================================================
# 16. Sanity Gate F — Random-B reproducibility
# ======================================================================

r1 = controlled_select_frontier(
    method="random_b",
    candidate_count=20,
    hop=0,
    plan_length=3,
    B=4,
    seed=42,
    group_key="example-question|plan0|topic0|hop0"
)

r2 = controlled_select_frontier(
    method="random_b",
    candidate_count=20,
    hop=0,
    plan_length=3,
    B=4,
    seed=42,
    group_key="example-question|plan0|topic0|hop0"
)

assert np.array_equal(
    r1["selected_indices"],
    r2["selected_indices"]
)

assert r1["retained_B"] == 4

print("Random-B reproducibility gate: PASSED")

# ======================================================================
# 17. Sanity Gate G — random seeds actually differ
# ======================================================================

seed_sets = []

for seed in RANDOM_BASELINE_SEEDS:
    r = controlled_select_frontier(
        method="random_b",
        candidate_count=50,
        hop=0,
        plan_length=3,
        B=5,
        seed=seed,
        group_key="seed-difference-check"
    )

    seed_sets.append(
        tuple(
            r["selected_indices"].tolist()
        )
    )

assert len(
    set(seed_sets)
) > 1

print("Random-seed differentiation gate: PASSED")

# ======================================================================
# 18. Sanity Gate H — Adaptive-Budget Random uses exact AFP budget
# ======================================================================

test_logits = np.asarray(
    [8.0, 2.0, 1.0, 0.0, -1.0]
)

afp_reference = controlled_select_frontier(
    method="afp",
    candidate_count=5,
    hop=0,
    plan_length=3,
    logits=test_logits,
    temperature=1.0,
    gamma_min=0.70
)

adaptive_random = controlled_select_frontier(
    method="adaptive_budget_random",
    candidate_count=5,
    hop=0,
    plan_length=3,
    logits=test_logits,
    temperature=1.0,
    gamma_min=0.70,
    seed=42,
    group_key="adaptive-budget-check"
)

assert (
    adaptive_random["retained_B"]
    ==
    afp_reference["retained_B"]
)

assert (
    adaptive_random["requested_B"]
    ==
    afp_reference["requested_B"]
)

print("Adaptive-budget Random budget-equivalence gate: PASSED")

# ======================================================================
# 19. Sanity Gate I — final-hop protection shared by all methods
# ======================================================================

for method in CONTROLLED_METHODS:
    kwargs = {}

    if method in [
        "fixed_top_b",
        "fixed_threshold",
        "adaptive_budget_random",
        "afp",
    ]:
        kwargs["logits"] = [5.0, 1.0, -1.0]

    if method in [
        "fixed_top_b",
        "random_b",
    ]:
        kwargs["B"] = 1

    if method == "fixed_threshold":
        kwargs["threshold"] = 0.99

    if method in [
        "adaptive_budget_random",
        "afp",
    ]:
        kwargs["temperature"] = 1.0
        kwargs["gamma_min"] = 0.70

    if method in [
        "random_b",
        "adaptive_budget_random",
    ]:
        kwargs["seed"] = 42
        kwargs["group_key"] = "final-hop-test"

    r = controlled_select_frontier(
        method=method,
        candidate_count=3,
        hop=2,
        plan_length=3,
        **kwargs
    )

    assert r["retained_B"] == 3
    assert r["pruned_count"] == 0

print("Shared final-hop protection gate: PASSED")

# ======================================================================
# 20. Sanity Gate J — singleton bypass shared
# ======================================================================

for method in CONTROLLED_METHODS:
    kwargs = {}

    if method in [
        "fixed_top_b",
        "fixed_threshold",
        "adaptive_budget_random",
        "afp",
    ]:
        kwargs["logits"] = [0.0]

    if method in [
        "fixed_top_b",
        "random_b",
    ]:
        kwargs["B"] = 1

    if method == "fixed_threshold":
        kwargs["threshold"] = 0.99

    if method in [
        "adaptive_budget_random",
        "afp",
    ]:
        kwargs["temperature"] = 1.0
        kwargs["gamma_min"] = 0.70

    if method in [
        "random_b",
        "adaptive_budget_random",
    ]:
        kwargs["seed"] = 42
        kwargs["group_key"] = "singleton-test"

    r = controlled_select_frontier(
        method=method,
        candidate_count=1,
        hop=0,
        plan_length=3,
        **kwargs
    )

    assert r["retained_B"] == 1
    assert r["pruned_count"] == 0

print("Shared singleton-bypass gate: PASSED")

# ======================================================================
# 21. Show policy characteristics
# ======================================================================

policy_table = [
    {
        "method": "RoG",
        "uses_scorer": False,
        "adaptive_budget": False,
        "random_selection": False,
    },
    {
        "method": "Fixed Top-B",
        "uses_scorer": True,
        "adaptive_budget": False,
        "random_selection": False,
    },
    {
        "method": "Fixed Threshold",
        "uses_scorer": True,
        "adaptive_budget": False,
        "random_selection": False,
    },
    {
        "method": "Random-B",
        "uses_scorer": False,
        "adaptive_budget": False,
        "random_selection": True,
    },
    {
        "method": "Adaptive-Budget Random",
        "uses_scorer": True,
        "adaptive_budget": True,
        "random_selection": True,
    },
    {
        "method": "AFP",
        "uses_scorer": True,
        "adaptive_budget": True,
        "random_selection": False,
    },
]

print("\n" + "=" * 92)
print("CONTROLLED METHOD CHARACTERISTICS")
print("=" * 92)

for p in policy_table:
    print(
        f"{p['method']:<24} "
        f"scorer={str(p['uses_scorer']):<5} "
        f"adaptive_budget={str(p['adaptive_budget']):<5} "
        f"random={str(p['random_selection']):<5}"
    )

# ======================================================================
# 22. Save baseline-definition manifest
# ======================================================================

BASELINE_DIR = (
    Path(RQ2_ROOT)
    / "09_controlled_baselines"
)

BASELINE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

baseline_manifest = {
    "baseline_version":
        CONTROLLED_BASELINE_VERSION,

    "baseline_spec_sha256":
        CONTROLLED_BASELINE_SPEC_SHA256,

    "feature_spec_sha256":
        AFP_FEATURE_SPEC_SHA256,

    "selector_spec_sha256":
        AFP_SELECTOR_SPEC_SHA256,

    "scorer_selection_manifest":
        str(
            FINAL_SCORER_DIR
            / "afp_scorer_selection_manifest.json"
        ),

    "methods":
        CONTROLLED_METHODS,

    "random_seeds":
        RANDOM_BASELINE_SEEDS,

    "shared_controls": {
        "same_candidate_frontier": True,
        "same_relation_matching": True,
        "same_final_hop_protection": True,
        "same_singleton_bypass": True,
        "preserve_original_candidate_order_after_selection": True,
    },

    "fixed_top_b": {
        "hyperparameter_tuned": False,
        "cutoff_tie_expansion": True,
    },

    "fixed_threshold": {
        "hyperparameter_tuned": False,
        "empty_frontier_allowed": True,
        "forced_top1_fallback": False,
    },

    "random_b": {
        "hyperparameter_tuned": False,
        "seeds": RANDOM_BASELINE_SEEDS,
    },

    "adaptive_budget_random": {
        "temperature_tuned": False,
        "gamma_min_tuned": False,
        "budget_exactly_matches_afp": True,
    },

    "afp": {
        "temperature_tuned": False,
        "gamma_min_tuned": False,
    },

    "validation_traversal_run":
        False,

    "test_used":
        False,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

baseline_manifest_path = (
    BASELINE_DIR
    / "controlled_baseline_definition_manifest.json"
)

with open(
    baseline_manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        baseline_manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ======================================================================
# 23. Final report
# ======================================================================

print("\n" + "=" * 94)
print("=== RQ2 CELL 11: CONTROLLED BASELINE POLICIES IMPLEMENTED ===")
print("=" * 94)

print("RoG:                       YES")
print("Fixed Top-B:               YES")
print("Fixed Threshold:           YES")
print("Random-B:                  YES")
print("Adaptive-Budget Random:    YES")
print("AFP:                       YES")
print()
print("Shared final-hop protection: YES")
print("Shared singleton bypass:     YES")
print("Original candidate order:    PRESERVED after selection")
print("Random seeds:               ", RANDOM_BASELINE_SEEDS)
print()
print("Top-B tuned:                 NO")
print("Threshold tuned:             NO")
print("AFP temperature tuned:       NO")
print("AFP gamma_min tuned:         NO")
print("Validation traversal run:    NO")
print("TEST data/gold used:         NO")
print("Complete AFP frozen:         NO")
print()
print("Next: integrate all policies into the shared validation traversal.")

Controlled methods: ['rog', 'fixed_top_b', 'fixed_threshold', 'random_b', 'adaptive_budget_random', 'afp']
Random seeds: [42, 43, 44]
Baseline version: afp_controlled_baselines_v1
Baseline SHA256: 7170fa41d1bb1cd2...

RoG baseline gate: PASSED
Fixed Top-B gate: PASSED
Fixed Top-B tie-preservation gate: PASSED
Fixed-threshold gate: PASSED
Threshold-empty gate: PASSED
Random-B reproducibility gate: PASSED
Random-seed differentiation gate: PASSED
Adaptive-budget Random budget-equivalence gate: PASSED
Shared final-hop protection gate: PASSED
Shared singleton-bypass gate: PASSED

CONTROLLED METHOD CHARACTERISTICS
RoG                      scorer=False adaptive_budget=False random=False
Fixed Top-B              scorer=True  adaptive_budget=False random=False
Fixed Threshold          scorer=True  adaptive_budget=False random=False
Random-B                 scorer=False adaptive_budget=False random=True 
Adaptive-Budget Random   scorer=True  adaptive_budget=True  random=True 
AFP                

## Restore and Audit Validation Traversal Prerequisites

In [7]:
# ======================================================================
# RECOVER FROZEN VALIDATION PLANS FROM RQ1 ARTIFACTS
# ======================================================================
#
# PURPOSE
# -------
# Recover the EXACT persisted validation planning artifacts generated
# during RQ1 development.
#
# We DO NOT rerun:
#   - RoG planner
#   - MiniLM
#   - traversal
#   - tuning
#
# Instead:
#   1. Locate planning_<dataset>_validation.jsonl
#   2. Inspect its actual schema
#   3. Automatically identify the predicted-plan list field
#   4. Require exact frozen question/plan/empty-plan counts
#   5. Restore the rows + plan-access helper for Cell 12B
#
# NO TEST DATA.
# ======================================================================

import os
import json
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch


# ======================================================================
# 1. Project roots
# ======================================================================

RQ2_ROOT = Path("/kaggle/working/step3_rq2_dev_v1")

assert RQ2_ROOT.exists(), (
    f"Missing RQ2 root: {RQ2_ROOT}"
)

print("RQ2 root:", RQ2_ROOT)
print("Device:  ", "cuda" if torch.cuda.is_available() else "cpu")


# ======================================================================
# 2. Frozen validation expectations
# ======================================================================

FROZEN_VALIDATION_EXPECTATIONS = {
    "webqsp": {
        "questions": 246,
        "total_plans": 721,
        "empty_plans": 0,
        "nonempty_plans": 721,
    },

    "cwq": {
        "questions": 3519,
        "total_plans": 10536,
        "empty_plans": 7,
        "nonempty_plans": 10529,
    },
}


# ======================================================================
# 3. Locate exact RQ1 planning artifacts
# ======================================================================

def locate_rq1_validation_planning(dataset):
    filename = f"planning_{dataset}_validation.jsonl"

    preferred = [
        Path("/kaggle/working/step2_rq1_dev") / filename,
    ]

    # Known saved-notebook style location.
    input_root = Path("/kaggle/input")

    for p in preferred:
        if p.exists():
            return p

    if input_root.exists():
        matches = list(
            input_root.rglob(filename)
        )

        if matches:
            # Prefer paths containing step2_rq1_dev.
            matches = sorted(
                matches,
                key=lambda p: (
                    "step2_rq1_dev" not in str(p),
                    len(str(p))
                )
            )

            return matches[0]

    raise FileNotFoundError(
        f"Could not locate {filename}. "
        "Do NOT rerun the planner."
    )


WEBQSP_VAL_PLAN_PATH = locate_rq1_validation_planning(
    "webqsp"
)

CWQ_VAL_PLAN_PATH = locate_rq1_validation_planning(
    "cwq"
)

print("\nFrozen RQ1 planning artifacts:")
print("  WebQSP:", WEBQSP_VAL_PLAN_PATH)
print("  CWQ:   ", CWQ_VAL_PLAN_PATH)


# ======================================================================
# 4. Leakage gate
# ======================================================================

def assert_validation_only_path(path):
    text = str(path).lower()

    assert "validation" in text, (
        f"Not a validation artifact: {path}"
    )

    parts = (
        text
        .replace("\\", "/")
        .replace("-", "_")
        .replace(".", "_")
        .split("/")
    )

    for part in parts:
        tokens = part.split("_")

        assert "test" not in tokens, (
            f"TEST-like artifact rejected: {path}"
        )


assert_validation_only_path(
    WEBQSP_VAL_PLAN_PATH
)

assert_validation_only_path(
    CWQ_VAL_PLAN_PATH
)

print("\nValidation-only leakage gate: PASSED")


# ======================================================================
# 5. Load JSONL exactly as persisted
# ======================================================================

def load_jsonl(path):
    rows = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        for line_no, line in enumerate(
            f,
            start=1
        ):
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(
                    json.loads(line)
                )
            except Exception as e:
                raise RuntimeError(
                    f"JSON error in {path}, "
                    f"line {line_no}: {e}"
                )

    return rows


webqsp_val_plan_rows = load_jsonl(
    WEBQSP_VAL_PLAN_PATH
)

cwq_val_plan_rows = load_jsonl(
    CWQ_VAL_PLAN_PATH
)

print("\nRaw JSONL rows:")
print("  WebQSP:", len(webqsp_val_plan_rows))
print("  CWQ:   ", len(cwq_val_plan_rows))


# ======================================================================
# 6. Question-count fidelity gate
# ======================================================================

assert len(webqsp_val_plan_rows) == (
    FROZEN_VALIDATION_EXPECTATIONS[
        "webqsp"
    ][
        "questions"
    ]
)

assert len(cwq_val_plan_rows) == (
    FROZEN_VALIDATION_EXPECTATIONS[
        "cwq"
    ][
        "questions"
    ]
)

print("Frozen question-count gate: PASSED")


# ======================================================================
# 7. Show actual planning-row schema
# ======================================================================

def show_row_schema(dataset, rows):
    row = rows[0]

    print("\n" + "=" * 88)
    print(f"{dataset.upper()} FIRST PLANNING ROW SCHEMA")
    print("=" * 88)

    print("Top-level keys:")

    for key in row.keys():
        value = row[key]

        if isinstance(value, list):
            desc = f"list[{len(value)}]"

            if value:
                desc += (
                    f" -> {type(value[0]).__name__}"
                )

        elif isinstance(value, dict):
            desc = (
                "dict keys="
                + str(
                    list(value.keys())[:15]
                )
            )

        else:
            text = str(value)

            if len(text) > 100:
                text = text[:97] + "..."

            desc = (
                f"{type(value).__name__}: "
                f"{text}"
            )

        print(
            f"  {key:<30} {desc}"
        )


show_row_schema(
    "webqsp",
    webqsp_val_plan_rows
)

show_row_schema(
    "cwq",
    cwq_val_plan_rows
)


# ======================================================================
# 8. Enumerate list-valued dictionary field paths
# ======================================================================
#
# We inspect dictionary structure but do NOT descend into lists.
# This finds candidates such as:
#
#   ("predicted_paths",)
#   ("planning", "paths")
#   ("prediction", "relation_paths")
#
# ======================================================================

def enumerate_list_paths(
    obj,
    prefix=()
):
    paths = []

    if not isinstance(
        obj,
        dict
    ):
        return paths

    for key, value in obj.items():
        current = (
            *prefix,
            key
        )

        if isinstance(
            value,
            list
        ):
            paths.append(
                current
            )

        elif isinstance(
            value,
            dict
        ):
            paths.extend(
                enumerate_list_paths(
                    value,
                    current
                )
            )

    return paths


def get_nested_value(
    row,
    path
):
    value = row

    for key in path:
        if not isinstance(
            value,
            dict
        ):
            return None

        if key not in value:
            return None

        value = value[key]

    return value


def path_to_string(path):
    return ".".join(
        str(x)
        for x in path
    )


# ======================================================================
# 9. Empty-plan detector
# ======================================================================

def plan_is_empty(plan):
    if plan is None:
        return True

    if isinstance(
        plan,
        str
    ):
        text = plan.strip().lower()

        return text in {
            "",
            "[]",
            "()",
            "{}",
            "none",
            "null",
        }

    if isinstance(
        plan,
        (
            list,
            tuple,
            dict,
            set,
        )
    ):
        return len(plan) == 0

    return False


# ======================================================================
# 10. Evaluate every possible list field
# ======================================================================

SEMANTIC_PLAN_TOKENS = [
    "plan",
    "path",
    "relation",
    "prediction",
    "predict",
    "beam",
]


def evaluate_list_paths(
    dataset,
    rows
):
    expected = (
        FROZEN_VALIDATION_EXPECTATIONS[
            dataset
        ]
    )

    discovered_paths = set()

    # Inspect enough rows to catch optional nested structures.
    for row in rows[:min(
        100,
        len(rows)
    )]:
        for path in enumerate_list_paths(
            row
        ):
            discovered_paths.add(
                path
            )

    evaluations = []

    for path in sorted(
        discovered_paths
    ):
        values = []

        valid = True

        for row in rows:
            value = get_nested_value(
                row,
                path
            )

            if not isinstance(
                value,
                list
            ):
                valid = False
                break

            values.append(
                value
            )

        if not valid:
            continue

        total_items = sum(
            len(v)
            for v in values
        )

        empty_items = sum(
            int(
                plan_is_empty(item)
            )
            for value in values
            for item in value
        )

        path_text = (
            path_to_string(
                path
            )
            .lower()
        )

        semantic_score = sum(
            token in path_text
            for token
            in SEMANTIC_PLAN_TOKENS
        )

        exact_counts = (
            total_items
            == expected[
                "total_plans"
            ]
            and
            empty_items
            == expected[
                "empty_plans"
            ]
        )

        evaluations.append(
            {
                "path": path,
                "path_text":
                    path_to_string(
                        path
                    ),
                "total_items":
                    total_items,
                "empty_items":
                    empty_items,
                "semantic_score":
                    semantic_score,
                "exact_counts":
                    exact_counts,
            }
        )

    evaluations.sort(
        key=lambda x: (
            not x[
                "exact_counts"
            ],
            -x[
                "semantic_score"
            ],
            abs(
                x["total_items"]
                - expected["total_plans"]
            ),
            x["path_text"],
        )
    )

    return evaluations


webqsp_path_evaluations = (
    evaluate_list_paths(
        "webqsp",
        webqsp_val_plan_rows
    )
)

cwq_path_evaluations = (
    evaluate_list_paths(
        "cwq",
        cwq_val_plan_rows
    )
)


# ======================================================================
# 11. Print discovered field candidates
# ======================================================================

def print_path_candidates(
    dataset,
    evaluations,
    limit=15
):
    print("\n" + "=" * 96)
    print(
        f"{dataset.upper()} LIST-FIELD CANDIDATES"
    )
    print("=" * 96)

    print(
        f"{'field':<45}"
        f"{'items':>10}"
        f"{'empty':>10}"
        f"{'semantic':>11}"
        f"{'exact':>8}"
    )

    for item in evaluations[
        :limit
    ]:
        print(
            f"{item['path_text']:<45}"
            f"{item['total_items']:>10}"
            f"{item['empty_items']:>10}"
            f"{item['semantic_score']:>11}"
            f"{str(item['exact_counts']):>8}"
        )


print_path_candidates(
    "webqsp",
    webqsp_path_evaluations
)

print_path_candidates(
    "cwq",
    cwq_path_evaluations
)


# ======================================================================
# 12. Select exact frozen plan field
# ======================================================================

def select_exact_plan_path(
    dataset,
    evaluations
):
    exact = [
        x
        for x in evaluations
        if x["exact_counts"]
    ]

    assert len(exact) > 0, (
        f"{dataset.upper()}: no list field matches "
        "the frozen validation-plan counts."
    )

    # Require semantic relation to plans/predictions where possible.
    semantic_exact = [
        x
        for x in exact
        if x[
            "semantic_score"
        ] > 0
    ]

    if semantic_exact:
        exact = semantic_exact

    # Highest semantic score first.
    exact = sorted(
        exact,
        key=lambda x: (
            -x[
                "semantic_score"
            ],
            x[
                "path_text"
            ],
        )
    )

    # If multiple exact candidates remain, report them.
    if len(exact) > 1:
        print(
            f"\n{dataset.upper()}: "
            "multiple exact-count fields found:"
        )

        for x in exact:
            print(
                " ",
                x[
                    "path_text"
                ]
            )

        print(
            "Using highest-ranked semantic candidate:",
            exact[0][
                "path_text"
            ]
        )

    return exact[0]


webqsp_selected_plan_field = (
    select_exact_plan_path(
        "webqsp",
        webqsp_path_evaluations
    )
)

cwq_selected_plan_field = (
    select_exact_plan_path(
        "cwq",
        cwq_path_evaluations
    )
)

WEBQSP_VAL_PLAN_FIELD_PATH = (
    webqsp_selected_plan_field[
        "path"
    ]
)

CWQ_VAL_PLAN_FIELD_PATH = (
    cwq_selected_plan_field[
        "path"
    ]
)

print("\nSelected frozen plan fields:")
print(
    "  WebQSP:",
    path_to_string(
        WEBQSP_VAL_PLAN_FIELD_PATH
    )
)

print(
    "  CWQ:   ",
    path_to_string(
        CWQ_VAL_PLAN_FIELD_PATH
    )
)


# ======================================================================
# 13. Canonical frozen-plan accessor
# ======================================================================

def get_frozen_relation_plans(
    row,
    field_path
):
    plans = get_nested_value(
        row,
        field_path
    )

    assert isinstance(
        plans,
        list
    )

    return plans


# ======================================================================
# 14. Exact plan-count fidelity audit
# ======================================================================

def audit_restored_plans(
    dataset,
    rows,
    field_path
):
    expected = (
        FROZEN_VALIDATION_EXPECTATIONS[
            dataset
        ]
    )

    plan_lists = [
        get_frozen_relation_plans(
            row,
            field_path
        )
        for row in rows
    ]

    total_plans = sum(
        len(plans)
        for plans in plan_lists
    )

    empty_plans = sum(
        int(
            plan_is_empty(plan)
        )
        for plans in plan_lists
        for plan in plans
    )

    nonempty_plans = (
        total_plans
        - empty_plans
    )

    result = {
        "questions":
            len(rows),

        "total_plans":
            total_plans,

        "empty_plans":
            empty_plans,

        "nonempty_plans":
            nonempty_plans,
    }

    assert result == {
        "questions":
            expected[
                "questions"
            ],

        "total_plans":
            expected[
                "total_plans"
            ],

        "empty_plans":
            expected[
                "empty_plans"
            ],

        "nonempty_plans":
            expected[
                "nonempty_plans"
            ],
    }, (
        f"{dataset.upper()} frozen-plan "
        f"fidelity failure:\n"
        f"observed={result}\n"
        f"expected={expected}"
    )

    return result


webqsp_val_plan_audit = (
    audit_restored_plans(
        "webqsp",
        webqsp_val_plan_rows,
        WEBQSP_VAL_PLAN_FIELD_PATH
    )
)

cwq_val_plan_audit = (
    audit_restored_plans(
        "cwq",
        cwq_val_plan_rows,
        CWQ_VAL_PLAN_FIELD_PATH
    )
)

print(
    "\nExact frozen validation-plan "
    "count gate: PASSED"
)


# ======================================================================
# 15. Fingerprint ORIGINAL persisted JSONL files
# ======================================================================

def sha256_file(path):
    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


WEBQSP_VAL_PLAN_FILE_SHA256 = (
    sha256_file(
        WEBQSP_VAL_PLAN_PATH
    )
)

CWQ_VAL_PLAN_FILE_SHA256 = (
    sha256_file(
        CWQ_VAL_PLAN_PATH
    )
)

print("\nPersisted artifact SHA256:")
print(
    "  WebQSP:",
    WEBQSP_VAL_PLAN_FILE_SHA256
)

print(
    "  CWQ:   ",
    CWQ_VAL_PLAN_FILE_SHA256
)


# ======================================================================
# 16. Inspect a representative plan from each dataset
# ======================================================================

def first_nonempty_plan(
    rows,
    field_path
):
    for row_index, row in enumerate(
        rows
    ):
        plans = get_frozen_relation_plans(
            row,
            field_path
        )

        for plan_index, plan in enumerate(
            plans
        ):
            if not plan_is_empty(
                plan
            ):
                return (
                    row_index,
                    plan_index,
                    plan,
                )

    return None


webqsp_sample_plan = (
    first_nonempty_plan(
        webqsp_val_plan_rows,
        WEBQSP_VAL_PLAN_FIELD_PATH
    )
)

cwq_sample_plan = (
    first_nonempty_plan(
        cwq_val_plan_rows,
        CWQ_VAL_PLAN_FIELD_PATH
    )
)

print("\nRepresentative frozen plan objects:")

print(
    "  WebQSP:",
    webqsp_sample_plan
)

print(
    "  CWQ:   ",
    cwq_sample_plan
)


# ======================================================================
# 17. Verify selected scorer manifests still exist
# ======================================================================

FINAL_SCORER_DIR = (
    RQ2_ROOT
    / "07_final_scorer"
)

SCORER_SELECTION_MANIFEST = (
    FINAL_SCORER_DIR
    / "afp_scorer_selection_manifest.json"
)

SELECTOR_MANIFEST = (
    RQ2_ROOT
    / "08_adaptive_selector"
    / "adaptive_selector_definition_manifest.json"
)

BASELINE_MANIFEST = (
    RQ2_ROOT
    / "09_controlled_baselines"
    / "controlled_baseline_definition_manifest.json"
)

assert SCORER_SELECTION_MANIFEST.exists()
assert SELECTOR_MANIFEST.exists()
assert BASELINE_MANIFEST.exists()

with open(
    SCORER_SELECTION_MANIFEST,
    "r",
    encoding="utf-8"
) as f:
    scorer_selection_manifest_12a = (
        json.load(f)
    )

with open(
    SELECTOR_MANIFEST,
    "r",
    encoding="utf-8"
) as f:
    selector_manifest_12a = (
        json.load(f)
    )

with open(
    BASELINE_MANIFEST,
    "r",
    encoding="utf-8"
) as f:
    baseline_manifest_12a = (
        json.load(f)
    )

assert (
    scorer_selection_manifest_12a[
        "test_used_for_selection"
    ]
    is False
)

assert (
    scorer_selection_manifest_12a[
        "test_metrics_observed"
    ]
    is False
)

assert (
    selector_manifest_12a[
        "test_used"
    ]
    is False
)

assert (
    baseline_manifest_12a[
        "test_used"
    ]
    is False
)

print(
    "\nScorer/selector/baseline "
    "artifact gates: PASSED"
)


# ======================================================================
# 18. Verify Cell 10/11 policy functions
# ======================================================================

required_policy_functions = [
    "afp_select_from_logits",
    "afp_select_frontier",
    "controlled_select_frontier",
    "select_rog",
    "select_fixed_top_b",
    "select_fixed_threshold",
    "select_random_b",
    "select_adaptive_budget_random",
    "select_afp_policy",
]

missing = [
    name
    for name in required_policy_functions
    if name not in globals()
    or not callable(
        globals()[name]
    )
]

assert not missing, (
    "Missing Cell 10/11 functions: "
    + str(missing)
)

print(
    "Cell 10/11 policy functions: PASSED"
)


# ======================================================================
# 19. Discover likely validation/KG artifacts for Cell 12B
# ======================================================================

def discover_dataset_artifacts(
    dataset,
    limit=30
):
    aliases = {
        "webqsp": [
            "webqsp",
            "web_qsp",
        ],

        "cwq": [
            "cwq",
            "complexwebquestions",
            "complex_web_questions",
        ],
    }[
        dataset
    ]

    roots = [
        Path("/kaggle/working"),
        Path("/kaggle/input"),
    ]

    rows = []

    for root in roots:
        if not root.exists():
            continue

        for dirpath, dirnames, filenames in os.walk(
            root
        ):
            dirnames[:] = [
                d
                for d in dirnames
                if not d.startswith(".")
                and d != "__pycache__"
            ]

            for filename in filenames:
                path = (
                    Path(dirpath)
                    / filename
                )

                text = str(
                    path
                ).lower()

                if not any(
                    alias in text
                    for alias in aliases
                ):
                    continue

                # No test artifacts.
                parts = (
                    text
                    .replace("\\", "/")
                    .replace("-", "_")
                    .replace(".", "_")
                    .split("/")
                )

                if any(
                    "test" in part.split("_")
                    for part in parts
                ):
                    continue

                tags = []

                for token in [
                    "validation",
                    "val",
                    "graph",
                    "subgraph",
                    "kg",
                    "adj",
                    "profile",
                    "question",
                    "entity",
                    "relation",
                    "plan",
                ]:
                    if token in text:
                        tags.append(
                            token
                        )

                if not tags:
                    continue

                try:
                    size_mb = (
                        path.stat().st_size
                        / (1024 ** 2)
                    )
                except Exception:
                    size_mb = np.nan

                score = (
                    10 * int(
                        "graph" in tags
                        or "subgraph" in tags
                        or "kg" in tags
                    )
                    +
                    6 * int(
                        "validation" in tags
                        or "val" in tags
                    )
                    +
                    4 * int(
                        "profile" in tags
                    )
                )

                rows.append(
                    {
                        "path":
                            str(path),

                        "size_mb":
                            size_mb,

                        "tags":
                            ",".join(tags),

                        "score":
                            score,
                    }
                )

    df = pd.DataFrame(
        rows
    )

    if len(df) == 0:
        return df

    df = (
        df
        .drop_duplicates(
            subset=[
                "path"
            ]
        )
        .sort_values(
            [
                "score",
                "path",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .head(limit)
        .reset_index(
            drop=True
        )
    )

    return df


webqsp_artifact_inventory = (
    discover_dataset_artifacts(
        "webqsp"
    )
)

cwq_artifact_inventory = (
    discover_dataset_artifacts(
        "cwq"
    )
)


def print_inventory(
    dataset,
    df
):
    print("\n" + "=" * 100)
    print(
        f"{dataset.upper()} "
        "LIKELY VALIDATION / KG ARTIFACTS"
    )
    print("=" * 100)

    if len(df) == 0:
        print("NONE FOUND")
        return

    for _, row in df.iterrows():
        size = row[
            "size_mb"
        ]

        size_text = (
            f"{size:.2f} MB"
            if np.isfinite(size)
            else "?"
        )

        print(
            f"{size_text:>10}  "
            f"[{row['tags']}]  "
            f"{row['path']}"
        )


print_inventory(
    "webqsp",
    webqsp_artifact_inventory
)

print_inventory(
    "cwq",
    cwq_artifact_inventory
)


# ======================================================================
# 20. Inspect surviving traversal/data globals
# ======================================================================

TRAVERSAL_KEYWORDS = [
    "travers",
    "expand",
    "match",
    "candidate",
    "graph",
    "neighbor",
    "adj",
    "profile",
    "feature",
]

existing_traversal_callables = sorted(
    name
    for name, obj
    in globals().items()
    if callable(obj)
    and any(
        token in name.lower()
        for token
        in TRAVERSAL_KEYWORDS
    )
)

DATA_GLOBAL_KEYWORDS = [
    "webqsp",
    "cwq",
    "graph",
    "dataset",
    "validation",
    "val_data",
]

existing_data_globals = sorted(
    name
    for name, obj
    in globals().items()
    if not callable(obj)
    and any(
        token in name.lower()
        for token
        in DATA_GLOBAL_KEYWORDS
    )
    and not name.startswith("_")
)

print(
    "\nExisting traversal-related callables:"
)

print(
    existing_traversal_callables[:50]
    if existing_traversal_callables
    else "NONE"
)

print(
    "\nExisting data-related globals:"
)

print(
    existing_data_globals[:80]
    if existing_data_globals
    else "NONE"
)


# ======================================================================
# 21. Save recovery manifest
# ======================================================================

TRAVERSAL_DIR = (
    RQ2_ROOT
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CELL12A_RECOVERY_MANIFEST = {
    "cell":
        "RQ2_12A_R",

    "webqsp": {
        "artifact":
            str(
                WEBQSP_VAL_PLAN_PATH
            ),

        "artifact_sha256":
            WEBQSP_VAL_PLAN_FILE_SHA256,

        "plan_field":
            path_to_string(
                WEBQSP_VAL_PLAN_FIELD_PATH
            ),

        **webqsp_val_plan_audit,
    },

    "cwq": {
        "artifact":
            str(
                CWQ_VAL_PLAN_PATH
            ),

        "artifact_sha256":
            CWQ_VAL_PLAN_FILE_SHA256,

        "plan_field":
            path_to_string(
                CWQ_VAL_PLAN_FIELD_PATH
            ),

        **cwq_val_plan_audit,
    },

    "planner_rerun":
        False,

    "semantic_encoder_rerun":
        False,

    "validation_traversal_run":
        False,

    "hyperparameter_tuning_run":
        False,

    "test_loaded":
        False,

    "rog_fidelity_checked":
        False,
}

CELL12A_RECOVERY_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12a_frozen_plan_recovery.json"
)

with open(
    CELL12A_RECOVERY_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        CELL12A_RECOVERY_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 22. Final report
# ======================================================================

print("\n" + "=" * 98)
print(
    "=== RQ2 CELL 12A-R: FROZEN VALIDATION PLAN RECOVERY COMPLETE ==="
)
print("=" * 98)

print("\nWEBQSP")
print(
    "  artifact:       ",
    WEBQSP_VAL_PLAN_PATH
)
print(
    "  plan field:     ",
    path_to_string(
        WEBQSP_VAL_PLAN_FIELD_PATH
    )
)
print(
    "  questions:      ",
    webqsp_val_plan_audit[
        "questions"
    ]
)
print(
    "  total plans:    ",
    webqsp_val_plan_audit[
        "total_plans"
    ]
)
print(
    "  empty plans:    ",
    webqsp_val_plan_audit[
        "empty_plans"
    ]
)
print(
    "  executable:     ",
    webqsp_val_plan_audit[
        "nonempty_plans"
    ]
)

print("\nCWQ")
print(
    "  artifact:       ",
    CWQ_VAL_PLAN_PATH
)
print(
    "  plan field:     ",
    path_to_string(
        CWQ_VAL_PLAN_FIELD_PATH
    )
)
print(
    "  questions:      ",
    cwq_val_plan_audit[
        "questions"
    ]
)
print(
    "  total plans:    ",
    cwq_val_plan_audit[
        "total_plans"
    ]
)
print(
    "  empty plans:    ",
    cwq_val_plan_audit[
        "empty_plans"
    ]
)
print(
    "  executable:     ",
    cwq_val_plan_audit[
        "nonempty_plans"
    ]
)

print("\nIntegrity")
print(
    "  Exact frozen plan counts:       PASSED"
)
print(
    "  Persisted artifact fingerprint: SAVED"
)
print(
    "  Scorer artifacts:               PASSED"
)
print(
    "  Selector artifacts:             PASSED"
)
print(
    "  Baseline artifacts:             PASSED"
)
print(
    "  Cell 10/11 functions:           PASSED"
)

print("\nExecution status")
print(
    "  Planner rerun:                  NO"
)
print(
    "  MiniLM rerun:                   NO"
)
print(
    "  Validation traversal run:       NO"
)
print(
    "  Hyperparameter tuning:          NO"
)
print(
    "  TEST data/gold loaded:          NO"
)
print(
    "  RoG fidelity checked:           NO"
)

print(
    "\nRecovery manifest:",
    CELL12A_RECOVERY_MANIFEST_PATH
)

print(
    "\nNext: Cell 12B — shared validation traversal "
    "+ RoG fidelity gate."
)

RQ2 root: /kaggle/working/step3_rq2_dev_v1
Device:   cpu

Frozen RQ1 planning artifacts:
  WebQSP: /kaggle/working/step2_rq1_dev/planning_webqsp_validation.jsonl
  CWQ:    /kaggle/working/step2_rq1_dev/planning_cwq_validation.jsonl

Validation-only leakage gate: PASSED

Raw JSONL rows:
  WebQSP: 246
  CWQ:    3519
Frozen question-count gate: PASSED

WEBQSP FIRST PLANNING ROW SCHEMA
Top-level keys:
  id                             str: WebQTrn-9
  question                       str: how old is sacha baron cohen
  q_entity                       list[1] -> str
  a_entity                       list[1] -> str
  graph                          list[6293] -> list
  predicted_paths                list[3] -> list
  planning_time_sec              float: 1.4861440658569336

CWQ FIRST PLANNING ROW SCHEMA
Top-level keys:
  id                             str: WebQTrn-1430_ac053cda0a7424c48e4809c71171fbed
  question                       str: Who was the president in 1980 of the country that has Azad 

In [9]:
# ======================================================================
# EXACT RoG GRAPH SEMANTICS + SHARED TRAVERSAL FIDELITY
# ======================================================================
#
# FIX:
# Official RoG uses:
#
#   G = nx.Graph()
#   G.add_edge(h, t, relation=r.strip())
#
# Therefore:
#   - graph is UNDIRECTED
#   - only UNIQUE entity-pair neighbors exist
#   - repeated (h,t) edges overwrite relation attribute
#   - neighbor order follows first insertion
#
# This cell reproduces those semantics WITHOUT rerunning RoG planner.
#
# NO tuning.
# NO MiniLM.
# NO TEST.
# ======================================================================

import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd


# ======================================================================
# 1. Hard prerequisites
# ======================================================================

assert "webqsp_val_plan_rows" in globals()
assert "cwq_val_plan_rows" in globals()

assert len(webqsp_val_plan_rows) == 246
assert len(cwq_val_plan_rows) == 3519

assert "controlled_select_frontier" in globals()
assert callable(controlled_select_frontier)

assert "get_frozen_relation_plans" in globals()
assert callable(get_frozen_relation_plans)

print("Cell 12A-R prerequisites: PASSED")


# ======================================================================
# 2. Frozen RQ1 references
# ======================================================================

FROZEN_RQ1_CORE = {
    "webqsp": {
        "active_hop_rows": 971,
        "edges_examined": 341526,
        "candidate_branches": 7983,
    },

    "cwq": {
        "active_hop_rows": 16564,
        "edges_examined": 5257272,
        "candidate_branches": 247161,
    },
}

FROZEN_RQ1_REACHABILITY = {
    "webqsp": {
        "reachable_plans": 345,
        "reachable_questions": 205,
    },

    "cwq": {
        "reachable_plans": 3971,
        "reachable_questions": 2425,
    },
}

print("Frozen RQ1 references: LOADED")


# ======================================================================
# 3. Exact RoG graph construction
# ======================================================================
#
# Equivalent to:
#
#   G = nx.Graph()
#   for h, r, t in graph:
#       G.add_edge(h, t, relation=r.strip())
#
# Dict behavior is intentional:
#
#   - first appearance fixes neighbor insertion order
#   - later same-pair edges replace relation
#   - no parallel edges
#
# ======================================================================

def build_rog_adjacency(graph):
    adjacency = defaultdict(dict)

    for triple in graph:
        if (
            not isinstance(triple, (list, tuple))
            or len(triple) != 3
        ):
            continue

        h, r, t = triple

        h = str(h)
        t = str(t)
        r = str(r).strip()

        # ----------------------------------------------------------
        # NetworkX Graph.add_edge semantics:
        # same undirected pair => relation attr is overwritten.
        # Existing dictionary key keeps its insertion position.
        # ----------------------------------------------------------
        adjacency[h][t] = r
        adjacency[t][h] = r

    return adjacency


# ======================================================================
# 4. Initial prefix + extension helpers
# ======================================================================

def make_rog_initial_prefixes(topic_entities):
    return [
        {
            "entities": (str(entity),),
            "relations": tuple(),
        }
        for entity in topic_entities
    ]


def extend_rog_prefix(
    prefix,
    relation,
    next_entity
):
    return {
        "entities":
            prefix["entities"]
            + (str(next_entity),),

        "relations":
            prefix["relations"]
            + (str(relation),),
    }


# ======================================================================
# 5. Exact RoG candidate generation
# ======================================================================
#
# Equivalent to:
#
#   for neighbor in graph.neighbors(current_node):
#       rel = graph[current_node][neighbor]["relation"]
#       if rel == target_relation:
#           append...
#
# edges_examined = number of UNIQUE neighbors inspected.
# ======================================================================

def generate_rog_candidates(
    prefixes,
    target_relation,
    adjacency
):
    target_relation = str(
        target_relation
    ).strip()

    candidates = []
    edges_examined = 0

    for prefix in prefixes:
        current_entity = (
            prefix["entities"][-1]
        )

        neighbors = adjacency.get(
            current_entity,
            {}
        )

        # RoG checks every unique graph neighbor.
        edges_examined += len(
            neighbors
        )

        for neighbor, edge_relation in neighbors.items():

            if edge_relation != target_relation:
                continue

            candidates.append(
                extend_rog_prefix(
                    prefix=prefix,
                    relation=target_relation,
                    next_entity=neighbor
                )
            )

    return (
        candidates,
        edges_examined
    )


# ======================================================================
# 6. Exact unpruned RoG plan traversal
# ======================================================================

def traverse_rog_plan(
    row,
    plan,
    question_index,
    plan_index
):
    assert isinstance(plan, list)
    assert len(plan) > 0

    adjacency = build_rog_adjacency(
        row.get("graph", [])
    )

    prefixes = make_rog_initial_prefixes(
        row.get("q_entity", [])
    )

    hop_records = []

    for hop, relation in enumerate(plan):

        if len(prefixes) == 0:
            break

        active_prefixes = len(
            prefixes
        )

        candidates, edges_examined = (
            generate_rog_candidates(
                prefixes=prefixes,
                target_relation=relation,
                adjacency=adjacency
            )
        )

        hop_records.append(
            {
                "question_index":
                    question_index,

                "question_id":
                    str(
                        row.get(
                            "id",
                            question_index
                        )
                    ),

                "plan_index":
                    plan_index,

                "hop":
                    hop,

                "plan_length":
                    len(plan),

                "relation":
                    str(relation),

                "active_prefixes":
                    active_prefixes,

                "edges_examined":
                    edges_examined,

                "candidate_branches":
                    len(candidates),

                "unique_candidate_entities":
                    len({
                        p["entities"][-1]
                        for p in candidates
                    }),
            }
        )

        prefixes = candidates

    return (
        prefixes,
        hop_records
    )


# ======================================================================
# 7. Dataset-level exact RoG traversal
# ======================================================================

def run_exact_rog_validation(
    dataset,
    rows,
    plan_field_path
):
    all_hops = []

    total_plans = 0
    empty_plans = 0
    executable_plans = 0

    reachable_plans = 0
    reachable_questions = set()

    for q_idx, row in enumerate(rows):

        qid = str(
            row.get(
                "id",
                q_idx
            )
        )

        answers = {
            str(x)
            for x in row.get(
                "a_entity",
                []
            )
        }

        plans = get_frozen_relation_plans(
            row,
            plan_field_path
        )

        for plan_idx, plan in enumerate(
            plans
        ):
            total_plans += 1

            if (
                not isinstance(plan, list)
                or len(plan) == 0
            ):
                empty_plans += 1
                continue

            executable_plans += 1

            final_prefixes, hop_records = (
                traverse_rog_plan(
                    row=row,
                    plan=plan,
                    question_index=q_idx,
                    plan_index=plan_idx
                )
            )

            all_hops.extend(
                hop_records
            )

            final_entities = {
                p["entities"][-1]
                for p in final_prefixes
            }

            if (
                len(
                    final_entities
                    & answers
                )
                > 0
            ):
                reachable_plans += 1
                reachable_questions.add(
                    qid
                )

    hop_df = pd.DataFrame(
        all_hops
    )

    return {
        "dataset":
            dataset,

        "questions":
            len(rows),

        "total_plans":
            total_plans,

        "empty_plans":
            empty_plans,

        "executable_plans":
            executable_plans,

        "active_hop_rows":
            int(len(hop_df)),

        "edges_examined":
            int(
                hop_df[
                    "edges_examined"
                ].sum()
            ),

        "candidate_branches":
            int(
                hop_df[
                    "candidate_branches"
                ].sum()
            ),

        "reachable_plans":
            reachable_plans,

        "reachable_questions":
            len(
                reachable_questions
            ),

        "reachable_question_ids":
            sorted(
                reachable_questions
            ),

        "hop_df":
            hop_df,
    }


# ======================================================================
# 8. WebQSP exact RoG fidelity
# ======================================================================

print(
    "\n"
    + "=" * 94
)

print(
    "WEBQSP — OFFICIAL RoG GRAPH SEMANTICS FIDELITY"
)

print(
    "=" * 94
)

webqsp_exact_rog = (
    run_exact_rog_validation(
        dataset="webqsp",
        rows=webqsp_val_plan_rows,
        plan_field_path=
            WEBQSP_VAL_PLAN_FIELD_PATH
    )
)

print(
    "Observed core:",
    {
        k: webqsp_exact_rog[k]
        for k in [
            "active_hop_rows",
            "edges_examined",
            "candidate_branches",
        ]
    }
)

print(
    "Expected core:",
    FROZEN_RQ1_CORE[
        "webqsp"
    ]
)

for key, expected in (
    FROZEN_RQ1_CORE[
        "webqsp"
    ].items()
):
    assert (
        webqsp_exact_rog[key]
        == expected
    ), (
        f"WEBQSP mismatch: {key} | "
        f"observed={webqsp_exact_rog[key]} "
        f"expected={expected}"
    )

print(
    "WebQSP search-cost fidelity: PASSED"
)

print(
    "Observed reachability:",
    {
        "reachable_plans":
            webqsp_exact_rog[
                "reachable_plans"
            ],

        "reachable_questions":
            webqsp_exact_rog[
                "reachable_questions"
            ],
    }
)

print(
    "Expected reachability:",
    FROZEN_RQ1_REACHABILITY[
        "webqsp"
    ]
)

assert (
    webqsp_exact_rog[
        "reachable_plans"
    ]
    ==
    FROZEN_RQ1_REACHABILITY[
        "webqsp"
    ][
        "reachable_plans"
    ]
)

assert (
    webqsp_exact_rog[
        "reachable_questions"
    ]
    ==
    FROZEN_RQ1_REACHABILITY[
        "webqsp"
    ][
        "reachable_questions"
    ]
)

print(
    "WebQSP reachability fidelity: PASSED"
)


# ======================================================================
# 9. CWQ exact RoG fidelity
# ======================================================================

print(
    "\n"
    + "=" * 94
)

print(
    "CWQ — OFFICIAL RoG GRAPH SEMANTICS FIDELITY"
)

print(
    "=" * 94
)

cwq_exact_rog = (
    run_exact_rog_validation(
        dataset="cwq",
        rows=cwq_val_plan_rows,
        plan_field_path=
            CWQ_VAL_PLAN_FIELD_PATH
    )
)

print(
    "Observed core:",
    {
        k: cwq_exact_rog[k]
        for k in [
            "active_hop_rows",
            "edges_examined",
            "candidate_branches",
        ]
    }
)

print(
    "Expected core:",
    FROZEN_RQ1_CORE[
        "cwq"
    ]
)

for key, expected in (
    FROZEN_RQ1_CORE[
        "cwq"
    ].items()
):
    assert (
        cwq_exact_rog[key]
        == expected
    ), (
        f"CWQ mismatch: {key} | "
        f"observed={cwq_exact_rog[key]} "
        f"expected={expected}"
    )

print(
    "CWQ search-cost fidelity: PASSED"
)

print(
    "Observed reachability:",
    {
        "reachable_plans":
            cwq_exact_rog[
                "reachable_plans"
            ],

        "reachable_questions":
            cwq_exact_rog[
                "reachable_questions"
            ],
    }
)

print(
    "Expected reachability:",
    FROZEN_RQ1_REACHABILITY[
        "cwq"
    ]
)

assert (
    cwq_exact_rog[
        "reachable_plans"
    ]
    ==
    FROZEN_RQ1_REACHABILITY[
        "cwq"
    ][
        "reachable_plans"
    ]
)

assert (
    cwq_exact_rog[
        "reachable_questions"
    ]
    ==
    FROZEN_RQ1_REACHABILITY[
        "cwq"
    ][
        "reachable_questions"
    ]
)

print(
    "CWQ reachability fidelity: PASSED"
)


# ======================================================================
# 10. Plan-count gates
# ======================================================================

assert (
    webqsp_exact_rog[
        "total_plans"
    ] == 721
)

assert (
    webqsp_exact_rog[
        "empty_plans"
    ] == 0
)

assert (
    webqsp_exact_rog[
        "executable_plans"
    ] == 721
)

assert (
    cwq_exact_rog[
        "total_plans"
    ] == 10536
)

assert (
    cwq_exact_rog[
        "empty_plans"
    ] == 7
)

assert (
    cwq_exact_rog[
        "executable_plans"
    ] == 10529
)

print(
    "\nPlan-count fidelity: PASSED"
)


# ======================================================================
# 11. Shared controlled traversal
# ======================================================================
#
# THIS is the single engine used later by every pruning method.
#
# Only:
#
#     controlled_select_frontier(...)
#
# changes between methods.
#
# Scorer callback will be connected in Cell 12C.
# ======================================================================

SCORE_REQUIRED_METHODS = {
    "fixed_top_b",
    "fixed_threshold",
    "adaptive_budget_random",
    "afp",
}


def run_shared_validation_traversal(
    dataset,
    rows,
    plan_field_path,
    method,

    scorer_callback=None,

    B=None,
    threshold=None,

    temperature=None,
    gamma_min=None,

    seed=None,

    tie_tolerance=1e-8,

    collect_hop_records=True
):
    assert method in CONTROLLED_METHODS

    total_edges_examined = 0
    total_candidate_branches = 0
    total_retained_branches = 0
    total_pruned_branches = 0

    active_hop_rows = 0
    decision_hops = 0

    scorer_invocations = 0

    final_hop_protections = 0
    singleton_bypasses = 0
    tied_abstentions = 0
    cutoff_tie_expansions = 0
    empty_after_selection = 0

    total_plans = 0
    empty_plans = 0
    executable_plans = 0

    reachable_plans = 0
    reachable_question_ids = set()

    question_edge_counts = defaultdict(
        int
    )

    hop_records = []

    for q_idx, row in enumerate(
        rows
    ):
        qid = str(
            row.get(
                "id",
                q_idx
            )
        )

        answers = {
            str(x)
            for x in row.get(
                "a_entity",
                []
            )
        }

        adjacency = build_rog_adjacency(
            row.get(
                "graph",
                []
            )
        )

        plans = get_frozen_relation_plans(
            row,
            plan_field_path
        )

        for plan_idx, plan in enumerate(
            plans
        ):
            total_plans += 1

            if (
                not isinstance(plan, list)
                or len(plan) == 0
            ):
                empty_plans += 1
                continue

            executable_plans += 1

            prefixes = make_rog_initial_prefixes(
                row.get(
                    "q_entity",
                    []
                )
            )

            for hop, relation in enumerate(
                plan
            ):
                if len(prefixes) == 0:
                    break

                active_hop_rows += 1

                active_prefix_count = len(
                    prefixes
                )

                candidates, current_edges = (
                    generate_rog_candidates(
                        prefixes=prefixes,
                        target_relation=relation,
                        adjacency=adjacency
                    )
                )

                candidate_count = len(
                    candidates
                )

                total_edges_examined += (
                    current_edges
                )

                total_candidate_branches += (
                    candidate_count
                )

                question_edge_counts[
                    qid
                ] += current_edges

                # --------------------------------------------------
                # No relation-valid candidate.
                # --------------------------------------------------
                if candidate_count == 0:

                    if collect_hop_records:
                        hop_records.append(
                            {
                                "dataset":
                                    dataset,

                                "method":
                                    method,

                                "question_id":
                                    qid,

                                "question_index":
                                    q_idx,

                                "plan_index":
                                    plan_idx,

                                "hop":
                                    hop,

                                "plan_length":
                                    len(plan),

                                "relation":
                                    str(relation),

                                "active_prefixes":
                                    active_prefix_count,

                                "edges_examined":
                                    current_edges,

                                "candidate_branches":
                                    0,

                                "retained_branches":
                                    0,

                                "pruned_branches":
                                    0,

                                "selection_reason":
                                    "no_relation_match",
                            }
                        )

                    prefixes = []
                    break

                is_final_hop = (
                    hop
                    == len(plan) - 1
                )

                if (
                    not is_final_hop
                    and candidate_count > 1
                ):
                    decision_hops += 1

                # --------------------------------------------------
                # Compute logits only when this method/hop needs them.
                # --------------------------------------------------
                logits = None

                needs_scores = (
                    method
                    in SCORE_REQUIRED_METHODS
                    and
                    not is_final_hop
                    and
                    candidate_count > 1
                )

                if needs_scores:
                    assert (
                        scorer_callback
                        is not None
                    ), (
                        f"{method} requires scorer_callback."
                    )

                    logits = scorer_callback(
                        dataset=dataset,
                        row=row,
                        plan=plan,
                        plan_index=plan_idx,
                        hop=hop,
                        prefixes=prefixes,
                        candidates=candidates,
                        relation=str(
                            relation
                        ).strip(),
                        adjacency=adjacency,
                    )

                    logits = np.asarray(
                        logits,
                        dtype=np.float64
                    )

                    assert logits.shape == (
                        candidate_count,
                    )

                    assert np.all(
                        np.isfinite(
                            logits
                        )
                    )

                group_key = (
                    f"{dataset}|"
                    f"{qid}|"
                    f"plan={plan_idx}|"
                    f"hop={hop}"
                )

                selection = (
                    controlled_select_frontier(
                        method=method,

                        candidate_count=
                            candidate_count,

                        hop=hop,

                        plan_length=
                            len(plan),

                        logits=logits,

                        B=B,

                        threshold=
                            threshold,

                        temperature=
                            temperature,

                        gamma_min=
                            gamma_min,

                        seed=seed,

                        group_key=
                            group_key,

                        tie_tolerance=
                            tie_tolerance
                    )
                )

                selected_indices = np.asarray(
                    selection[
                        "selected_indices"
                    ],
                    dtype=np.int64
                )

                retained_count = len(
                    selected_indices
                )

                pruned_count = (
                    candidate_count
                    - retained_count
                )

                assert (
                    retained_count
                    ==
                    selection[
                        "retained_B"
                    ]
                )

                assert (
                    pruned_count
                    ==
                    selection[
                        "pruned_count"
                    ]
                )

                total_retained_branches += (
                    retained_count
                )

                total_pruned_branches += (
                    pruned_count
                )

                if selection.get(
                    "scorer_invoked",
                    False
                ):
                    scorer_invocations += 1

                reason = selection.get(
                    "reason",
                    ""
                )

                if (
                    reason
                    == "final_hop_protection"
                ):
                    final_hop_protections += 1

                if (
                    reason
                    == "singleton_bypass"
                ):
                    singleton_bypasses += 1

                if (
                    reason
                    == "all_scores_tied_retain_all"
                ):
                    tied_abstentions += 1

                if "tie_expansion" in reason:
                    cutoff_tie_expansions += 1

                if retained_count == 0:
                    empty_after_selection += 1
                    prefixes = []

                else:
                    prefixes = [
                        candidates[
                            int(i)
                        ]
                        for i
                        in selected_indices
                    ]

                if collect_hop_records:
                    hop_records.append(
                        {
                            "dataset":
                                dataset,

                            "method":
                                method,

                            "question_id":
                                qid,

                            "question_index":
                                q_idx,

                            "plan_index":
                                plan_idx,

                            "hop":
                                hop,

                            "plan_length":
                                len(plan),

                            "relation":
                                str(
                                    relation
                                ).strip(),

                            "active_prefixes":
                                active_prefix_count,

                            "edges_examined":
                                current_edges,

                            "candidate_branches":
                                candidate_count,

                            "retained_branches":
                                retained_count,

                            "pruned_branches":
                                pruned_count,

                            "selection_reason":
                                reason,
                        }
                    )

            # ------------------------------------------------------
            # Plan-level answer reachability
            # ------------------------------------------------------
            if len(prefixes) > 0:

                final_entities = {
                    p["entities"][-1]
                    for p in prefixes
                }

                if (
                    len(
                        final_entities
                        & answers
                    )
                    > 0
                ):
                    reachable_plans += 1

                    reachable_question_ids.add(
                        qid
                    )

    hop_df = (
        pd.DataFrame(
            hop_records
        )
        if collect_hop_records
        else None
    )

    return {
        "dataset":
            dataset,

        "method":
            method,

        "questions":
            len(rows),

        "total_plans":
            total_plans,

        "empty_plans":
            empty_plans,

        "executable_plans":
            executable_plans,

        "active_hop_rows":
            active_hop_rows,

        "decision_hops":
            decision_hops,

        "edges_examined":
            total_edges_examined,

        "candidate_branches":
            total_candidate_branches,

        "retained_branches":
            total_retained_branches,

        "pruned_branches":
            total_pruned_branches,

        "scorer_invocations":
            scorer_invocations,

        "final_hop_protections":
            final_hop_protections,

        "singleton_bypasses":
            singleton_bypasses,

        "tied_abstentions":
            tied_abstentions,

        "cutoff_tie_expansions":
            cutoff_tie_expansions,

        "empty_after_selection":
            empty_after_selection,

        "reachable_plans":
            reachable_plans,

        "reachable_questions":
            len(
                reachable_question_ids
            ),

        "reachable_question_ids":
            sorted(
                reachable_question_ids
            ),

        "question_edge_counts":
            dict(
                question_edge_counts
            ),

        "hop_df":
            hop_df,
    }


# ======================================================================
# 12. Shared-engine RoG fidelity
# ======================================================================

print(
    "\n"
    + "=" * 94
)

print(
    "SHARED ENGINE — RoG FIDELITY"
)

print(
    "=" * 94
)

webqsp_shared_rog = (
    run_shared_validation_traversal(
        dataset="webqsp",
        rows=webqsp_val_plan_rows,
        plan_field_path=
            WEBQSP_VAL_PLAN_FIELD_PATH,
        method="rog",
        scorer_callback=None
    )
)

cwq_shared_rog = (
    run_shared_validation_traversal(
        dataset="cwq",
        rows=cwq_val_plan_rows,
        plan_field_path=
            CWQ_VAL_PLAN_FIELD_PATH,
        method="rog",
        scorer_callback=None
    )
)


def assert_rog_fidelity(
    dataset,
    result
):
    core = FROZEN_RQ1_CORE[
        dataset
    ]

    reach = FROZEN_RQ1_REACHABILITY[
        dataset
    ]

    for key, expected in core.items():

        assert (
            result[key]
            == expected
        ), (
            f"{dataset.upper()} "
            f"shared-engine mismatch {key}: "
            f"{result[key]} != {expected}"
        )

    assert (
        result[
            "reachable_plans"
        ]
        ==
        reach[
            "reachable_plans"
        ]
    )

    assert (
        result[
            "reachable_questions"
        ]
        ==
        reach[
            "reachable_questions"
        ]
    )

    assert (
        result[
            "pruned_branches"
        ] == 0
    )

    assert (
        result[
            "scorer_invocations"
        ] == 0
    )

    print(
        f"{dataset.upper()} shared "
        "RoG fidelity: PASSED"
    )


assert_rog_fidelity(
    "webqsp",
    webqsp_shared_rog
)

assert_rog_fidelity(
    "cwq",
    cwq_shared_rog
)


# ======================================================================
# 13. Save fidelity artifacts
# ======================================================================

TRAVERSAL_DIR = (
    Path(RQ2_ROOT)
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

webqsp_shared_rog[
    "hop_df"
].to_csv(
    TRAVERSAL_DIR
    / "webqsp_shared_rog_validation_hops.csv",
    index=False
)

cwq_shared_rog[
    "hop_df"
].to_csv(
    TRAVERSAL_DIR
    / "cwq_shared_rog_validation_hops.csv",
    index=False
)


def compact_result(result):
    return {
        k: v
        for k, v
        in result.items()
        if k not in {
            "hop_df",
            "reachable_question_ids",
            "question_edge_counts",
        }
    }


CELL12B_RECOVERY_MANIFEST = {
    "cell":
        "RQ2_12B_R",

    "graph_semantics":
        "official_rog_networkx_graph_equivalent",

    "graph_type":
        "undirected_simple_graph",

    "duplicate_pair_behavior":
        "last_relation_attribute_overwrites",

    "neighbor_behavior":
        "unique_neighbors",

    "relation_matching":
        "exact_stored_relation_match",

    "webqsp":
        compact_result(
            webqsp_shared_rog
        ),

    "cwq":
        compact_result(
            cwq_shared_rog
        ),

    "search_fidelity_passed":
        True,

    "reachability_fidelity_passed":
        True,

    "shared_engine_defined":
        True,

    "planner_rerun":
        False,

    "semantic_encoder_rerun":
        False,

    "hyperparameter_tuning_run":
        False,

    "test_loaded":
        False,

    "complete_afp_frozen":
        False,
}

CELL12B_RECOVERY_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12b_exact_rog_shared_traversal.json"
)

with open(
    CELL12B_RECOVERY_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL12B_RECOVERY_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 14. Final report
# ======================================================================

print(
    "\n"
    + "=" * 98
)

print(
    "=== RQ2 CELL 12B-R: EXACT RoG SHARED TRAVERSAL FIDELITY PASSED ==="
)

print(
    "=" * 98
)

print("\nGraph semantics")
print(
    "  Undirected simple graph:          YES"
)
print(
    "  Unique entity-pair edges:         YES"
)
print(
    "  Duplicate-pair relation overwrite:YES"
)
print(
    "  Exact relation matching:          YES"
)

print("\nWEBQSP")
print(
    "  Active hop rows:       ",
    webqsp_shared_rog[
        "active_hop_rows"
    ]
)
print(
    "  Examined edges:        ",
    webqsp_shared_rog[
        "edges_examined"
    ]
)
print(
    "  Candidate branches:    ",
    webqsp_shared_rog[
        "candidate_branches"
    ]
)
print(
    "  Reachable plans:       ",
    webqsp_shared_rog[
        "reachable_plans"
    ]
)
print(
    "  Reachable questions:   ",
    webqsp_shared_rog[
        "reachable_questions"
    ]
)

print("\nCWQ")
print(
    "  Active hop rows:       ",
    cwq_shared_rog[
        "active_hop_rows"
    ]
)
print(
    "  Examined edges:        ",
    cwq_shared_rog[
        "edges_examined"
    ]
)
print(
    "  Candidate branches:    ",
    cwq_shared_rog[
        "candidate_branches"
    ]
)
print(
    "  Reachable plans:       ",
    cwq_shared_rog[
        "reachable_plans"
    ]
)
print(
    "  Reachable questions:   ",
    cwq_shared_rog[
        "reachable_questions"
    ]
)

print("\nIntegrity")
print(
    "  Search-cost fidelity:  PASSED"
)
print(
    "  Reachability fidelity: PASSED"
)
print(
    "  Shared engine:         DEFINED"
)

print("\nExecution")
print(
    "  Planner rerun:         NO"
)
print(
    "  MiniLM rerun:          NO"
)
print(
    "  Pruning run:           NO"
)
print(
    "  Tuning run:            NO"
)
print(
    "  TEST loaded:           NO"
)
print(
    "  AFP fully frozen:      NO"
)

print(
    "\nManifest:",
    CELL12B_RECOVERY_MANIFEST_PATH
)

print(
    "\nNext: Cell 12C — online Feature-v2 scorer "
    "integration fidelity gate."
)

Cell 12A-R prerequisites: PASSED
Frozen RQ1 references: LOADED

WEBQSP — OFFICIAL RoG GRAPH SEMANTICS FIDELITY
Observed core: {'active_hop_rows': 971, 'edges_examined': 341526, 'candidate_branches': 7983}
Expected core: {'active_hop_rows': 971, 'edges_examined': 341526, 'candidate_branches': 7983}
WebQSP search-cost fidelity: PASSED
Observed reachability: {'reachable_plans': 345, 'reachable_questions': 205}
Expected reachability: {'reachable_plans': 345, 'reachable_questions': 205}
WebQSP reachability fidelity: PASSED

CWQ — OFFICIAL RoG GRAPH SEMANTICS FIDELITY
Observed core: {'active_hop_rows': 16564, 'edges_examined': 5257272, 'candidate_branches': 247161}
Expected core: {'active_hop_rows': 16564, 'edges_examined': 5257272, 'candidate_branches': 247161}
CWQ search-cost fidelity: PASSED
Observed reachability: {'reachable_plans': 3971, 'reachable_questions': 2425}
Expected reachability: {'reachable_plans': 3971, 'reachable_questions': 2425}
CWQ reachability fidelity: PASSED

Plan-coun

In [10]:
# ======================================================================
# RECOVER EXACT FROZEN FEATURE-v2 RUNTIME SPECIFICATION
# ======================================================================
#
# PURPOSE
# -------
# Before implementing ONLINE AFP scoring inside the dynamic traversal,
# recover as much of the EXACT frozen Feature-v2 implementation as
# possible from:
#
#   - feature manifests
#   - validation supervision manifests/records
#   - saved NPZ structure
#   - saved notebook/source files, if present
#   - semantic embedding/cache artifacts, if present
#   - surviving Python globals
#
# WHY:
#   We must NOT re-invent Feature-v2 from feature names alone.
#   Cell 12C-B must reproduce the cached validation features before
#   being allowed to score dynamically changed frontiers.
#
# THIS CELL:
#   - does NOT compute new features
#   - does NOT run MiniLM
#   - does NOT run pruning
#   - does NOT tune hyperparameters
#   - does NOT touch TEST
# ======================================================================

import os
import re
import json
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd


# ======================================================================
# 1. Roots
# ======================================================================

RQ2_ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

FEATURE_DIR = (
    RQ2_ROOT
    / "03_features"
)

assert FEATURE_DIR.exists()

print("RQ2 root:     ", RQ2_ROOT)
print("Feature root: ", FEATURE_DIR)


# ======================================================================
# 2. Expected frozen Feature-v2 identity
# ======================================================================

EXPECTED_FEATURE_VERSION = (
    "afp_features_v2_masked_entity_semantics"
)

EXPECTED_FEATURE_DIM = 27

EXPECTED_FEATURE_SHA256 = (
    "738985d1232a8ac5935c397ed95eca23377bc59b4a99494547fa7779062ade86"
)

EXPECTED_FEATURE_NAMES = [
    "sem_q_candidate",
    "sem_candidate_surface_available",
    "sem_q_current_entity",
    "sem_current_surface_available",
    "sem_q_current_relation",
    "sem_q_full_plan",
    "sem_q_remaining_suffix",
    "sem_candidate_current_relation",

    "path_q_prefix_entity_mean",
    "path_candidate_prefix_entity_mean",
    "path_prefix_surface_fraction",
    "path_candidate_repeats_entity",
    "path_candidate_occurrence_fraction",
    "path_unique_entity_ratio",
    "path_relation_repeat_fraction_before",

    "struct_log_candidate_count",
    "struct_log_unique_candidate_entities",
    "struct_log_contributing_parents",
    "struct_log_parent_fanout",
    "struct_parent_frontier_share",
    "struct_log_endpoint_multiplicity",
    "struct_endpoint_frontier_share",
    "struct_duplicate_endpoint_ratio",

    "prog_hop_fraction",
    "prog_remaining_fraction",
    "prog_log_plan_length",
    "prog_penultimate_indicator",
]

assert len(EXPECTED_FEATURE_NAMES) == 27

print("\nExpected Feature-v2 identity:")
print("  version:", EXPECTED_FEATURE_VERSION)
print("  dim:    ", EXPECTED_FEATURE_DIM)
print("  SHA256: ", EXPECTED_FEATURE_SHA256)


# ======================================================================
# 3. Locate frozen manifests
# ======================================================================

WEBQSP_VAL_FEATURE_MANIFEST = (
    FEATURE_DIR
    / "webqsp"
    / "webqsp_validation_afp_features_v2_manifest.json"
)

CWQ_VAL_FEATURE_MANIFEST = (
    FEATURE_DIR
    / "cwq"
    / "cwq_validation_afp_features_v2_manifest.json"
)

WEBQSP_VAL_FEATURE_NPZ = (
    FEATURE_DIR
    / "webqsp"
    / "webqsp_validation_afp_features_v2.npz"
)

CWQ_VAL_FEATURE_NPZ = (
    FEATURE_DIR
    / "cwq"
    / "cwq_validation_afp_features_v2.npz"
)

WEBQSP_LABEL_MANIFEST = (
    FEATURE_DIR
    / "validation_supervision"
    / "webqsp_validation_decision_labels_manifest.json"
)

CWQ_LABEL_MANIFEST = (
    FEATURE_DIR
    / "validation_supervision"
    / "cwq_validation_decision_labels_manifest.json"
)

WEBQSP_LABEL_FILE = (
    FEATURE_DIR
    / "validation_supervision"
    / "webqsp_validation_decision_labels.jsonl"
)

CWQ_LABEL_FILE = (
    FEATURE_DIR
    / "validation_supervision"
    / "cwq_validation_decision_labels.jsonl"
)

for path in [
    WEBQSP_VAL_FEATURE_MANIFEST,
    CWQ_VAL_FEATURE_MANIFEST,
    WEBQSP_VAL_FEATURE_NPZ,
    CWQ_VAL_FEATURE_NPZ,
    WEBQSP_LABEL_MANIFEST,
    CWQ_LABEL_MANIFEST,
    WEBQSP_LABEL_FILE,
    CWQ_LABEL_FILE,
]:
    assert path.exists(), (
        f"Missing artifact: {path}"
    )

print("\nFrozen feature/supervision artifacts: FOUND")


# ======================================================================
# 4. JSON helpers
# ======================================================================

def load_json(path):
    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        return json.load(f)


def load_jsonl(path):
    rows = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        for line in f:
            line = line.strip()

            if line:
                rows.append(
                    json.loads(line)
                )

    return rows


webqsp_feature_manifest_12c = load_json(
    WEBQSP_VAL_FEATURE_MANIFEST
)

cwq_feature_manifest_12c = load_json(
    CWQ_VAL_FEATURE_MANIFEST
)

webqsp_label_manifest_12c = load_json(
    WEBQSP_LABEL_MANIFEST
)

cwq_label_manifest_12c = load_json(
    CWQ_LABEL_MANIFEST
)


# ======================================================================
# 5. Pretty-print manifest structure
# ======================================================================

def print_json_structure(
    obj,
    prefix="",
    depth=0,
    max_depth=4
):
    if depth > max_depth:
        return

    if isinstance(obj, dict):
        for key, value in obj.items():

            full = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            if isinstance(value, dict):
                print(
                    f"  {full}: dict[{len(value)}]"
                )

                print_json_structure(
                    value,
                    full,
                    depth + 1,
                    max_depth
                )

            elif isinstance(value, list):
                print(
                    f"  {full}: list[{len(value)}]"
                )

                if (
                    len(value) <= 40
                    and all(
                        not isinstance(
                            x,
                            (dict, list)
                        )
                        for x in value
                    )
                ):
                    print(
                        "    ",
                        value
                    )

            else:
                text = str(value)

                if len(text) > 180:
                    text = (
                        text[:177]
                        + "..."
                    )

                print(
                    f"  {full}: {text}"
                )


print(
    "\n"
    + "=" * 100
)

print(
    "WEBQSP FEATURE MANIFEST STRUCTURE"
)

print(
    "=" * 100
)

print_json_structure(
    webqsp_feature_manifest_12c
)


print(
    "\n"
    + "=" * 100
)

print(
    "CWQ FEATURE MANIFEST STRUCTURE"
)

print(
    "=" * 100
)

print_json_structure(
    cwq_feature_manifest_12c
)


# ======================================================================
# 6. Search manifests recursively for important terms
# ======================================================================

IMPORTANT_TERMS = [
    "feature",
    "semantic",
    "surface",
    "encoder",
    "minilm",
    "embedding",
    "prefix",
    "candidate",
    "struct",
    "progress",
    "log1p",
    "cosine",
    "relation",
    "suffix",
    "mask",
    "mid",
]


def flatten_json(
    obj,
    prefix=""
):
    output = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            new_prefix = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            output.extend(
                flatten_json(
                    value,
                    new_prefix
                )
            )

    elif isinstance(obj, list):

        if all(
            not isinstance(
                x,
                (dict, list)
            )
            for x in obj
        ):
            output.append(
                (
                    prefix,
                    obj
                )
            )

        else:
            for i, value in enumerate(
                obj
            ):
                output.extend(
                    flatten_json(
                        value,
                        f"{prefix}[{i}]"
                    )
                )

    else:
        output.append(
            (
                prefix,
                obj
            )
        )

    return output


def print_relevant_manifest_entries(
    name,
    manifest
):
    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{name} — RELEVANT MANIFEST ENTRIES"
    )

    print(
        "=" * 100
    )

    flattened = flatten_json(
        manifest
    )

    shown = set()

    for path, value in flattened:
        combined = (
            path
            + " "
            + str(value)
        ).lower()

        if not any(
            term in combined
            for term
            in IMPORTANT_TERMS
        ):
            continue

        key = (
            path,
            str(value)
        )

        if key in shown:
            continue

        shown.add(
            key
        )

        text = str(value)

        if len(text) > 500:
            text = (
                text[:497]
                + "..."
            )

        print(
            f"{path}: {text}"
        )


print_relevant_manifest_entries(
    "WEBQSP",
    webqsp_feature_manifest_12c
)

print_relevant_manifest_entries(
    "CWQ",
    cwq_feature_manifest_12c
)


# ======================================================================
# 7. NPZ schema inspection
# ======================================================================

def inspect_npz(
    dataset,
    path
):
    data = np.load(
        path,
        allow_pickle=True
    )

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{dataset.upper()} FEATURE NPZ SCHEMA"
    )

    print(
        "=" * 100
    )

    for key in data.files:
        arr = data[key]

        print(
            f"{key:<30} "
            f"shape={str(arr.shape):<18} "
            f"dtype={arr.dtype}"
        )

        if (
            arr.ndim == 1
            and len(arr) <= 30
        ):
            print(
                "   ",
                arr.tolist()
            )

    return {
        key: data[key]
        for key in data.files
    }


webqsp_npz_12c = inspect_npz(
    "webqsp",
    WEBQSP_VAL_FEATURE_NPZ
)

cwq_npz_12c = inspect_npz(
    "cwq",
    CWQ_VAL_FEATURE_NPZ
)


# ======================================================================
# 8. Basic frozen shape gates
# ======================================================================

def resolve_npz_array(
    d,
    aliases
):
    for key in aliases:
        if key in d:
            return d[key]

    return None


webqsp_X_12c = resolve_npz_array(
    webqsp_npz_12c,
    [
        "X",
        "features",
        "x",
    ]
)

cwq_X_12c = resolve_npz_array(
    cwq_npz_12c,
    [
        "X",
        "features",
        "x",
    ]
)

assert webqsp_X_12c is not None
assert cwq_X_12c is not None

assert webqsp_X_12c.shape == (
    966,
    27
)

assert cwq_X_12c.shape == (
    18688,
    27
)

print(
    "\nFrozen validation feature shape gate: PASSED"
)


# ======================================================================
# 9. Feature-column numeric summary
# ======================================================================

def feature_numeric_summary(
    dataset,
    X
):
    assert X.shape[1] == 27

    rows = []

    for j, name in enumerate(
        EXPECTED_FEATURE_NAMES
    ):
        col = np.asarray(
            X[:, j],
            dtype=np.float64
        )

        rows.append(
            {
                "index":
                    j,

                "feature":
                    name,

                "min":
                    float(
                        np.min(col)
                    ),

                "max":
                    float(
                        np.max(col)
                    ),

                "mean":
                    float(
                        np.mean(col)
                    ),

                "std":
                    float(
                        np.std(col)
                    ),

                "zero_rate":
                    float(
                        np.mean(
                            np.isclose(
                                col,
                                0.0
                            )
                        )
                    ),

                "unique_rounded_6":
                    int(
                        len(
                            np.unique(
                                np.round(
                                    col,
                                    6
                                )
                            )
                        )
                    ),
            }
        )

    df = pd.DataFrame(
        rows
    )

    print(
        "\n"
        + "=" * 110
    )

    print(
        f"{dataset.upper()} FEATURE NUMERIC SUMMARY"
    )

    print(
        "=" * 110
    )

    print(
        df.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

    return df


webqsp_feature_numeric_12c = (
    feature_numeric_summary(
        "webqsp",
        webqsp_X_12c
    )
)

cwq_feature_numeric_12c = (
    feature_numeric_summary(
        "cwq",
        cwq_X_12c
    )
)


# ======================================================================
# 10. Validation supervision schema
# ======================================================================

webqsp_labels_12c = load_jsonl(
    WEBQSP_LABEL_FILE
)

cwq_labels_12c = load_jsonl(
    CWQ_LABEL_FILE
)

print(
    "\nValidation supervision rows:"
)

print(
    "  WebQSP:",
    len(webqsp_labels_12c)
)

print(
    "  CWQ:   ",
    len(cwq_labels_12c)
)


def describe_json_record(
    dataset,
    row
):
    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{dataset.upper()} VALIDATION SUPERVISION SAMPLE"
    )

    print(
        "=" * 100
    )

    for key, value in row.items():

        if isinstance(value, list):
            preview = value[:5]

            print(
                f"{key:<35} "
                f"list[{len(value)}] "
                f"{preview}"
            )

        elif isinstance(value, dict):
            print(
                f"{key:<35} "
                f"dict keys={list(value.keys())[:20]}"
            )

        else:
            text = str(value)

            if len(text) > 250:
                text = (
                    text[:247]
                    + "..."
                )

            print(
                f"{key:<35} "
                f"{type(value).__name__}: {text}"
            )


if webqsp_labels_12c:
    describe_json_record(
        "webqsp",
        webqsp_labels_12c[0]
    )

if cwq_labels_12c:
    describe_json_record(
        "cwq",
        cwq_labels_12c[0]
    )


# ======================================================================
# 11. Supervision manifest structures
# ======================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "WEBQSP VALIDATION SUPERVISION MANIFEST"
)

print(
    "=" * 100
)

print_json_structure(
    webqsp_label_manifest_12c
)


print(
    "\n"
    + "=" * 100
)

print(
    "CWQ VALIDATION SUPERVISION MANIFEST"
)

print(
    "=" * 100
)

print_json_structure(
    cwq_label_manifest_12c
)


# ======================================================================
# 12. Search filesystem for original Feature-v2 source
# ======================================================================
#
# Look for literal frozen feature names/version in:
#
#   .py
#   .ipynb
#   .md
#   .txt
#   .json
#
# Large graph/data files are skipped.
# ======================================================================

SOURCE_ROOTS = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

SOURCE_EXTENSIONS = {
    ".py",
    ".ipynb",
    ".md",
    ".txt",
    ".json",
}

SOURCE_NEEDLES = [
    "afp_features_v2_masked_entity_semantics",
    "sem_q_candidate",
    "path_candidate_prefix_entity_mean",
    "struct_log_endpoint_multiplicity",
    "prog_penultimate_indicator",
]

MAX_SOURCE_FILE_MB = 25


def search_source_files():
    hits = []

    for root in SOURCE_ROOTS:

        if not root.exists():
            continue

        for dirpath, dirnames, filenames in os.walk(
            root
        ):
            dirnames[:] = [
                d
                for d in dirnames
                if not d.startswith(".")
                and d not in {
                    "__pycache__",
                    "node_modules",
                }
            ]

            for filename in filenames:
                path = (
                    Path(dirpath)
                    / filename
                )

                if (
                    path.suffix.lower()
                    not in SOURCE_EXTENSIONS
                ):
                    continue

                text_path = str(
                    path
                ).lower()

                # Explicit TEST leakage safety.
                normalized_parts = re.split(
                    r"[/\\_\-.]+",
                    text_path
                )

                if "test" in normalized_parts:
                    continue

                try:
                    size_mb = (
                        path.stat().st_size
                        / (1024 ** 2)
                    )

                    if (
                        size_mb
                        > MAX_SOURCE_FILE_MB
                    ):
                        continue

                    content = path.read_text(
                        encoding="utf-8",
                        errors="ignore"
                    )

                except Exception:
                    continue

                matched = [
                    needle
                    for needle
                    in SOURCE_NEEDLES
                    if needle in content
                ]

                if matched:
                    hits.append(
                        {
                            "path":
                                str(path),

                            "size_mb":
                                size_mb,

                            "matched":
                                matched,
                        }
                    )

    hits.sort(
        key=lambda x: (
            -len(
                x["matched"]
            ),
            x["size_mb"],
            x["path"],
        )
    )

    return hits


feature_source_hits_12c = (
    search_source_files()
)

print(
    "\n"
    + "=" * 100
)

print(
    "FEATURE-v2 SOURCE SEARCH"
)

print(
    "=" * 100
)

if not feature_source_hits_12c:
    print(
        "No saved source file containing Feature-v2 literals found."
    )

else:
    for hit in feature_source_hits_12c[
        :30
    ]:
        print(
            f"{hit['size_mb']:.3f} MB  "
            f"{hit['path']}"
        )

        print(
            "   matched:",
            hit[
                "matched"
            ]
        )


# ======================================================================
# 13. Extract relevant notebook/code snippets when possible
# ======================================================================

def extract_source_snippets(
    path,
    needles,
    context_chars=2500
):
    path = Path(
        path
    )

    text = path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    snippets = []

    for needle in needles:

        pos = text.find(
            needle
        )

        if pos < 0:
            continue

        start = max(
            0,
            pos - context_chars
        )

        end = min(
            len(text),
            pos + context_chars
        )

        snippets.append(
            {
                "needle":
                    needle,

                "snippet":
                    text[
                        start:end
                    ],
            }
        )

    return snippets


print(
    "\n"
    + "=" * 100
)

print(
    "TOP SOURCE SNIPPETS"
)

print(
    "=" * 100
)

if feature_source_hits_12c:

    for hit in feature_source_hits_12c[
        :3
    ]:

        print(
            "\nSOURCE:",
            hit["path"]
        )

        snippets = (
            extract_source_snippets(
                hit["path"],
                hit["matched"]
            )
        )

        for snippet in snippets[
            :3
        ]:

            print(
                "\n--- around:",
                snippet[
                    "needle"
                ],
                "---"
            )

            print(
                snippet[
                    "snippet"
                ][:6000]
            )


# ======================================================================
# 14. Search for semantic embedding/cache artifacts
# ======================================================================

SEMANTIC_FILE_TOKENS = [
    "embedding",
    "embeddings",
    "semantic",
    "minilm",
    "sentence",
    "encoder",
    "cache",
]


def discover_semantic_artifacts():
    hits = []

    for root in SOURCE_ROOTS:

        if not root.exists():
            continue

        for dirpath, dirnames, filenames in os.walk(
            root
        ):
            dirnames[:] = [
                d
                for d in dirnames
                if not d.startswith(".")
                and d != "__pycache__"
            ]

            for filename in filenames:
                path = (
                    Path(dirpath)
                    / filename
                )

                text = str(
                    path
                ).lower()

                if not any(
                    token in text
                    for token
                    in SEMANTIC_FILE_TOKENS
                ):
                    continue

                # Avoid TEST files.
                parts = re.split(
                    r"[/\\_\-.]+",
                    text
                )

                if "test" in parts:
                    continue

                try:
                    size_mb = (
                        path.stat().st_size
                        / (1024 ** 2)
                    )
                except Exception:
                    size_mb = np.nan

                hits.append(
                    {
                        "path":
                            str(path),

                        "size_mb":
                            size_mb,
                    }
                )

    # Deduplicate
    unique = {
        x["path"]: x
        for x in hits
    }

    hits = list(
        unique.values()
    )

    hits.sort(
        key=lambda x: (
            x["path"]
        )
    )

    return hits


semantic_artifacts_12c = (
    discover_semantic_artifacts()
)

print(
    "\n"
    + "=" * 100
)

print(
    "SEMANTIC / EMBEDDING / CACHE ARTIFACTS"
)

print(
    "=" * 100
)

if not semantic_artifacts_12c:
    print(
        "No obvious semantic-cache artifact found."
    )

else:
    for hit in semantic_artifacts_12c[
        :50
    ]:

        size = hit[
            "size_mb"
        ]

        size_text = (
            f"{size:.3f} MB"
            if np.isfinite(size)
            else "?"
        )

        print(
            f"{size_text:>12}  "
            f"{hit['path']}"
        )


# ======================================================================
# 15. Search current globals for encoder/cache/extractor objects
# ======================================================================

GLOBAL_TOKENS = [
    "feature",
    "semantic",
    "embed",
    "encoder",
    "minilm",
    "sentence",
    "surface",
    "cache",
]


runtime_globals_12c = []

for name, obj in globals().items():

    if name.startswith(
        "_"
    ):
        continue

    low = name.lower()

    if any(
        token in low
        for token
        in GLOBAL_TOKENS
    ):

        runtime_globals_12c.append(
            {
                "name":
                    name,

                "type":
                    type(
                        obj
                    ).__name__,

                "callable":
                    callable(
                        obj
                    ),
            }
        )


runtime_globals_12c = sorted(
    runtime_globals_12c,
    key=lambda x:
        x["name"]
)

print(
    "\n"
    + "=" * 100
)

print(
    "CURRENT FEATURE/SEMANTIC-RELATED GLOBALS"
)

print(
    "=" * 100
)

for row in runtime_globals_12c:
    print(
        f"{row['name']:<50} "
        f"type={row['type']:<25} "
        f"callable={row['callable']}"
    )


# ======================================================================
# 16. Determine whether exact runtime implementation is recoverable
# ======================================================================

SOURCE_FOUND = (
    len(
        feature_source_hits_12c
    ) > 0
)

SEMANTIC_CACHE_FOUND = (
    len(
        semantic_artifacts_12c
    ) > 0
)

EXTRACTOR_GLOBALS = [
    row["name"]
    for row in runtime_globals_12c
    if row[
        "callable"
    ]
    and any(
        token
        in row["name"].lower()
        for token
        in [
            "feature",
            "embed",
            "semantic",
            "surface",
        ]
    )
]


# ======================================================================
# 17. Save audit manifest
# ======================================================================

TRAVERSAL_DIR = (
    RQ2_ROOT
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CELL12C_A_MANIFEST = {
    "cell":
        "RQ2_12C_A",

    "purpose":
        "recover_exact_frozen_feature_v2_runtime_specification",

    "feature_version":
        EXPECTED_FEATURE_VERSION,

    "feature_dim":
        EXPECTED_FEATURE_DIM,

    "feature_spec_sha256":
        EXPECTED_FEATURE_SHA256,

    "webqsp_validation_shape":
        list(
            webqsp_X_12c.shape
        ),

    "cwq_validation_shape":
        list(
            cwq_X_12c.shape
        ),

    "source_hits":
        feature_source_hits_12c,

    "semantic_artifact_count":
        len(
            semantic_artifacts_12c
        ),

    "extractor_globals":
        EXTRACTOR_GLOBALS,

    "source_found":
        SOURCE_FOUND,

    "semantic_cache_candidate_found":
        SEMANTIC_CACHE_FOUND,

    "new_feature_computation":
        False,

    "semantic_encoder_run":
        False,

    "pruning_run":
        False,

    "hyperparameter_tuning_run":
        False,

    "test_loaded":
        False,
}

CELL12C_A_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12c_a_feature_runtime_recovery_audit.json"
)

with open(
    CELL12C_A_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL12C_A_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )


# ======================================================================
# 18. Final report
# ======================================================================

print(
    "\n"
    + "=" * 104
)

print(
    "=== RQ2 CELL 12C-A: FROZEN FEATURE-v2 RUNTIME RECOVERY AUDIT COMPLETE ==="
)

print(
    "=" * 104
)

print(
    "\nFrozen Feature-v2"
)

print(
    "  Version:                     ",
    EXPECTED_FEATURE_VERSION
)

print(
    "  Dimension:                   ",
    EXPECTED_FEATURE_DIM
)

print(
    "  SHA256:                      ",
    EXPECTED_FEATURE_SHA256
)

print(
    "\nValidation caches"
)

print(
    "  WebQSP:                      ",
    webqsp_X_12c.shape
)

print(
    "  CWQ:                         ",
    cwq_X_12c.shape
)

print(
    "\nRecovery"
)

print(
    "  Feature source found:        ",
    SOURCE_FOUND
)

print(
    "  Semantic/cache candidates:   ",
    len(
        semantic_artifacts_12c
    )
)

print(
    "  Extractor globals found:     ",
    EXTRACTOR_GLOBALS
)

print(
    "\nExecution"
)

print(
    "  New features computed:       NO"
)

print(
    "  MiniLM/encoder run:          NO"
)

print(
    "  Pruning run:                 NO"
)

print(
    "  Hyperparameter tuning:       NO"
)

print(
    "  TEST loaded:                 NO"
)

print(
    "\nManifest:",
    CELL12C_A_MANIFEST_PATH
)

print(
    "\nNext: use this audit to implement Cell 12C-B "
    "and require exact reproduction of cached Feature-v2 values "
    "before enabling online AFP scoring."
)

RQ2 root:      /kaggle/working/step3_rq2_dev_v1
Feature root:  /kaggle/working/step3_rq2_dev_v1/03_features

Expected Feature-v2 identity:
  version: afp_features_v2_masked_entity_semantics
  dim:     27
  SHA256:  738985d1232a8ac5935c397ed95eca23377bc59b4a99494547fa7779062ade86

Frozen feature/supervision artifacts: FOUND

WEBQSP FEATURE MANIFEST STRUCTURE
  dataset: webqsp
  split: validation
  feature_version: afp_features_v2_masked_entity_semantics
  feature_spec_sha256: 738985d1232a8ac5935c397ed95eca23377bc59b4a99494547fa7779062ade86
  feature_dim: 27
  semantic_encoder: sentence-transformers/all-MiniLM-L6-v2
  raw_mid_policy: zero_embedding_plus_availability
  decision_only: True
  n_groups: 87
  n_branches: 966
  positive_branches: 319
  negative_branches: 647
  positive_rate: 0.3302277432712215
  npz_sha256: de6b1ed170f931812561aaee141439c22fb23bfdb63cd025f637caa6e1f7edeb
  created_utc: 2026-08-31T16:40:38.107811+00:00

CWQ FEATURE MANIFEST STRUCTURE
  dataset: cwq
  split: val

RuntimeError: dictionary changed size during iteration

In [11]:
# ======================================================================
# A RECOVERY — CONTINUE FROM SECTION 15
# ======================================================================
#
# Fixes:
#   RuntimeError: dictionary changed size during iteration
#
# Everything before Section 15 from the previous run remains valid.
# ======================================================================

import json
from pathlib import Path


# ======================================================================
# 15. Search current globals for encoder/cache/extractor objects
# ======================================================================

GLOBAL_TOKENS = [
    "feature",
    "semantic",
    "embed",
    "encoder",
    "minilm",
    "sentence",
    "surface",
    "cache",
]

runtime_globals_12c = []

# IMPORTANT:
# Snapshot globals BEFORE iterating.
# This prevents Jupyter/global-scope assignments from modifying the
# dictionary being iterated.
global_snapshot_12c = list(
    globals().items()
)

for name, obj in global_snapshot_12c:

    if name.startswith("_"):
        continue

    low = name.lower()

    if any(
        token in low
        for token in GLOBAL_TOKENS
    ):
        runtime_globals_12c.append(
            {
                "name":
                    name,

                "type":
                    type(obj).__name__,

                "callable":
                    callable(obj),
            }
        )


runtime_globals_12c = sorted(
    runtime_globals_12c,
    key=lambda x: x["name"]
)


print(
    "\n"
    + "=" * 100
)

print(
    "CURRENT FEATURE/SEMANTIC-RELATED GLOBALS"
)

print(
    "=" * 100
)

for row in runtime_globals_12c:

    print(
        f"{row['name']:<50} "
        f"type={row['type']:<25} "
        f"callable={row['callable']}"
    )


# ======================================================================
# 16. Determine exact runtime-recovery status
# ======================================================================

SOURCE_FOUND = (
    len(
        feature_source_hits_12c
    ) > 0
)

SEMANTIC_CACHE_FOUND = (
    len(
        semantic_artifacts_12c
    ) > 0
)

EXTRACTOR_GLOBALS = [
    row["name"]
    for row in runtime_globals_12c
    if row["callable"]
    and any(
        token in row["name"].lower()
        for token in [
            "feature",
            "embed",
            "semantic",
            "surface",
        ]
    )
]


# ======================================================================
# 17. Additional recovery interpretation
# ======================================================================
#
# We found the exact saved notebook source containing Feature-v2.
# This is sufficient to reconstruct the original feature code itself.
#
# However:
#   semantic cache candidate was not found by filename search.
#
# Therefore Cell 12C-B must separately determine whether:
#
#   A) MiniLM model/cache is locally available, or
#   B) original semantic embeddings can be recovered from notebook
#      artifacts, or
#   C) semantic encoding must be rerun.
#
# We will NOT silently change the feature representation.
# ======================================================================

FEATURE_RUNTIME_RECOVERY_STATUS = {
    "exact_feature_source_found":
        SOURCE_FOUND,

    "feature_source_primary":
        (
            feature_source_hits_12c[0]["path"]
            if SOURCE_FOUND
            else None
        ),

    "semantic_cache_candidate_found":
        SEMANTIC_CACHE_FOUND,

    "semantic_cache_candidates":
        len(
            semantic_artifacts_12c
        ),

    "extractor_globals":
        EXTRACTOR_GLOBALS,

    "safe_to_reconstruct_feature_code":
        SOURCE_FOUND,

    "safe_to_claim_online_semantic_runtime_ready":
        False,
}


print(
    "\n"
    + "=" * 100
)

print(
    "FEATURE-v2 RUNTIME RECOVERY STATUS"
)

print(
    "=" * 100
)

print(
    "Exact Feature-v2 source found:       ",
    SOURCE_FOUND
)

if SOURCE_FOUND:
    print(
        "Primary source:                    ",
        feature_source_hits_12c[0]["path"]
    )

print(
    "Semantic/cache candidates found:     ",
    len(
        semantic_artifacts_12c
    )
)

print(
    "Feature/semantic callable globals:   ",
    EXTRACTOR_GLOBALS
)

print(
    "Feature code reconstructable:        ",
    SOURCE_FOUND
)

print(
    "Online semantic runtime ready:       ",
    "NOT YET VERIFIED"
)


# ======================================================================
# 18. Save corrected audit manifest
# ======================================================================

TRAVERSAL_DIR = (
    RQ2_ROOT
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CELL12C_A_MANIFEST = {
    "cell":
        "RQ2_12C_A",

    "purpose":
        "recover_exact_frozen_feature_v2_runtime_specification",

    "feature_version":
        EXPECTED_FEATURE_VERSION,

    "feature_dim":
        EXPECTED_FEATURE_DIM,

    "feature_spec_sha256":
        EXPECTED_FEATURE_SHA256,

    "webqsp_validation_shape":
        list(
            webqsp_X_12c.shape
        ),

    "cwq_validation_shape":
        list(
            cwq_X_12c.shape
        ),

    "source_hits":
        feature_source_hits_12c,

    "primary_feature_source":
        (
            feature_source_hits_12c[0]["path"]
            if SOURCE_FOUND
            else None
        ),

    "semantic_artifact_count":
        len(
            semantic_artifacts_12c
        ),

    "extractor_globals":
        EXTRACTOR_GLOBALS,

    "source_found":
        SOURCE_FOUND,

    "semantic_cache_candidate_found":
        SEMANTIC_CACHE_FOUND,

    "runtime_recovery_status":
        FEATURE_RUNTIME_RECOVERY_STATUS,

    "new_feature_computation":
        False,

    "semantic_encoder_run":
        False,

    "pruning_run":
        False,

    "hyperparameter_tuning_run":
        False,

    "test_loaded":
        False,
}

CELL12C_A_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12c_a_feature_runtime_recovery_audit.json"
)

with open(
    CELL12C_A_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL12C_A_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )


# ======================================================================
# 19. Final report
# ======================================================================

print(
    "\n"
    + "=" * 104
)

print(
    "=== RQ2 CELL 12C-A: FROZEN FEATURE-v2 RUNTIME RECOVERY AUDIT COMPLETE ==="
)

print(
    "=" * 104
)

print("\nFrozen Feature-v2")

print(
    "  Version:                     ",
    EXPECTED_FEATURE_VERSION
)

print(
    "  Dimension:                   ",
    EXPECTED_FEATURE_DIM
)

print(
    "  SHA256:                      ",
    EXPECTED_FEATURE_SHA256
)


print("\nValidation caches")

print(
    "  WebQSP:                      ",
    webqsp_X_12c.shape
)

print(
    "  CWQ:                         ",
    cwq_X_12c.shape
)


print("\nRecovery")

print(
    "  Exact feature source found:  ",
    SOURCE_FOUND
)

print(
    "  Primary source:              ",
    (
        feature_source_hits_12c[0]["path"]
        if SOURCE_FOUND
        else "NONE"
    )
)

print(
    "  Semantic/cache candidates:   ",
    len(
        semantic_artifacts_12c
    )
)

print(
    "  Extractor globals found:     ",
    EXTRACTOR_GLOBALS
)


print("\nExecution")

print(
    "  New features computed:       NO"
)

print(
    "  MiniLM/encoder run:          NO"
)

print(
    "  Pruning run:                 NO"
)

print(
    "  Hyperparameter tuning:       NO"
)

print(
    "  TEST loaded:                 NO"
)


print(
    "\nManifest:",
    CELL12C_A_MANIFEST_PATH
)

print(
    "\nNext: Cell 12C-B — reconstruct exact Feature-v2 runtime "
    "from the saved notebook source and verify online features "
    "against the frozen validation NPZ before pruning."
)


CURRENT FEATURE/SEMANTIC-RELATED GLOBALS
AFPFeatureStandardizer                             type=type                      callable=True
AFP_FEATURE_DIM                                    type=int                       callable=False
AFP_FEATURE_SPEC_SHA256                            type=str                       callable=False
AFP_FEATURE_VERSION                                type=str                       callable=False
AFP_SEMANTIC_ENCODER_NAME                          type=str                       callable=False
CWQ_VAL_FEATURE_MANIFEST                           type=PosixPath                 callable=False
CWQ_VAL_FEATURE_NPZ                                type=PosixPath                 callable=False
EXPECTED_FEATURE_DIM                               type=int                       callable=False
EXPECTED_FEATURE_NAMES                             type=list                      callable=False
EXPECTED_FEATURE_SHA                               type=str                       call

In [13]:
# ======================================================================
# STATIC RECOVERY OF EXACT FEATURE-v2 SOURCE
# ======================================================================
#
# FIX:
#   Previous B1 executed top-level assignments from Cell 167.
#   One of those instantiated MockSemanticEncoder, whose definition
#   lives elsewhere in the notebook.
#
# This recovery cell performs STATIC SOURCE ANALYSIS ONLY.
#
# It does NOT execute Cell 167.
# Therefore:
#   - no MockSemanticEncoder dependency
#   - no MiniLM
#   - no feature computation
#   - no pruning
#   - no tuning
#   - no TEST
# ======================================================================

import ast
import json
import re
from pathlib import Path


# ======================================================================
# 1. Load saved notebook
# ======================================================================

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb"
)

assert NOTEBOOK_PATH.exists()

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:
    saved_notebook_12cb = json.load(f)

print("Notebook:", NOTEBOOK_PATH)
print("Cells:   ", len(saved_notebook_12cb["cells"]))


# ======================================================================
# 2. Collect code cells
# ======================================================================

code_cells_12cb = []

for cell_index, cell in enumerate(
    saved_notebook_12cb["cells"]
):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    code_cells_12cb.append(
        {
            "cell_index": cell_index,
            "source": source,
        }
    )

print("Code cells:", len(code_cells_12cb))


# ======================================================================
# 3. Locate exact Feature-v2 definition cell
# ======================================================================

feature_definition_candidates = []

for item in code_cells_12cb:
    source = item["source"]

    if (
        'AFP_FEATURE_VERSION = "afp_features_v2_masked_entity_semantics"'
        in source
        and "sem_q_candidate" in source
        and "struct_log_endpoint_multiplicity" in source
        and "prog_penultimate_indicator" in source
    ):
        feature_definition_candidates.append(
            item
        )

assert len(feature_definition_candidates) == 1, (
    "Expected exactly one Feature-v2 definition cell; "
    f"found {len(feature_definition_candidates)}"
)

FEATURE_DEFINITION_CELL = (
    feature_definition_candidates[0]
)

FEATURE_SOURCE_12CB = (
    FEATURE_DEFINITION_CELL["source"]
)

FEATURE_CELL_INDEX_12CB = (
    FEATURE_DEFINITION_CELL["cell_index"]
)

print(
    "\nFeature-v2 definition cell:",
    FEATURE_CELL_INDEX_12CB
)


# ======================================================================
# 4. Parse source WITHOUT executing it
# ======================================================================

feature_tree_12cb = ast.parse(
    FEATURE_SOURCE_12CB,
    filename="saved_notebook_feature_v2"
)

print("Static AST parse: PASSED")


# ======================================================================
# 5. Static constant extractor
# ======================================================================

def get_assignment_node(
    tree,
    variable_name
):
    for node in tree.body:

        if isinstance(node, ast.Assign):
            for target in node.targets:

                if (
                    isinstance(target, ast.Name)
                    and target.id == variable_name
                ):
                    return node.value

        elif isinstance(node, ast.AnnAssign):

            if (
                isinstance(node.target, ast.Name)
                and node.target.id == variable_name
            ):
                return node.value

    return None


def literal_assignment(
    tree,
    variable_name
):
    node = get_assignment_node(
        tree,
        variable_name
    )

    if node is None:
        return None

    try:
        return ast.literal_eval(
            node
        )
    except Exception:
        return None


# ======================================================================
# 6. Recover literal Feature-v2 identity
# ======================================================================

STATIC_FEATURE_VERSION = (
    literal_assignment(
        feature_tree_12cb,
        "AFP_FEATURE_VERSION"
    )
)

STATIC_SEMANTIC_ENCODER = (
    literal_assignment(
        feature_tree_12cb,
        "AFP_SEMANTIC_ENCODER_NAME"
    )
)

STATIC_SEMANTIC_DIM = (
    literal_assignment(
        feature_tree_12cb,
        "AFP_SEMANTIC_EXPECTED_DIM"
    )
)

print("\nStatic identity recovery:")
print(
    "  feature version:  ",
    STATIC_FEATURE_VERSION
)
print(
    "  encoder:          ",
    STATIC_SEMANTIC_ENCODER
)
print(
    "  semantic dim:     ",
    STATIC_SEMANTIC_DIM
)

assert (
    STATIC_FEATURE_VERSION
    ==
    EXPECTED_FEATURE_VERSION
)

assert (
    STATIC_SEMANTIC_ENCODER
    ==
    "sentence-transformers/all-MiniLM-L6-v2"
)

assert (
    STATIC_SEMANTIC_DIM
    == 384
)

print("Static identity gates: PASSED")


# ======================================================================
# 7. Recover exact feature-name lists statically
# ======================================================================

FEATURE_NAME_VARIABLES = [
    "AFP_SEMANTIC_FEATURE_NAMES",
    "AFP_PATH_FEATURE_NAMES",
    "AFP_STRUCTURAL_FEATURE_NAMES",
    "AFP_PROGRESS_FEATURE_NAMES",
]

static_feature_groups_12cb = {}

for variable in FEATURE_NAME_VARIABLES:

    value = literal_assignment(
        feature_tree_12cb,
        variable
    )

    assert isinstance(
        value,
        list
    ), (
        f"Could not statically recover {variable}"
    )

    static_feature_groups_12cb[
        variable
    ] = value


STATIC_FEATURE_NAMES = (
    static_feature_groups_12cb[
        "AFP_SEMANTIC_FEATURE_NAMES"
    ]
    +
    static_feature_groups_12cb[
        "AFP_PATH_FEATURE_NAMES"
    ]
    +
    static_feature_groups_12cb[
        "AFP_STRUCTURAL_FEATURE_NAMES"
    ]
    +
    static_feature_groups_12cb[
        "AFP_PROGRESS_FEATURE_NAMES"
    ]
)

assert len(
    STATIC_FEATURE_NAMES
) == 27

assert (
    STATIC_FEATURE_NAMES
    ==
    EXPECTED_FEATURE_NAMES
)

print(
    "Feature-name/order static gate: PASSED"
)


# ======================================================================
# 8. Verify frozen SHA from already persisted manifest
# ======================================================================
#
# We do NOT need to execute the hashing expression inside Cell 167.
# The persisted feature artifact already carries the frozen SHA.
# ======================================================================

assert (
    webqsp_feature_manifest_12c[
        "feature_spec_sha256"
    ]
    ==
    EXPECTED_FEATURE_SHA256
)

assert (
    cwq_feature_manifest_12c[
        "feature_spec_sha256"
    ]
    ==
    EXPECTED_FEATURE_SHA256
)

print(
    "Persisted Feature-v2 SHA gate: PASSED"
)


# ======================================================================
# 9. Enumerate exact function definitions in Feature-v2 cell
# ======================================================================

feature_functions_12cb = []
feature_classes_12cb = []

for node in feature_tree_12cb.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        )
    ):

        source_segment = (
            ast.get_source_segment(
                FEATURE_SOURCE_12CB,
                node
            )
            or ""
        )

        # Reconstruct readable signature directly from source line.
        first_line = (
            source_segment
            .splitlines()[0]
            if source_segment
            else node.name
        )

        feature_functions_12cb.append(
            {
                "name":
                    node.name,

                "line":
                    int(
                        getattr(
                            node,
                            "lineno",
                            -1
                        )
                    ),

                "end_line":
                    int(
                        getattr(
                            node,
                            "end_lineno",
                            -1
                        )
                    ),

                "first_line":
                    first_line,

                "source":
                    source_segment,
            }
        )

    elif isinstance(
        node,
        ast.ClassDef
    ):

        feature_classes_12cb.append(
            {
                "name":
                    node.name,

                "line":
                    int(
                        getattr(
                            node,
                            "lineno",
                            -1
                        )
                    ),

                "source":
                    (
                        ast.get_source_segment(
                            FEATURE_SOURCE_12CB,
                            node
                        )
                        or ""
                    ),
            }
        )


print(
    "\n"
    + "=" * 105
)

print(
    "FUNCTIONS RECOVERED FROM EXACT FEATURE-v2 CELL"
)

print(
    "=" * 105
)

for item in feature_functions_12cb:
    print(
        f"line={item['line']:<5} "
        f"{item['first_line']}"
    )


print(
    "\nClasses defined directly in Feature-v2 cell:"
)

if feature_classes_12cb:
    for item in feature_classes_12cb:
        print(
            f"line={item['line']:<5} "
            f"class {item['name']}"
        )
else:
    print("NONE")


# ======================================================================
# 10. Rank likely feature-builder functions by SOURCE CONTENT
# ======================================================================

BUILDER_MARKERS = [
    "candidate_entity",
    "prefix_entities",
    "semantic_encoder",
    "candidate_count",
    "AFP_FEATURE_DIM",
    "AFP_FEATURE_NAMES",
    "struct_log_candidate_count",
    "prog_hop_fraction",
]

feature_builder_candidates_12cb = []

for item in feature_functions_12cb:

    source = item[
        "source"
    ]

    score = sum(
        marker in source
        for marker
        in BUILDER_MARKERS
    )

    low_name = item[
        "name"
    ].lower()

    if any(
        token in low_name
        for token in [
            "feature",
            "candidate",
            "branch",
            "extract",
            "build",
        ]
    ):
        score += 2

    if score > 0:
        feature_builder_candidates_12cb.append(
            {
                "name":
                    item["name"],

                "line":
                    item["line"],

                "score":
                    score,

                "first_line":
                    item[
                        "first_line"
                    ],

                "source":
                    source,
            }
        )


feature_builder_candidates_12cb = sorted(
    feature_builder_candidates_12cb,
    key=lambda x: (
        -x["score"],
        x["line"]
    )
)


print(
    "\n"
    + "=" * 105
)

print(
    "LIKELY FEATURE BUILDER FUNCTIONS"
)

print(
    "=" * 105
)

for item in feature_builder_candidates_12cb:
    print(
        f"score={item['score']:<3} "
        f"line={item['line']:<5} "
        f"{item['first_line']}"
    )


# ======================================================================
# 11. Print top builder SOURCE exactly
# ======================================================================

TOP_FEATURE_BUILDER_12CB = (
    feature_builder_candidates_12cb[0]
    if feature_builder_candidates_12cb
    else None
)

print(
    "\n"
    + "=" * 105
)

print(
    "TOP FEATURE BUILDER SOURCE"
)

print(
    "=" * 105
)

if TOP_FEATURE_BUILDER_12CB is None:

    print(
        "No likely builder found."
    )

else:

    print(
        "Name:",
        TOP_FEATURE_BUILDER_12CB[
            "name"
        ]
    )

    print(
        "Line:",
        TOP_FEATURE_BUILDER_12CB[
            "line"
        ]
    )

    print()

    print(
        TOP_FEATURE_BUILDER_12CB[
            "source"
        ]
    )


# ======================================================================
# 12. Search entire notebook for MockSemanticEncoder
# ======================================================================

mock_semantic_locations_12cb = []

for item in code_cells_12cb:

    if "MockSemanticEncoder" in item[
        "source"
    ]:

        mock_semantic_locations_12cb.append(
            item[
                "cell_index"
            ]
        )


print(
    "\nMockSemanticEncoder appears in cells:",
    mock_semantic_locations_12cb
)


# ======================================================================
# 13. Search whole notebook for semantic encoder classes/functions
# ======================================================================

SEMANTIC_MARKERS = [
    "SentenceTransformer",
    "semantic_encoder",
    "SemanticEncoder",
    "MockSemanticEncoder",
    "all-MiniLM-L6-v2",
]


semantic_source_cells_12cb = []

for item in code_cells_12cb:

    source = item[
        "source"
    ]

    score = sum(
        marker.lower()
        in source.lower()
        for marker
        in SEMANTIC_MARKERS
    )

    if score > 0:

        semantic_source_cells_12cb.append(
            {
                "cell_index":
                    item[
                        "cell_index"
                    ],

                "score":
                    score,

                "source":
                    source,
            }
        )


semantic_source_cells_12cb = sorted(
    semantic_source_cells_12cb,
    key=lambda x: (
        -x["score"],
        x["cell_index"]
    )
)


print(
    "\n"
    + "=" * 105
)

print(
    "SEMANTIC ENCODER SOURCE-CELL CANDIDATES"
)

print(
    "=" * 105
)

for item in semantic_source_cells_12cb[
    :20
]:

    print(
        f"cell={item['cell_index']:<5} "
        f"score={item['score']}"
    )


# ======================================================================
# 14. Static inventory of semantic-related definitions
# ======================================================================

semantic_definition_inventory_12cb = []

for item in semantic_source_cells_12cb:

    try:
        tree = ast.parse(
            item[
                "source"
            ]
        )
    except SyntaxError:
        continue

    for node in tree.body:

        if isinstance(
            node,
            ast.ClassDef
        ):

            low = node.name.lower()

            if any(
                token in low
                for token in [
                    "semantic",
                    "embed",
                    "encoder",
                    "mock",
                ]
            ):

                semantic_definition_inventory_12cb.append(
                    {
                        "cell":
                            item[
                                "cell_index"
                            ],

                        "type":
                            "class",

                        "name":
                            node.name,

                        "source":
                            (
                                ast.get_source_segment(
                                    item[
                                        "source"
                                    ],
                                    node
                                )
                                or ""
                            ),
                    }
                )

        elif isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        ):

            low = node.name.lower()

            if any(
                token in low
                for token in [
                    "semantic",
                    "embed",
                    "encoder",
                    "surface",
                ]
            ):

                semantic_definition_inventory_12cb.append(
                    {
                        "cell":
                            item[
                                "cell_index"
                            ],

                        "type":
                            "function",

                        "name":
                            node.name,

                        "source":
                            (
                                ast.get_source_segment(
                                    item[
                                        "source"
                                    ],
                                    node
                                )
                                or ""
                            ),
                    }
                )


print(
    "\n"
    + "=" * 105
)

print(
    "SEMANTIC/ENCODER DEFINITIONS FOUND IN SAVED NOTEBOOK"
)

print(
    "=" * 105
)

for item in semantic_definition_inventory_12cb:

    print(
        f"cell={item['cell']:<5} "
        f"{item['type']:<10} "
        f"{item['name']}"
    )


# ======================================================================
# 15. Print sources of relevant semantic classes/functions
# ======================================================================

print(
    "\n"
    + "=" * 105
)

print(
    "RELEVANT SEMANTIC DEFINITION SOURCES"
)

print(
    "=" * 105
)

for item in semantic_definition_inventory_12cb:

    print(
        "\n"
        + "-" * 100
    )

    print(
        f"Cell {item['cell']} | "
        f"{item['type']} {item['name']}"
    )

    print(
        "-" * 100
    )

    print(
        item[
            "source"
        ]
    )


# ======================================================================
# 16. Check whether SentenceTransformer package/model is available
#     WITHOUT loading a model
# ======================================================================

sentence_transformers_importable_12cb = False
sentence_transformers_version_12cb = None

try:

    import sentence_transformers

    sentence_transformers_importable_12cb = True

    sentence_transformers_version_12cb = getattr(
        sentence_transformers,
        "__version__",
        "unknown"
    )

except Exception:

    sentence_transformers_importable_12cb = False


print(
    "\nSentence-transformers importable:",
    sentence_transformers_importable_12cb
)

print(
    "Version:",
    sentence_transformers_version_12cb
)


# ======================================================================
# 17. Save static recovery manifest
# ======================================================================

TRAVERSAL_DIR = (
    Path(RQ2_ROOT)
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CELL12C_B1_STATIC_MANIFEST = {
    "cell":
        "RQ2_12C_B1_R",

    "recovery_mode":
        "static_source_analysis_only",

    "feature_source":
        str(
            NOTEBOOK_PATH
        ),

    "feature_definition_cell":
        FEATURE_CELL_INDEX_12CB,

    "feature_version":
        STATIC_FEATURE_VERSION,

    "semantic_encoder":
        STATIC_SEMANTIC_ENCODER,

    "semantic_dim":
        STATIC_SEMANTIC_DIM,

    "feature_dim":
        len(
            STATIC_FEATURE_NAMES
        ),

    "feature_names_match":
        True,

    "feature_sha_manifest_match":
        True,

    "functions":
        [
            {
                "name":
                    x["name"],

                "line":
                    x["line"],

                "first_line":
                    x["first_line"],
            }
            for x in
            feature_functions_12cb
        ],

    "builder_candidates":
        [
            {
                "name":
                    x["name"],

                "line":
                    x["line"],

                "score":
                    x["score"],

                "first_line":
                    x["first_line"],
            }
            for x in
            feature_builder_candidates_12cb
        ],

    "semantic_definitions":
        [
            {
                "cell":
                    x["cell"],

                "type":
                    x["type"],

                "name":
                    x["name"],
            }
            for x in
            semantic_definition_inventory_12cb
        ],

    "sentence_transformers_importable":
        sentence_transformers_importable_12cb,

    "sentence_transformers_version":
        sentence_transformers_version_12cb,

    "notebook_code_executed":
        False,

    "minilm_loaded":
        False,

    "new_features_computed":
        False,

    "pruning_run":
        False,

    "hyperparameter_tuning":
        False,

    "test_loaded":
        False,
}


CELL12C_B1_STATIC_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12c_b1_static_feature_source_recovery.json"
)


with open(
    CELL12C_B1_STATIC_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL12C_B1_STATIC_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )


# ======================================================================
# 18. Final report
# ======================================================================

print(
    "\n"
    + "=" * 108
)

print(
    "=== RQ2 CELL 12C-B1-R: EXACT FEATURE-v2 SOURCE STATICALLY RECOVERED ==="
)

print(
    "=" * 108
)

print(
    "Feature definition cell:    ",
    FEATURE_CELL_INDEX_12CB
)

print(
    "Feature version:            ",
    STATIC_FEATURE_VERSION
)

print(
    "Semantic encoder:           ",
    STATIC_SEMANTIC_ENCODER
)

print(
    "Semantic dimension:         ",
    STATIC_SEMANTIC_DIM
)

print(
    "Feature dimension:          ",
    len(
        STATIC_FEATURE_NAMES
    )
)

print(
    "Feature name/order gate:    PASS"
)

print(
    "Feature SHA manifest gate:  PASS"
)

print(
    "Notebook Feature cell run:  NO"
)

print(
    "MiniLM loaded:              NO"
)

print(
    "New features computed:      NO"
)

print(
    "Pruning run:                NO"
)

print(
    "Hyperparameter tuning:      NO"
)

print(
    "TEST loaded:                NO"
)

print(
    "\nManifest:",
    CELL12C_B1_STATIC_MANIFEST_PATH
)

print(
    "\nNext: use the recovered builder + encoder definitions "
    "to construct the runtime scorer and reproduce frozen NPZ "
    "features before any tuning."
)

Notebook: /kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb
Cells:    190
Code cells: 94

Feature-v2 definition cell: 167
Static AST parse: PASSED

Static identity recovery:
  feature version:   afp_features_v2_masked_entity_semantics
  encoder:           sentence-transformers/all-MiniLM-L6-v2
  semantic dim:      384
Static identity gates: PASSED
Feature-name/order static gate: PASSED
Persisted Feature-v2 SHA gate: PASSED

FUNCTIONS RECOVERED FROM EXACT FEATURE-v2 CELL
line=66    def is_raw_freebase_id(x):
line=77    def has_readable_entity_surface(
line=105   def normalize_surface_text(x):
line=130   def relation_surface_text(relation):
line=136   def entity_surface_text(
line=168   def relation_sequence_text(relations):
line=182   def safe_l2_normalize(
line=208   def safe_cosine(a, b):
line=227   def mean_embedding(
line=261   def get_safe_entity_embedding(
line=480   def extract_afp_candidate_features(
line=964   def extract_afp_group_features(
line=1056  def spli

In [14]:
# ======================================================================
# STATIC RECOVERY OF EXACT SEMANTIC RUNTIME + GROUP BUILDER
# ======================================================================
#
# PURPOSE
# -------
# Recover, without executing:
#
#   1. Exact semantic-encoder implementation/configuration.
#   2. Exact extract_afp_group_features(...) source.
#   3. Exact notebook call sites used to build Feature-v2.
#   4. Any encoder constructor / device / batch / normalization settings.
#
# NO MiniLM loading.
# NO new features.
# NO pruning.
# NO tuning.
# NO TEST.
# ======================================================================

import ast
import json
import re
from pathlib import Path


# ======================================================================
# 1. Reload saved notebook independently
# ======================================================================

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb"
)

assert NOTEBOOK_PATH.exists()

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:
    nb_12cb2 = json.load(f)


def notebook_cell_source(cell_index):
    cell = nb_12cb2["cells"][cell_index]

    assert cell["cell_type"] == "code"

    return "".join(
        cell.get("source", [])
    )


# Known from B1-R
CELL_FEATURE = 167
CELL_SEMANTIC_CANDIDATE = 154
CELL_BUILD_CANDIDATE = 173

feature_source = notebook_cell_source(
    CELL_FEATURE
)

semantic_source_154 = notebook_cell_source(
    CELL_SEMANTIC_CANDIDATE
)

build_source_173 = notebook_cell_source(
    CELL_BUILD_CANDIDATE
)

print("Notebook:", NOTEBOOK_PATH)
print("Feature cell:", CELL_FEATURE)
print("Semantic candidate cell:", CELL_SEMANTIC_CANDIDATE)
print("Build candidate cell:", CELL_BUILD_CANDIDATE)


# ======================================================================
# 2. Generic exact AST source extractor
# ======================================================================

def parse_source(source, label):
    try:
        return ast.parse(
            source,
            filename=label
        )
    except SyntaxError as e:
        raise RuntimeError(
            f"AST parse failed for {label}: {e}"
        )


feature_tree = parse_source(
    feature_source,
    "feature_cell_167"
)

semantic_tree_154 = parse_source(
    semantic_source_154,
    "semantic_cell_154"
)

build_tree_173 = parse_source(
    build_source_173,
    "build_cell_173"
)


def exact_source(source, node):
    return (
        ast.get_source_segment(
            source,
            node
        )
        or ""
    )


# ======================================================================
# 3. Recover exact extract_afp_group_features source
# ======================================================================

group_builder_node = None

for node in feature_tree.body:
    if (
        isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        )
        and node.name
        == "extract_afp_group_features"
    ):
        group_builder_node = node
        break


assert group_builder_node is not None

GROUP_BUILDER_SOURCE_12CB2 = (
    exact_source(
        feature_source,
        group_builder_node
    )
)

print(
    "\n"
    + "=" * 110
)

print(
    "EXACT extract_afp_group_features SOURCE"
)

print(
    "=" * 110
)

print(
    GROUP_BUILDER_SOURCE_12CB2
)


# ======================================================================
# 4. Inventory ALL top-level definitions in Cell 154
# ======================================================================

semantic_defs_154 = []

for node in semantic_tree_154.body:

    if isinstance(
        node,
        ast.ClassDef
    ):
        semantic_defs_154.append(
            {
                "type":
                    "class",

                "name":
                    node.name,

                "line":
                    node.lineno,

                "source":
                    exact_source(
                        semantic_source_154,
                        node
                    ),
            }
        )

    elif isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        )
    ):
        semantic_defs_154.append(
            {
                "type":
                    "function",

                "name":
                    node.name,

                "line":
                    node.lineno,

                "source":
                    exact_source(
                        semantic_source_154,
                        node
                    ),
            }
        )


print(
    "\n"
    + "=" * 110
)

print(
    "ALL TOP-LEVEL DEFINITIONS IN SEMANTIC CELL 154"
)

print(
    "=" * 110
)

if not semantic_defs_154:
    print("NONE")

for item in semantic_defs_154:
    first_line = (
        item["source"]
        .splitlines()[0]
        if item["source"]
        else item["name"]
    )

    print(
        f"line={item['line']:<5} "
        f"{item['type']:<10} "
        f"{first_line}"
    )


# ======================================================================
# 5. Print exact Cell-154 definitions
# ======================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "EXACT SEMANTIC CELL 154 DEFINITION SOURCES"
)

print(
    "=" * 110
)

for item in semantic_defs_154:

    print(
        "\n"
        + "-" * 105
    )

    print(
        f"{item['type'].upper()} "
        f"{item['name']} "
        f"(line {item['line']})"
    )

    print(
        "-" * 105
    )

    print(
        item["source"]
    )


# ======================================================================
# 6. Find relevant imports in Cell 154
# ======================================================================

semantic_imports_154 = []

for node in semantic_tree_154.body:

    if isinstance(
        node,
        (
            ast.Import,
            ast.ImportFrom,
        )
    ):
        semantic_imports_154.append(
            exact_source(
                semantic_source_154,
                node
            )
        )


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 154 IMPORTS"
)

print(
    "=" * 110
)

for src in semantic_imports_154:
    print(src)


# ======================================================================
# 7. Extract relevant top-level assignments / expressions
# ======================================================================

RUNTIME_TERMS = [
    "semantic",
    "encoder",
    "sentence",
    "transformer",
    "minilm",
    "model",
    "device",
    "batch",
    "cache",
    "normalize",
    "embedding",
]


def relevant_top_level_statements(
    source,
    tree
):
    rows = []

    for node in tree.body:

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
                ast.Import,
                ast.ImportFrom,
            )
        ):
            continue

        segment = exact_source(
            source,
            node
        )

        low = segment.lower()

        if any(
            term in low
            for term in RUNTIME_TERMS
        ):
            rows.append(
                {
                    "line":
                        getattr(
                            node,
                            "lineno",
                            -1
                        ),

                    "source":
                        segment,
                }
            )

    return rows


semantic_runtime_statements_154 = (
    relevant_top_level_statements(
        semantic_source_154,
        semantic_tree_154
    )
)

build_runtime_statements_173 = (
    relevant_top_level_statements(
        build_source_173,
        build_tree_173
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 154 RELEVANT RUNTIME STATEMENTS"
)

print(
    "=" * 110
)

if not semantic_runtime_statements_154:
    print("NONE")

for item in semantic_runtime_statements_154:
    print(
        f"\n[line {item['line']}]\n"
        f"{item['source']}"
    )


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 173 RELEVANT RUNTIME STATEMENTS"
)

print(
    "=" * 110
)

if not build_runtime_statements_173:
    print("NONE")

for item in build_runtime_statements_173:
    print(
        f"\n[line {item['line']}]\n"
        f"{item['source']}"
    )


# ======================================================================
# 8. Search ALL notebook cells for encoder construction
# ======================================================================

SEARCH_PATTERNS = [
    "SentenceTransformer(",
    "semantic_encoder =",
    "semantic_encoder=",
    "SemanticEncoder(",
    "Cached",
    "MiniLM",
    "all-MiniLM-L6-v2",
]

encoder_runtime_hits = []

for cell_index, cell in enumerate(
    nb_12cb2["cells"]
):
    if cell.get(
        "cell_type"
    ) != "code":
        continue

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    matched = [
        pattern
        for pattern in SEARCH_PATTERNS
        if pattern.lower()
        in source.lower()
    ]

    if matched:
        encoder_runtime_hits.append(
            {
                "cell":
                    cell_index,

                "matched":
                    matched,

                "source":
                    source,
            }
        )


print(
    "\n"
    + "=" * 110
)

print(
    "NOTEBOOK CELLS CONTAINING SEMANTIC ENCODER CONSTRUCTION"
)

print(
    "=" * 110
)

for hit in encoder_runtime_hits:

    print(
        f"\nCELL {hit['cell']} "
        f"| matched={hit['matched']}"
    )

    print(
        "-" * 100
    )

    # Print only relevant lines + nearby context.
    lines = hit[
        "source"
    ].splitlines()

    relevant_indices = []

    for i, line in enumerate(
        lines
    ):
        low = line.lower()

        if any(
            pattern.lower()
            in low
            for pattern
            in SEARCH_PATTERNS
        ):
            relevant_indices.extend(
                range(
                    max(0, i - 5),
                    min(
                        len(lines),
                        i + 12
                    )
                )
            )

    relevant_indices = sorted(
        set(
            relevant_indices
        )
    )

    for i in relevant_indices:
        print(
            f"{i+1:04d}: "
            f"{lines[i]}"
        )


# ======================================================================
# 9. Find exact Feature-v2 call sites across notebook
# ======================================================================

CALL_TARGETS = {
    "extract_afp_candidate_features",
    "extract_afp_group_features",
    "collect_semantic_texts",
}


feature_call_sites = []

for cell_index, cell in enumerate(
    nb_12cb2["cells"]
):

    if cell.get(
        "cell_type"
    ) != "code":
        continue

    source = "".join(
        cell.get(
            "source",
            []
        )
    )

    try:
        tree = ast.parse(
            source
        )
    except Exception:
        continue

    for node in ast.walk(
        tree
    ):
        if not isinstance(
            node,
            ast.Call
        ):
            continue

        function_name = None

        if isinstance(
            node.func,
            ast.Name
        ):
            function_name = (
                node.func.id
            )

        elif isinstance(
            node.func,
            ast.Attribute
        ):
            function_name = (
                node.func.attr
            )

        if (
            function_name
            not in CALL_TARGETS
        ):
            continue

        segment = exact_source(
            source,
            node
        )

        feature_call_sites.append(
            {
                "cell":
                    cell_index,

                "line":
                    getattr(
                        node,
                        "lineno",
                        -1
                    ),

                "function":
                    function_name,

                "source":
                    segment,
            }
        )


print(
    "\n"
    + "=" * 110
)

print(
    "EXACT FEATURE-v2 CALL SITES"
)

print(
    "=" * 110
)

for item in feature_call_sites:

    print(
        f"\ncell={item['cell']} "
        f"line={item['line']} "
        f"function={item['function']}"
    )

    print(
        item["source"]
    )


# ======================================================================
# 10. Recover other builder/helper definitions from Cell 173
# ======================================================================

defs_173 = []

for node in build_tree_173.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
            ast.ClassDef,
        )
    ):

        defs_173.append(
            {
                "name":
                    node.name,

                "type":
                    (
                        "class"
                        if isinstance(
                            node,
                            ast.ClassDef
                        )
                        else "function"
                    ),

                "line":
                    node.lineno,

                "source":
                    exact_source(
                        build_source_173,
                        node
                    ),
            }
        )


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 173 DEFINITIONS"
)

print(
    "=" * 110
)

for item in defs_173:

    first_line = (
        item["source"]
        .splitlines()[0]
        if item["source"]
        else item["name"]
    )

    print(
        f"line={item['line']:<5} "
        f"{item['type']:<10} "
        f"{first_line}"
    )


# ======================================================================
# 11. Print build/helper definitions likely needed for reproduction
# ======================================================================

BUILD_HELPER_TERMS = [
    "semantic",
    "feature",
    "decision",
    "group",
    "cache",
    "encoder",
    "build",
]


print(
    "\n"
    + "=" * 110
)

print(
    "RELEVANT CELL 173 HELPER SOURCES"
)

print(
    "=" * 110
)

for item in defs_173:

    low = item[
        "name"
    ].lower()

    if not any(
        term in low
        for term
        in BUILD_HELPER_TERMS
    ):
        continue

    print(
        "\n"
        + "-" * 100
    )

    print(
        f"{item['type'].upper()} "
        f"{item['name']} "
        f"(line {item['line']})"
    )

    print(
        "-" * 100
    )

    print(
        item[
            "source"
        ]
    )


# ======================================================================
# 12. Static check for encoder normalization / batching behavior
# ======================================================================

combined_relevant_source = "\n".join(
    [
        semantic_source_154,
        build_source_173,
    ]
)

runtime_flags_12cb2 = {
    "uses_sentence_transformer":
        "SentenceTransformer"
        in combined_relevant_source,

    "uses_encode":
        ".encode("
        in combined_relevant_source,

    "normalize_embeddings_true":
        bool(
            re.search(
                r"normalize_embeddings\s*=\s*True",
                combined_relevant_source
            )
        ),

    "normalize_embeddings_false":
        bool(
            re.search(
                r"normalize_embeddings\s*=\s*False",
                combined_relevant_source
            )
        ),

    "batch_size_explicit":
        "batch_size"
        in combined_relevant_source,

    "device_explicit":
        "device"
        in combined_relevant_source,

    "cache_explicit":
        "cache"
        in combined_relevant_source.lower(),
}


print(
    "\n"
    + "=" * 110
)

print(
    "SEMANTIC RUNTIME FLAGS"
)

print(
    "=" * 110
)

for key, value in (
    runtime_flags_12cb2.items()
):
    print(
        f"{key:<35} {value}"
    )


# ======================================================================
# 13. Save static recovery manifest
# ======================================================================

TRAVERSAL_DIR = (
    Path(RQ2_ROOT)
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CELL12C_B2_MANIFEST = {
    "cell":
        "RQ2_12C_B2",

    "mode":
        "static_source_recovery",

    "feature_definition_cell":
        CELL_FEATURE,

    "semantic_candidate_cell":
        CELL_SEMANTIC_CANDIDATE,

    "build_candidate_cell":
        CELL_BUILD_CANDIDATE,

    "group_builder_found":
        group_builder_node
        is not None,

    "semantic_definitions":
        [
            {
                "type":
                    x["type"],

                "name":
                    x["name"],

                "line":
                    x["line"],
            }
            for x in semantic_defs_154
        ],

    "feature_call_sites":
        [
            {
                "cell":
                    x["cell"],

                "line":
                    x["line"],

                "function":
                    x["function"],
            }
            for x in feature_call_sites
        ],

    "runtime_flags":
        runtime_flags_12cb2,

    "notebook_code_executed":
        False,

    "model_loaded":
        False,

    "new_features_computed":
        False,

    "pruning_run":
        False,

    "hyperparameter_tuning":
        False,

    "test_loaded":
        False,
}


CELL12C_B2_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12c_b2_semantic_runtime_static_recovery.json"
)


with open(
    CELL12C_B2_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL12C_B2_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )


# ======================================================================
# 14. Final report
# ======================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "=== RQ2 CELL 12C-B2: EXACT SEMANTIC RUNTIME STATIC RECOVERY COMPLETE ==="
)

print(
    "=" * 110
)

print(
    "Group Feature-v2 builder found: ",
    group_builder_node is not None
)

print(
    "Cell-154 definitions found:      ",
    len(
        semantic_defs_154
    )
)

print(
    "Feature-build call sites found:  ",
    len(
        feature_call_sites
    )
)

print(
    "Notebook code executed:          NO"
)

print(
    "MiniLM loaded:                   NO"
)

print(
    "New features computed:           NO"
)

print(
    "Pruning run:                     NO"
)

print(
    "Hyperparameter tuning:           NO"
)

print(
    "TEST loaded:                     NO"
)

print(
    "\nManifest:",
    CELL12C_B2_MANIFEST_PATH
)

print(
    "\nNext: instantiate the EXACT recovered semantic encoder on CPU, "
    "rebuild the cached validation decision-group features, and require "
    "numerical equality with the frozen NPZ before enabling online AFP."
)

Notebook: /kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb
Feature cell: 167
Semantic candidate cell: 154
Build candidate cell: 173

EXACT extract_afp_group_features SOURCE
def extract_afp_group_features(
    question_id,
    question,
    plan,
    hop,
    candidate_rows,
    semantic_encoder,
    entity_name_map=None
):

    assert len(
        candidate_rows
    ) > 1


    candidates = []

    for row in candidate_rows:

        candidates.append(
            {
                "prefix_entities":
                    list(
                        row[
                            "prefix_entities"
                        ]
                    ),

                "candidate_entity":
                    row[
                        "candidate_entity"
                    ],

                "parent_prefix_index":
                    int(
                        row[
                            "parent_prefix_index"
                        ]
                    ),
     

In [16]:
# ======================================================================
# LOCATE HOW FrozenMiniLMEncoder WAS DEFINED
# ======================================================================
#
# STATIC ONLY.
# No model loading.
# No feature computation.
# No pruning.
# No tuning.
# No TEST.
# ======================================================================

import ast
import json
import re
from pathlib import Path

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb"
)

assert NOTEBOOK_PATH.exists()

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:
    nb = json.load(f)

TARGET = "FrozenMiniLMEncoder"

print("Notebook:", NOTEBOOK_PATH)
print("Cells:", len(nb["cells"]))


# ======================================================================
# 1. Find EVERY textual occurrence
# ======================================================================

text_hits = []

for cell_idx, cell in enumerate(
    nb["cells"]
):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(
        cell.get("source", [])
    )

    if TARGET not in src:
        continue

    lines = src.splitlines()

    for line_idx, line in enumerate(
        lines
    ):
        if TARGET in line:

            start = max(
                0,
                line_idx - 12
            )

            end = min(
                len(lines),
                line_idx + 20
            )

            context = "\n".join(
                f"{i+1:04d}: {lines[i]}"
                for i in range(
                    start,
                    end
                )
            )

            text_hits.append(
                {
                    "cell":
                        cell_idx,

                    "line":
                        line_idx + 1,

                    "context":
                        context,
                }
            )


print(
    "\n"
    + "=" * 100
)

print(
    "ALL TEXTUAL OCCURRENCES OF FrozenMiniLMEncoder"
)

print(
    "=" * 100
)

print(
    "Total occurrences:",
    len(text_hits)
)

for hit in text_hits:

    print(
        "\n"
        + "-" * 95
    )

    print(
        f"CELL {hit['cell']} "
        f"| LINE {hit['line']}"
    )

    print(
        "-" * 95
    )

    print(
        hit["context"]
    )


# ======================================================================
# 2. AST search for all ways the symbol could be introduced
# ======================================================================

ast_hits = []

for cell_idx, cell in enumerate(
    nb["cells"]
):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(
        cell.get("source", [])
    )

    try:
        tree = ast.parse(src)
    except Exception:
        continue

    for node in ast.walk(
        tree
    ):

        # ----------------------------------------------------------
        # class FrozenMiniLMEncoder:
        # ----------------------------------------------------------
        if (
            isinstance(node, ast.ClassDef)
            and node.name == TARGET
        ):
            ast_hits.append(
                {
                    "cell":
                        cell_idx,

                    "kind":
                        "ClassDef",

                    "line":
                        getattr(
                            node,
                            "lineno",
                            -1
                        ),

                    "source":
                        ast.get_source_segment(
                            src,
                            node
                        ) or "",
                }
            )

        # ----------------------------------------------------------
        # def FrozenMiniLMEncoder(...):
        # ----------------------------------------------------------
        elif (
            isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                )
            )
            and node.name == TARGET
        ):
            ast_hits.append(
                {
                    "cell":
                        cell_idx,

                    "kind":
                        "FunctionDef",

                    "line":
                        getattr(
                            node,
                            "lineno",
                            -1
                        ),

                    "source":
                        ast.get_source_segment(
                            src,
                            node
                        ) or "",
                }
            )

        # ----------------------------------------------------------
        # FrozenMiniLMEncoder = ...
        # ----------------------------------------------------------
        elif isinstance(
            node,
            ast.Assign
        ):

            target_names = []

            for target in node.targets:

                if isinstance(
                    target,
                    ast.Name
                ):
                    target_names.append(
                        target.id
                    )

            if TARGET in target_names:

                ast_hits.append(
                    {
                        "cell":
                            cell_idx,

                        "kind":
                            "Assign",

                        "line":
                            getattr(
                                node,
                                "lineno",
                                -1
                            ),

                        "source":
                            ast.get_source_segment(
                                src,
                                node
                            ) or "",
                    }
                )

        # ----------------------------------------------------------
        # FrozenMiniLMEncoder: X = ...
        # ----------------------------------------------------------
        elif (
            isinstance(
                node,
                ast.AnnAssign
            )
            and isinstance(
                node.target,
                ast.Name
            )
            and node.target.id == TARGET
        ):

            ast_hits.append(
                {
                    "cell":
                        cell_idx,

                    "kind":
                        "AnnAssign",

                    "line":
                        getattr(
                            node,
                            "lineno",
                            -1
                        ),

                    "source":
                        ast.get_source_segment(
                            src,
                            node
                        ) or "",
                }
            )

        # ----------------------------------------------------------
        # from x import FrozenMiniLMEncoder
        # ----------------------------------------------------------
        elif isinstance(
            node,
            ast.ImportFrom
        ):

            for alias in node.names:

                if (
                    alias.name == TARGET
                    or alias.asname == TARGET
                ):

                    ast_hits.append(
                        {
                            "cell":
                                cell_idx,

                            "kind":
                                "ImportFrom",

                            "line":
                                getattr(
                                    node,
                                    "lineno",
                                    -1
                                ),

                            "source":
                                ast.get_source_segment(
                                    src,
                                    node
                                ) or "",
                        }
                    )

        # ----------------------------------------------------------
        # import something as FrozenMiniLMEncoder
        # ----------------------------------------------------------
        elif isinstance(
            node,
            ast.Import
        ):

            for alias in node.names:

                if (
                    alias.name == TARGET
                    or alias.asname == TARGET
                ):

                    ast_hits.append(
                        {
                            "cell":
                                cell_idx,

                            "kind":
                                "Import",

                            "line":
                                getattr(
                                    node,
                                    "lineno",
                                    -1
                                ),

                            "source":
                                ast.get_source_segment(
                                    src,
                                    node
                                ) or "",
                        }
                    )


print(
    "\n"
    + "=" * 100
)

print(
    "AST DEFINITIONS / IMPORTS / ASSIGNMENTS"
)

print(
    "=" * 100
)

if not ast_hits:
    print("NONE FOUND")

else:

    for hit in ast_hits:

        print(
            f"\ncell={hit['cell']} "
            f"line={hit['line']} "
            f"kind={hit['kind']}"
        )

        print(
            hit["source"]
        )


# ======================================================================
# 3. Search for likely encoder classes even if name differs
# ======================================================================

encoder_like_defs = []

TOKENS = [
    "encoder",
    "embedding",
    "minilm",
    "sentence",
    "transformer",
    "cache",
]


for cell_idx, cell in enumerate(
    nb["cells"]
):

    if cell.get("cell_type") != "code":
        continue

    src = "".join(
        cell.get("source", [])
    )

    try:
        tree = ast.parse(
            src
        )
    except Exception:
        continue

    for node in tree.body:

        if isinstance(
            node,
            ast.ClassDef
        ):

            name_low = (
                node.name.lower()
            )

            body_src = (
                ast.get_source_segment(
                    src,
                    node
                )
                or ""
            )

            body_low = (
                body_src.lower()
            )

            score = sum(
                token in name_low
                or token in body_low
                for token in TOKENS
            )

            if score > 0:

                encoder_like_defs.append(
                    {
                        "cell":
                            cell_idx,

                        "name":
                            node.name,

                        "line":
                            node.lineno,

                        "score":
                            score,

                        "source":
                            body_src,
                    }
                )


encoder_like_defs = sorted(
    encoder_like_defs,
    key=lambda x: (
        -x["score"],
        x["cell"],
        x["line"],
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "ENCODER-LIKE CLASS DEFINITIONS"
)

print(
    "=" * 100
)

if not encoder_like_defs:
    print("NONE FOUND")

else:

    for item in encoder_like_defs[
        :20
    ]:

        first = (
            item["source"]
            .splitlines()[0]
            if item["source"]
            else item["name"]
        )

        print(
            f"score={item['score']:<3} "
            f"cell={item['cell']:<4} "
            f"line={item['line']:<5} "
            f"{first}"
        )


# ======================================================================
# 4. Search for SentenceTransformer creation / .encode behavior
# ======================================================================

runtime_hits = []

PATTERNS = [
    "SentenceTransformer(",
    ".encode(",
    "normalize_embeddings",
    "convert_to_numpy",
    "show_progress_bar",
    "batch_size",
    "self.cache",
    "self.model",
    "self.dim",
]


for cell_idx, cell in enumerate(
    nb["cells"]
):

    if cell.get("cell_type") != "code":
        continue

    src = "".join(
        cell.get("source", [])
    )

    matched = [
        p
        for p in PATTERNS
        if p.lower()
        in src.lower()
    ]

    if not matched:
        continue

    runtime_hits.append(
        {
            "cell":
                cell_idx,

            "matched":
                matched,

            "source":
                src,
        }
    )


print(
    "\n"
    + "=" * 100
)

print(
    "SENTENCE-TRANSFORMER / ENCODER RUNTIME CELLS"
)

print(
    "=" * 100
)

for hit in runtime_hits:

    print(
        f"\nCELL {hit['cell']} "
        f"| matched={hit['matched']}"
    )

    lines = hit[
        "source"
    ].splitlines()

    keep = set()

    for i, line in enumerate(
        lines
    ):

        if any(
            pattern.lower()
            in line.lower()
            for pattern
            in PATTERNS
        ):

            for j in range(
                max(0, i - 8),
                min(
                    len(lines),
                    i + 18
                )
            ):
                keep.add(j)

    for j in sorted(
        keep
    ):
        print(
            f"{j+1:04d}: "
            f"{lines[j]}"
        )


# ======================================================================
# 5. Search for dynamic creation via exec/eval
# ======================================================================

dynamic_hits = []

for cell_idx, cell in enumerate(
    nb["cells"]
):

    if cell.get("cell_type") != "code":
        continue

    src = "".join(
        cell.get("source", [])
    )

    if (
        "exec(" in src
        or "eval(" in src
    ):

        dynamic_hits.append(
            {
                "cell":
                    cell_idx,

                "source":
                    src,
            }
        )


print(
    "\n"
    + "=" * 100
)

print(
    "EXEC / EVAL CELLS"
)

print(
    "=" * 100
)

if not dynamic_hits:
    print("NONE")

else:

    for hit in dynamic_hits:
        print(
            f"\nCELL {hit['cell']}"
        )

        print(
            hit["source"]
        )


# ======================================================================
# 6. Final diagnosis
# ======================================================================

print(
    "\n"
    + "=" * 104
)

print(
    "=== FrozenMiniLMEncoder DEFINITION DIAGNOSTIC COMPLETE ==="
)

print(
    "=" * 104
)

print(
    "Text occurrences:            ",
    len(text_hits)
)

print(
    "AST definitions/imports:     ",
    len(ast_hits)
)

print(
    "Encoder-like classes:        ",
    len(encoder_like_defs)
)

print(
    "Runtime encoder cells:       ",
    len(runtime_hits)
)

print(
    "Dynamic exec/eval cells:     ",
    len(dynamic_hits)
)

print(
    "\nMiniLM loaded:               NO"
)

print(
    "Features recomputed:         NO"
)

print(
    "Pruning run:                NO"
)

print(
    "Hyperparameter tuning:      NO"
)

print(
    "TEST loaded:                NO"
)

Notebook: /kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb
Cells: 190

ALL TEXTUAL OCCURRENCES OF FrozenMiniLMEncoder
Total occurrences: 2

-----------------------------------------------------------------------------------------------
CELL 173 | LINE 75
-----------------------------------------------------------------------------------------------
0063: assert AFP_USE_KGE_CORE is False
0064: 
0065: required_objects = [
0066:     "webqsp_train",
0067:     "cwq_train",
0068:     "webqsp_branch_manifest",
0069:     "cwq_branch_manifest",
0070:     "build_graph",
0071:     "relation_valid_neighbors",
0072:     "call_suffix_dp",
0073:     "dp_is_reachable",
0074:     "extract_afp_group_features",
0075:     "FrozenMiniLMEncoder",
0076: ]
0077: 
0078: missing = [x for x in required_objects if x not in globals()]
0079: 
0080: assert not missing, (
0081:     "Missing required previous-cell objects: "
0082:     + ", ".join(missing)
0083: )
0084: 
0085: print("Feature version:"

In [17]:
# ======================================================================
# BEHAVIORAL RECOVERY OF MINILM RUNTIME
# ======================================================================
#
# FrozenMiniLMEncoder source is absent from the saved notebook.
#
# Therefore we recover its observable behavior using:
#   - exact frozen MiniLM model
#   - exact Feature-v2 extractor from Cell 167
#   - exact frozen validation supervision
#   - frozen validation Feature-v2 NPZ
#
# We compare:
#   A. raw SentenceTransformer embeddings
#   B. L2-normalized SentenceTransformer embeddings
#
# The frozen NPZ decides SOFTWARE FIDELITY only.
#
# NO pruning.
# NO selector tuning.
# NO TEST.
# ======================================================================

import ast
import json
import math
import re
from pathlib import Path
from collections import Counter
from typing import Optional, Dict

import numpy as np
import torch
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer


# ======================================================================
# 1. Paths + hard gates
# ======================================================================

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb"
)

RQ2_ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

FEATURE_DIR = (
    RQ2_ROOT
    / "03_features"
)

assert NOTEBOOK_PATH.exists()
assert FEATURE_DIR.exists()

assert len(webqsp_val_plan_rows) == 246
assert len(cwq_val_plan_rows) == 3519

print("Notebook:", NOTEBOOK_PATH)
print("Runtime device: CPU")


# ======================================================================
# 2. Recover exact Feature-v2 functions from Cell 167
# ======================================================================

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:
    nb = json.load(f)

FEATURE_CELL = 167

feature_source = "".join(
    nb["cells"][FEATURE_CELL]["source"]
)

tree = ast.parse(
    feature_source
)

REQUIRED_FUNCTIONS = [
    "is_raw_freebase_id",
    "has_readable_entity_surface",
    "normalize_surface_text",
    "relation_surface_text",
    "entity_surface_text",
    "relation_sequence_text",
    "safe_l2_normalize",
    "safe_cosine",
    "mean_embedding",
    "get_safe_entity_embedding",
    "extract_afp_candidate_features",
    "extract_afp_group_features",
]

nodes = {}

for node in tree.body:
    if (
        isinstance(node, ast.FunctionDef)
        and node.name in REQUIRED_FUNCTIONS
    ):
        nodes[node.name] = node

missing = [
    name
    for name in REQUIRED_FUNCTIONS
    if name not in nodes
]

assert not missing, (
    "Missing Feature-v2 functions: "
    + str(missing)
)


feature_ns = {
    "np": np,
    "math": math,
    "re": re,
    "Counter": Counter,
    "Optional": Optional,
    "Dict": Dict,

    "AFP_FEATURE_DIM": 27,

    "FREEBASE_ID_RE":
        re.compile(
            r"^(?:m|g)\.[A-Za-z0-9_\-]+$"
        ),

    "__name__":
        "recovered_feature_v2",
}

module = ast.Module(
    body=[
        nodes[name]
        for name in REQUIRED_FUNCTIONS
    ],
    type_ignores=[]
)

ast.fix_missing_locations(
    module
)

exec(
    compile(
        module,
        filename="recovered_feature_v2",
        mode="exec"
    ),
    feature_ns
)


extract_group = (
    feature_ns[
        "extract_afp_group_features"
    ]
)

relation_surface_text = (
    feature_ns[
        "relation_surface_text"
    ]
)

relation_sequence_text = (
    feature_ns[
        "relation_sequence_text"
    ]
)

has_readable_entity_surface = (
    feature_ns[
        "has_readable_entity_surface"
    ]
)

entity_surface_text = (
    feature_ns[
        "entity_surface_text"
    ]
)

safe_l2_normalize = (
    feature_ns[
        "safe_l2_normalize"
    ]
)

print(
    "Exact Feature-v2 extractor recovery: PASSED"
)


# ======================================================================
# 3. Frozen artifacts
# ======================================================================

WEBQSP_LABEL_FILE = (
    FEATURE_DIR
    / "validation_supervision"
    / "webqsp_validation_decision_labels.jsonl"
)

CWQ_LABEL_FILE = (
    FEATURE_DIR
    / "validation_supervision"
    / "cwq_validation_decision_labels.jsonl"
)

WEBQSP_NPZ = (
    FEATURE_DIR
    / "webqsp"
    / "webqsp_validation_afp_features_v2.npz"
)

CWQ_NPZ = (
    FEATURE_DIR
    / "cwq"
    / "cwq_validation_afp_features_v2.npz"
)

for path in [
    WEBQSP_LABEL_FILE,
    CWQ_LABEL_FILE,
    WEBQSP_NPZ,
    CWQ_NPZ,
]:
    assert path.exists()


# ======================================================================
# 4. Exact decision-group iterator
# ======================================================================

def iter_groups(labels_file):

    current_id = None
    current_rows = []

    with open(
        labels_file,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            row = json.loads(
                line
            )

            if (
                row.get(
                    "decision_opportunity",
                    True
                )
                is False
            ):
                continue

            gid = row[
                "group_id"
            ]

            if (
                current_id is not None
                and gid != current_id
            ):
                assert len(
                    current_rows
                ) > 1

                yield current_rows

                current_rows = []

            current_id = gid

            current_rows.append(
                row
            )

    if current_rows:

        assert len(
            current_rows
        ) > 1

        yield current_rows


# ======================================================================
# 5. Collect exact validation semantic inventory
# ======================================================================

def collect_semantic_inventory(
    labels_file,
    dataset_rows
):

    questions = {}
    entities = {}
    relations = {}
    plans = {}
    suffixes = {}

    groups = 0
    branches = 0

    for rows in iter_groups(
        labels_file
    ):

        groups += 1
        branches += len(rows)

        first = rows[0]

        source_index = int(
            first["source_index"]
        )

        rec = dataset_rows[
            source_index
        ]

        assert str(
            rec["id"]
        ) == str(
            first["question_id"]
        )

        qid = str(
            first["question_id"]
        )

        questions[qid] = (
            rec["question"]
        )

        plan = list(
            first["plan"]
        )

        hop = int(
            first["hop"]
        )

        for relation in plan:

            relations[
                str(relation)
            ] = relation_surface_text(
                relation
            )

        plan_key = "||".join(
            str(x)
            for x in plan
        )

        plans[
            plan_key
        ] = relation_sequence_text(
            plan
        )

        suffix = plan[
            hop + 1:
        ]

        suffix_key = "||".join(
            str(x)
            for x in suffix
        )

        suffixes[
            suffix_key
        ] = relation_sequence_text(
            suffix
        )

        for row in rows:

            candidate = row[
                "candidate_entity"
            ]

            if has_readable_entity_surface(
                candidate
            ):

                entities[
                    str(candidate)
                ] = entity_surface_text(
                    candidate
                )

            for entity in row[
                "prefix_entities"
            ]:

                if has_readable_entity_surface(
                    entity
                ):

                    entities[
                        str(entity)
                    ] = entity_surface_text(
                        entity
                    )

    return {
        "questions": questions,
        "entities": entities,
        "relations": relations,
        "plans": plans,
        "suffixes": suffixes,
        "groups": groups,
        "branches": branches,
    }


print(
    "\nCollecting validation semantic inventory..."
)

wq_inventory = (
    collect_semantic_inventory(
        WEBQSP_LABEL_FILE,
        webqsp_val_plan_rows
    )
)

cwq_inventory = (
    collect_semantic_inventory(
        CWQ_LABEL_FILE,
        cwq_val_plan_rows
    )
)


def merge_maps(*maps):

    out = {}

    for mapping in maps:
        out.update(
            mapping
        )

    return out


ALL_QUESTIONS = merge_maps(
    wq_inventory["questions"],
    cwq_inventory["questions"]
)

ALL_ENTITIES = merge_maps(
    wq_inventory["entities"],
    cwq_inventory["entities"]
)

ALL_RELATIONS = merge_maps(
    wq_inventory["relations"],
    cwq_inventory["relations"]
)

ALL_PLANS = merge_maps(
    wq_inventory["plans"],
    cwq_inventory["plans"]
)

ALL_SUFFIXES = merge_maps(
    wq_inventory["suffixes"],
    cwq_inventory["suffixes"]
)


print("\nInventory")
print(
    "  Questions:         ",
    len(ALL_QUESTIONS)
)
print(
    "  Readable entities: ",
    len(ALL_ENTITIES)
)
print(
    "  Relations:         ",
    len(ALL_RELATIONS)
)
print(
    "  Plans:             ",
    len(ALL_PLANS)
)
print(
    "  Suffixes:          ",
    len(ALL_SUFFIXES)
)


# ======================================================================
# 6. Load exact frozen MiniLM model
# ======================================================================

MODEL_NAME = (
    "sentence-transformers/"
    "all-MiniLM-L6-v2"
)

print(
    "\nLoading:",
    MODEL_NAME
)

minilm_model = (
    SentenceTransformer(
        MODEL_NAME,
        device="cpu"
    )
)

MINILM_DIM = int(
    minilm_model
    .get_sentence_embedding_dimension()
)

assert MINILM_DIM == 384

print(
    "Embedding dimension:",
    MINILM_DIM
)


# ======================================================================
# 7. Encode raw embeddings exactly once
# ======================================================================
#
# Original Cell 173 used:
#   batch_size = 256
#
# We preserve category separation:
#   question
#   entity
#   relation
#   plan
#   suffix
#
# normalize_embeddings=False gives the raw SentenceTransformer output.
# The second candidate behavior is obtained by L2-normalizing these
# same vectors, so MiniLM inference is performed only once.
# ======================================================================

RAW_CACHE = {}


def prefill_raw(
    namespace,
    mapping
):

    items = list(
        mapping.items()
    )

    if not items:
        return

    identifiers = [
        str(k)
        for k, _
        in items
    ]

    texts = [
        str(v)
        for _, v
        in items
    ]

    print(
        f"Encoding {namespace:<10}: "
        f"{len(texts)}"
    )

    embeddings = (
        minilm_model.encode(
            texts,
            batch_size=256,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=False
        )
    )

    embeddings = np.asarray(
        embeddings,
        dtype=np.float32
    )

    assert embeddings.shape == (
        len(texts),
        384
    )

    for identifier, vector in zip(
        identifiers,
        embeddings
    ):

        RAW_CACHE[
            (
                namespace,
                identifier
            )
        ] = np.asarray(
            vector,
            dtype=np.float32
        )


print(
    "\nBuilding raw validation semantic cache..."
)

prefill_raw(
    "question",
    ALL_QUESTIONS
)

prefill_raw(
    "entity",
    ALL_ENTITIES
)

prefill_raw(
    "relation",
    ALL_RELATIONS
)

prefill_raw(
    "plan",
    ALL_PLANS
)

prefill_raw(
    "suffix",
    ALL_SUFFIXES
)

print(
    "\nRaw cache entries:",
    len(RAW_CACHE)
)


# ======================================================================
# 8. Behavioral encoder wrapper
# ======================================================================

class RecoveredMiniLMEncoder:

    def __init__(
        self,
        model,
        raw_cache,
        vector_mode
    ):
        assert vector_mode in {
            "raw",
            "unit_normalized",
        }

        self.model = model
        self.raw_cache = raw_cache
        self.vector_mode = (
            vector_mode
        )

        self.dim = 384

        # Runtime cache follows expected API.
        self.cache = {}


    def _transform(
        self,
        vector
    ):
        vector = np.asarray(
            vector,
            dtype=np.float32
        )

        if (
            self.vector_mode
            == "raw"
        ):
            return vector.copy()

        return safe_l2_normalize(
            vector
        )


    def get(
        self,
        namespace,
        identifier,
        raw_text
    ):

        key = (
            str(namespace),
            str(identifier)
        )

        if key in self.cache:
            return self.cache[
                key
            ]

        # Prefer precomputed raw vector.
        if key in self.raw_cache:

            raw = self.raw_cache[
                key
            ]

        else:
            # This path will later support dynamic frontiers.
            raw = self.model.encode(
                [str(raw_text)],
                batch_size=1,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=False
            )[0]

            raw = np.asarray(
                raw,
                dtype=np.float32
            )

            self.raw_cache[
                key
            ] = raw

        output = self._transform(
            raw
        )

        self.cache[
            key
        ] = output

        return output


encoder_raw = (
    RecoveredMiniLMEncoder(
        model=minilm_model,
        raw_cache=RAW_CACHE,
        vector_mode="raw"
    )
)

encoder_unit = (
    RecoveredMiniLMEncoder(
        model=minilm_model,
        raw_cache=RAW_CACHE,
        vector_mode=
            "unit_normalized"
    )
)


# ======================================================================
# 9. Rebuild validation feature matrix
# ======================================================================

def rebuild_features(
    dataset_name,
    dataset_rows,
    labels_file,
    encoder
):

    blocks = []

    group_ptr = [0]

    group_source_index = []
    group_hop = []
    group_plan_length = []
    group_candidate_count = []

    for rows in tqdm(
        iter_groups(
            labels_file
        ),
        desc=(
            f"{dataset_name} "
            f"{encoder.vector_mode}"
        )
    ):

        first = rows[0]

        source_index = int(
            first[
                "source_index"
            ]
        )

        rec = dataset_rows[
            source_index
        ]

        assert str(
            rec["id"]
        ) == str(
            first[
                "question_id"
            ]
        )

        plan = list(
            first[
                "plan"
            ]
        )

        hop = int(
            first[
                "hop"
            ]
        )

        X_group = extract_group(
            question_id=
                first[
                    "question_id"
                ],

            question=
                rec[
                    "question"
                ],

            plan=
                plan,

            hop=
                hop,

            candidate_rows=
                rows,

            semantic_encoder=
                encoder,

            entity_name_map=
                None
        )

        assert X_group.shape == (
            len(rows),
            27
        )

        blocks.append(
            X_group
        )

        group_ptr.append(
            group_ptr[-1]
            + len(rows)
        )

        group_source_index.append(
            source_index
        )

        group_hop.append(
            hop
        )

        group_plan_length.append(
            len(plan)
        )

        group_candidate_count.append(
            len(rows)
        )

    return {
        "X":
            np.concatenate(
                blocks,
                axis=0
            ).astype(
                np.float32
            ),

        "group_ptr":
            np.asarray(
                group_ptr,
                dtype=np.int64
            ),

        "group_source_index":
            np.asarray(
                group_source_index,
                dtype=np.int32
            ),

        "group_hop":
            np.asarray(
                group_hop,
                dtype=np.int16
            ),

        "group_plan_length":
            np.asarray(
                group_plan_length,
                dtype=np.int16
            ),

        "group_candidate_count":
            np.asarray(
                group_candidate_count,
                dtype=np.int32
            ),
    }


# ======================================================================
# 10. Frozen arrays
# ======================================================================

def load_npz(path):

    z = np.load(
        path,
        allow_pickle=False
    )

    return {
        k: z[k]
        for k in z.files
    }


wq_frozen = load_npz(
    WEBQSP_NPZ
)

cwq_frozen = load_npz(
    CWQ_NPZ
)


# ======================================================================
# 11. Rebuild BOTH plausible runtime behaviors
# ======================================================================

print(
    "\nRebuilding RAW behavior..."
)

wq_raw = rebuild_features(
    "webqsp",
    webqsp_val_plan_rows,
    WEBQSP_LABEL_FILE,
    encoder_raw
)

cwq_raw = rebuild_features(
    "cwq",
    cwq_val_plan_rows,
    CWQ_LABEL_FILE,
    encoder_raw
)


print(
    "\nRebuilding UNIT-NORMALIZED behavior..."
)

wq_unit = rebuild_features(
    "webqsp",
    webqsp_val_plan_rows,
    WEBQSP_LABEL_FILE,
    encoder_unit
)

cwq_unit = rebuild_features(
    "cwq",
    cwq_val_plan_rows,
    CWQ_LABEL_FILE,
    encoder_unit
)


# ======================================================================
# 12. Metadata fidelity
# ======================================================================

META_KEYS = [
    "group_ptr",
    "group_source_index",
    "group_hop",
    "group_plan_length",
    "group_candidate_count",
]


def assert_metadata(
    dataset,
    rebuilt,
    frozen
):

    for key in META_KEYS:

        assert np.array_equal(
            rebuilt[key],
            frozen[key]
        ), (
            f"{dataset}: "
            f"metadata mismatch {key}"
        )


assert_metadata(
    "WebQSP/raw",
    wq_raw,
    wq_frozen
)

assert_metadata(
    "WebQSP/unit",
    wq_unit,
    wq_frozen
)

assert_metadata(
    "CWQ/raw",
    cwq_raw,
    cwq_frozen
)

assert_metadata(
    "CWQ/unit",
    cwq_unit,
    cwq_frozen
)

print(
    "\nMetadata fidelity: PASSED"
)


# ======================================================================
# 13. Numerical comparison
# ======================================================================

SEMANTIC_COLUMNS = [
    0, 2, 4, 5, 6, 7, 8, 9
]

SYMBOLIC_COLUMNS = [
    i
    for i in range(27)
    if i not in SEMANTIC_COLUMNS
]


def compare_features(
    rebuilt,
    frozen
):

    A = np.asarray(
        rebuilt["X"],
        dtype=np.float64
    )

    B = np.asarray(
        frozen["X"],
        dtype=np.float64
    )

    assert A.shape == B.shape

    diff = np.abs(
        A - B
    )

    sem = diff[
        :,
        SEMANTIC_COLUMNS
    ]

    sym = diff[
        :,
        SYMBOLIC_COLUMNS
    ]

    return {
        "max_all":
            float(
                diff.max()
            ),

        "mean_all":
            float(
                diff.mean()
            ),

        "max_semantic":
            float(
                sem.max()
            ),

        "mean_semantic":
            float(
                sem.mean()
            ),

        "max_symbolic":
            float(
                sym.max()
            ),

        "mean_symbolic":
            float(
                sym.mean()
            ),
    }


results = {
    "webqsp_raw":
        compare_features(
            wq_raw,
            wq_frozen
        ),

    "webqsp_unit":
        compare_features(
            wq_unit,
            wq_frozen
        ),

    "cwq_raw":
        compare_features(
            cwq_raw,
            cwq_frozen
        ),

    "cwq_unit":
        compare_features(
            cwq_unit,
            cwq_frozen
        ),
}


print(
    "\n"
    + "=" * 100
)

print(
    "FEATURE-v2 BEHAVIORAL FIDELITY"
)

print(
    "=" * 100
)

for name, values in (
    results.items()
):

    print(
        f"\n{name}"
    )

    for key, value in (
        values.items()
    ):

        print(
            f"  {key:<18} "
            f"{value:.10g}"
        )


# ======================================================================
# 14. Determine which embedding behavior matches frozen features
# ======================================================================

raw_score = (
    results[
        "webqsp_raw"
    ][
        "mean_semantic"
    ]
    +
    results[
        "cwq_raw"
    ][
        "mean_semantic"
    ]
)

unit_score = (
    results[
        "webqsp_unit"
    ][
        "mean_semantic"
    ]
    +
    results[
        "cwq_unit"
    ][
        "mean_semantic"
    ]
)


if raw_score <= unit_score:

    SELECTED_VECTOR_MODE = (
        "raw"
    )

    AFP_RUNTIME_SEMANTIC_ENCODER = (
        encoder_raw
    )

    selected_results = [
        results["webqsp_raw"],
        results["cwq_raw"],
    ]

else:

    SELECTED_VECTOR_MODE = (
        "unit_normalized"
    )

    AFP_RUNTIME_SEMANTIC_ENCODER = (
        encoder_unit
    )

    selected_results = [
        results["webqsp_unit"],
        results["cwq_unit"],
    ]


print(
    "\nSelected runtime behavior:",
    SELECTED_VECTOR_MODE
)


# ======================================================================
# 15. Strict software-fidelity gate
# ======================================================================

SYMBOLIC_ATOL = 1e-7

# CPU vs original CUDA MiniLM may differ slightly.
SEMANTIC_ATOL = 5e-5


for result in selected_results:

    assert (
        result[
            "max_symbolic"
        ]
        <= SYMBOLIC_ATOL
    ), (
        "Symbolic Feature-v2 reproduction failed."
    )

    assert (
        result[
            "max_semantic"
        ]
        <= SEMANTIC_ATOL
    ), (
        "Semantic Feature-v2 reproduction failed. "
        "Do NOT increase tolerance automatically."
    )


print(
    "Feature-v2 behavioral fidelity: PASSED"
)


# ======================================================================
# 16. Difference between the two candidate behaviors
# ======================================================================

RAW_UNIT_WEBQSP_MAX = float(
    np.max(
        np.abs(
            wq_raw["X"]
            -
            wq_unit["X"]
        )
    )
)

RAW_UNIT_CWQ_MAX = float(
    np.max(
        np.abs(
            cwq_raw["X"]
            -
            cwq_unit["X"]
        )
    )
)

print(
    "\nRaw vs unit-normalized feature difference"
)

print(
    "  WebQSP max:",
    RAW_UNIT_WEBQSP_MAX
)

print(
    "  CWQ max:   ",
    RAW_UNIT_CWQ_MAX
)


# ======================================================================
# 17. Freeze recovered runtime objects for B4
# ======================================================================

AFP_RUNTIME_FEATURE_EXTRACTOR = (
    extract_group
)

AFP_RUNTIME_FEATURE_VERSION = (
    "afp_features_v2_masked_entity_semantics"
)

AFP_RUNTIME_FEATURE_DIM = 27

AFP_RUNTIME_MINILM_MODEL = (
    minilm_model
)

AFP_RUNTIME_VECTOR_MODE = (
    SELECTED_VECTOR_MODE
)


# ======================================================================
# 18. Save recovery manifest
# ======================================================================

TRAVERSAL_DIR = (
    RQ2_ROOT
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

manifest = {
    "cell":
        "RQ2_12C_B3_R",

    "recovery_reason":
        (
            "FrozenMiniLMEncoder definition absent "
            "from persisted notebook source"
        ),

    "recovery_type":
        "behavioral_feature_reproduction",

    "semantic_model":
        MODEL_NAME,

    "runtime_device":
        "cpu",

    "batch_size":
        256,

    "candidate_vector_modes": [
        "raw",
        "unit_normalized",
    ],

    "selected_vector_mode":
        SELECTED_VECTOR_MODE,

    "webqsp_raw":
        results[
            "webqsp_raw"
        ],

    "webqsp_unit":
        results[
            "webqsp_unit"
        ],

    "cwq_raw":
        results[
            "cwq_raw"
        ],

    "cwq_unit":
        results[
            "cwq_unit"
        ],

    "raw_unit_webqsp_max":
        RAW_UNIT_WEBQSP_MAX,

    "raw_unit_cwq_max":
        RAW_UNIT_CWQ_MAX,

    "symbolic_atol":
        SYMBOLIC_ATOL,

    "semantic_atol":
        SEMANTIC_ATOL,

    "feature_fidelity_passed":
        True,

    "software_fidelity_only":
        True,

    "validation_labels_used_for_model_selection":
        False,

    "pruning_run":
        False,

    "hyperparameter_tuning_run":
        False,

    "test_loaded":
        False,

    "complete_afp_frozen":
        False,
}


manifest_path = (
    TRAVERSAL_DIR
    / "cell12c_b3_behavioral_minilm_recovery.json"
)


with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 19. Final report
# ======================================================================

print(
    "\n"
    + "=" * 106
)

print(
    "=== RQ2 CELL 12C-B3-R: MINILM RUNTIME BEHAVIOR RECOVERED ==="
)

print(
    "=" * 106
)

print(
    "FrozenMiniLMEncoder source:     MISSING"
)

print(
    "Recovery basis:                 FROZEN Feature-v2 NPZ"
)

print(
    "Semantic model:                 ",
    MODEL_NAME
)

print(
    "Selected embedding behavior:    ",
    SELECTED_VECTOR_MODE
)

print(
    "Feature-v2 fidelity:            PASS"
)

print(
    "Runtime extractor ready:        YES"
)

print(
    "Runtime semantic encoder ready: YES"
)

print(
    "\nPruning run:                    NO"
)

print(
    "Hyperparameter tuning:          NO"
)

print(
    "TEST loaded:                    NO"
)

print(
    "Complete AFP frozen:            NO"
)

print(
    "\nManifest:",
    manifest_path
)

print(
    "\nNext: Cell 12C-B4 — attach train-only standardizer + "
    "selected scorer checkpoint, verify online logits, then "
    "Cell 13 hyperparameter tuning."
)

Notebook: /kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb
Runtime device: CPU
Exact Feature-v2 extractor recovery: PASSED


Inventory
  Questions:          882
  Readable entities:  2637
  Relations:          274
  Plans:              292
  Suffixes:           190

Loading: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...


Rebuilding UNIT-NORMALIZED behavior...


webqsp unit_normalized: 0it [00:00, ?it/s]

cwq unit_normalized: 0it [00:00, ?it/s]


Metadata fidelity: PASSED

FEATURE-v2 BEHAVIORAL FIDELITY

webqsp_raw
  max_all            3.576278687e-07
  mean_all           1.56384136e-08
  max_semantic       3.576278687e-07
  mean_semantic      5.277964589e-08
  max_symbolic       0
  mean_symbolic      0

webqsp_unit
  max_all            3.576278687e-07
  mean_all           1.56059912e-08
  max_semantic       3.576278687e-07
  mean_semantic      5.267022031e-08
  max_symbolic       0
  mean_symbolic      0

cwq_raw
  max_all            0.4881388545
  mean_all           0.0001131997791
  max_semantic       0.4881388545
  mean_semantic      0.0003820492545
  max_symbolic       0
  mean_symbolic      0

cwq_unit
  max_all            0.4881388545
  mean_all           0.0001131997428
  max_semantic       0.4881388545
  mean_semantic      0.0003820491321
  max_symbolic       0
  mean_symbolic      0

Selected runtime behavior: unit_normalized


AssertionError: Semantic Feature-v2 reproduction failed. Do NOT increase tolerance automatically.

In [18]:
# ======================================================================
# 3.12C-B3-DIAG LOCALIZE CWQ SEMANTIC FIDELITY FAILURE
# ======================================================================
#
# Uses existing:
#   cwq_unit
#   cwq_frozen
#   cwq_val_plan_rows
#
# No MiniLM rerun.
# No pruning.
# No tuning.
# No TEST.
# ======================================================================

import numpy as np
import pandas as pd


FEATURE_NAMES = [
    "sem_q_candidate",                     # 0
    "sem_candidate_surface_available",     # 1
    "sem_q_current_entity",                # 2
    "sem_current_surface_available",       # 3
    "sem_q_current_relation",              # 4
    "sem_q_full_plan",                     # 5
    "sem_q_remaining_suffix",              # 6
    "sem_candidate_current_relation",      # 7
    "path_q_prefix_entity_mean",            # 8
    "path_candidate_prefix_entity_mean",    # 9
    "path_prefix_surface_fraction",         # 10
    "path_candidate_repeats_entity",        # 11
    "path_candidate_occurrence_fraction",   # 12
    "path_unique_entity_ratio",             # 13
    "path_relation_repeat_fraction_before", # 14
    "struct_log_candidate_count",           # 15
    "struct_log_unique_candidate_entities", # 16
    "struct_log_contributing_parents",      # 17
    "struct_log_parent_fanout",             # 18
    "struct_parent_frontier_share",         # 19
    "struct_log_endpoint_multiplicity",     # 20
    "struct_endpoint_frontier_share",       # 21
    "struct_duplicate_endpoint_ratio",      # 22
    "prog_hop_fraction",                    # 23
    "prog_remaining_fraction",              # 24
    "prog_log_plan_length",                 # 25
    "prog_penultimate_indicator",           # 26
]

assert len(FEATURE_NAMES) == 27

A = np.asarray(
    cwq_unit["X"],
    dtype=np.float64
)

B = np.asarray(
    cwq_frozen["X"],
    dtype=np.float64
)

assert A.shape == B.shape

diff = np.abs(A - B)

FIDELITY_ATOL = 5e-5


# ======================================================================
# 1. Global discrepancy summary
# ======================================================================

bad_mask = diff > FIDELITY_ATOL

bad_rows = np.flatnonzero(
    np.any(
        bad_mask,
        axis=1
    )
)

bad_cells = np.argwhere(
    bad_mask
)

print("=" * 100)
print("CWQ SEMANTIC FIDELITY FAILURE LOCALIZATION")
print("=" * 100)

print("Feature matrix shape:       ", A.shape)
print("Bad feature cells (>5e-5):  ", len(bad_cells))
print("Bad branch rows:             ", len(bad_rows))
print(
    "Bad branch-row rate:        ",
    f"{100 * len(bad_rows) / len(A):.4f}%"
)
print("Maximum difference:          ", diff.max())


# ======================================================================
# 2. Which feature columns are failing?
# ======================================================================

column_rows = []

for j, name in enumerate(FEATURE_NAMES):

    col_diff = diff[:, j]

    count = int(
        np.sum(
            col_diff > FIDELITY_ATOL
        )
    )

    column_rows.append(
        {
            "feature_index": j,
            "feature": name,
            "bad_values": count,
            "max_abs_diff":
                float(
                    col_diff.max()
                ),
            "mean_abs_diff":
                float(
                    col_diff.mean()
                ),
        }
    )


column_df = pd.DataFrame(
    column_rows
)

column_df = column_df[
    column_df["bad_values"] > 0
].sort_values(
    [
        "bad_values",
        "max_abs_diff",
    ],
    ascending=False
)


print(
    "\n"
    + "=" * 100
)

print(
    "FAILING FEATURE COLUMNS"
)

print(
    "=" * 100
)

if len(column_df) == 0:
    print("NONE")
else:
    print(
        column_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.9f}"
        )
    )


# ======================================================================
# 3. Map branch rows -> decision groups
# ======================================================================

ptr = np.asarray(
    cwq_frozen[
        "group_ptr"
    ],
    dtype=np.int64
)

source_indices = np.asarray(
    cwq_frozen[
        "group_source_index"
    ],
    dtype=np.int64
)

hops = np.asarray(
    cwq_frozen[
        "group_hop"
    ],
    dtype=np.int64
)

plan_lengths = np.asarray(
    cwq_frozen[
        "group_plan_length"
    ],
    dtype=np.int64
)

candidate_counts = np.asarray(
    cwq_frozen[
        "group_candidate_count"
    ],
    dtype=np.int64
)


def branch_to_group(row_idx):
    return int(
        np.searchsorted(
            ptr[1:],
            row_idx,
            side="right"
        )
    )


bad_groups = sorted(
    set(
        branch_to_group(i)
        for i in bad_rows
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "AFFECTED GROUPS"
)

print(
    "=" * 100
)

print(
    "Bad groups:",
    len(bad_groups),
    "/",
    len(ptr) - 1,
    "=",
    f"{100 * len(bad_groups)/(len(ptr)-1):.4f}%"
)


group_summary = []

for g in bad_groups:

    s = int(ptr[g])
    e = int(ptr[g + 1])

    group_diff = diff[
        s:e
    ]

    source_index = int(
        source_indices[g]
    )

    rec_plan = (
        cwq_val_plan_rows[
            source_index
        ]
    )

    group_summary.append(
        {
            "group_index":
                g,

            "source_index":
                source_index,

            "question_id":
                str(
                    rec_plan.get(
                        "id",
                        ""
                    )
                ),

            "hop":
                int(
                    hops[g]
                ),

            "plan_length":
                int(
                    plan_lengths[g]
                ),

            "candidate_count":
                int(
                    candidate_counts[g]
                ),

            "max_abs_diff":
                float(
                    group_diff.max()
                ),

            "bad_cells":
                int(
                    np.sum(
                        group_diff
                        > FIDELITY_ATOL
                    )
                ),
        }
    )


group_df = pd.DataFrame(
    group_summary
).sort_values(
    "max_abs_diff",
    ascending=False
)


print(
    group_df.head(
        50
    ).to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ======================================================================
# 4. Inspect the largest individual differences
# ======================================================================

flat_order = np.argsort(
    diff.ravel()
)[::-1]

print(
    "\n"
    + "=" * 110
)

print(
    "TOP 30 INDIVIDUAL FEATURE DIFFERENCES"
)

print(
    "=" * 110
)

shown = 0

for flat_idx in flat_order:

    row_idx, feature_idx = (
        np.unravel_index(
            flat_idx,
            diff.shape
        )
    )

    d = float(
        diff[
            row_idx,
            feature_idx
        ]
    )

    if d <= FIDELITY_ATOL:
        break

    g = branch_to_group(
        row_idx
    )

    src_idx = int(
        source_indices[g]
    )

    qid = str(
        cwq_val_plan_rows[
            src_idx
        ].get(
            "id",
            ""
        )
    )

    print(
        f"row={row_idx:<6} "
        f"group={g:<5} "
        f"source={src_idx:<5} "
        f"feature={feature_idx:02d} "
        f"{FEATURE_NAMES[feature_idx]:<38} "
        f"rebuilt={A[row_idx, feature_idx]: .7f} "
        f"frozen={B[row_idx, feature_idx]: .7f} "
        f"diff={d:.7f} "
        f"id={qid}"
    )

    shown += 1

    if shown >= 30:
        break


# ======================================================================
# 5. Diagnostic pattern
# ======================================================================

QUESTION_DEPENDENT = {
    0,  # q-candidate
    2,  # q-current entity
    4,  # q-current relation
    5,  # q-full plan
    6,  # q-suffix
    8,  # q-prefix mean
}

NONQUESTION_SEMANTIC = {
    7,  # candidate-current relation
    9,  # candidate-prefix mean
}


bad_feature_indices = set(
    int(x[1])
    for x in bad_cells
)

question_only_failure = (
    len(bad_feature_indices) > 0
    and bad_feature_indices.issubset(
        QUESTION_DEPENDENT
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "FAILURE PATTERN"
)

print(
    "=" * 100
)

print(
    "Failing feature indices:",
    sorted(
        bad_feature_indices
    )
)

print(
    "Only question-dependent semantic features fail:",
    question_only_failure
)

print(
    "Candidate/entity-only semantic failures:",
    sorted(
        bad_feature_indices
        & NONQUESTION_SEMANTIC
    )
)


# ======================================================================
# 6. Compare actual validation dataset question text if available
# ======================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "CWQ DATASET vs FROZEN PLAN QUESTION TEXT CHECK"
)

print(
    "=" * 100
)


if "cwq_val" not in globals():

    print(
        "cwq_val object is not currently loaded."
    )

    print(
        "Cannot yet compare the original validation dataset text "
        "against frozen planning-row text."
    )

else:

    assert len(
        cwq_val
    ) == len(
        cwq_val_plan_rows
    )

    question_mismatches = []

    for i in range(
        len(cwq_val)
    ):

        dataset_id = str(
            cwq_val[i]["id"]
        )

        plan_id = str(
            cwq_val_plan_rows[i]["id"]
        )

        assert dataset_id == plan_id

        q_dataset = str(
            cwq_val[i][
                "question"
            ]
        )

        q_plan = str(
            cwq_val_plan_rows[i][
                "question"
            ]
        )

        if q_dataset != q_plan:

            question_mismatches.append(
                {
                    "source_index":
                        i,

                    "id":
                        dataset_id,

                    "dataset_question":
                        q_dataset,

                    "plan_question":
                        q_plan,

                    "dataset_repr":
                        repr(
                            q_dataset
                        ),

                    "plan_repr":
                        repr(
                            q_plan
                        ),
                }
            )


    print(
        "Exact question-text mismatches:",
        len(
            question_mismatches
        )
    )


    bad_source_set = set(
        int(
            source_indices[g]
        )
        for g in bad_groups
    )

    mismatch_source_set = set(
        x[
            "source_index"
        ]
        for x in question_mismatches
    )


    print(
        "Bad feature source indices:",
        len(
            bad_source_set
        )
    )

    print(
        "Question-mismatch source indices:",
        len(
            mismatch_source_set
        )
    )

    print(
        "Intersection:",
        len(
            bad_source_set
            & mismatch_source_set
        )
    )


    if question_mismatches:

        print(
            "\nFirst question-text mismatches:"
        )

        for item in question_mismatches[
            :20
        ]:

            marker = (
                " <-- FEATURE FAILURE"
                if item[
                    "source_index"
                ]
                in bad_source_set
                else ""
            )

            print(
                "\nsource_index=",
                item[
                    "source_index"
                ],
                " id=",
                item[
                    "id"
                ],
                marker,
                sep=""
            )

            print(
                "dataset:",
                item[
                    "dataset_repr"
                ]
            )

            print(
                "plan:   ",
                item[
                    "plan_repr"
                ]
            )


# ======================================================================
# 7. Final diagnosis summary
# ======================================================================

print(
    "\n"
    + "=" * 104
)

print(
    "=== CWQ FEATURE FIDELITY DIAGNOSTIC COMPLETE ==="
)

print(
    "=" * 104
)

print(
    "Bad branch rows:              ",
    len(
        bad_rows
    )
)

print(
    "Bad decision groups:          ",
    len(
        bad_groups
    )
)

print(
    "Maximum semantic difference:  ",
    float(
        diff.max()
    )
)

print(
    "Question-only failure pattern:",
    question_only_failure
)

print(
    "\nNo tolerance changed."
)

print(
    "No MiniLM rerun."
)

print(
    "No pruning."
)

print(
    "No hyperparameter tuning."
)

print(
    "No TEST."
)

CWQ SEMANTIC FIDELITY FAILURE LOCALIZATION
Feature matrix shape:        (18688, 27)
Bad feature cells (>5e-5):   4754
Bad branch rows:              839
Bad branch-row rate:         4.4895%
Maximum difference:           0.4881388545036316

FAILING FEATURE COLUMNS
 feature_index                   feature  bad_values  max_abs_diff  mean_abs_diff
             2      sem_q_current_entity         839   0.274980783    0.000811751
             8 path_q_prefix_entity_mean         839   0.274980783    0.000793556
             6    sem_q_remaining_suffix         839   0.040323436    0.000360753
             4    sem_q_current_relation         839   0.037393421    0.000351435
             5           sem_q_full_plan         804   0.034087270    0.000334386
             0           sem_q_candidate         594   0.488138855    0.000404427

AFFECTED GROUPS
Bad groups: 70 / 1352 = 5.1775%
 group_index  source_index                                    question_id  hop  plan_length  candidate_count  max_

In [20]:
# ======================================================================
# FAST CWQ TRAIN/VALIDATION QUESTION-ID COLLISION AUDIT
# ======================================================================
#
# Uses only id + question columns.
# Does NOT materialize the huge graph field row-by-row.
# CPU only.
# ======================================================================

from collections import defaultdict

question_versions = defaultdict(list)

print("\nReading CWQ TRAIN id/question columns...")

train_ids = cwq_train_exact["id"]
train_questions = cwq_train_exact["question"]

assert len(train_ids) == len(train_questions)
assert len(train_ids) == 27639

for i, (qid, q) in enumerate(
    zip(
        train_ids,
        train_questions
    )
):
    question_versions[
        str(qid)
    ].append(
        (
            "train",
            i,
            str(q)
        )
    )


print("Reading CWQ VALIDATION id/question columns...")

val_ids = cwq_val_exact["id"]
val_questions = cwq_val_exact["question"]

assert len(val_ids) == len(val_questions)
assert len(val_ids) == 3519

for i, (qid, q) in enumerate(
    zip(
        val_ids,
        val_questions
    )
):
    question_versions[
        str(qid)
    ].append(
        (
            "validation",
            i,
            str(q)
        )
    )


# --------------------------------------------------------------
# Find IDs associated with multiple DISTINCT question strings
# --------------------------------------------------------------

collision_rows = []

for qid, versions in question_versions.items():

    unique_texts = {
        x[2]
        for x in versions
    }

    if len(unique_texts) <= 1:
        continue

    val_sources = [
        x[1]
        for x in versions
        if x[0] == "validation"
    ]

    collision_rows.append(
        {
            "question_id":
                qid,

            "n_occurrences":
                len(versions),

            "n_unique_texts":
                len(unique_texts),

            "validation_source_indices":
                val_sources,

            "versions":
                versions,
        }
    )


collision_val_sources = set()

for item in collision_rows:

    collision_val_sources.update(
        int(x)
        for x in item[
            "validation_source_indices"
        ]
    )


print(
    "\n"
    + "=" * 105
)

print(
    "CWQ TRAIN/VALIDATION QUESTION-ID COLLISION AUDIT"
)

print(
    "=" * 105
)

print(
    "IDs with >1 distinct question text:",
    len(collision_rows)
)

print(
    "Validation source indices involved:",
    len(collision_val_sources)
)

print(
    "Intersection with failed feature sources:",
    len(
        collision_val_sources
        & bad_source_set
    )
)


print(
    "\nFailed sources:",
    len(bad_source_set)
)

print(
    "Failed sources explained by collisions:",
    len(
        bad_source_set
        & collision_val_sources
    )
)


# --------------------------------------------------------------
# Show relevant collisions
# --------------------------------------------------------------

shown = 0

for item in collision_rows:

    relevant_sources = (
        set(
            item[
                "validation_source_indices"
            ]
        )
        & bad_source_set
    )

    if not relevant_sources:
        continue

    print(
        "\n"
        + "-" * 100
    )

    print(
        "question_id:",
        item[
            "question_id"
        ]
    )

    print(
        "affected validation indices:",
        sorted(
            relevant_sources
        )
    )

    for split, idx, text in item[
        "versions"
    ]:

        print(
            f"  {split:<10} "
            f"index={idx:<6} "
            f"{repr(text)}"
        )

    shown += 1

    if shown >= 30:
        break


# --------------------------------------------------------------
# Root-cause classification
# --------------------------------------------------------------

if (
    bad_source_set
    and bad_source_set.issubset(
        mismatch_source_set
    )
):

    ROOT_CAUSE = (
        "planning_row_question_text_differs_from_"
        "original_validation_dataset"
    )

elif bad_source_set.issubset(
    mismatch_source_set
    | collision_val_sources
):

    ROOT_CAUSE = (
        "mixed_question_text_source_and_or_"
        "question_id_cache_collision"
    )

else:

    ROOT_CAUSE = (
        "not_fully_explained_yet"
    )


print(
    "\n"
    + "=" * 108
)

print(
    "=== CWQ QUESTION-EMBEDDING SOURCE DIAGNOSTIC COMPLETE ==="
)

print(
    "=" * 108
)

print(
    "Failed feature source questions:",
    len(bad_source_set)
)

print(
    "Plan-vs-dataset text mismatches:",
    len(mismatch_source_set)
)

print(
    "Question-ID collision sources:",
    len(collision_val_sources)
)

print(
    "Failed sources explained by collisions:",
    len(
        bad_source_set
        & collision_val_sources
    )
)

print(
    "\nROOT CAUSE CLASSIFICATION:"
)

print(
    ROOT_CAUSE
)

print("\nMiniLM rerun:          NO")
print("Feature rebuild:       NO")
print("Pruning:               NO")
print("Hyperparameter tuning: NO")
print("TEST used for dev:     NO")


Reading CWQ TRAIN id/question columns...
Reading CWQ VALIDATION id/question columns...

CWQ TRAIN/VALIDATION QUESTION-ID COLLISION AUDIT
IDs with >1 distinct question text: 0
Validation source indices involved: 0
Intersection with failed feature sources: 0

Failed sources: 45
Failed sources explained by collisions: 0

=== CWQ QUESTION-EMBEDDING SOURCE DIAGNOSTIC COMPLETE ===
Failed feature source questions: 45
Plan-vs-dataset text mismatches: 0
Question-ID collision sources: 0
Failed sources explained by collisions: 0

ROOT CAUSE CLASSIFICATION:
not_fully_explained_yet

MiniLM rerun:          NO
Feature rebuild:       NO
Pruning:               NO
Hyperparameter tuning: NO
TEST used for dev:     NO


In [21]:
# ======================================================================
# 3.12C-B3-DIAG3 — MINILM QUESTION TRUNCATION / MAX-LENGTH AUDIT
# ======================================================================
#
# PURPOSE
# -------
# Determine whether the missing FrozenMiniLMEncoder used a different
# max_seq_length / truncation behavior.
#
# Uses ONLY:
#   - CWQ VALIDATION
#   - the 45 already-identified affected questions
#   - frozen validation Feature-v2 values as SOFTWARE-FIDELITY reference
#
# No scorer.
# No pruning.
# No selector tuning.
# No TEST examples accessed.
# ======================================================================

import numpy as np
import pandas as pd
import torch


# ======================================================================
# 1. Hard prerequisites
# ======================================================================

required = [
    "minilm_model",
    "cwq_val_exact",
    "cwq_frozen",
    "cwq_val_plan_rows",
    "bad_groups",
    "bad_source_set",
    "source_indices",
    "safe_l2_normalize",
]

missing = [
    x for x in required
    if x not in globals()
]

assert not missing, (
    "Missing previous diagnostic objects: "
    + ", ".join(missing)
)

assert len(cwq_val_exact) == 3519
assert len(bad_source_set) == 45
assert len(bad_groups) == 70

tokenizer = minilm_model.tokenizer

print("Affected CWQ source questions:", len(bad_source_set))
print("Affected decision groups:     ", len(bad_groups))
print("Current MiniLM max_seq_length:", minilm_model.max_seq_length)


# ======================================================================
# 2. Exact token lengths WITHOUT truncation
# ======================================================================

all_token_lengths = []

for i in range(len(cwq_val_exact)):

    q = str(
        cwq_val_exact[i]["question"]
    )

    encoded = tokenizer(
        q,
        add_special_tokens=True,
        truncation=False
    )

    n_tokens = len(
        encoded["input_ids"]
    )

    all_token_lengths.append(
        n_tokens
    )


all_token_lengths = np.asarray(
    all_token_lengths,
    dtype=np.int32
)

bad_indices = np.asarray(
    sorted(bad_source_set),
    dtype=np.int32
)

bad_token_lengths = (
    all_token_lengths[
        bad_indices
    ]
)


# ======================================================================
# 3. Token-length profile
# ======================================================================

print("\n" + "=" * 100)
print("CWQ QUESTION TOKEN-LENGTH PROFILE")
print("=" * 100)

print("\nALL VALIDATION")
print("  n:      ", len(all_token_lengths))
print("  mean:   ", float(all_token_lengths.mean()))
print("  median: ", float(np.median(all_token_lengths)))
print("  p90:    ", float(np.quantile(all_token_lengths, 0.90)))
print("  p95:    ", float(np.quantile(all_token_lengths, 0.95)))
print("  p99:    ", float(np.quantile(all_token_lengths, 0.99)))
print("  max:    ", int(all_token_lengths.max()))

print("\nAFFECTED 45 QUESTIONS")
print("  n:      ", len(bad_token_lengths))
print("  mean:   ", float(bad_token_lengths.mean()))
print("  median: ", float(np.median(bad_token_lengths)))
print("  min:    ", int(bad_token_lengths.min()))
print("  max:    ", int(bad_token_lengths.max()))


for threshold in [
    64,
    96,
    128,
    160,
    192,
    256,
]:

    all_over = int(
        np.sum(
            all_token_lengths > threshold
        )
    )

    bad_over = int(
        np.sum(
            bad_token_lengths > threshold
        )
    )

    print(
        f"\n> {threshold:<3} tokens:"
        f"  all={all_over:<4}"
        f"  affected={bad_over:<3}"
        f" / {len(bad_token_lengths)}"
    )


# ======================================================================
# 4. Print affected question lengths
# ======================================================================

affected_length_rows = []

for idx in sorted(
    bad_source_set
):

    rec = cwq_val_exact[
        idx
    ]

    affected_length_rows.append(
        {
            "source_index":
                int(idx),

            "question_id":
                str(
                    rec["id"]
                ),

            "n_tokens":
                int(
                    all_token_lengths[
                        idx
                    ]
                ),

            "question":
                str(
                    rec["question"]
                ),
        }
    )


affected_length_df = (
    pd.DataFrame(
        affected_length_rows
    )
    .sort_values(
        "n_tokens",
        ascending=False
    )
)


print("\n" + "=" * 110)
print("AFFECTED QUESTION LENGTHS")
print("=" * 110)

print(
    affected_length_df[
        [
            "source_index",
            "n_tokens",
            "question_id",
            "question",
        ]
    ].to_string(
        index=False
    )
)


# ======================================================================
# 5. Reconstruct affected group metadata
# ======================================================================

ptr = np.asarray(
    cwq_frozen["group_ptr"],
    dtype=np.int64
)

assert len(ptr) == 1353


# We need the exact decision-group rows.
assert "iter_groups" in globals(), (
    "iter_groups() from B3-R is missing."
)

assert "CWQ_LABEL_FILE" in globals(), (
    "CWQ_LABEL_FILE from B3-R is missing."
)


all_cwq_group_rows = list(
    iter_groups(
        CWQ_LABEL_FILE
    )
)

assert len(
    all_cwq_group_rows
) == 1352


affected_groups = {}

for g in bad_groups:

    rows = all_cwq_group_rows[
        g
    ]

    first = rows[0]

    src_idx = int(
        first["source_index"]
    )

    assert src_idx in bad_source_set

    affected_groups[
        g
    ] = rows


# ======================================================================
# 6. Need the currently-correct NON-question semantic embeddings
# ======================================================================
#
# encoder_unit already reproduced all candidate/entity-only semantics.
# We use it only for relation / plan / suffix reference embeddings.
# ======================================================================

assert "encoder_unit" in globals()

base_encoder = encoder_unit


def cosine_np(a, b):

    a = np.asarray(
        a,
        dtype=np.float32
    )

    b = np.asarray(
        b,
        dtype=np.float32
    )

    na = float(
        np.linalg.norm(a)
    )

    nb = float(
        np.linalg.norm(b)
    )

    if na <= 1e-12 or nb <= 1e-12:
        return 0.0

    return float(
        np.dot(a, b)
        / (na * nb)
    )


# ======================================================================
# 7. Software-fidelity reference
# ======================================================================
#
# For each affected group we compare:
#
# feature 4: cos(q, current_relation)
# feature 5: cos(q, full_plan)
# feature 6: cos(q, remaining_suffix)
#
# These three depend on q but NOT candidate identity.
# Therefore one row per group is sufficient.
# ======================================================================

REFERENCE_FEATURES = [
    4,
    5,
    6,
]


def evaluate_question_embeddings(
    q_embedding_by_source
):

    diffs = []

    per_feature = {
        4: [],
        5: [],
        6: [],
    }

    for g in bad_groups:

        rows = affected_groups[
            g
        ]

        first = rows[0]

        source_index = int(
            first["source_index"]
        )

        q_emb = q_embedding_by_source[
            source_index
        ]

        plan = list(
            first["plan"]
        )

        hop = int(
            first["hop"]
        )

        current_relation = (
            plan[hop]
        )

        remaining_suffix = (
            plan[
                hop + 1:
            ]
        )

        relation_emb = (
            base_encoder.get(
                "relation",
                current_relation,
                feature_ns[
                    "relation_surface_text"
                ](
                    current_relation
                )
            )
        )

        plan_key = "||".join(
            str(x)
            for x in plan
        )

        full_plan_text = (
            feature_ns[
                "relation_sequence_text"
            ](
                plan
            )
        )

        plan_emb = (
            base_encoder.get(
                "plan",
                plan_key,
                full_plan_text
            )
        )

        suffix_key = "||".join(
            str(x)
            for x in remaining_suffix
        )

        suffix_text = (
            feature_ns[
                "relation_sequence_text"
            ](
                remaining_suffix
            )
        )

        suffix_emb = (
            base_encoder.get(
                "suffix",
                suffix_key,
                suffix_text
            )
        )

        predicted = {
            4:
                cosine_np(
                    q_emb,
                    relation_emb
                ),

            5:
                cosine_np(
                    q_emb,
                    plan_emb
                ),

            6:
                cosine_np(
                    q_emb,
                    suffix_emb
                ),
        }

        frozen_row = int(
            ptr[g]
        )

        for feature_idx in REFERENCE_FEATURES:

            observed = float(
                cwq_frozen["X"][
                    frozen_row,
                    feature_idx
                ]
            )

            delta = abs(
                predicted[
                    feature_idx
                ]
                - observed
            )

            diffs.append(
                delta
            )

            per_feature[
                feature_idx
            ].append(
                delta
            )


    diffs = np.asarray(
        diffs,
        dtype=np.float64
    )

    return {
        "n_values":
            int(
                len(diffs)
            ),

        "mean_abs":
            float(
                diffs.mean()
            ),

        "max_abs":
            float(
                diffs.max()
            ),

        "p95_abs":
            float(
                np.quantile(
                    diffs,
                    0.95
                )
            ),

        "f4_mean":
            float(
                np.mean(
                    per_feature[4]
                )
            ),

        "f5_mean":
            float(
                np.mean(
                    per_feature[5]
                )
            ),

        "f6_mean":
            float(
                np.mean(
                    per_feature[6]
                )
            ),
    }


# ======================================================================
# 8. Test PREDECLARED plausible max_seq_length behaviors
# ======================================================================
#
# These are software-fidelity candidates, NOT AFP hyperparameters.
#
# Common sentence-transformer truncation limits:
#   64, 128, 256
#
# Also retain the currently loaded model value.
# ======================================================================

ORIGINAL_RUNTIME_MAX_SEQ = int(
    minilm_model.max_seq_length
)

candidate_max_lengths = sorted(
    set(
        [
            64,
            128,
            256,
            ORIGINAL_RUNTIME_MAX_SEQ,
        ]
    )
)


affected_questions = [
    str(
        cwq_val_exact[idx][
            "question"
        ]
    )
    for idx in sorted(
        bad_source_set
    )
]

affected_indices_sorted = sorted(
    bad_source_set
)


max_length_results = {}


print("\n" + "=" * 100)
print("MAX-SEQUENCE-LENGTH SOFTWARE-FIDELITY TEST")
print("=" * 100)


for max_len in candidate_max_lengths:

    print(
        f"\nTesting max_seq_length={max_len} ..."
    )

    minilm_model.max_seq_length = int(
        max_len
    )

    with torch.inference_mode():

        E = minilm_model.encode(
            affected_questions,
            batch_size=64,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=False
        )

    E = np.asarray(
        E,
        dtype=np.float32
    )

    assert E.shape == (
        len(
            affected_indices_sorted
        ),
        384
    )

    q_map = {
        int(idx):
            E[j]
        for j, idx in enumerate(
            affected_indices_sorted
        )
    }

    result = evaluate_question_embeddings(
        q_map
    )

    max_length_results[
        int(max_len)
    ] = result

    print(
        f"  mean abs diff: "
        f"{result['mean_abs']:.10f}"
    )

    print(
        f"  max abs diff:  "
        f"{result['max_abs']:.10f}"
    )

    print(
        f"  p95 abs diff:  "
        f"{result['p95_abs']:.10f}"
    )

    print(
        "  feature means: "
        f"f4={result['f4_mean']:.10f} | "
        f"f5={result['f5_mean']:.10f} | "
        f"f6={result['f6_mean']:.10f}"
    )


# Always restore current model setting.
minilm_model.max_seq_length = (
    ORIGINAL_RUNTIME_MAX_SEQ
)


# ======================================================================
# 9. Rank candidate behaviors
# ======================================================================

ranking_rows = []

for max_len, result in (
    max_length_results.items()
):

    ranking_rows.append(
        {
            "max_seq_length":
                int(max_len),

            "mean_abs_diff":
                result[
                    "mean_abs"
                ],

            "max_abs_diff":
                result[
                    "max_abs"
                ],

            "p95_abs_diff":
                result[
                    "p95_abs"
                ],
        }
    )


ranking_df = (
    pd.DataFrame(
        ranking_rows
    )
    .sort_values(
        [
            "mean_abs_diff",
            "max_abs_diff",
        ]
    )
)


print("\n" + "=" * 100)
print("MAX-LENGTH FIDELITY RANKING")
print("=" * 100)

print(
    ranking_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.10f}"
    )
)


best_row = ranking_df.iloc[0]

BEST_MAX_SEQ_LENGTH = int(
    best_row[
        "max_seq_length"
    ]
)

BEST_MEAN_DIFF = float(
    best_row[
        "mean_abs_diff"
    ]
)

BEST_MAX_DIFF = float(
    best_row[
        "max_abs_diff"
    ]
)


print(
    "\nBest candidate max_seq_length:",
    BEST_MAX_SEQ_LENGTH
)

print(
    "Best mean difference:",
    BEST_MEAN_DIFF
)

print(
    "Best max difference:",
    BEST_MAX_DIFF
)


# ======================================================================
# 10. Classification
# ======================================================================

FIDELITY_ATOL = 5e-5

if BEST_MAX_DIFF <= FIDELITY_ATOL:

    MAX_LENGTH_EXPLAINS_FAILURE = True

    print(
        "\nRESULT: max_seq_length behavior "
        "FULLY explains the frozen CWQ features."
    )

else:

    MAX_LENGTH_EXPLAINS_FAILURE = False

    print(
        "\nRESULT: max_seq_length alone does NOT "
        "fully explain the frozen CWQ features."
    )


print("\n" + "=" * 108)
print("=== MINILM QUESTION-RUNTIME DIAGNOSTIC COMPLETE ===")
print("=" * 108)

print(
    "Current runtime max_seq_length: ",
    ORIGINAL_RUNTIME_MAX_SEQ
)

print(
    "Best candidate max_seq_length:  ",
    BEST_MAX_SEQ_LENGTH
)

print(
    "Failure explained:              ",
    MAX_LENGTH_EXPLAINS_FAILURE
)

print(
    "\nMiniLM encoded affected validation questions only."
)

print(
    "No scorer."
)

print(
    "No pruning."
)

print(
    "No AFP hyperparameter tuning."
)

print(
    "No TEST examples accessed."
)

Affected CWQ source questions: 45
Affected decision groups:      70
Current MiniLM max_seq_length: 256

CWQ QUESTION TOKEN-LENGTH PROFILE

ALL VALIDATION
  n:       3519
  mean:    18.429667519181585
  median:  18.0
  p90:     24.0
  p95:     25.0
  p99:     29.0
  max:     39

AFFECTED 45 QUESTIONS
  n:       45
  mean:    20.733333333333334
  median:  20.0
  min:     15
  max:     30

> 64  tokens:  all=0     affected=0   / 45

> 96  tokens:  all=0     affected=0   / 45

> 128 tokens:  all=0     affected=0   / 45

> 160 tokens:  all=0     affected=0   / 45

> 192 tokens:  all=0     affected=0   / 45

> 256 tokens:  all=0     affected=0   / 45

AFFECTED QUESTION LENGTHS
 source_index  n_tokens                                    question_id                                                                                                              question
         1313        30  WebQTrn-3012_da5f6afb91d1bbe3f283829b281fe01b Of the countries that share a border with China, which count

In [22]:
# ======================================================================
# 3.12C-B3-DIAG4 — QUESTION NORMALIZATION FIDELITY AUDIT
# ======================================================================
#
# Hypothesis:
#   Missing FrozenMiniLMEncoder normalized raw_text using the existing
#   Feature-v2 normalize_surface_text() before SentenceTransformer.encode.
#
# Why plausible:
#   questions  -> stored RAW in semantic inventory
#   entities   -> already normalized
#   relations  -> already normalized
#   plans      -> already normalized
#   suffixes   -> already normalized
#
# Therefore an encoder-side normalizer would selectively alter QUESTION
# embeddings, matching the observed failure pattern.
#
# VALIDATION SOFTWARE-FIDELITY ONLY.
# No scorer.
# No pruning.
# No AFP hyperparameter tuning.
# No TEST examples accessed.
# ======================================================================

import numpy as np
import pandas as pd
import torch


# ======================================================================
# 1. Hard prerequisites
# ======================================================================

required = [
    "minilm_model",
    "cwq_val_exact",
    "cwq_frozen",
    "bad_source_set",
    "bad_groups",
    "encoder_unit",
    "extract_group",
    "iter_groups",
    "CWQ_LABEL_FILE",
    "feature_ns",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing previous B3 diagnostic objects: "
    + ", ".join(missing)
)


normalize_surface_text_fn = (
    feature_ns[
        "normalize_surface_text"
    ]
)

assert callable(
    normalize_surface_text_fn
)

tokenizer = (
    minilm_model.tokenizer
)

print(
    "Affected CWQ source questions:",
    len(bad_source_set)
)

print(
    "Affected decision groups:",
    len(bad_groups)
)


# ======================================================================
# 2. Compare RAW vs Feature-v2-normalized tokenization
# ======================================================================

token_change_rows = []

for source_index in range(
    len(cwq_val_exact)
):

    raw_question = str(
        cwq_val_exact[
            source_index
        ][
            "question"
        ]
    )

    normalized_question = (
        normalize_surface_text_fn(
            raw_question
        )
    )

    raw_ids = tokenizer(
        raw_question,
        add_special_tokens=True,
        truncation=False
    )[
        "input_ids"
    ]

    normalized_ids = tokenizer(
        normalized_question,
        add_special_tokens=True,
        truncation=False
    )[
        "input_ids"
    ]

    tokens_differ = (
        raw_ids
        != normalized_ids
    )

    if tokens_differ:

        token_change_rows.append(
            {
                "source_index":
                    int(
                        source_index
                    ),

                "question_id":
                    str(
                        cwq_val_exact[
                            source_index
                        ][
                            "id"
                        ]
                    ),

                "raw_question":
                    raw_question,

                "normalized_question":
                    normalized_question,

                "raw_token_count":
                    len(
                        raw_ids
                    ),

                "normalized_token_count":
                    len(
                        normalized_ids
                    ),
            }
        )


token_change_df = (
    pd.DataFrame(
        token_change_rows
    )
)

token_change_sources = {
    int(x)
    for x in (
        token_change_df[
            "source_index"
        ].tolist()
        if len(token_change_df)
        else []
    )
}


# ======================================================================
# 3. Compare token-change set with the 45 failure sources
# ======================================================================

intersection = (
    token_change_sources
    & bad_source_set
)

bad_not_changed = (
    bad_source_set
    - token_change_sources
)

changed_not_bad = (
    token_change_sources
    - bad_source_set
)


print(
    "\n"
    + "=" * 106
)

print(
    "RAW QUESTION vs normalize_surface_text() TOKENIZATION"
)

print(
    "=" * 106
)

print(
    "Validation questions:",
    len(cwq_val_exact)
)

print(
    "Questions whose token IDs change:",
    len(
        token_change_sources
    )
)

print(
    "Known failed source questions:",
    len(
        bad_source_set
    )
)

print(
    "Intersection:",
    len(
        intersection
    )
)

print(
    "Failed but tokenization unchanged:",
    len(
        bad_not_changed
    )
)

print(
    "Tokenization changed but feature did not fail:",
    len(
        changed_not_bad
    )
)


if bad_source_set:

    recall = (
        len(intersection)
        /
        len(bad_source_set)
    )

else:
    recall = 0.0


if token_change_sources:

    precision = (
        len(intersection)
        /
        len(token_change_sources)
    )

else:
    precision = 0.0


print(
    f"\nFailure-source recall:    "
    f"{100*recall:.2f}%"
)

print(
    f"Failure-source precision: "
    f"{100*precision:.2f}%"
)

print(
    "Exact set equality:",
    token_change_sources
    == bad_source_set
)


# ======================================================================
# 4. Show affected normalization examples
# ======================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "NORMALIZATION EXAMPLES AMONG FAILED QUESTIONS"
)

print(
    "=" * 110
)


shown = 0

for _, row in token_change_df.iterrows():

    src = int(
        row[
            "source_index"
        ]
    )

    if src not in bad_source_set:
        continue

    print(
        "\n"
        + "-" * 104
    )

    print(
        "source_index:",
        src
    )

    print(
        "id:",
        row[
            "question_id"
        ]
    )

    print(
        "RAW:       ",
        repr(
            row[
                "raw_question"
            ]
        )
    )

    print(
        "NORMALIZED:",
        repr(
            row[
                "normalized_question"
            ]
        )
    )

    print(
        "token counts:",
        row[
            "raw_token_count"
        ],
        "->",
        row[
            "normalized_token_count"
        ]
    )

    shown += 1

    if shown >= 25:
        break


# ======================================================================
# 5. Encode normalized versions of ONLY the 45 affected questions
# ======================================================================

affected_indices = sorted(
    bad_source_set
)

affected_normalized_questions = [
    normalize_surface_text_fn(
        str(
            cwq_val_exact[
                idx
            ][
                "question"
            ]
        )
    )
    for idx in affected_indices
]


print(
    "\nEncoding normalized affected questions only..."
)


with torch.inference_mode():

    normalized_embeddings = (
        minilm_model.encode(
            affected_normalized_questions,
            batch_size=64,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=False
        )
    )


normalized_embeddings = np.asarray(
    normalized_embeddings,
    dtype=np.float32
)

assert normalized_embeddings.shape == (
    len(
        affected_indices
    ),
    384
)


normalized_qid_embedding = {}

normalized_source_embedding = {}

for j, source_index in enumerate(
    affected_indices
):

    qid = str(
        cwq_val_exact[
            source_index
        ][
            "id"
        ]
    )

    normalized_qid_embedding[
        qid
    ] = (
        normalized_embeddings[
            j
        ]
    )

    normalized_source_embedding[
        int(
            source_index
        )
    ] = (
        normalized_embeddings[
            j
        ]
    )


# ======================================================================
# 6. Encoder override
# ======================================================================
#
# QUESTION:
#   use the normalized-question embedding above.
#
# Everything else:
#   delegate to encoder_unit, which already reproduced candidate/entity/
#   relation/plan/suffix semantics.
# ======================================================================

class NormalizedQuestionEncoder:

    def __init__(
        self,
        question_embedding_by_id,
        fallback_encoder
    ):

        self.question_embedding_by_id = (
            question_embedding_by_id
        )

        self.fallback_encoder = (
            fallback_encoder
        )

        self.dim = 384


    def get(
        self,
        namespace,
        identifier,
        raw_text
    ):

        namespace = str(
            namespace
        )

        identifier = str(
            identifier
        )

        if (
            namespace
            == "question"
            and identifier
            in self.question_embedding_by_id
        ):

            return np.asarray(
                self.question_embedding_by_id[
                    identifier
                ],
                dtype=np.float32
            )

        return (
            self.fallback_encoder.get(
                namespace,
                identifier,
                raw_text
            )
        )


normalized_question_encoder = (
    NormalizedQuestionEncoder(
        question_embedding_by_id=
            normalized_qid_embedding,

        fallback_encoder=
            encoder_unit
    )
)


# ======================================================================
# 7. Reload exact validation decision groups
# ======================================================================

cwq_group_rows_diag4 = list(
    iter_groups(
        CWQ_LABEL_FILE
    )
)

assert len(
    cwq_group_rows_diag4
) == 1352


group_ptr = np.asarray(
    cwq_frozen[
        "group_ptr"
    ],
    dtype=np.int64
)


# ======================================================================
# 8. Rebuild ONLY the 70 previously failing groups
# ======================================================================

all_differences = []

semantic_differences = []

symbolic_differences = []

SEMANTIC_COLUMNS = [
    0, 2, 4, 5, 6, 7, 8, 9
]

SYMBOLIC_COLUMNS = [
    i
    for i in range(27)
    if i not in SEMANTIC_COLUMNS
]


group_results = []


for g in sorted(
    bad_groups
):

    rows = (
        cwq_group_rows_diag4[
            g
        ]
    )

    first = rows[0]

    source_index = int(
        first[
            "source_index"
        ]
    )

    assert (
        source_index
        in bad_source_set
    )

    rec = (
        cwq_val_exact[
            source_index
        ]
    )

    assert str(
        rec["id"]
    ) == str(
        first[
            "question_id"
        ]
    )

    plan = list(
        first[
            "plan"
        ]
    )

    hop = int(
        first[
            "hop"
        ]
    )


    X_rebuilt = extract_group(
        question_id=
            first[
                "question_id"
            ],

        question=
            rec[
                "question"
            ],

        plan=
            plan,

        hop=
            hop,

        candidate_rows=
            rows,

        semantic_encoder=
            normalized_question_encoder,

        entity_name_map=
            None
    )


    start = int(
        group_ptr[
            g
        ]
    )

    end = int(
        group_ptr[
            g + 1
        ]
    )

    X_frozen = np.asarray(
        cwq_frozen[
            "X"
        ][
            start:end
        ],
        dtype=np.float32
    )


    assert X_rebuilt.shape == (
        X_frozen.shape
    )


    diff = np.abs(
        X_rebuilt.astype(
            np.float64
        )
        -
        X_frozen.astype(
            np.float64
        )
    )


    sem_diff = diff[
        :,
        SEMANTIC_COLUMNS
    ]

    sym_diff = diff[
        :,
        SYMBOLIC_COLUMNS
    ]


    all_differences.append(
        diff.ravel()
    )

    semantic_differences.append(
        sem_diff.ravel()
    )

    symbolic_differences.append(
        sym_diff.ravel()
    )


    group_results.append(
        {
            "group_index":
                int(g),

            "source_index":
                source_index,

            "question_id":
                str(
                    rec[
                        "id"
                    ]
                ),

            "max_all":
                float(
                    diff.max()
                ),

            "max_semantic":
                float(
                    sem_diff.max()
                ),

            "max_symbolic":
                float(
                    sym_diff.max()
                ),
        }
    )


all_differences = np.concatenate(
    all_differences
)

semantic_differences = np.concatenate(
    semantic_differences
)

symbolic_differences = np.concatenate(
    symbolic_differences
)


# ======================================================================
# 9. Fidelity report
# ======================================================================

print(
    "\n"
    + "=" * 108
)

print(
    "NORMALIZED-QUESTION FEATURE-v2 FIDELITY"
)

print(
    "=" * 108
)

print(
    "Groups rebuilt:",
    len(
        group_results
    )
)

print(
    "Max all-feature difference:",
    float(
        all_differences.max()
    )
)

print(
    "Mean all-feature difference:",
    float(
        all_differences.mean()
    )
)

print(
    "Max semantic difference:",
    float(
        semantic_differences.max()
    )
)

print(
    "Mean semantic difference:",
    float(
        semantic_differences.mean()
    )
)

print(
    "Max symbolic difference:",
    float(
        symbolic_differences.max()
    )
)


group_result_df = (
    pd.DataFrame(
        group_results
    )
    .sort_values(
        "max_semantic",
        ascending=False
    )
)


print(
    "\nWorst groups after normalization:"
)

print(
    group_result_df.head(
        20
    ).to_string(
        index=False,
        float_format=lambda x: f"{x:.10g}"
    )
)


# ======================================================================
# 10. Strong software-fidelity gate
# ======================================================================

SEMANTIC_ATOL = 5e-5
SYMBOLIC_ATOL = 1e-7

NORMALIZATION_EXPLAINS_FAILURE = (
    float(
        semantic_differences.max()
    )
    <= SEMANTIC_ATOL

    and

    float(
        symbolic_differences.max()
    )
    <= SYMBOLIC_ATOL
)


print(
    "\n"
    + "=" * 108
)

print(
    "QUESTION NORMALIZATION ROOT-CAUSE TEST"
)

print(
    "=" * 108
)

print(
    "Token-change exact set equality:",
    token_change_sources
    == bad_source_set
)

print(
    "Feature-fidelity gate:",
    NORMALIZATION_EXPLAINS_FAILURE
)


if NORMALIZATION_EXPLAINS_FAILURE:

    QUESTION_RUNTIME_ROOT_CAUSE = (
        "FrozenMiniLMEncoder normalized raw question text "
        "before MiniLM encoding"
    )

    print(
        "\nRESULT: QUESTION NORMALIZATION "
        "FULLY EXPLAINS THE CWQ FAILURE."
    )

else:

    QUESTION_RUNTIME_ROOT_CAUSE = (
        "question normalization alone does not fully explain failure"
    )

    print(
        "\nRESULT: QUESTION NORMALIZATION DOES NOT "
        "FULLY EXPLAIN THE CWQ FAILURE."
    )


print(
    "\nRoot-cause classification:"
)

print(
    QUESTION_RUNTIME_ROOT_CAUSE
)


print(
    "\nMiniLM encoded only 45 affected validation questions."
)

print(
    "No scorer."
)

print(
    "No pruning."
)

print(
    "No AFP hyperparameter tuning."
)

print(
    "No TEST examples accessed."
)

Affected CWQ source questions: 45
Affected decision groups: 70

RAW QUESTION vs normalize_surface_text() TOKENIZATION
Validation questions: 3519
Questions whose token IDs change: 265
Known failed source questions: 45
Intersection: 45
Failed but tokenization unchanged: 0
Tokenization changed but feature did not fail: 220

Failure-source recall:    100.00%
Failure-source precision: 16.98%
Exact set equality: False

NORMALIZATION EXAMPLES AMONG FAILED QUESTIONS

--------------------------------------------------------------------------------------------------------
source_index: 76
id: WebQTrn-836_4dbb23937c062d0f3a3496d2fbf26a12
RAW:        "What major religion in the UK has a place of worship named St. Mary's Cathedral, Batticaloa?"
NORMALIZED: "what major religion in the uk has a place of worship named st mary's cathedral, batticaloa?"
token counts: 25 -> 24

--------------------------------------------------------------------------------------------------------
source_index: 347
id: W

In [23]:
# ======================================================================
# 3.12C-B3-FINAL
# BEHAVIORALLY RECOVERED MINILM RUNTIME + FULL FEATURE-v2 FIDELITY GATE
# ======================================================================
#
# RECOVERED BEHAVIOR
# ------------------
# The original FrozenMiniLMEncoder source was not persisted in the
# notebook. Validation diagnostics established that its observable
# behavior includes:
#
#       normalize_surface_text(raw_text)
#               ↓
#             MiniLM
#
# Evidence:
#   - WebQSP already reproduced to ~3e-7.
#   - CWQ failure occurred ONLY in q-dependent semantic features.
#   - Question normalization reduced CWQ failing-group max error from
#       0.4881388545
#     to
#       2.980232e-07.
#
# THIS CELL:
#   1. Defines recovered runtime encoder.
#   2. Prefills validation semantic inventory.
#   3. Rebuilds ALL frozen validation Feature-v2 groups.
#   4. Requires full WebQSP + CWQ numerical fidelity.
#   5. Exposes runtime encoder/extractor for Cell 12C-B4.
#
# NO scorer selection.
# NO pruning.
# NO AFP hyperparameter tuning.
# NO TEST examples accessed.
# ======================================================================

import json
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm


# ======================================================================
# 1. Hard prerequisites
# ======================================================================

required = [
    "minilm_model",
    "feature_ns",
    "extract_group",
    "iter_groups",
    "WEBQSP_LABEL_FILE",
    "CWQ_LABEL_FILE",
    "WEBQSP_NPZ",
    "CWQ_NPZ",
    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing B3/diagnostic objects: "
    + ", ".join(missing)
)


normalize_surface_text_runtime = (
    feature_ns[
        "normalize_surface_text"
    ]
)

relation_surface_text_runtime = (
    feature_ns[
        "relation_surface_text"
    ]
)

relation_sequence_text_runtime = (
    feature_ns[
        "relation_sequence_text"
    ]
)

has_readable_entity_surface_runtime = (
    feature_ns[
        "has_readable_entity_surface"
    ]
)

entity_surface_text_runtime = (
    feature_ns[
        "entity_surface_text"
    ]
)

assert callable(
    normalize_surface_text_runtime
)

assert callable(
    extract_group
)


print(
    "Recovered normalization function: READY"
)

print(
    "MiniLM embedding dimension:",
    minilm_model.get_embedding_dimension()
)


# ======================================================================
# 2. Resolve validation question records
# ======================================================================
#
# Original Cell 173 used the validation dataset's question field.
#
# CWQ exact dataset was loaded during diagnostics.
#
# For WebQSP:
#   use webqsp_val if still present;
#   otherwise frozen planning rows contain the aligned question text
#   already shown to reproduce the frozen matrix.
# ======================================================================

if (
    "webqsp_val" in globals()
    and globals()["webqsp_val"] is not None
):

    webqsp_val_runtime = globals()[
        "webqsp_val"
    ]

    WEBQSP_RUNTIME_SOURCE = (
        "existing_global:webqsp_val"
    )

else:

    webqsp_val_runtime = (
        webqsp_val_plan_rows
    )

    WEBQSP_RUNTIME_SOURCE = (
        "frozen_validation_planning_rows"
    )


if (
    "cwq_val_exact" in globals()
    and globals()["cwq_val_exact"] is not None
):

    cwq_val_runtime = (
        cwq_val_exact
    )

    CWQ_RUNTIME_SOURCE = (
        "exact_validation_dataset"
    )

elif (
    "cwq_val" in globals()
    and globals()["cwq_val"] is not None
):

    cwq_val_runtime = (
        globals()["cwq_val"]
    )

    CWQ_RUNTIME_SOURCE = (
        "existing_global:cwq_val"
    )

else:

    cwq_val_runtime = (
        cwq_val_plan_rows
    )

    CWQ_RUNTIME_SOURCE = (
        "frozen_validation_planning_rows"
    )


assert len(
    webqsp_val_runtime
) == 246

assert len(
    cwq_val_runtime
) == 3519


print(
    "\nValidation sources:"
)

print(
    "  WebQSP:",
    WEBQSP_RUNTIME_SOURCE
)

print(
    "  CWQ:   ",
    CWQ_RUNTIME_SOURCE
)


# ======================================================================
# 3. Exact validation group iterator alias
# ======================================================================

iter_runtime_groups = (
    iter_groups
)


# ======================================================================
# 4. Collect semantic inventory exactly as Feature-v2 expects
# ======================================================================

def collect_runtime_semantic_inventory(
    labels_file,
    dataset_rows
):

    questions = {}
    entities = {}
    relations = {}
    plans = {}
    suffixes = {}

    n_groups = 0
    n_branches = 0

    for rows in iter_runtime_groups(
        labels_file
    ):

        n_groups += 1
        n_branches += len(
            rows
        )

        first = rows[0]

        source_index = int(
            first[
                "source_index"
            ]
        )

        rec = dataset_rows[
            source_index
        ]

        assert str(
            rec["id"]
        ) == str(
            first[
                "question_id"
            ]
        )


        # --------------------------------------------------------------
        # CRITICAL:
        # Store the RAW question here.
        #
        # The recovered semantic encoder performs normalization.
        # This mirrors the original Cell-173 inventory behavior.
        # --------------------------------------------------------------

        question_id = str(
            first[
                "question_id"
            ]
        )

        questions[
            question_id
        ] = str(
            rec[
                "question"
            ]
        )


        plan = list(
            first[
                "plan"
            ]
        )

        hop = int(
            first[
                "hop"
            ]
        )


        for relation in plan:

            relations[
                str(
                    relation
                )
            ] = (
                relation_surface_text_runtime(
                    relation
                )
            )


        plan_key = "||".join(
            str(x)
            for x in plan
        )

        plans[
            plan_key
        ] = (
            relation_sequence_text_runtime(
                plan
            )
        )


        suffix = (
            plan[
                hop + 1:
            ]
        )

        suffix_key = "||".join(
            str(x)
            for x in suffix
        )

        suffixes[
            suffix_key
        ] = (
            relation_sequence_text_runtime(
                suffix
            )
        )


        for row in rows:

            candidate = row[
                "candidate_entity"
            ]

            if (
                has_readable_entity_surface_runtime(
                    candidate
                )
            ):

                entities[
                    str(
                        candidate
                    )
                ] = (
                    entity_surface_text_runtime(
                        candidate
                    )
                )


            for entity in row[
                "prefix_entities"
            ]:

                if (
                    has_readable_entity_surface_runtime(
                        entity
                    )
                ):

                    entities[
                        str(
                            entity
                        )
                    ] = (
                        entity_surface_text_runtime(
                            entity
                        )
                    )


    return {
        "questions":
            questions,

        "entities":
            entities,

        "relations":
            relations,

        "plans":
            plans,

        "suffixes":
            suffixes,

        "n_groups":
            n_groups,

        "n_branches":
            n_branches,
    }


print(
    "\nCollecting full validation semantic inventory..."
)


webqsp_inventory_final = (
    collect_runtime_semantic_inventory(
        WEBQSP_LABEL_FILE,
        webqsp_val_runtime
    )
)

cwq_inventory_final = (
    collect_runtime_semantic_inventory(
        CWQ_LABEL_FILE,
        cwq_val_runtime
    )
)


assert (
    webqsp_inventory_final[
        "n_groups"
    ]
    == 87
)

assert (
    webqsp_inventory_final[
        "n_branches"
    ]
    == 966
)

assert (
    cwq_inventory_final[
        "n_groups"
    ]
    == 1352
)

assert (
    cwq_inventory_final[
        "n_branches"
    ]
    == 18688
)


print(
    "WebQSP:",
    webqsp_inventory_final[
        "n_groups"
    ],
    "groups |",
    webqsp_inventory_final[
        "n_branches"
    ],
    "branches"
)

print(
    "CWQ:   ",
    cwq_inventory_final[
        "n_groups"
    ],
    "groups |",
    cwq_inventory_final[
        "n_branches"
    ],
    "branches"
)


# ======================================================================
# 5. Merge semantic inventory
# ======================================================================

def merge_runtime_dicts(
    *dicts
):

    result = {}

    for d in dicts:

        result.update(
            d
        )

    return result


RUNTIME_QUESTIONS = (
    merge_runtime_dicts(
        webqsp_inventory_final[
            "questions"
        ],
        cwq_inventory_final[
            "questions"
        ],
    )
)

RUNTIME_ENTITIES = (
    merge_runtime_dicts(
        webqsp_inventory_final[
            "entities"
        ],
        cwq_inventory_final[
            "entities"
        ],
    )
)

RUNTIME_RELATIONS = (
    merge_runtime_dicts(
        webqsp_inventory_final[
            "relations"
        ],
        cwq_inventory_final[
            "relations"
        ],
    )
)

RUNTIME_PLANS = (
    merge_runtime_dicts(
        webqsp_inventory_final[
            "plans"
        ],
        cwq_inventory_final[
            "plans"
        ],
    )
)

RUNTIME_SUFFIXES = (
    merge_runtime_dicts(
        webqsp_inventory_final[
            "suffixes"
        ],
        cwq_inventory_final[
            "suffixes"
        ],
    )
)


print(
    "\nSemantic inventory"
)

print(
    "  Questions:         ",
    len(
        RUNTIME_QUESTIONS
    )
)

print(
    "  Readable entities: ",
    len(
        RUNTIME_ENTITIES
    )
)

print(
    "  Relations:         ",
    len(
        RUNTIME_RELATIONS
    )
)

print(
    "  Plans:             ",
    len(
        RUNTIME_PLANS
    )
)

print(
    "  Suffixes:          ",
    len(
        RUNTIME_SUFFIXES
    )
)


# ======================================================================
# 6. Behaviorally recovered FrozenMiniLM runtime
# ======================================================================
#
# Important:
#
# The lost wrapper source cannot be claimed to have been recovered
# textually.
#
# What IS recovered and verified is its observable feature behavior:
#
#       raw_text
#          ↓
#       normalize_surface_text
#          ↓
#       all-MiniLM-L6-v2
#
# We use unit-normalized MiniLM vectors.
#
# Feature-v2 cosine operations and validation fidelity determine the
# relevant observable behavior.
# ======================================================================

class BehaviorallyRecoveredMiniLMEncoder:

    def __init__(
        self,
        model,
        batch_size=256
    ):

        self.model = model

        self.batch_size = int(
            batch_size
        )

        self.dim = int(
            model.get_embedding_dimension()
        )

        assert self.dim == 384

        self.cache = {}


    def _normalize_text(
        self,
        raw_text
    ):

        return (
            normalize_surface_text_runtime(
                raw_text
            )
        )


    def prefill(
        self,
        namespace,
        mapping
    ):

        namespace = str(
            namespace
        )

        items = list(
            mapping.items()
        )

        missing_items = []

        for identifier, raw_text in items:

            key = (
                namespace,
                str(
                    identifier
                )
            )

            if key not in self.cache:

                missing_items.append(
                    (
                        str(
                            identifier
                        ),
                        raw_text
                    )
                )


        if not missing_items:
            return


        texts = [
            self._normalize_text(
                raw_text
            )
            for _, raw_text
            in missing_items
        ]


        embeddings = (
            self.model.encode(
                texts,
                batch_size=
                    self.batch_size,

                show_progress_bar=
                    False,

                convert_to_numpy=
                    True,

                normalize_embeddings=
                    True
            )
        )


        embeddings = np.asarray(
            embeddings,
            dtype=np.float32
        )


        assert embeddings.shape == (
            len(
                missing_items
            ),
            self.dim
        )


        for (
            (
                identifier,
                _
            ),
            vector
        ) in zip(
            missing_items,
            embeddings
        ):

            self.cache[
                (
                    namespace,
                    identifier
                )
            ] = np.asarray(
                vector,
                dtype=np.float32
            )


    def get(
        self,
        namespace,
        identifier,
        raw_text
    ):

        namespace = str(
            namespace
        )

        identifier = str(
            identifier
        )

        key = (
            namespace,
            identifier
        )


        if key not in self.cache:

            text = self._normalize_text(
                raw_text
            )

            vector = (
                self.model.encode(
                    [text],
                    batch_size=1,
                    show_progress_bar=False,
                    convert_to_numpy=True,
                    normalize_embeddings=True
                )[0]
            )

            self.cache[
                key
            ] = np.asarray(
                vector,
                dtype=np.float32
            )


        return self.cache[
            key
        ]


# ======================================================================
# 7. Instantiate recovered runtime
# ======================================================================

AFP_RECOVERED_SEMANTIC_ENCODER = (
    BehaviorallyRecoveredMiniLMEncoder(
        model=
            minilm_model,

        batch_size=
            256
    )
)


print(
    "\nPrefilling recovered semantic runtime..."
)


for namespace, mapping in [
    (
        "question",
        RUNTIME_QUESTIONS
    ),
    (
        "entity",
        RUNTIME_ENTITIES
    ),
    (
        "relation",
        RUNTIME_RELATIONS
    ),
    (
        "plan",
        RUNTIME_PLANS
    ),
    (
        "suffix",
        RUNTIME_SUFFIXES
    ),
]:

    print(
        f"  {namespace:<10}"
        f"{len(mapping):>6}"
    )

    AFP_RECOVERED_SEMANTIC_ENCODER.prefill(
        namespace,
        mapping
    )


print(
    "\nRecovered semantic cache entries:",
    len(
        AFP_RECOVERED_SEMANTIC_ENCODER.cache
    )
)


# ======================================================================
# 8. Full validation Feature-v2 reconstruction
# ======================================================================

def rebuild_full_validation_features(
    dataset_name,
    dataset_rows,
    labels_file,
    encoder
):

    X_blocks = []
    y_blocks = []

    group_ptr = [
        0
    ]

    group_source_index = []
    group_hop = []
    group_plan_length = []
    group_candidate_count = []


    n_groups = 0


    for rows in tqdm(
        iter_runtime_groups(
            labels_file
        ),
        desc=(
            f"{dataset_name} full "
            "Feature-v2 fidelity"
        )
    ):

        first = rows[
            0
        ]

        source_index = int(
            first[
                "source_index"
            ]
        )

        rec = dataset_rows[
            source_index
        ]


        assert str(
            rec[
                "id"
            ]
        ) == str(
            first[
                "question_id"
            ]
        )


        plan = list(
            first[
                "plan"
            ]
        )

        hop = int(
            first[
                "hop"
            ]
        )


        X_group = extract_group(
            question_id=
                first[
                    "question_id"
                ],

            question=
                rec[
                    "question"
                ],

            plan=
                plan,

            hop=
                hop,

            candidate_rows=
                rows,

            semantic_encoder=
                encoder,

            entity_name_map=
                None
        )


        y_group = np.asarray(
            [
                int(
                    row[
                        "label"
                    ]
                )
                for row in rows
            ],
            dtype=np.uint8
        )


        assert X_group.shape == (
            len(
                rows
            ),
            27
        )


        X_blocks.append(
            X_group
        )

        y_blocks.append(
            y_group
        )


        group_ptr.append(
            group_ptr[-1]
            + len(
                rows
            )
        )


        group_source_index.append(
            source_index
        )

        group_hop.append(
            hop
        )

        group_plan_length.append(
            len(
                plan
            )
        )

        group_candidate_count.append(
            len(
                rows
            )
        )


        n_groups += 1


    return {
        "X":
            np.concatenate(
                X_blocks,
                axis=0
            ).astype(
                np.float32
            ),

        "y":
            np.concatenate(
                y_blocks,
                axis=0
            ).astype(
                np.uint8
            ),

        "group_ptr":
            np.asarray(
                group_ptr,
                dtype=np.int64
            ),

        "group_source_index":
            np.asarray(
                group_source_index,
                dtype=np.int32
            ),

        "group_hop":
            np.asarray(
                group_hop,
                dtype=np.int16
            ),

        "group_plan_length":
            np.asarray(
                group_plan_length,
                dtype=np.int16
            ),

        "group_candidate_count":
            np.asarray(
                group_candidate_count,
                dtype=np.int32
            ),

        "n_groups":
            int(
                n_groups
            ),
    }


print(
    "\nRebuilding COMPLETE WebQSP validation matrix..."
)

webqsp_runtime_features = (
    rebuild_full_validation_features(
        dataset_name=
            "webqsp",

        dataset_rows=
            webqsp_val_runtime,

        labels_file=
            WEBQSP_LABEL_FILE,

        encoder=
            AFP_RECOVERED_SEMANTIC_ENCODER
    )
)


print(
    "\nRebuilding COMPLETE CWQ validation matrix..."
)

cwq_runtime_features = (
    rebuild_full_validation_features(
        dataset_name=
            "cwq",

        dataset_rows=
            cwq_val_runtime,

        labels_file=
            CWQ_LABEL_FILE,

        encoder=
            AFP_RECOVERED_SEMANTIC_ENCODER
    )
)


# ======================================================================
# 9. Load frozen Feature-v2 references
# ======================================================================

def load_frozen_npz_final(
    path
):

    z = np.load(
        path,
        allow_pickle=False
    )

    return {
        key:
            z[
                key
            ]
        for key in z.files
    }


webqsp_frozen_final = (
    load_frozen_npz_final(
        WEBQSP_NPZ
    )
)

cwq_frozen_final = (
    load_frozen_npz_final(
        CWQ_NPZ
    )
)


# ======================================================================
# 10. Exact metadata gates
# ======================================================================

META_KEYS = [
    "y",
    "group_ptr",
    "group_source_index",
    "group_hop",
    "group_plan_length",
    "group_candidate_count",
]


def exact_metadata_gate(
    dataset_name,
    rebuilt,
    frozen
):

    print(
        f"\n{dataset_name.upper()} metadata"
    )

    for key in META_KEYS:

        assert np.array_equal(
            rebuilt[
                key
            ],
            frozen[
                key
            ]
        ), (
            f"{dataset_name}: "
            f"{key} mismatch"
        )

        print(
            f"  {key:<25} PASS"
        )


exact_metadata_gate(
    "webqsp",
    webqsp_runtime_features,
    webqsp_frozen_final
)

exact_metadata_gate(
    "cwq",
    cwq_runtime_features,
    cwq_frozen_final
)


# ======================================================================
# 11. Numerical fidelity
# ======================================================================

SEMANTIC_COLUMNS = [
    0,
    2,
    4,
    5,
    6,
    7,
    8,
    9,
]

SYMBOLIC_COLUMNS = [
    i
    for i in range(
        27
    )
    if i not in (
        SEMANTIC_COLUMNS
    )
]


def full_feature_fidelity_report(
    dataset_name,
    rebuilt,
    frozen
):

    A = np.asarray(
        rebuilt[
            "X"
        ],
        dtype=np.float64
    )

    B = np.asarray(
        frozen[
            "X"
        ],
        dtype=np.float64
    )

    assert A.shape == B.shape


    diff = np.abs(
        A - B
    )

    sem = diff[
        :,
        SEMANTIC_COLUMNS
    ]

    sym = diff[
        :,
        SYMBOLIC_COLUMNS
    ]


    report = {
        "shape":
            list(
                A.shape
            ),

        "max_all":
            float(
                diff.max()
            ),

        "mean_all":
            float(
                diff.mean()
            ),

        "p99_all":
            float(
                np.quantile(
                    diff,
                    0.99
                )
            ),

        "max_semantic":
            float(
                sem.max()
            ),

        "mean_semantic":
            float(
                sem.mean()
            ),

        "max_symbolic":
            float(
                sym.max()
            ),

        "mean_symbolic":
            float(
                sym.mean()
            ),

        "n_values_gt_5e_5":
            int(
                np.sum(
                    diff
                    > 5e-5
                )
            ),
    }


    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{dataset_name.upper()} FULL FEATURE-v2 FIDELITY"
    )

    print(
        "=" * 100
    )


    for key, value in report.items():

        print(
            f"{key:<24}",
            value
        )


    return report


webqsp_full_fidelity = (
    full_feature_fidelity_report(
        "webqsp",
        webqsp_runtime_features,
        webqsp_frozen_final
    )
)

cwq_full_fidelity = (
    full_feature_fidelity_report(
        "cwq",
        cwq_runtime_features,
        cwq_frozen_final
    )
)


# ======================================================================
# 12. HARD software-fidelity gate
# ======================================================================

SEMANTIC_ATOL = 5e-5
SYMBOLIC_ATOL = 1e-7


for dataset_name, result in [
    (
        "WebQSP",
        webqsp_full_fidelity
    ),
    (
        "CWQ",
        cwq_full_fidelity
    ),
]:

    assert (
        result[
            "max_semantic"
        ]
        <= SEMANTIC_ATOL
    ), (
        f"{dataset_name}: semantic fidelity failed."
    )


    assert (
        result[
            "max_symbolic"
        ]
        <= SYMBOLIC_ATOL
    ), (
        f"{dataset_name}: symbolic fidelity failed."
    )


    assert (
        result[
            "n_values_gt_5e_5"
        ]
        == 0
    )


print(
    "\nFULL Feature-v2 software fidelity: PASSED"
)


# ======================================================================
# 13. Expose FINAL online runtime objects
# ======================================================================

AFP_RUNTIME_SEMANTIC_ENCODER = (
    AFP_RECOVERED_SEMANTIC_ENCODER
)

AFP_RUNTIME_FEATURE_EXTRACTOR = (
    extract_group
)

AFP_RUNTIME_FEATURE_VERSION = (
    "afp_features_v2_masked_entity_semantics"
)

AFP_RUNTIME_FEATURE_DIM = (
    27
)

AFP_RUNTIME_SEMANTIC_MODEL = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

AFP_RUNTIME_TEXT_POLICY = (
    "normalize_surface_text_before_minilm"
)

AFP_RUNTIME_RECOVERY_STATUS = (
    "behaviorally_recovered_and_feature_fidelity_verified"
)


# ======================================================================
# 14. Save final fidelity manifest
# ======================================================================

TRAVERSAL_DIR = (
    Path(
        "/kaggle/working/"
        "step3_rq2_dev_v1/"
        "10_validation_traversal"
    )
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


B3_FINAL_MANIFEST = {
    "cell":
        "RQ2_12C_B3_FINAL",

    "feature_version":
        AFP_RUNTIME_FEATURE_VERSION,

    "feature_dim":
        27,

    "semantic_encoder":
        AFP_RUNTIME_SEMANTIC_MODEL,

    "original_wrapper_source_available":
        False,

    "recovery_type":
        "behavioral_software_fidelity",

    "recovered_text_policy":
        AFP_RUNTIME_TEXT_POLICY,

    "recovered_vector_policy":
        "unit_normalized_minilm_embedding",

    "recovery_evidence": {
        "cwq_initial_max_semantic_error":
            0.4881388545036316,

        "cwq_normalized_question_diagnostic_max_error":
            2.980232238769531e-07,
    },

    "webqsp":
        webqsp_full_fidelity,

    "cwq":
        cwq_full_fidelity,

    "semantic_atol":
        SEMANTIC_ATOL,

    "symbolic_atol":
        SYMBOLIC_ATOL,

    "metadata_fidelity":
        True,

    "full_feature_fidelity":
        True,

    "online_feature_runtime_ready":
        True,

    "scorer_integration_ready":
        False,

    "pruning_run":
        False,

    "afp_hyperparameter_tuning_run":
        False,

    "test_examples_accessed":
        False,

    "complete_afp_frozen":
        False,
}


B3_FINAL_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12c_b3_final_feature_runtime_fidelity.json"
)


with open(
    B3_FINAL_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        B3_FINAL_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 15. Final report
# ======================================================================

print(
    "\n"
    + "=" * 112
)

print(
    "=== RQ2 CELL 12C-B3-FINAL: FEATURE-v2 ONLINE RUNTIME VERIFIED ==="
)

print(
    "=" * 112
)


print(
    "\nRecovered semantic behavior:"
)

print(
    "  raw text"
)

print(
    "    -> normalize_surface_text"
)

print(
    "    -> all-MiniLM-L6-v2"
)

print(
    "    -> unit-normalized embedding"
)


print(
    "\nWebQSP full validation:"
)

print(
    "  branches:",
    webqsp_runtime_features[
        "X"
    ].shape[
        0
    ]
)

print(
    "  max semantic error:",
    webqsp_full_fidelity[
        "max_semantic"
    ]
)

print(
    "  max symbolic error:",
    webqsp_full_fidelity[
        "max_symbolic"
    ]
)


print(
    "\nCWQ full validation:"
)

print(
    "  branches:",
    cwq_runtime_features[
        "X"
    ].shape[
        0
    ]
)

print(
    "  max semantic error:",
    cwq_full_fidelity[
        "max_semantic"
    ]
)

print(
    "  max symbolic error:",
    cwq_full_fidelity[
        "max_symbolic"
    ]
)


print(
    "\nRuntime status:"
)

print(
    "  Exact Feature-v2 code:      VERIFIED"
)

print(
    "  Semantic behavior:          RECOVERED"
)

print(
    "  Full frozen-NPZ fidelity:   PASS"
)

print(
    "  Online feature runtime:     READY"
)

print(
    "  Scorer integration:         NEXT"
)

print(
    "  Pruning run:                NO"
)

print(
    "  Hyperparameter tuning:      NO"
)

print(
    "  TEST examples accessed:     NO"
)

print(
    "  Complete AFP frozen:        NO"
)


print(
    "\nManifest:",
    B3_FINAL_MANIFEST_PATH
)

print(
    "\nNext: Cell 12C-B4 — frozen standardizer + selected scorer "
    "checkpoint integration and online-logit fidelity gate."
)

Recovered normalization function: READY
MiniLM embedding dimension: 384

Validation sources:
  WebQSP: frozen_validation_planning_rows
  CWQ:    exact_validation_dataset

WebQSP: 87 groups | 966 branches
CWQ:    1352 groups | 18688 branches

Semantic inventory
  Questions:          882
  Readable entities:  2637
  Relations:          274
  Plans:              292
  Suffixes:           190

Prefilling recovered semantic runtime...
  question     882
  entity      2637
  relation     274
  plan         292
  suffix       190

Recovered semantic cache entries: 4275

Rebuilding COMPLETE WebQSP validation matrix...


webqsp full Feature-v2 fidelity: 0it [00:00, ?it/s]


Rebuilding COMPLETE CWQ validation matrix...


cwq full Feature-v2 fidelity: 0it [00:00, ?it/s]


WEBQSP metadata
  y                         PASS
  group_ptr                 PASS
  group_source_index        PASS
  group_hop                 PASS
  group_plan_length         PASS
  group_candidate_count     PASS

CWQ metadata
  y                         PASS
  group_ptr                 PASS
  group_source_index        PASS
  group_hop                 PASS
  group_plan_length         PASS
  group_candidate_count     PASS

WEBQSP FULL FEATURE-v2 FIDELITY
shape                    [966, 27]
max_all                  3.5762786865234375e-07
mean_all                 1.6312285219084568e-08
p99_all                  2.0559877157210816e-07
max_semantic             3.5762786865234375e-07
mean_semantic            5.5053962614410415e-08
max_symbolic             0.0
mean_symbolic            0.0
n_values_gt_5e_5         0

CWQ FULL FEATURE-v2 FIDELITY
shape                    [18688, 27]
max_all                  5.364418029785156e-07
mean_all                 1.8326335775376288e-08
p99_all           

In [25]:
# ======================================================================
# RQ2 CELL 12C-B4 — REVISED
# FROZEN STANDARDIZER + SELECTED SCORER INTEGRATION
# + ONLINE LOGIT FIDELITY GATE
# ======================================================================
#
# PURPOSE
# -------
# 1. Recover the exact persisted AFPScorer definition.
# 2. Recover its required AFP_* namespace constants safely.
# 3. Load the already-selected development checkpoints:
#
#       WebQSP: H=32, branch_bce, seed=42
#       CWQ:    H=64, branch_bce, seed=42
#
# 4. Restore frozen TRAIN-only standardizers.
# 5. Verify selected checkpoint identities.
# 6. Compare scorer logits from:
#
#       frozen Feature-v2 NPZ
#                 vs
#       recovered online Feature-v2
#
# 7. Build the gold-free runtime scoring callback for Cell 13.
# 8. Spot-check actual group-level online scoring.
#
# IMPORTANT
# ---------
# SOFTWARE/RUNTIME FIDELITY ONLY.
#
# NO scorer re-selection.
# NO pruning.
# NO AFP hyperparameter tuning.
# NO TEST examples accessed.
# ======================================================================

import ast
import hashlib
import inspect
import json
from pathlib import Path
from typing import *

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ======================================================================
# 1. HARD PREREQUISITES FROM B3-FINAL
# ======================================================================

required_b3_objects = [
    "AFP_RUNTIME_SEMANTIC_ENCODER",
    "AFP_RUNTIME_FEATURE_EXTRACTOR",
    "AFP_RUNTIME_FEATURE_VERSION",
    "AFP_RUNTIME_FEATURE_DIM",

    "webqsp_runtime_features",
    "cwq_runtime_features",

    "webqsp_frozen_final",
    "cwq_frozen_final",

    "WEBQSP_LABEL_FILE",
    "CWQ_LABEL_FILE",

    "webqsp_val_runtime",
    "cwq_val_runtime",

    "iter_runtime_groups",
]

missing_b3_objects = [
    name
    for name in required_b3_objects
    if name not in globals()
]

assert not missing_b3_objects, (
    "Missing B3-FINAL objects:\n  "
    + "\n  ".join(missing_b3_objects)
    + "\nDo NOT continue. B3-FINAL runtime objects are required."
)

assert int(
    AFP_RUNTIME_FEATURE_DIM
) == 27

DEVICE = torch.device(
    "cpu"
)

print(
    "Runtime device:",
    DEVICE
)

print(
    "Feature runtime:",
    AFP_RUNTIME_FEATURE_VERSION
)

print(
    "Feature dimension:",
    AFP_RUNTIME_FEATURE_DIM
)


# ======================================================================
# 2. ARTIFACT PATHS
# ======================================================================

ROOT = Path(
    "/kaggle/working/"
    "step3_rq2_dev_v1"
)

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/"
    "mdsadmansamikhan/rog-ap/"
    "__notebook__.ipynb"
)

FINAL_SCORER_DIR = (
    ROOT
    / "07_final_scorer"
)

WEBQSP_CKPT = (
    FINAL_SCORER_DIR
    / "webqsp_afp_scorer_selected.pt"
)

CWQ_CKPT = (
    FINAL_SCORER_DIR
    / "cwq_afp_scorer_selected.pt"
)


assert NOTEBOOK_PATH.exists(), (
    f"Notebook not found: {NOTEBOOK_PATH}"
)

assert WEBQSP_CKPT.exists(), (
    f"WebQSP checkpoint not found: {WEBQSP_CKPT}"
)

assert CWQ_CKPT.exists(), (
    f"CWQ checkpoint not found: {CWQ_CKPT}"
)


print(
    "\nSelected checkpoints:"
)

print(
    "  WebQSP:",
    WEBQSP_CKPT
)

print(
    "  CWQ:   ",
    CWQ_CKPT
)


# ======================================================================
# 3. SHA256 IDENTITY GATE
# ======================================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


webqsp_ckpt_sha = sha256_file(
    WEBQSP_CKPT
)

cwq_ckpt_sha = sha256_file(
    CWQ_CKPT
)


print(
    "\nCheckpoint SHA256"
)

print(
    "  WebQSP:",
    webqsp_ckpt_sha
)

print(
    "  CWQ:   ",
    cwq_ckpt_sha
)


# Selected scorer artifacts frozen at Cell 9C.
assert webqsp_ckpt_sha.startswith(
    "bee65146403d5656"
), (
    "WebQSP selected checkpoint identity mismatch."
)

assert cwq_ckpt_sha.startswith(
    "91a531c057bb02d0"
), (
    "CWQ selected checkpoint identity mismatch."
)


print(
    "Selected checkpoint identity: PASSED"
)


# ======================================================================
# 4. LOAD PERSISTED NOTEBOOK + RECOVER CLASS SOURCE
# ======================================================================

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    nb_b4 = json.load(
        f
    )


def recover_class_source(
    notebook,
    class_name
):

    matches = []

    for cell_idx, cell in enumerate(
        notebook["cells"]
    ):

        if (
            cell.get(
                "cell_type"
            )
            != "code"
        ):
            continue

        source = "".join(
            cell.get(
                "source",
                []
            )
        )

        try:

            tree = ast.parse(
                source
            )

        except Exception:

            continue

        for node in tree.body:

            if not (
                isinstance(
                    node,
                    ast.ClassDef
                )
                and
                node.name
                == class_name
            ):
                continue

            class_source = (
                ast.get_source_segment(
                    source,
                    node
                )
            )

            if class_source:

                matches.append(
                    {
                        "cell_idx":
                            int(
                                cell_idx
                            ),

                        "source":
                            class_source,
                    }
                )

    return matches


scorer_matches = (
    recover_class_source(
        nb_b4,
        "AFPScorer"
    )
)

standardizer_matches = (
    recover_class_source(
        nb_b4,
        "AFPFeatureStandardizer"
    )
)


assert len(
    scorer_matches
) >= 1, (
    "Persisted AFPScorer class definition not found."
)

assert len(
    standardizer_matches
) >= 1, (
    "Persisted AFPFeatureStandardizer class definition not found."
)


# Latest persisted definition.
scorer_match = (
    scorer_matches[
        -1
    ]
)

standardizer_match = (
    standardizer_matches[
        -1
    ]
)

scorer_cell_idx = (
    scorer_match[
        "cell_idx"
    ]
)

scorer_source = (
    scorer_match[
        "source"
    ]
)

std_cell_idx = (
    standardizer_match[
        "cell_idx"
    ]
)

std_source = (
    standardizer_match[
        "source"
    ]
)


print(
    "\nRecovered classes:"
)

print(
    "  AFPScorer:",
    f"cell {scorer_cell_idx}"
)

print(
    "  AFPFeatureStandardizer:",
    f"cell {std_cell_idx}"
)


# ======================================================================
# 5. RECOVER CLASS NAMESPACE DEPENDENCIES
# ======================================================================
#
# Previous B4 failure:
#
#     NameError: AFP_INPUT_DIM is not defined
#
# The persisted class uses AFP_* constants in constructor defaults.
#
# Here we:
#   - start from the verified current runtime namespace,
#   - recover literal AFP_* constants from the class source cell,
#   - explicitly bind AFP_INPUT_DIM = 27 from verified Feature-v2.
#
# No notebook training code is executed.
# ======================================================================

class_ns = dict(
    globals()
)

class_ns.update(
    {
        "np":
            np,

        "torch":
            torch,

        "nn":
            nn,

        "F":
            F,

        "Path":
            Path,

        "Optional":
            Optional,

        "Dict":
            Dict,

        "List":
            List,

        "Tuple":
            Tuple,

        "Any":
            Any,
    }
)


def recover_literal_afp_constants_from_cell(
    notebook,
    cell_idx
):

    source = "".join(
        notebook[
            "cells"
        ][
            cell_idx
        ].get(
            "source",
            []
        )
    )

    tree = ast.parse(
        source
    )

    recovered = {}


    for node in tree.body:

        # --------------------------------------------------------------
        # AFP_SOMETHING = literal
        # --------------------------------------------------------------

        if isinstance(
            node,
            ast.Assign
        ):

            try:

                value = ast.literal_eval(
                    node.value
                )

            except Exception:

                continue


            for target in node.targets:

                if (
                    isinstance(
                        target,
                        ast.Name
                    )
                    and
                    target.id.startswith(
                        "AFP_"
                    )
                ):

                    recovered[
                        target.id
                    ] = value


        # --------------------------------------------------------------
        # AFP_SOMETHING: type = literal
        # --------------------------------------------------------------

        elif isinstance(
            node,
            ast.AnnAssign
        ):

            if not (
                isinstance(
                    node.target,
                    ast.Name
                )
                and
                node.target.id.startswith(
                    "AFP_"
                )
            ):

                continue

            if node.value is None:
                continue


            try:

                value = ast.literal_eval(
                    node.value
                )

            except Exception:

                continue


            recovered[
                node.target.id
            ] = value


    return recovered


dependency_cells = sorted(
    set(
        [
            scorer_cell_idx,
            std_cell_idx,
        ]
    )
)


recovered_afp_constants = {}


for cell_idx in dependency_cells:

    recovered_afp_constants.update(
        recover_literal_afp_constants_from_cell(
            nb_b4,
            cell_idx
        )
    )


for name, value in (
    recovered_afp_constants.items()
):

    class_ns[
        name
    ] = value


# ----------------------------------------------------------------------
# Verified dependency, NOT guessed.
# ----------------------------------------------------------------------

class_ns[
    "AFP_INPUT_DIM"
] = int(
    AFP_RUNTIME_FEATURE_DIM
)


print(
    "\nRecovered scorer namespace dependencies:"
)

print(
    "  AFP_INPUT_DIM =",
    class_ns[
        "AFP_INPUT_DIM"
    ]
)


if recovered_afp_constants:

    print(
        "  Persisted literal AFP_* constants:"
    )

    for name in sorted(
        recovered_afp_constants
    ):

        print(
            f"    {name} = "
            f"{recovered_afp_constants[name]!r}"
        )

else:

    print(
        "  No additional literal AFP_* constants found."
    )


# ======================================================================
# 6. EXECUTE ONLY THE EXACT PERSISTED CLASS DEFINITIONS
# ======================================================================

exec(
    scorer_source,
    class_ns
)

exec(
    std_source,
    class_ns
)


AFPScorerExact = (
    class_ns[
        "AFPScorer"
    ]
)

AFPFeatureStandardizerExact = (
    class_ns[
        "AFPFeatureStandardizer"
    ]
)


print(
    "\nExact persisted classes: LOADED"
)


print(
    "\nAFPScorer signature:"
)

print(
    inspect.signature(
        AFPScorerExact
    )
)


print(
    "AFPFeatureStandardizer signature:"
)

print(
    inspect.signature(
        AFPFeatureStandardizerExact
    )
)


scorer_signature = (
    inspect.signature(
        AFPScorerExact
    )
)


assert (
    "input_dim"
    in scorer_signature.parameters
), (
    "Persisted AFPScorer does not expose input_dim."
)


input_default = (
    scorer_signature
    .parameters[
        "input_dim"
    ]
    .default
)


if (
    input_default
    is not
    inspect._empty
):

    assert int(
        input_default
    ) == 27, (
        "Persisted scorer input dimension "
        "does not equal Feature-v2 dimension 27."
    )


print(
    "AFPScorer input-dimension gate: PASSED"
)


# ======================================================================
# 7. LOAD SELECTED CHECKPOINTS ON CPU
# ======================================================================

def torch_load_cpu(
    path
):

    try:

        return torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

    except TypeError:

        return torch.load(
            path,
            map_location="cpu"
        )


webqsp_ckpt_obj = (
    torch_load_cpu(
        WEBQSP_CKPT
    )
)

cwq_ckpt_obj = (
    torch_load_cpu(
        CWQ_CKPT
    )
)


print(
    "\nCheckpoint top-level types:"
)

print(
    "  WebQSP:",
    type(
        webqsp_ckpt_obj
    )
)

print(
    "  CWQ:   ",
    type(
        cwq_ckpt_obj
    )
)


if isinstance(
    webqsp_ckpt_obj,
    dict
):

    print(
        "  WebQSP keys:",
        sorted(
            webqsp_ckpt_obj.keys()
        )
    )


if isinstance(
    cwq_ckpt_obj,
    dict
):

    print(
        "  CWQ keys:",
        sorted(
            cwq_ckpt_obj.keys()
        )
    )


# ======================================================================
# 8. EXTRACT MODEL STATE_DICT
# ======================================================================

MODEL_STATE_KEYS = [
    "model_state_dict",
    "state_dict",
    "model_state",
    "scorer_state_dict",
    "scorer_state",
]


def looks_like_state_dict(
    obj
):

    if not isinstance(
        obj,
        dict
    ):
        return False

    if len(
        obj
    ) == 0:
        return False

    tensor_count = sum(
        int(
            torch.is_tensor(
                value
            )
        )
        for value in obj.values()
    )

    if tensor_count == 0:
        return False

    return any(
        str(
            key
        ).endswith(
            "weight"
        )
        for key in obj.keys()
    )


def extract_model_state(
    checkpoint
):

    if looks_like_state_dict(
        checkpoint
    ):

        return checkpoint


    assert isinstance(
        checkpoint,
        dict
    )


    # Direct known locations.
    for key in MODEL_STATE_KEYS:

        if key not in checkpoint:
            continue

        value = checkpoint[
            key
        ]

        if looks_like_state_dict(
            value
        ):

            return value


    # One nested level.
    for parent_key, parent_value in (
        checkpoint.items()
    ):

        if not isinstance(
            parent_value,
            dict
        ):
            continue

        for key in MODEL_STATE_KEYS:

            if key not in parent_value:
                continue

            value = parent_value[
                key
            ]

            if looks_like_state_dict(
                value
            ):

                return value


    raise AssertionError(
        "Could not locate scorer state_dict "
        "inside selected checkpoint."
    )


webqsp_model_state = (
    extract_model_state(
        webqsp_ckpt_obj
    )
)

cwq_model_state = (
    extract_model_state(
        cwq_ckpt_obj
    )
)


print(
    "\nModel state keys"
)

print(
    "  WebQSP:",
    list(
        webqsp_model_state.keys()
    )
)

print(
    "  CWQ:   ",
    list(
        cwq_model_state.keys()
    )
)


# ======================================================================
# 9. INFER FROZEN MLP ARCHITECTURE DIRECTLY FROM WEIGHTS
# ======================================================================

def infer_mlp_dimensions(
    state_dict
):

    matrix_weights = []

    for key, value in (
        state_dict.items()
    ):

        if (
            torch.is_tensor(
                value
            )
            and
            value.ndim == 2
        ):

            matrix_weights.append(
                (
                    str(
                        key
                    ),
                    tuple(
                        value.shape
                    )
                )
            )


    assert len(
        matrix_weights
    ) == 2, (
        "Expected exactly two 2-D Linear weight tensors; "
        f"found {matrix_weights}"
    )


    first_candidates = [
        item
        for item in matrix_weights
        if item[
            1
        ][
            1
        ] == 27
    ]


    assert len(
        first_candidates
    ) == 1, (
        "Could not uniquely identify first 27-D Linear layer."
    )


    first_key, first_shape = (
        first_candidates[
            0
        ]
    )


    hidden_dim = int(
        first_shape[
            0
        ]
    )


    second_candidates = [
        item
        for item in matrix_weights
        if item[
            1
        ] == (
            1,
            hidden_dim
        )
    ]


    assert len(
        second_candidates
    ) == 1, (
        "Could not uniquely identify output Linear layer."
    )


    second_key = (
        second_candidates[
            0
        ][
            0
        ]
    )


    return {
        "input_dim":
            27,

        "hidden_dim":
            hidden_dim,

        "output_dim":
            1,

        "first_weight_key":
            first_key,

        "second_weight_key":
            second_key,
    }


webqsp_arch = (
    infer_mlp_dimensions(
        webqsp_model_state
    )
)

cwq_arch = (
    infer_mlp_dimensions(
        cwq_model_state
    )
)


print(
    "\nRecovered scorer architectures"
)

print(
    "  WebQSP:",
    webqsp_arch
)

print(
    "  CWQ:   ",
    cwq_arch
)


assert (
    webqsp_arch[
        "hidden_dim"
    ]
    == 32
), (
    "WebQSP selected scorer should be H=32."
)

assert (
    cwq_arch[
        "hidden_dim"
    ]
    == 64
), (
    "CWQ selected scorer should be H=64."
)


print(
    "Selected H=32/H=64 architecture gate: PASSED"
)


# ======================================================================
# 10. INSTANTIATE EXACT PERSISTED SCORER
# ======================================================================

def instantiate_exact_scorer(
    hidden_dim
):

    signature = (
        inspect.signature(
            AFPScorerExact
        )
    )

    kwargs = {}


    for name, param in (
        signature.parameters.items()
    ):

        lname = str(
            name
        ).lower()


        if lname in {
            "input_dim",
            "in_dim",
            "feature_dim",
            "n_features",
        }:

            kwargs[
                name
            ] = 27


        elif lname in {
            "hidden_dim",
            "hidden_size",
            "hidden",
        }:

            kwargs[
                name
            ] = int(
                hidden_dim
            )


        elif lname in {
            "dropout",
            "dropout_p",
            "dropout_rate",
        }:

            kwargs[
                name
            ] = 0.0


        elif (
            param.default
            is not
            inspect._empty
        ):

            # Preserve persisted default.
            continue


        else:

            raise AssertionError(
                "Unknown required AFPScorer constructor "
                f"parameter: {name}"
            )


    model = AFPScorerExact(
        **kwargs
    )


    model = model.to(
        DEVICE
    )

    model.eval()

    return model


webqsp_scorer = (
    instantiate_exact_scorer(
        hidden_dim=32
    )
)

cwq_scorer = (
    instantiate_exact_scorer(
        hidden_dim=64
    )
)


webqsp_scorer.load_state_dict(
    webqsp_model_state,
    strict=True
)

cwq_scorer.load_state_dict(
    cwq_model_state,
    strict=True
)


webqsp_scorer.eval()
cwq_scorer.eval()


print(
    "\nExact scorer state restoration: PASSED"
)


# ======================================================================
# 11. RECOVER FROZEN TRAIN-ONLY STANDARDIZER
# ======================================================================

def to_numpy_float32(
    value
):

    if torch.is_tensor(
        value
    ):

        value = (
            value
            .detach()
            .cpu()
            .numpy()
        )

    return np.asarray(
        value,
        dtype=np.float32
    )


def find_mean_std_dict(
    obj,
    path="root"
):

    matches = []


    if not isinstance(
        obj,
        dict
    ):

        return matches


    lowercase_keys = {
        str(
            key
        ).lower():
            key
        for key in obj.keys()
    }


    possible_mean_keys = [
        "mean",
        "mean_",
        "feature_mean",
        "feature_means",
    ]

    possible_std_keys = [
        "std",
        "std_",
        "feature_std",
        "feature_stds",
        "scale",
        "scale_",
    ]


    mean_key = next(
        (
            lowercase_keys[
                name
            ]
            for name in possible_mean_keys
            if name in lowercase_keys
        ),
        None
    )

    std_key = next(
        (
            lowercase_keys[
                name
            ]
            for name in possible_std_keys
            if name in lowercase_keys
        ),
        None
    )


    if (
        mean_key is not None
        and
        std_key is not None
    ):

        try:

            mean = to_numpy_float32(
                obj[
                    mean_key
                ]
            )

            std = to_numpy_float32(
                obj[
                    std_key
                ]
            )

        except Exception:

            mean = None
            std = None


        if (
            mean is not None
            and
            std is not None
            and
            mean.shape == (
                27,
            )
            and
            std.shape == (
                27,
            )
        ):

            matches.append(
                {
                    "path":
                        path,

                    "mean":
                        mean,

                    "std":
                        std,
                }
            )


    for key, value in (
        obj.items()
    ):

        if isinstance(
            value,
            dict
        ):

            matches.extend(
                find_mean_std_dict(
                    value,
                    path=(
                        f"{path}.{key}"
                    )
                )
            )


    return matches


def extract_standardizer_state(
    checkpoint,
    dataset_name
):

    matches = find_mean_std_dict(
        checkpoint
    )


    assert len(
        matches
    ) >= 1, (
        f"{dataset_name}: no frozen 27-D "
        "standardizer mean/std found."
    )


    reference = (
        matches[
            0
        ]
    )


    # Multiple copies are acceptable only if identical.
    for other in matches[
        1:
    ]:

        assert np.array_equal(
            reference[
                "mean"
            ],
            other[
                "mean"
            ]
        ), (
            f"{dataset_name}: inconsistent standardizer means."
        )

        assert np.array_equal(
            reference[
                "std"
            ],
            other[
                "std"
            ]
        ), (
            f"{dataset_name}: inconsistent standardizer stds."
        )


    return reference


webqsp_std_state = (
    extract_standardizer_state(
        webqsp_ckpt_obj,
        "WebQSP"
    )
)

cwq_std_state = (
    extract_standardizer_state(
        cwq_ckpt_obj,
        "CWQ"
    )
)


print(
    "\nFrozen standardizer state"
)

print(
    "  WebQSP source:",
    webqsp_std_state[
        "path"
    ]
)

print(
    "  CWQ source:   ",
    cwq_std_state[
        "path"
    ]
)


for dataset_name, state in [
    (
        "WebQSP",
        webqsp_std_state
    ),
    (
        "CWQ",
        cwq_std_state
    ),
]:

    assert state[
        "mean"
    ].shape == (
        27,
    )

    assert state[
        "std"
    ].shape == (
        27,
    )

    assert np.all(
        np.isfinite(
            state[
                "mean"
            ]
        )
    )

    assert np.all(
        np.isfinite(
            state[
                "std"
            ]
        )
    )

    assert np.all(
        state[
            "std"
        ] > 0
    ), (
        f"{dataset_name}: standardizer contains non-positive std."
    )


print(
    "Frozen standardizer validity: PASSED"
)


# ======================================================================
# 12. STANDARDIZATION FUNCTION
# ======================================================================

def apply_frozen_standardizer(
    X,
    state
):

    X = np.asarray(
        X,
        dtype=np.float32
    )


    mean = np.asarray(
        state[
            "mean"
        ],
        dtype=np.float32
    )

    std = np.asarray(
        state[
            "std"
        ],
        dtype=np.float32
    )


    assert X.ndim == 2
    assert X.shape[
        1
    ] == 27


    return (
        (
            X
            - mean
        )
        /
        std
    ).astype(
        np.float32
    )


# ======================================================================
# 13. SCORER LOGIT INFERENCE
# ======================================================================

def scorer_logits(
    model,
    X_standardized
):

    X_standardized = np.asarray(
        X_standardized,
        dtype=np.float32
    )


    assert X_standardized.ndim == 2
    assert X_standardized.shape[
        1
    ] == 27


    with torch.inference_mode():

        tensor = (
            torch.from_numpy(
                X_standardized
            )
            .to(
                DEVICE
            )
        )


        output = model(
            tensor
        )


        logits = (
            output
            .reshape(
                -1
            )
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )


    assert logits.shape == (
        X_standardized.shape[
            0
        ],
    )


    return logits


# ======================================================================
# 14. FROZEN NPZ vs RECOVERED ONLINE LOGIT FIDELITY
# ======================================================================

def sigmoid_numpy(
    logits
):

    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    return (
        1.0
        /
        (
            1.0
            +
            np.exp(
                -logits
            )
        )
    )


def evaluate_logit_fidelity(
    dataset_name,
    frozen_features,
    runtime_features,
    standardizer_state,
    scorer
):

    X_frozen = np.asarray(
        frozen_features[
            "X"
        ],
        dtype=np.float32
    )

    X_runtime = np.asarray(
        runtime_features[
            "X"
        ],
        dtype=np.float32
    )


    assert X_frozen.shape == (
        X_runtime.shape
    )


    Z_frozen = (
        apply_frozen_standardizer(
            X_frozen,
            standardizer_state
        )
    )

    Z_runtime = (
        apply_frozen_standardizer(
            X_runtime,
            standardizer_state
        )
    )


    logits_frozen = (
        scorer_logits(
            scorer,
            Z_frozen
        )
    )

    logits_runtime = (
        scorer_logits(
            scorer,
            Z_runtime
        )
    )


    probs_frozen = sigmoid_numpy(
        logits_frozen
    )

    probs_runtime = sigmoid_numpy(
        logits_runtime
    )


    feature_diff = np.abs(
        X_frozen.astype(
            np.float64
        )
        -
        X_runtime.astype(
            np.float64
        )
    )

    standardized_diff = np.abs(
        Z_frozen.astype(
            np.float64
        )
        -
        Z_runtime.astype(
            np.float64
        )
    )

    logit_diff = np.abs(
        logits_frozen.astype(
            np.float64
        )
        -
        logits_runtime.astype(
            np.float64
        )
    )

    probability_diff = np.abs(
        probs_frozen
        -
        probs_runtime
    )


    report = {
        "n_branches":
            int(
                X_frozen.shape[
                    0
                ]
            ),

        "max_feature_diff":
            float(
                feature_diff.max()
            ),

        "mean_feature_diff":
            float(
                feature_diff.mean()
            ),

        "max_standardized_diff":
            float(
                standardized_diff.max()
            ),

        "mean_standardized_diff":
            float(
                standardized_diff.mean()
            ),

        "max_logit_diff":
            float(
                logit_diff.max()
            ),

        "mean_logit_diff":
            float(
                logit_diff.mean()
            ),

        "max_probability_diff":
            float(
                probability_diff.max()
            ),

        "mean_probability_diff":
            float(
                probability_diff.mean()
            ),
    }


    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{dataset_name.upper()} "
        "SCORER LOGIT FIDELITY"
    )

    print(
        "=" * 100
    )


    for key, value in (
        report.items()
    ):

        print(
            f"{key:<30}",
            value
        )


    return {
        "report":
            report,

        "logits_frozen":
            logits_frozen,

        "logits_runtime":
            logits_runtime,

        "standardized_frozen":
            Z_frozen,

        "standardized_runtime":
            Z_runtime,
    }


webqsp_logit_result = (
    evaluate_logit_fidelity(
        dataset_name=
            "webqsp",

        frozen_features=
            webqsp_frozen_final,

        runtime_features=
            webqsp_runtime_features,

        standardizer_state=
            webqsp_std_state,

        scorer=
            webqsp_scorer
    )
)


cwq_logit_result = (
    evaluate_logit_fidelity(
        dataset_name=
            "cwq",

        frozen_features=
            cwq_frozen_final,

        runtime_features=
            cwq_runtime_features,

        standardizer_state=
            cwq_std_state,

        scorer=
            cwq_scorer
    )
)


webqsp_logit_fidelity = (
    webqsp_logit_result[
        "report"
    ]
)

cwq_logit_fidelity = (
    cwq_logit_result[
        "report"
    ]
)

webqsp_frozen_logits = (
    webqsp_logit_result[
        "logits_frozen"
    ]
)

webqsp_runtime_logits = (
    webqsp_logit_result[
        "logits_runtime"
    ]
)

cwq_frozen_logits = (
    cwq_logit_result[
        "logits_frozen"
    ]
)

cwq_runtime_logits = (
    cwq_logit_result[
        "logits_runtime"
    ]
)


# ======================================================================
# 15. HARD LOGIT FIDELITY GATE
# ======================================================================
#
# Tiny Feature-v2 float32 differences can be amplified by standardized
# low-variance columns.
#
# Observable scorer logits are therefore the main software-fidelity gate.
#
# Do NOT loosen this automatically if it fails.
# ======================================================================

LOGIT_ATOL = 1e-4


for dataset_name, result in [
    (
        "WebQSP",
        webqsp_logit_fidelity
    ),
    (
        "CWQ",
        cwq_logit_fidelity
    ),
]:

    assert (
        result[
            "max_logit_diff"
        ]
        <= LOGIT_ATOL
    ), (
        f"{dataset_name}: scorer logit fidelity failed. "
        "Do NOT increase LOGIT_ATOL automatically."
    )


print(
    "\nFrozen-NPZ -> online-logit fidelity: PASSED"
)


# ======================================================================
# 16. EXPOSE FINAL RUNTIME SCORERS/STANDARDIZERS
# ======================================================================

AFP_RUNTIME_SCORERS = {
    "webqsp":
        webqsp_scorer,

    "cwq":
        cwq_scorer,
}


AFP_RUNTIME_STANDARDIZERS = {
    "webqsp": {
        "mean":
            webqsp_std_state[
                "mean"
            ].copy(),

        "std":
            webqsp_std_state[
                "std"
            ].copy(),
    },

    "cwq": {
        "mean":
            cwq_std_state[
                "mean"
            ].copy(),

        "std":
            cwq_std_state[
                "std"
            ].copy(),
    },
}


# ======================================================================
# 17. GOLD-FREE ONLINE GROUP SCORING CALLBACK
# ======================================================================
#
# This is the scorer interface for the later online validation traversal.
#
# Inputs available at inference time:
#   - question ID
#   - question text
#   - relation plan
#   - hop
#   - current candidate group
#
# NO:
#   - answer entities
#   - gold labels
#   - suffix reachability labels
# ======================================================================

def afp_runtime_score_group(
    dataset_name,
    question_id,
    question,
    plan,
    hop,
    candidate_rows
):

    dataset_key = str(
        dataset_name
    ).strip().lower()


    assert dataset_key in {
        "webqsp",
        "cwq",
    }


    assert len(
        candidate_rows
    ) >= 1


    X = (
        AFP_RUNTIME_FEATURE_EXTRACTOR(
            question_id=
                question_id,

            question=
                question,

            plan=
                list(
                    plan
                ),

            hop=
                int(
                    hop
                ),

            candidate_rows=
                candidate_rows,

            semantic_encoder=
                AFP_RUNTIME_SEMANTIC_ENCODER,

            entity_name_map=
                None
        )
    )


    X = np.asarray(
        X,
        dtype=np.float32
    )


    assert X.shape == (
        len(
            candidate_rows
        ),
        27
    )


    Z = (
        apply_frozen_standardizer(
            X,
            AFP_RUNTIME_STANDARDIZERS[
                dataset_key
            ]
        )
    )


    logits = (
        scorer_logits(
            AFP_RUNTIME_SCORERS[
                dataset_key
            ],
            Z
        )
    )


    assert logits.shape == (
        len(
            candidate_rows
        ),
    )


    return {
        "features":
            X,

        "standardized_features":
            Z,

        "logits":
            logits,
    }


AFP_RUNTIME_SCORE_GROUP = (
    afp_runtime_score_group
)


# ======================================================================
# 18. GROUP-LEVEL CALLBACK SPOT CHECK
# ======================================================================

def callback_spotcheck(
    dataset_name,
    dataset_rows,
    labels_file,
    frozen_features,
    frozen_logits,
    requested_group_indices
):

    groups = list(
        iter_runtime_groups(
            labels_file
        )
    )


    group_ptr = np.asarray(
        frozen_features[
            "group_ptr"
        ],
        dtype=np.int64
    )


    assert len(
        group_ptr
    ) == (
        len(
            groups
        )
        + 1
    )


    results = []


    for group_index in (
        requested_group_indices
    ):

        rows = groups[
            group_index
        ]

        first = rows[
            0
        ]


        source_index = int(
            first[
                "source_index"
            ]
        )


        rec = dataset_rows[
            source_index
        ]


        assert str(
            rec[
                "id"
            ]
        ) == str(
            first[
                "question_id"
            ]
        )


        runtime_output = (
            AFP_RUNTIME_SCORE_GROUP(
                dataset_name=
                    dataset_name,

                question_id=
                    first[
                        "question_id"
                    ],

                question=
                    rec[
                        "question"
                    ],

                plan=
                    first[
                        "plan"
                    ],

                hop=
                    first[
                        "hop"
                    ],

                candidate_rows=
                    rows
            )
        )


        start = int(
            group_ptr[
                group_index
            ]
        )

        end = int(
            group_ptr[
                group_index
                + 1
            ]
        )


        reference_logits = (
            frozen_logits[
                start:end
            ]
        )


        runtime_logits_group = (
            runtime_output[
                "logits"
            ]
        )


        assert reference_logits.shape == (
            runtime_logits_group.shape
        )


        max_logit_diff = float(
            np.max(
                np.abs(
                    reference_logits.astype(
                        np.float64
                    )
                    -
                    runtime_logits_group.astype(
                        np.float64
                    )
                )
            )
        )


        results.append(
            {
                "group_index":
                    int(
                        group_index
                    ),

                "source_index":
                    source_index,

                "hop":
                    int(
                        first[
                            "hop"
                        ]
                    ),

                "candidate_count":
                    int(
                        len(
                            rows
                        )
                    ),

                "max_logit_diff":
                    max_logit_diff,
            }
        )


    return results


webqsp_n_groups = (
    len(
        webqsp_frozen_final[
            "group_ptr"
        ]
    )
    - 1
)

cwq_n_groups = (
    len(
        cwq_frozen_final[
            "group_ptr"
        ]
    )
    - 1
)


webqsp_spot_groups = sorted(
    set(
        [
            0,
            webqsp_n_groups // 4,
            webqsp_n_groups // 2,
            (
                3
                * webqsp_n_groups
            )
            // 4,
            webqsp_n_groups - 1,
        ]
    )
)


cwq_spot_groups = sorted(
    set(
        [
            0,
            cwq_n_groups // 4,
            cwq_n_groups // 2,
            (
                3
                * cwq_n_groups
            )
            // 4,
            cwq_n_groups - 1,
        ]
    )
)


webqsp_callback_check = (
    callback_spotcheck(
        dataset_name=
            "webqsp",

        dataset_rows=
            webqsp_val_runtime,

        labels_file=
            WEBQSP_LABEL_FILE,

        frozen_features=
            webqsp_frozen_final,

        frozen_logits=
            webqsp_frozen_logits,

        requested_group_indices=
            webqsp_spot_groups
    )
)


cwq_callback_check = (
    callback_spotcheck(
        dataset_name=
            "cwq",

        dataset_rows=
            cwq_val_runtime,

        labels_file=
            CWQ_LABEL_FILE,

        frozen_features=
            cwq_frozen_final,

        frozen_logits=
            cwq_frozen_logits,

        requested_group_indices=
            cwq_spot_groups
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "ONLINE CALLBACK SPOT-CHECK"
)

print(
    "=" * 100
)


for dataset_name, rows in [
    (
        "WebQSP",
        webqsp_callback_check
    ),
    (
        "CWQ",
        cwq_callback_check
    ),
]:

    print(
        f"\n{dataset_name}"
    )


    for row in rows:

        print(
            "  "
            f"group={row['group_index']:<5} "
            f"hop={row['hop']:<2} "
            f"n={row['candidate_count']:<4} "
            f"max_logit_diff="
            f"{row['max_logit_diff']:.10g}"
        )


        assert (
            row[
                "max_logit_diff"
            ]
            <= LOGIT_ATOL
        )


print(
    "\nOnline scorer callback: PASSED"
)


# ======================================================================
# 19. FINAL RUNTIME OBJECT GATES
# ======================================================================

assert callable(
    AFP_RUNTIME_SCORE_GROUP
)

assert set(
    AFP_RUNTIME_SCORERS.keys()
) == {
    "webqsp",
    "cwq",
}

assert set(
    AFP_RUNTIME_STANDARDIZERS.keys()
) == {
    "webqsp",
    "cwq",
}


AFP_RUNTIME_SCORER_READY = True


print(
    "\nRuntime scorer objects: READY"
)


# ======================================================================
# 20. SAVE B4 MANIFEST
# ======================================================================

TRAVERSAL_DIR = (
    ROOT
    / "10_validation_traversal"
)

TRAVERSAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


B4_MANIFEST = {
    "cell":
        "RQ2_12C_B4_REVISED",

    "runtime_device":
        "cpu",

    "feature_version":
        AFP_RUNTIME_FEATURE_VERSION,

    "feature_dim":
        27,

    "scorer_version":
        "afp_mlp_v1",

    "selected_checkpoints": {
        "webqsp": {
            "path":
                str(
                    WEBQSP_CKPT
                ),

            "sha256":
                webqsp_ckpt_sha,

            "hidden_dim":
                32,

            "loss":
                "branch_bce",

            "deployment_seed":
                42,
        },

        "cwq": {
            "path":
                str(
                    CWQ_CKPT
                ),

            "sha256":
                cwq_ckpt_sha,

            "hidden_dim":
                64,

            "loss":
                "branch_bce",

            "deployment_seed":
                42,
        },
    },

    "persisted_class_recovery": {
        "afp_scorer_cell":
            int(
                scorer_cell_idx
            ),

        "afp_standardizer_cell":
            int(
                std_cell_idx
            ),

        "afp_input_dim":
            27,

        "class_namespace_dependency_recovery":
            True,
    },

    "standardization": {
        "source":
            "selected_checkpoint_train_only_state",

        "formula":
            "(X - mean) / std",
    },

    "webqsp_logit_fidelity":
        webqsp_logit_fidelity,

    "cwq_logit_fidelity":
        cwq_logit_fidelity,

    "logit_atol":
        LOGIT_ATOL,

    "callback_spotcheck": {
        "webqsp":
            webqsp_callback_check,

        "cwq":
            cwq_callback_check,
    },

    "online_score_callback_ready":
        True,

    "gold_used_by_runtime_callback":
        False,

    "pruning_run":
        False,

    "afp_hyperparameter_tuning_run":
        False,

    "test_examples_accessed":
        False,

    "complete_afp_frozen":
        False,
}


B4_MANIFEST_PATH = (
    TRAVERSAL_DIR
    / "cell12c_b4_scorer_runtime_fidelity.json"
)


with open(
    B4_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        B4_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 21. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 116
)

print(
    "=== RQ2 CELL 12C-B4: ONLINE AFP SCORER RUNTIME VERIFIED ==="
)

print(
    "=" * 116
)


print(
    "\nWebQSP selected scorer:"
)

print(
    "  hidden dimension: 32"
)

print(
    "  loss:             branch_bce"
)

print(
    "  deployment seed:  42"
)

print(
    "  checkpoint SHA:  ",
    webqsp_ckpt_sha
)


print(
    "\nCWQ selected scorer:"
)

print(
    "  hidden dimension: 64"
)

print(
    "  loss:             branch_bce"
)

print(
    "  deployment seed:  42"
)

print(
    "  checkpoint SHA:  ",
    cwq_ckpt_sha
)


print(
    "\nRuntime integration:"
)

print(
    "  Exact persisted scorer class: RESTORED"
)

print(
    "  AFP_INPUT_DIM dependency:     RESTORED"
)

print(
    "  Frozen train standardizers:   RESTORED"
)

print(
    "  Selected scorer weights:      RESTORED"
)

print(
    "  Frozen-vs-online logits:      VERIFIED"
)

print(
    "  Online group callback:        VERIFIED"
)

print(
    "  Gold in runtime callback:     NO"
)

print(
    "  Pruning run:                  NO"
)

print(
    "  Hyperparameter tuning:        NO"
)

print(
    "  TEST examples accessed:       NO"
)

print(
    "  Complete AFP frozen:          NO"
)


print(
    "\nManifest:",
    B4_MANIFEST_PATH
)


print(
    "\nNEXT STEP:"
)

print(
    "Cell 13 — VALIDATION-ONLY hyperparameter tuning "
    "for AFP (T, gamma_min) and controlled baselines."
)

Runtime device: cpu
Feature runtime: afp_features_v2_masked_entity_semantics
Feature dimension: 27

Selected checkpoints:
  WebQSP: /kaggle/working/step3_rq2_dev_v1/07_final_scorer/webqsp_afp_scorer_selected.pt
  CWQ:    /kaggle/working/step3_rq2_dev_v1/07_final_scorer/cwq_afp_scorer_selected.pt

Checkpoint SHA256
  WebQSP: bee65146403d565661b5105c41831af3e81d54ed5fba1b2a99bcb37382420d9f
  CWQ:    91a531c057bb02d0b311ef8b78bf02248a63cd66e57fe8959ef86e7297d9d0a1
Selected checkpoint identity: PASSED

Recovered classes:
  AFPScorer: cell 175
  AFPFeatureStandardizer: cell 175

Recovered scorer namespace dependencies:
  AFP_INPUT_DIM = 27
  Persisted literal AFP_* constants:
    AFP_DROPOUT = 0.0
    AFP_HIDDEN_CANDIDATES = [32, 64, 128]
    AFP_SCORER_VERSION = 'afp_mlp_v1'

Exact persisted classes: LOADED

AFPScorer signature:
(input_dim=27, hidden_dim=64, dropout=0.0)
AFPFeatureStandardizer signature:
(eps=1e-08)
AFPScorer input-dimension gate: PASSED

Checkpoint top-level types:
  WebQ